# ECL-global_full_null — ECLIPSE activity-conditioning control | training seed 1

**Baby → Sports**, two fresh 64D/500-epoch runs. Only the null-conditioning rule changes from Full ECLIPSE:

\[
M_{active}=|\{u:h_u>0\}|,\quad \mu^{global}_{ij}=n_i n_j/M_{active},\quad
r^{global}_{ij}=\frac{[c_{ij}-\mu^{global}_{ij}]_+}{c_{ij}+\mu^{global}_{ij}+4},\quad
\widetilde W_{ij}=W_{ij}\exp(r^{global}_{ij}).
\]

Full-user pooled excess, **no user halves and no activity conditioning**. Native mode `pooled_global_excess`, bins `none`, halves0, delta1/tau2; WMSG+CSE eta.1+LC.01+MI/FSC/BSC and per-dataset settings match Full. Inactive TRAIN users do not enter M_active. This is neither Count-only nor `unstratified_replication`.

- Attach processed Baby/Sports inputs before Save & Run All; enable GPU/Internet for setup. RUN_MODE=`train` starts jobs only in the dataset sections. Jobs run sequentially on GPU_INDEX. Electronics is excluded from this plan.
- Source is the recorded Full commit **plus three declared, SHA-locked estimator files embedded here**. A private checkout is used; existing Full/Count checkouts are not patched. No unpublished GitHub commit is required. All other recorded source hashes must still match Full.
- `RUN_MODE='report_only'` accepts only corresponding Global-full receipts; no clone/install/model/CUDA/training in that mode. Never relabel Full/Count receipts as this arm.
- Native VALID NDCG@20 selects the checkpoint; report selected TEST. Keep 500epochs; no TEST-based tuning or automatic retries/resume/deletion.
- PDFs/PNG300dpi, tables/source data/captions and ZIP+checksum are generated after measured runs. A session has one actual seed/dataset, no invented SD/p-values or missing-as-zero values.


## 1. Session configuration — one source of truth


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import sys
import uuid

ARM = 'global_full_null'
SEED = 1
RUN_MODE = 'train'  # or 'report_only'
EXISTING_ARTIFACT_DIRS = {}  # report_only: {'baby':'/path/native_run', ...}
EXISTING_STDOUT_LOGS = {}  # report_only: {'baby':'/path/stdout.log', ...}; missing log -> timing NOT_MEASURED
NOTEBOOK_SOURCE_PATH = None  # optional exact submitted .ipynb to include
SELECTED_DATASETS = ('baby', 'sports')
GPU_INDEX = 0
INSTALL_MISSING_DEPENDENCIES = True
AUTO_DOWNLOAD_DATA = False  # Optional outside Kaggle only; attach inputs before Save & Run All.
DATASET_HANDLE = 'rainyle/stair-datasets-mmrec'
DATASET_SEARCH_ROOTS = []  # Optional parent folders; default input/cache paths are searched automatically.
STRICT_RUNTIME = True  # Require the seed-1 Python major/minor, Torch and FreeRec versions.
RUNTIME_COMMIT = '5212263e87c5c1cfe6e48c91afa90201a48f5929'
REPO_URL = 'https://github.com/ThanhChuong12/STAIR-Enhanced.git'
REPO_OVERRIDE = None  # Disabled for this arm: use an isolated base-plus-overlay checkout.
DATASET_SOURCE_OVERRIDES = {}  # Example: {'baby': '/kaggle/input/.../baby'}
if RUN_MODE not in ('train','report_only'): raise ValueError('Invalid RUN_MODE')
if RUN_MODE=='report_only' and set(SELECTED_DATASETS)-set(EXISTING_ARTIFACT_DIRS):
    raise ValueError('Set EXISTING_ARTIFACT_DIRS for every selected dataset.')

if isinstance(SEED, bool) or SEED not in (1, 2, 3):
    raise ValueError('SEED must be 1,2 or3; default job is Global-full null seed1.')
if not SELECTED_DATASETS or len(set(SELECTED_DATASETS)) != len(SELECTED_DATASETS):
    raise ValueError('Select one or more datasets without duplicates.')
if set(SELECTED_DATASETS)-{'baby','sports'}:
    raise ValueError('This follow-up notebook supports Baby/Sports only; Electronics is outside the Global-full plan.')
if 'electronics' in SELECTED_DATASETS:
    raise ValueError('Global-full null plan: Baby/Sports only for seeds1/2/3.')
if isinstance(GPU_INDEX, bool) or not isinstance(GPU_INDEX, int) or GPU_INDEX < 0:
    raise ValueError('GPU_INDEX must be a non-negative integer.')

WORK_DIR = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()
SESSION_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:8]
RUN_ROOT = WORK_DIR / 'eclipse_runs' / f'eclipse_{ARM}_seed{SEED}_{SESSION_ID}'
RUN_ROOT.mkdir(parents=True,exist_ok=False)
for name in ('logs', 'artifacts', 'configs', 'figures', 'tables', 'provenance','helpers'):
    (RUN_ROOT / name).mkdir(parents=True, exist_ok=True)
DATA_ROOT = WORK_DIR / 'eclipse_data'
GRAPH_CACHE_ROOT = WORK_DIR / 'eclipse_graph_cache' / RUNTIME_COMMIT[:12] / ARM / 'f62a74a254ef'
GRAPH_CACHE_ROOT.mkdir(parents=True, exist_ok=True)

PRIMARY_METRICS = ('RECALL@10', 'RECALL@20', 'NDCG@10', 'NDCG@20')
DATASET_TITLES = {'baby': 'Amazon Baby', 'sports': 'Amazon Sports', 'electronics': 'Amazon Electronics'}
DATASET_FOLDERS = {k: f'Amazon2014{v}_550_MMRec' for k, v in
                   [('baby', 'Baby'), ('sports', 'Sports'), ('electronics', 'Electronics')]}
print(f'ECLIPSE Global-full null | seed={SEED} | datasets={SELECTED_DATASETS}')
print(f'Run directory: {RUN_ROOT}')
print('Training starts only in the dataset training cells below.')

SESSION_STEM=f'eclipse_{ARM}_seed{SEED}'
def stem_for(key):
    if key not in DATASET_FOLDERS:raise ValueError(f'Unknown dataset: {key}')
    return f'eclipse_{key}_{ARM}_seed{SEED}'
ARTIFACT_PATHS={key:(Path(EXISTING_ARTIFACT_DIRS[key]).expanduser().resolve() if RUN_MODE=='report_only'
                   else RUN_ROOT/'artifacts'/stem_for(key)) for key in SELECTED_DATASETS}
LOG_PATHS={key:((Path(EXISTING_STDOUT_LOGS[key]).expanduser().resolve() if EXISTING_STDOUT_LOGS.get(key) else None)
               if RUN_MODE=='report_only' else RUN_ROOT/'logs'/f'{stem_for(key)}.log') for key in SELECTED_DATASETS}
if ARM!='global_full_null':raise ValueError('This notebook is locked to Global-full null.')


## 2. Frozen Full protocol and declared Global-full extension
Input/candidate/count/raw semantic/CF hashes remain fixed. Null expectation, one-pool assignment, scores and calibrated final graph may differ. Full source references remain immutable; only three declared estimator files have new source digests.


In [ ]:
import math, re, time, shutil
def file_sha256(path):
    digest=hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda:handle.read(1024*1024),b''):digest.update(chunk)
    return digest.hexdigest()

def write_json(path,payload):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    temporary=path.with_name(path.name+'.tmp')
    temporary.write_text(json.dumps(payload,indent=2,ensure_ascii=False,allow_nan=False)+'\n',encoding='utf-8')
    temporary.replace(path)

SEED1_REFERENCE=json.loads('{"baby":{"commit":"5212263e87c5c1cfe6e48c91afa90201a48f5929","python":"3.13.15","torch":"2.11.0+cu128","freerec":"0.9.7","gpu":"Tesla T4","v8_config":{"embedding_dim":64,"num_layers":3,"gamma":0.1,"lr":0.001,"weight_decay":0.3,"batch_size":1024,"epochs":500,"beta1":0.9,"beta2":0.999,"mfiles":["textual_modality.pkl","visual_modality.pkl"],"num_neighbors":[5,1],"eval_freq":5,"lambda_nlgcl":0.01,"nlgcl_tau":0.2,"nlgcl_G":1,"nlgcl_alpha":0.5,"eta":0.1,"k_cf":5,"c_min":2,"t_shrinkage":5.0,"v8_arm":"WMSG-core","semantic_mode":"weighted","edge_power":2.0,"edge_floor":0.05,"semantic_mix":1.0,"cl_chunk_size":1024,"knn_chunk_size":1024,"knn_device":"cpu","cf_block_size":64,"cf_memory_budget_mib":128.0},"icaart_config":{"evidence_mode":"pooled_activity_excess","evidence_activity_bins":"log2","evidence_halves":0,"evidence_seed":null,"evidence_tau":2.0,"evidence_delta":1.0,"evidence_candidate_block":8192,"evidence_placebo_seed":20261016},"features":[{"file":"textual_modality.pkl","sha256":"ceda3582c04f4e276e8d4bdcc2146f2634f6ca171e1f9dfafeb219dc5cfe96eb"},{"file":"visual_modality.pkl","sha256":"763bf9510c74fd5c6a45ee8164ca9f1ababc5c355c1757fc3acbdcc5d46af0a5"}],"knn":[{"features":"ceda3582c04f4e276e8d4bdcc2146f2634f6ca171e1f9dfafeb219dc5cfe96eb","k":5,"chunk":1024,"algorithm":"cosine_fp32_diagonal_minus10_torch_topk_v1","torch":"2.11.0+cu128","device":"cpu","tf32":false,"source":"640f26b80b5b5eabf9bdcd44bac7a9c0544571168b31eefb7cbf96e59c166038","fingerprint":"2cb2ad16db4996b65a179bbb221bbf09ac3eb390cf2cacf2cb3b26fd471efa40","cache_hit":false,"candidate_sha256":"5bfeae7c9bb0a96be4ca20803e20d0d84a57068aeb322749bd132f0ce2691827"},{"features":"763bf9510c74fd5c6a45ee8164ca9f1ababc5c355c1757fc3acbdcc5d46af0a5","k":1,"chunk":1024,"algorithm":"cosine_fp32_diagonal_minus10_torch_topk_v1","torch":"2.11.0+cu128","device":"cpu","tf32":false,"source":"640f26b80b5b5eabf9bdcd44bac7a9c0544571168b31eefb7cbf96e59c166038","fingerprint":"a50d5b1b5a586f38a4a5aec66d128556d43046d899d631a8756b476d47e04586","cache_hit":false,"candidate_sha256":"ce97e02fbb82f94038c40fd2ae8e911c387c951b7710ee29eb2b144fc3cd7b08"}],"contract":{"config":{"embedding_dim":64,"num_layers":3,"gamma":0.1,"num_neighbors":[5,1],"mfiles":["textual_modality.pkl","visual_modality.pkl"],"lr":0.001,"weight_decay":0.3,"beta1":0.9,"beta2":0.999,"batch_size":1024,"seed":1,"eval_freq":5,"ranking":"full","which4best":"NDCG@20","num_workers":4,"nlgcl_G":1,"nlgcl_tau":0.2,"nlgcl_alpha":0.5,"eta":0.1,"k_cf":5,"c_min":2,"t_shrinkage":5.0,"v8_arm":"WMSG-core","semantic_mode":"weighted","edge_power":2.0,"edge_floor":0.05,"semantic_mix":1.0,"alignment":"off","anchor_seed":2718,"anchor_per_stratum":1024,"relation_gate":"off","relation_strength":0.0,"relation_threshold":0.7,"relation_temperature":0.1,"bsc_residual":0.0,"lambda_dirichlet":0.0,"placebo_seed":1,"knn_device":"cpu","knn_chunk_size":1024,"cl_chunk_size":1024,"metadata_fingerprint":null},"splits":{"test.txt":"307162309a9fae52172dabbbb0f373e845af5e73740d987b70bb0f406d5ac411","train.txt":"bcced50dea427734d76b031f1de140f219d67ba6bb6bc5eac85d7b3be51645bb","valid.txt":"c44687959888d29e2d931aa3de1210f7f708e204d169b7680e5adfcf0a183a3a"},"candidate_hashes":["5bfeae7c9bb0a96be4ca20803e20d0d84a57068aeb322749bd132f0ce2691827","ce97e02fbb82f94038c40fd2ae8e911c387c951b7710ee29eb2b144fc3cd7b08"],"sources":{"stair5_v8.py":"dea612a122c82f7cc6c4c1ba069510fa1242fa3819f00b6cbfe0d37b9ed9c502","stair5_v8_alignment.py":"8d6bb9eb25dfe2426d2a358ec82268d27c9b4913eef0472b6fda83fbd231607d","stair5_v8_graph.py":"2a7c931740ceeac177ce3fc6dc2cdae479eb75673a8ef1b115122ba4a0826ae7","stair5_v4_graph.py":"eb5d2b2355871711931d597bbe2d01f520b174852430e922be268826c7252241","stair5_v4.py":"640f26b80b5b5eabf9bdcd44bac7a9c0544571168b31eefb7cbf96e59c166038","stair5_v4_objectives.py":"1afd078a22aad8635778cfa61f05628beaecd180cd2c6c17ed56f72a6a94182e","stair5_v4_utils.py":"39930cb54af3f5cdb65983f05b3b03cabe502673b9d70889b0ac7c1d22274b1a","freerec_compat.py":"de46f3dd9bc11bb69acd5f151d175968e73d980be37ad36d5afc636b0b9c5e90","stair5_v8_smoother.py":"c4f1ebab6008ffa5f7720007a17854040f0a7211fbd6087fc9dedd26d10d7663","AdamW.py":"43108b6264cd5b1fa723d0a731faff0269d05fb91bcf525445c9bd0406443715","main_stair5_v8.py":"036b45594a8c0b17e32c5401c5b84a93e85992fadae6303aa0db9eeaea8331b8","stair_icaart.py":"77b862894f9aa91eba8fa96ad54a38eb6e29a25d9e134b7718a58f4bb5851014","stair_icaart_evidence.py":"11487e0fef87077cde7863b58847bcf22adeebef6e5378ae32f71f366d88c41a","stair_icaart_graph.py":"33bd582c8cd3b298eed1ce47ad99837ec9f0cd220778d684ae3d96161a455aad","stair_icaart_config.py":"fe5bcaa711ba64fd40d431db27ae6278e7c937ce05f57786b6f7262668061a51","main_stair_icaart.py":"b95eb5423ac8947da290bd13562e0aeb433adb8742ca0f0246139b4880aeec3f","stair_icaart_reporting.py":"6902360e9ac92dd511ea87e988ad622cae51886eb0ac183b170ce7df1b5065cc"},"data_fingerprint":"de38b811bc05e4838c2f52d025ee3464447a1af4e1a1844747635cddafd13a9e","graph_fingerprint":"b8013795d439064965dcd6946a44751d9a275ca0cefdc5231192066582d757a8"},"provenance":"logs/GD6/STAIR-PEEC/stair_icaart/baby_STAIR_ICAART_pooled_activity_excess_d1.0_seed1","evidence_invariants":{"raw_semantic_fingerprint":"6c9010ff253ecd73e031c024db783e2ac3922be074560d5f1827ec30bcd32064","assignment_hash":"9dd175d6e15c8d0bf6733797390191bea57263f06d6e99b57b3eb3555aba32a4","counts_shape":[29926],"null_expectation_shape":[29926],"counts_hash":"a2b9aa0811aaf132948b4478ead0aadbada027b3c8d80c77d3e020a6c563d8eb","null_expectation_hash":"1c56431d4ab4f2179e7ef1efeba35924f3f6b881ddd6b15864df8dc6468e7c35","train_presence_hash":"5ac41072b49e5a21d02fc06e5c5f87f9a215561bd010e5c8f58eebdbb109344b","num_raw_train_interactions":118551,"num_deduplicated_train_interactions":118551,"num_users":19445,"num_items":7050,"group_sizes":[0,5998,9065,3545,746,86,5],"baseline_v8_fingerprint":"eab753510c9d830425ba0f1f44f6d4964aa0143c574d4707fb2e7c9d93532041","cf":{"raw_cf_fingerprint":"f93a45f8fd1e3db84017cd47c5a4293f26a7a48a537db8cc4a99c0fa79df44a7","normalized_cf_fingerprint":"213d645c5f3b302a7ffc2539849238967edc409bc495f7ac87317cc179ba7314","cf_cache_key":"8d678b20109a0b4f099ad7fee46585d5f59aec9fda227f55b5e4ece87c9fafa0"}}},"sports":{"commit":"5212263e87c5c1cfe6e48c91afa90201a48f5929","python":"3.13.15","torch":"2.11.0+cu128","freerec":"0.9.7","gpu":"Tesla T4","v8_config":{"embedding_dim":64,"num_layers":3,"gamma":0.2,"lr":0.001,"weight_decay":0.1,"batch_size":1024,"epochs":500,"beta1":0.9,"beta2":0.999,"mfiles":["textual_modality.pkl","visual_modality.pkl"],"num_neighbors":[5,1],"eval_freq":5,"lambda_nlgcl":0.01,"nlgcl_tau":0.2,"nlgcl_G":1,"nlgcl_alpha":0.5,"eta":0.1,"k_cf":5,"c_min":2,"t_shrinkage":5.0,"v8_arm":"WMSG-core","semantic_mode":"weighted","edge_power":2.0,"edge_floor":0.05,"semantic_mix":1.0,"cl_chunk_size":1024,"knn_chunk_size":1024,"knn_device":"cpu","cf_block_size":64,"cf_memory_budget_mib":128.0},"icaart_config":{"evidence_mode":"pooled_activity_excess","evidence_activity_bins":"log2","evidence_halves":0,"evidence_seed":null,"evidence_tau":2.0,"evidence_delta":1.0,"evidence_candidate_block":8192,"evidence_placebo_seed":20261016},"features":[{"file":"textual_modality.pkl","sha256":"ca99972f3a2c7a9ead14e9a0d649351411722bbd98e920b4c5fe9f8ffa1564a2"},{"file":"visual_modality.pkl","sha256":"5d2ea8187bbc52f806616404815930061acefe6cb22f353e3f02f286c95b9959"}],"knn":[{"features":"ca99972f3a2c7a9ead14e9a0d649351411722bbd98e920b4c5fe9f8ffa1564a2","k":5,"chunk":609,"algorithm":"cosine_fp32_diagonal_minus10_torch_topk_v1","torch":"2.11.0+cu128","device":"cpu","tf32":false,"source":"640f26b80b5b5eabf9bdcd44bac7a9c0544571168b31eefb7cbf96e59c166038","fingerprint":"9dacd55f8d2be69d2bebff457c2c83345135e05f90eb44c1c4f0b8d62b4347e7","cache_hit":false,"candidate_sha256":"bbe9e918209a03b5186a74cbd78ba474f420dfaab4eb6c2095c7c578ac0ca5af"},{"features":"5d2ea8187bbc52f806616404815930061acefe6cb22f353e3f02f286c95b9959","k":1,"chunk":609,"algorithm":"cosine_fp32_diagonal_minus10_torch_topk_v1","torch":"2.11.0+cu128","device":"cpu","tf32":false,"source":"640f26b80b5b5eabf9bdcd44bac7a9c0544571168b31eefb7cbf96e59c166038","fingerprint":"9a875c92e35fbe67a4e43ac1298d898fed91f7160d449e4d9e8d5b86899889e5","cache_hit":false,"candidate_sha256":"e3270692a18ae7d11d50a83d30b7f071f4a095e630714c01b2a0f6b4cff4e8a5"}],"contract":{"config":{"embedding_dim":64,"num_layers":3,"gamma":0.2,"num_neighbors":[5,1],"mfiles":["textual_modality.pkl","visual_modality.pkl"],"lr":0.001,"weight_decay":0.1,"beta1":0.9,"beta2":0.999,"batch_size":1024,"seed":1,"eval_freq":5,"ranking":"full","which4best":"NDCG@20","num_workers":4,"nlgcl_G":1,"nlgcl_tau":0.2,"nlgcl_alpha":0.5,"eta":0.1,"k_cf":5,"c_min":2,"t_shrinkage":5.0,"v8_arm":"WMSG-core","semantic_mode":"weighted","edge_power":2.0,"edge_floor":0.05,"semantic_mix":1.0,"alignment":"off","anchor_seed":2718,"anchor_per_stratum":1024,"relation_gate":"off","relation_strength":0.0,"relation_threshold":0.7,"relation_temperature":0.1,"bsc_residual":0.0,"lambda_dirichlet":0.0,"placebo_seed":1,"knn_device":"cpu","knn_chunk_size":1024,"cl_chunk_size":1024,"metadata_fingerprint":null},"splits":{"test.txt":"218501613d897422d24d2035e268af027f4ca77688a3f836080b197621f5610a","train.txt":"2491704e8543dbdeb9046cec0928c6734aa80521b1026afd53e0d296f4f7427c","valid.txt":"6dabc7efcc14de50909c6b1995f7bf9862d6233a3440d593d389e1bf91fef56e"},"candidate_hashes":["bbe9e918209a03b5186a74cbd78ba474f420dfaab4eb6c2095c7c578ac0ca5af","e3270692a18ae7d11d50a83d30b7f071f4a095e630714c01b2a0f6b4cff4e8a5"],"sources":{"stair5_v8.py":"dea612a122c82f7cc6c4c1ba069510fa1242fa3819f00b6cbfe0d37b9ed9c502","stair5_v8_alignment.py":"8d6bb9eb25dfe2426d2a358ec82268d27c9b4913eef0472b6fda83fbd231607d","stair5_v8_graph.py":"2a7c931740ceeac177ce3fc6dc2cdae479eb75673a8ef1b115122ba4a0826ae7","stair5_v4_graph.py":"eb5d2b2355871711931d597bbe2d01f520b174852430e922be268826c7252241","stair5_v4.py":"640f26b80b5b5eabf9bdcd44bac7a9c0544571168b31eefb7cbf96e59c166038","stair5_v4_objectives.py":"1afd078a22aad8635778cfa61f05628beaecd180cd2c6c17ed56f72a6a94182e","stair5_v4_utils.py":"39930cb54af3f5cdb65983f05b3b03cabe502673b9d70889b0ac7c1d22274b1a","freerec_compat.py":"de46f3dd9bc11bb69acd5f151d175968e73d980be37ad36d5afc636b0b9c5e90","stair5_v8_smoother.py":"c4f1ebab6008ffa5f7720007a17854040f0a7211fbd6087fc9dedd26d10d7663","AdamW.py":"43108b6264cd5b1fa723d0a731faff0269d05fb91bcf525445c9bd0406443715","main_stair5_v8.py":"036b45594a8c0b17e32c5401c5b84a93e85992fadae6303aa0db9eeaea8331b8","stair_icaart.py":"77b862894f9aa91eba8fa96ad54a38eb6e29a25d9e134b7718a58f4bb5851014","stair_icaart_evidence.py":"11487e0fef87077cde7863b58847bcf22adeebef6e5378ae32f71f366d88c41a","stair_icaart_graph.py":"33bd582c8cd3b298eed1ce47ad99837ec9f0cd220778d684ae3d96161a455aad","stair_icaart_config.py":"fe5bcaa711ba64fd40d431db27ae6278e7c937ce05f57786b6f7262668061a51","main_stair_icaart.py":"b95eb5423ac8947da290bd13562e0aeb433adb8742ca0f0246139b4880aeec3f","stair_icaart_reporting.py":"6902360e9ac92dd511ea87e988ad622cae51886eb0ac183b170ce7df1b5065cc"},"data_fingerprint":"383929cee0a814ae8211bde6c15ef0abb6ecd4499822bdf5f3579cb87fa04643","graph_fingerprint":"6b7be789b969cd17a17459deea25bfdf5bfcf518a008b02b7c2925a2d3f88f6c"},"provenance":"logs/GD6/STAIR-PEEC/stair_icaart/sports_STAIR_ICAART_pooled_activity_excess_d1.0_seed1","evidence_invariants":{"raw_semantic_fingerprint":"7be2cf2f50ab128cb070eef6f853982741f6ff52e8229c45d227d352c114c2ce","assignment_hash":"68fc8c356709aa20dca248289216a90d298c182672e9438b5b888b1bf27281bd","counts_shape":[78872],"null_expectation_shape":[78872],"counts_hash":"fa866fab24c551f62ef527c0ea90b2c5f87886a3186ac7387f0a1000e5deaec2","null_expectation_hash":"0cb8ef4d164d8c28ada6f5b897df36c44a60c8bd3fcb23b396ca367ca346a42a","train_presence_hash":"3823a2e457063833e1ee27cef6622c24d20880c706a5f078ff8474f05dad20f7","num_raw_train_interactions":218409,"num_deduplicated_train_interactions":218409,"num_users":35598,"num_items":18357,"group_sizes":[0,11416,16324,6216,1430,190,21,1],"baseline_v8_fingerprint":"cc0ff8deb6bbdac1c9295e27781dbb96de422731779794d79b4e1fd355ee75ea","cf":{"raw_cf_fingerprint":"aa9eb264792a316ef7f6caf6015cf260ab2e0544846f3ce6f2e6d633c1c4c4dd","normalized_cf_fingerprint":"b72c22e849dd477b641135cd2196e090472d56b8396ab37adff9b6137c7c949b","cf_cache_key":"20b7e1854aac1bf2b3cafa9d06a1b7265da6d19c326e037a617d1c14ef919195"}}}}')
write_json(RUN_ROOT/'provenance'/'protocol_reference.json',SEED1_REFERENCE)
write_json(RUN_ROOT/'run_manifest.json',{'method':'ECLIPSE','arm_id':ARM,'training_seed':SEED,'datasets':list(SELECTED_DATASETS),'mode':RUN_MODE,'status':'PREPARED','session_id':SESSION_ID,'expected_algorithm_diff':{'evidence_mode':['pooled_activity_excess','pooled_global_excess']}})

GLOBAL_SOURCE_OVERLAY=json.loads('{"base_commit":"5212263e87c5c1cfe6e48c91afa90201a48f5929","arm":"global_full_null","files":{"stair_icaart_config.py":{"base_sha256":"fe5bcaa711ba64fd40d431db27ae6278e7c937ce05f57786b6f7262668061a51","sha256":"b1dd36f1247753930c3785ae48e91ee91bfba3c3c85854500a840cbaf5288098","content":"\\"\\"\\"Versioned, dependency-free contracts for PEEC and legacy REEC controls.\\n\\nPlatform: STAIR5-v8 / WMSG-core (64-dimensional primary).\\nCalibrates pre-BSC semantic graph weights using pooled-full evidence or historical\\nhalf-conditioned controls; all use binary TRAIN presence.\\n\\"\\"\\"\\n\\nfrom collections.abc import Mapping\\nimport math\\nfrom typing import Any, Dict, Tuple\\n\\n\\nICAART_OPTIONS: Tuple[Tuple[str, type, Any], ...] = (\\n    (\\"evidence-mode\\", str, \\"replicated_excess\\"),\\n    (\\"evidence-activity-bins\\", str, \\"log2\\"),\\n    (\\"evidence-halves\\", int, 2),\\n    (\\"evidence-seed\\", int, 20261016),\\n    (\\"evidence-tau\\", float, 2.0),\\n    (\\"evidence-delta\\", float, 1.0),\\n    (\\"evidence-candidate-block\\", int, 8192),\\n    (\\"evidence-placebo-seed\\", int, 20261016),\\n)\\n\\nICAART_NAMES: Tuple[str, ...] = tuple(flag.replace(\\"-\\", \\"_\\") for flag, _, _ in ICAART_OPTIONS)\\nDEFAULTS: Dict[str, Any] = {flag.replace(\\"-\\", \\"_\\"): value for flag, _, value in ICAART_OPTIONS}\\n\\nALLOWED_EVIDENCE_MODES: Tuple[str, ...] = (\\n    \\"replicated_excess\\",\\n    \\"v8_exact\\",\\n    \\"pooled_excess\\",\\n    \\"unstratified_replication\\",\\n    \\"replicated_count\\",\\n    \\"degree_stratified_shuffle\\",\\n    \\"uniform_multiplier\\",\\n    \\"pooled_activity_excess\\",\\n    \\"pooled_activity_count\\",\\n    \\"pooled_activity_shuffle\\",\\n    \\"pooled_global_excess\\",\\n)\\n\\nFULL_ACTIVITY_MODES = frozenset((\\n    \\"pooled_activity_excess\\", \\"pooled_activity_count\\", \\"pooled_activity_shuffle\\",\\n    \\"pooled_global_excess\\",  # Full-user pooling, without activity conditioning.\\n))\\nEVIDENCE_CONTRACT_VERSION = 2\\n\\n\\ndef evidence_identity(values: Mapping) -> Dict[str, Any]:\\n    \\"\\"\\"Describe the active estimator, without mislabeling pooled scores as replication.\\"\\"\\"\\n    mode = values[\\"evidence_mode\\"]\\n    full = mode in FULL_ACTIVITY_MODES\\n    return {\\n        \\"contract_version\\": EVIDENCE_CONTRACT_VERSION,\\n        \\"architecture\\": \\"STAIR-PEEC-versionICAART-R2\\" if full else \\"STAIR-REEC-versionICAART\\",\\n        \\"null_scope\\": \\"global_full\\" if mode == \\"pooled_global_excess\\" else \\"full_activity\\" if full else (\\n            \\"unstratified_halves\\" if mode == \\"unstratified_replication\\" else \\"half_activity\\"\\n        ),\\n        \\"active_partition\\": not full,\\n        \\"active_halves\\": 0 if full else 2,\\n        \\"score_kind\\": mode,\\n        \\"pooled_regularizer\\": 2.0 * values[\\"evidence_tau\\"] if full or mode == \\"pooled_excess\\" else None,\\n        \\"normalization\\": \\"actual_degree\\",\\n    }\\n\\n\\ndef _get(config: Any, name: str, default: Any = None) -> Any:\\n    \\"\\"\\"Retrieve attribute or dict key uniformly.\\"\\"\\"\\n    return config.get(name, default) if isinstance(config, Mapping) else getattr(config, name, default)\\n\\n\\ndef validate_icaart_config(config: Any) -> Dict[str, Any]:\\n    \\"\\"\\"Validate PEEC and legacy-control contracts before model execution.\\n\\n    Enforces strict mathematical bounds on evidence estimation and preserves\\n    the unchanged STAIR5-v8 WMSG-core backbone contracts.\\n    \\"\\"\\"\\n    values = {name: _get(config, name, default) for name, default in DEFAULTS.items()}\\n\\n    mode = values[\\"evidence_mode\\"]\\n    if mode not in ALLOWED_EVIDENCE_MODES:\\n        raise ValueError(f\\"evidence_mode must be one of {ALLOWED_EVIDENCE_MODES}; got {mode!r}\\")\\n\\n    expected_bins = \\"none\\" if mode == \\"pooled_global_excess\\" else \\"log2\\"\\n    if values[\\"evidence_activity_bins\\"] != expected_bins:\\n        raise ValueError(f\\"{mode} requires evidence_activity_bins={expected_bins!r}\\")\\n\\n    halves = values[\\"evidence_halves\\"]\\n    full = mode in FULL_ACTIVITY_MODES\\n    expected_halves = 0 if full else 2\\n    if halves is None:\\n        values[\\"evidence_halves\\"] = expected_halves\\n        halves = expected_halves\\n    if isinstance(halves, bool) or not isinstance(halves, int) or halves != expected_halves:\\n        raise ValueError(f\\"{mode} requires evidence_halves={expected_halves}\\")\\n\\n    for int_field in (\\"evidence_seed\\", \\"evidence_placebo_seed\\", \\"evidence_candidate_block\\"):\\n        if int_field == \\"evidence_seed\\" and full:\\n            # No hashing, partition or graph identity depends on this inactive seed.\\n            values[int_field] = None\\n            continue\\n        val = values[int_field]\\n        if isinstance(val, bool) or not isinstance(val, int) or val < 0:\\n            raise ValueError(f\\"{int_field} must be a non-negative integer\\")\\n    if values[\\"evidence_candidate_block\\"] < 1:\\n        raise ValueError(\\"evidence_candidate_block must be at least 1\\")\\n\\n    if isinstance(values[\\"evidence_tau\\"], bool) or isinstance(values[\\"evidence_delta\\"], bool):\\n        raise ValueError(\\"evidence_tau and evidence_delta must be numeric, not boolean\\")\\n    tau = float(values[\\"evidence_tau\\"])\\n    if not math.isfinite(tau) or not math.isfinite(2.0 * tau) or tau <= 0:\\n        raise ValueError(f\\"evidence_tau must be finite and > 0; got {tau}\\")\\n    values[\\"evidence_tau\\"] = tau\\n\\n    delta = float(values[\\"evidence_delta\\"])\\n    if not math.isfinite(delta) or not 0.0 <= delta <= 2.0:\\n        raise ValueError(f\\"evidence_delta must be finite and in [0.0, 2.0]; got {delta}\\")\\n    values[\\"evidence_delta\\"] = delta\\n\\n    # Backbone contract validations: STAIR5-v8 WMSG-core constraints\\n    required_backbone = {\\n        \\"embedding_dim\\": (64, 256),\\n        \\"num_layers\\": (3,),\\n        \\"nlgcl_G\\": (1,),\\n        \\"lambda_nlgcl\\": (0.01,),\\n        \\"nlgcl_tau\\": (0.2,),\\n        \\"nlgcl_alpha\\": (0.5,),\\n        \\"v8_arm\\": (\\"WMSG-core\\",),\\n        \\"alignment\\": (\\"off\\",),\\n        \\"relation_gate\\": (\\"off\\",),\\n        \\"relation_strength\\": (0.0,),\\n        \\"bsc_residual\\": (0.0,),\\n        \\"lambda_dirichlet\\": (0.0,),\\n        \\"semantic_mode\\": (\\"weighted\\",),\\n        \\"semantic_mix\\": (1.0,),\\n        \\"edge_power\\": (2.0,),\\n        \\"edge_floor\\": (0.05,),\\n        \\"knn_device\\": (\\"cpu\\",),\\n    }\\n\\n    for name, allowed in required_backbone.items():\\n        val = _get(config, name, allowed[0])\\n        if isinstance(val, bool) or val not in allowed:\\n            raise ValueError(f\\"STAIR-REEC requires {name} in {allowed}; got {val!r}\\")\\n\\n    neighbors = _get(config, \\"num_neighbors\\", [5, 1])\\n    if isinstance(neighbors, str):\\n        neighbors = list(map(int, neighbors.split(\\"-\\")))\\n    modalities = _get(config, \\"mfiles\\", [\\"textual_modality.pkl\\", \\"visual_modality.pkl\\"])\\n    if isinstance(modalities, str):\\n        modalities = modalities.split(\\",\\")\\n    if list(neighbors) != [5, 1] or len(modalities) != 2:\\n        raise ValueError(\\"STAIR-REEC requires two modalities [text, vision] with num_neighbors=[5, 1]\\")\\n    if \\"visual\\" in str(modalities[0]).lower() or \\"textual\\" in str(modalities[1]).lower():\\n        raise ValueError(\\"Modality order must be [text, vision], not [vision, text]\\")\\n\\n    return values\\n\\n\\ndef analytical_icaart_payload(n_items: int, n_edges: int, *, pooled: bool = False) -> Dict[str, Any]:\\n    \\"\\"\\"Analytical memory footprint of graph and temporary evidence structures.\\"\\"\\"\\n    if n_items <= 0 or n_edges < 0:\\n        raise ValueError(\\"Require positive item count and non-negative edge count\\")\\n    return {\\n        \\"semantic_directed_csr_bytes\\": 8 * (n_items + 1) + 24 * n_edges,\\n        \\"edge_count_convention\\": \\"n_edges counts undirected semantic pairs; final CF union is additional\\",\\n        \\"evidence_pairs_int64_bytes\\": 16 * n_edges,\\n        \\"evidence_counts_int64_bytes\\": (8 if pooled else 16) * n_edges,\\n        \\"evidence_mu_fp64_bytes\\": (8 if pooled else 16) * n_edges,\\n        \\"evidence_score_fp64_bytes\\": 8 * n_edges,\\n        \\"evidence_weight_fp64_bytes\\": 8 * n_edges,\\n        \\"additional_trainable_parameters\\": 0,\\n        \\"scope\\": \\"All evidence structures reside transiently on CPU and are freed before GPU training.\\",\\n    }\\n"},"models/stair_icaart_evidence.py":{"base_sha256":"11487e0fef87077cde7863b58847bcf22adeebef6e5378ae32f71f366d88c41a","sha256":"8c0d2a22b32a7a4c827fa522e91e40c66997713327b6c57866155563c1ab15f2","content":"\\"\\"\\"models/stair_icaart_evidence.py — Full-activity pooled evidence and legacy replicated controls.\\n====================================================================================================\\nImplements:\\n1. Deterministic activity stratification b(u) = floor(log2(h_u)) on deduplicated TRAIN data.\\n2. Disjoint user splitting f in {0, 1} with deterministic seeded hash (no Python randomized hash).\\n3. Candidate-only observed co-interactions c_ijf and activity-null expectations mu_ijf.\\n4. Full-activity pooled excess with kappa=2*tau, preserving all legacy half-based controls.\\n\\nContracts:\\n- Runs strictly on CPU in bounded blocks (default candidate_block = 8192).\\n- Never allocates dense N x N matrices or full R.T @ R.\\n- Zero gradients, zero learned parameters.\\n\\"\\"\\"\\n\\nimport hashlib\\nimport math\\nfrom typing import Dict, Optional, Tuple\\n\\nimport numpy as np\\nimport scipy.sparse as sp\\n\\n\\ndef deterministic_user_hash(evidence_seed: int, user_id: int) -> int:\\n    \\"\\"\\"Compute a deterministic 64-bit unsigned integer hash for (evidence_seed, user_id).\\n\\n    Uses SHA-256 to guarantee bitwise reproducibility across platforms and Python runs,\\n    completely independent of Python\'s randomized hash() seed or PyTorch global RNG.\\n    \\"\\"\\"\\n    token = f\\"{evidence_seed}:{user_id}\\".encode(\\"ascii\\")\\n    return int.from_bytes(hashlib.sha256(token).digest()[:8], \\"little\\")\\n\\n\\ndef build_activity_replication(\\n    R_csr: sp.csr_matrix,\\n    evidence_seed: int = 20261016,\\n) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:\\n    \\"\\"\\"Stratifies users by train activity and partitions active users into disjoint halves.\\n\\n    Mathematical formulation (Eq 3-4):\\n    - h_u = sum_i R_{ui} (train activity only)\\n    - Zero-history users (h_u == 0): excluded from strata, assigned user_bin = -1, half = -1.\\n    - Active users (h_u >= 1): b(u) = floor(log2(h_u)).\\n    - Within each non-empty stratum b: sort active users by (deterministic_hash, user_id),\\n      then alternate assignment f in {0, 1}.\\n    - Singleton strata (|U_b| == 1): deterministically assigned to half 0.\\n\\n    Args:\\n        R_csr: Deduplicated binary interaction matrix [M, N] in CSR format (CPU).\\n        evidence_seed: Integer seed for user hashing. Default 20261016.\\n\\n    Returns:\\n        user_bin: int64 array [M] of activity strata (-1 for inactive).\\n        half: int64 array [M] of user halves {0, 1} (-1 for inactive).\\n        group_sizes: int64 array [2, B] of user counts per (half, stratum).\\n    \\"\\"\\"\\n    if not sp.isspmatrix_csr(R_csr):\\n        raise TypeError(\\"R_csr must be a scipy.sparse.csr_matrix on CPU\\")\\n    if isinstance(evidence_seed, bool) or not isinstance(evidence_seed, (int, np.integer)) or evidence_seed < 0:\\n        raise ValueError(\\"evidence_seed must be a non-negative integer\\")\\n    R_csr = _canonical_presence(R_csr, \\"csr\\")\\n\\n    M, N = R_csr.shape\\n    if M <= 0 or N <= 0:\\n        raise ValueError(f\\"Invalid matrix shape: ({M}, {N})\\")\\n\\n    # h_u: user interaction count (train activity)\\n    h_u = np.diff(R_csr.indptr).astype(np.int64)\\n\\n    user_bin = np.full(M, -1, dtype=np.int64)\\n    half = np.full(M, -1, dtype=np.int64)\\n\\n    active_mask = h_u > 0\\n    active_users = np.flatnonzero(active_mask)\\n\\n    if len(active_users) == 0:\\n        return user_bin, half, np.zeros((2, 0), dtype=np.int64)\\n\\n    # b(u) = floor(log2(max(1, h_u)))\\n    active_bins = np.floor(np.log2(h_u[active_users])).astype(np.int64)\\n    user_bin[active_users] = active_bins\\n\\n    B = int(np.max(active_bins)) + 1\\n    group_sizes = np.zeros((2, B), dtype=np.int64)\\n\\n    for b in range(B):\\n        u_in_b = active_users[active_bins == b]\\n        num_u = len(u_in_b)\\n        if num_u == 0:\\n            continue\\n        elif num_u == 1:\\n            u_single = u_in_b[0]\\n            half[u_single] = 0\\n            group_sizes[0, b] += 1\\n        else:\\n            # Deterministic seeded hash with user_id tie-breaker\\n            hashes = np.fromiter(\\n                (deterministic_user_hash(evidence_seed, int(u)) for u in u_in_b),\\n                dtype=np.uint64, count=num_u,\\n            )\\n            order = np.lexsort((u_in_b, hashes))\\n            sorted_u = u_in_b[order]\\n            half[sorted_u] = np.arange(num_u, dtype=np.int64) % 2\\n            group_sizes[:, b] = ((num_u + 1) // 2, num_u // 2)\\n\\n    return user_bin, half, group_sizes\\n\\n\\ndef _canonical_presence(matrix: sp.spmatrix, format: str) -> sp.spmatrix:\\n    \\"\\"\\"Return private binary, sorted storage without changing caller training data.\\"\\"\\"\\n    if not np.isfinite(matrix.data).all() or np.any(matrix.data < 0):\\n        raise ValueError(\\"Interaction counts must be finite and non-negative\\")\\n    # Boolean duplicate reduction avoids int8 count overflow inside this conversion.\\n    result = matrix.astype(bool).asformat(format, copy=True)\\n    result.sum_duplicates()\\n    result.eliminate_zeros()\\n    result.sort_indices()\\n    return result\\n\\n\\ndef build_activity_strata(R_csr: sp.csr_matrix) -> Tuple[np.ndarray, np.ndarray]:\\n    \\"\\"\\"Full-user log2 activity strata; inactive users are excluded, with no RNG or split.\\"\\"\\"\\n    if not sp.isspmatrix_csr(R_csr):\\n        raise TypeError(\\"R_csr must be a scipy.sparse.csr_matrix\\")\\n    if min(R_csr.shape) <= 0:\\n        raise ValueError(\\"Interaction dimensions must be positive\\")\\n    matrix = _canonical_presence(R_csr, \\"csr\\")\\n    activity = np.diff(matrix.indptr).astype(np.int64)\\n    bins = np.full(matrix.shape[0], -1, dtype=np.int64)\\n    active = activity > 0\\n    bins[active] = np.floor(np.log2(activity[active])).astype(np.int64)\\n    sizes = np.bincount(bins[active]).astype(np.int64)\\n    return bins, sizes\\n\\n\\ndef candidate_pooled_statistics(\\n    R_csc: sp.csc_matrix,\\n    E_sem: np.ndarray,\\n    user_bin: np.ndarray,\\n    group_sizes: np.ndarray,\\n    candidate_block: int = 8192,\\n    *, null_scope: str = \\"full_activity\\",\\n) -> Tuple[np.ndarray, np.ndarray]:\\n    \\"\\"\\"Compute full-user c[E], mu[E] on semantic candidates, without half-conditioned nulls.\\n\\n    mu_ij = sum_b n_ib*n_jb/M_b, using binary TRAIN presence. With\\n    null_scope=\'global_full\', one pool contains all and only active TRAIN users,\\n    giving mu_ij=n_i*n_j/M_active. Observed overlap counting is identical.\\n    Integer counters\\n    are promoted before multiplication. Temporary expectation arrays are block-by-bin,\\n    never item-by-item. Posting searches execute in NumPy, not per-history pair loops.\\n    \\"\\"\\"\\n    if not sp.isspmatrix_csc(R_csc):\\n        raise TypeError(\\"R_csc must be a scipy.sparse.csc_matrix\\")\\n    if (isinstance(candidate_block, bool)\\n            or not isinstance(candidate_block, (int, np.integer)) or candidate_block < 1):\\n        raise ValueError(\\"candidate_block must be a positive integer\\")\\n    matrix = _canonical_presence(R_csc, \\"csc\\")\\n    n_users, n_items = matrix.shape\\n    if min(matrix.shape) <= 0:\\n        raise ValueError(\\"Interaction dimensions must be positive\\")\\n    pairs = np.asarray(E_sem)\\n    if (pairs.ndim != 2 or pairs.shape[1] != 2 or pairs.dtype.kind not in \\"iu\\"\\n            or np.any(pairs < 0) or np.any(pairs >= n_items)\\n            or np.any(pairs[:, 0] >= pairs[:, 1])):\\n        raise ValueError(\\"Candidate pairs must be integer [E,2], in catalog, with i < j\\")\\n    pairs = pairs.astype(np.int64, copy=False)\\n    if len(np.unique(pairs[:, 0] * n_items + pairs[:, 1])) != len(pairs):\\n        raise ValueError(\\"Candidate pairs must be unique\\")\\n    if null_scope not in (\\"full_activity\\", \\"global_full\\"):\\n        raise ValueError(\\"Unknown full-user null scope\\")\\n    bins, sizes = np.asarray(user_bin), np.asarray(group_sizes)\\n    activity = np.bincount(matrix.indices, minlength=n_users)\\n    active = activity > 0\\n    expected_bins = np.full(n_users, -1, dtype=np.int64)\\n    if null_scope == \\"global_full\\":\\n        expected_bins[active] = 0\\n        expected_sizes = np.array([np.count_nonzero(active)], dtype=np.int64)\\n    else:\\n        expected_bins[active] = np.floor(np.log2(activity[active])).astype(np.int64)\\n        expected_sizes = np.bincount(expected_bins[active])\\n    if (bins.shape != (n_users,) or bins.dtype.kind not in \\"iu\\"\\n            or sizes.ndim != 1 or sizes.dtype.kind not in \\"iu\\"\\n            or not np.array_equal(bins, expected_bins)\\n            or not np.array_equal(sizes, expected_sizes)):\\n        raise ValueError(\\"Null groups must match the selected full-user TRAIN scope exactly\\")\\n\\n    counts = np.zeros(len(pairs), dtype=np.int64)\\n    expected = np.zeros(len(pairs), dtype=np.float64)\\n    if not len(pairs) or not active.any():\\n        return counts, expected\\n    n_bins = len(sizes)\\n    item_ids = np.repeat(np.arange(n_items, dtype=np.int64), np.diff(matrix.indptr))\\n    flat = item_ids * n_bins + bins[matrix.indices]\\n    item_counts = np.bincount(flat, minlength=n_items * n_bins).reshape(n_items, n_bins)\\n    del flat, item_ids\\n    inverse = np.zeros(n_bins, dtype=np.float64)\\n    np.divide(1.0, sizes, out=inverse, where=sizes > 0)\\n    for start in range(0, len(pairs), candidate_block):\\n        stop = min(start + candidate_block, len(pairs))\\n        left_ids, right_ids = pairs[start:stop].T\\n        for offset, (i, j) in enumerate(zip(left_ids, right_ids)):\\n            left = matrix.indices[matrix.indptr[i]:matrix.indptr[i + 1]]\\n            right = matrix.indices[matrix.indptr[j]:matrix.indptr[j + 1]]\\n            if len(left) > len(right):\\n                left, right = right, left\\n            positions = np.searchsorted(right, left)\\n            valid = positions < len(right)\\n            selected = left[valid]\\n            counts[start + offset] = np.count_nonzero(right[positions[valid]] == selected)\\n        products = item_counts[left_ids].astype(np.float64)\\n        products *= item_counts[right_ids].astype(np.float64)\\n        expected[start:stop] = products @ inverse\\n    if np.any((counts > 0) & (expected <= 0)):\\n        raise ArithmeticError(\\"Positive overlap must imply positive null expectation\\")\\n    return counts, expected\\n\\n\\ndef degree_stratified_permutation(\\n    scores: np.ndarray, degrees: np.ndarray, pairs: np.ndarray, seed: int,\\n) -> np.ndarray:\\n    \\"\\"\\"Shuffle the supplied score family on unordered endpoint-degree quintile pairs.\\"\\"\\"\\n    scores, degrees, pairs = map(np.asarray, (scores, degrees, pairs))\\n    if (scores.ndim != 1 or not np.isfinite(scores).all()\\n            or degrees.ndim != 1 or not np.isfinite(degrees).all() or np.any(degrees < 0)\\n            or pairs.shape != (len(scores), 2) or pairs.dtype.kind not in \\"iu\\"\\n            or np.any(pairs < 0) or np.any(pairs >= len(degrees))\\n            or np.any(pairs[:, 0] >= pairs[:, 1])):\\n        raise ValueError(\\"Invalid scores, train degrees or unordered candidate pairs\\")\\n    if isinstance(seed, bool) or not isinstance(seed, (int, np.integer)) or seed < 0:\\n        raise ValueError(\\"Permutation seed must be a non-negative integer\\")\\n    if not len(scores):\\n        return scores.astype(np.float64, copy=True)\\n    cuts = np.unique(np.quantile(degrees, [.2, .4, .6, .8]))\\n    strata = np.searchsorted(cuts, degrees, side=\\"right\\")\\n    left, right = strata[pairs[:, 0]], strata[pairs[:, 1]]\\n    groups = np.minimum(left, right) * 5 + np.maximum(left, right)\\n    rng = np.random.default_rng(seed)\\n    result = scores.astype(np.float64, copy=True)\\n    for group in np.unique(groups):\\n        indices = np.flatnonzero(groups == group)\\n        result[indices] = rng.permutation(scores[indices])\\n    return result\\n\\n\\ndef pooled_activity_score(\\n    c: np.ndarray, mu: np.ndarray, tau: float = 2.0,\\n    mode: str = \\"pooled_activity_excess\\", *, degrees: Optional[np.ndarray] = None,\\n    E_sem: Optional[np.ndarray] = None, placebo_seed: int = 20261016,\\n) -> np.ndarray:\\n    \\"\\"\\"PEEC-full: [c-mu]+/(c+mu+2*tau), or its explicitly labeled count/placebo controls.\\n\\n    The vectors are full-stratum statistics, not sums of half-specific expectations.\\n    tau retains its legacy count-scale convention; pooled kappa is exactly 2*tau.\\n    \\"\\"\\"\\n    from stair_icaart_config import FULL_ACTIVITY_MODES\\n    if mode not in FULL_ACTIVITY_MODES:\\n        raise ValueError(f\\"Unknown full-activity evidence mode: {mode}\\")\\n    if isinstance(tau, bool) or not math.isfinite(tau) or tau <= 0 or not math.isfinite(2 * tau):\\n        raise ValueError(\\"tau and pooled kappa must be finite and positive\\")\\n    c, mu = np.asarray(c), np.asarray(mu)\\n    if (c.ndim != 1 or c.dtype.kind not in \\"iu\\" or mu.shape != c.shape\\n            or np.any(c < 0) or not np.isfinite(mu).all() or np.any(mu < 0)):\\n        raise ValueError(\\"Full counts must be integer [E]; expectations must be finite nonnegative [E]\\")\\n    if np.any((c > 0) & (mu <= 0)):\\n        raise ArithmeticError(\\"Positive overlap must imply positive null expectation\\")\\n    counts = c.astype(np.float64)\\n    expected = mu.astype(np.float64)\\n    if mode == \\"pooled_activity_count\\":\\n        scores = counts / (counts + 2.0 * tau)\\n    else:\\n        scores = np.maximum(0.0, counts - expected) / (counts + expected + 2.0 * tau)\\n    if not np.isfinite(scores).all() or np.any(scores < 0) or np.any(scores >= 1):\\n        raise ArithmeticError(\\"Pooled evidence score must lie in [0,1)\\")\\n    if mode == \\"pooled_activity_shuffle\\":\\n        if degrees is None or E_sem is None:\\n            raise ValueError(\\"Pooled placebo requires TRAIN item degrees and candidate pairs\\")\\n        scores = degree_stratified_permutation(scores, degrees, E_sem, placebo_seed)\\n    return scores\\n\\n\\ndef candidate_excess_statistics(\\n    R_csc: sp.csc_matrix,\\n    E_sem: np.ndarray,\\n    half: np.ndarray,\\n    user_bin: np.ndarray,\\n    group_sizes: np.ndarray,\\n    candidate_block: int = 8192,\\n    mode: str = \\"replicated_excess\\",\\n) -> Tuple[np.ndarray, np.ndarray]:\\n    \\"\\"\\"Computes observed co-interaction counts c_ijf and conditional null expectations mu_ijf.\\n\\n    Only evaluates unique undirected candidate pairs (i, j) in E_sem with i < j.\\n    Uses bounded block processing (default candidate_block = 8192) to strictly limit memory.\\n\\n    Mathematical formulation (Eq 4):\\n    - M_fb = |{u : f(u) = f, b(u) = b}|\\n    - n_ifb = sum_{u: f(u)=f, b(u)=b} R_{ui}\\n    - c_ijf = sum_{u: f(u)=f} R_{ui} R_{uj}\\n    - mu_ijf = sum_{b: M_fb > 0} (n_ifb * n_jfb) / M_fb\\n\\n    Args:\\n        R_csc: Binary interaction matrix [M, N] in CSC format (CPU).\\n        E_sem: int64 array [E, 2] of candidate item pairs (i, j) with i < j.\\n        half: int64 array [M] of user halves {0, 1} (-1 for inactive).\\n        user_bin: int64 array [M] of user strata (-1 for inactive).\\n        group_sizes: int64 array [2, B] of user group sizes.\\n        candidate_block: Maximum candidate edges processed per chunk.\\n        mode: Evidence mode (supports \'unstratified_replication\' ablation).\\n\\n    Returns:\\n        c: int64 array [E, 2] of observed co-occurrence counts.\\n        mu: float64 array [E, 2] of null expectation values.\\n    \\"\\"\\"\\n    if not sp.isspmatrix_csc(R_csc):\\n        raise TypeError(\\"R_csc must be a scipy.sparse.csc_matrix on CPU\\")\\n    if (isinstance(candidate_block, bool)\\n            or not isinstance(candidate_block, (int, np.integer)) or candidate_block < 1):\\n        raise ValueError(\\"candidate_block must be a positive integer\\")\\n    if mode not in (\\"replicated_excess\\", \\"v8_exact\\", \\"pooled_excess\\",\\n                    \\"unstratified_replication\\", \\"replicated_count\\",\\n                    \\"degree_stratified_shuffle\\", \\"uniform_multiplier\\"):\\n        raise ValueError(f\\"Unknown evidence mode: {mode}\\")\\n    R = _canonical_presence(R_csc, \\"csc\\")\\n    M, N = R.shape\\n    E_sem = np.asarray(E_sem)\\n    if E_sem.ndim != 2 or E_sem.shape[1] != 2 or E_sem.dtype.kind not in \\"iu\\":\\n        raise ValueError(\\"E_sem must be an integer array of shape [E, 2]\\")\\n    if np.any(E_sem < 0) or np.any(E_sem >= N) or np.any(E_sem[:, 0] >= E_sem[:, 1]):\\n        raise ValueError(\\"Candidate pairs must be in catalog and strictly satisfy i < j\\")\\n    if N and len(np.unique(E_sem[:, 0].astype(np.int64) * N + E_sem[:, 1])) != len(E_sem):\\n        raise ValueError(\\"Candidate pairs must be unique\\")\\n    half, user_bin, group_sizes = map(np.asarray, (half, user_bin, group_sizes))\\n    if (half.shape != (M,) or user_bin.shape != (M,)\\n            or half.dtype.kind not in \\"iu\\" or user_bin.dtype.kind not in \\"iu\\"\\n            or group_sizes.ndim != 2 or group_sizes.shape[0] != 2\\n            or group_sizes.dtype.kind not in \\"iu\\" or np.any(group_sizes < 0)):\\n        raise ValueError(\\"Invalid user assignments or group_sizes shape/dtype\\")\\n    B = group_sizes.shape[1]\\n    active = np.bincount(R.indices, minlength=M) > 0\\n    if (np.any(~np.isin(half[active], (0, 1))) or np.any(user_bin[active] < 0)\\n            or np.any(user_bin[active] >= B)\\n            or np.any(half[~active] != -1) or np.any(user_bin[~active] != -1)):\\n        raise ValueError(\\"Active users require valid groups; inactive users require group ID -1\\")\\n    actual_sizes = np.bincount(\\n        half[active] * B + user_bin[active], minlength=2 * B,\\n    ).reshape(2, B)\\n    if not np.array_equal(actual_sizes, group_sizes):\\n        raise ArithmeticError(\\"group_sizes do not match the disjoint active-user assignments\\")\\n\\n    E = len(E_sem)\\n    c = np.zeros((E, 2), dtype=np.int64)\\n    mu = np.zeros((E, 2), dtype=np.float64)\\n    if E == 0:\\n        return c, mu\\n    if not active.any():\\n        return c, mu\\n\\n    # One streaming incidence pass; no split R0/R1 matrices and no M x block product.\\n    item_ids = np.repeat(np.arange(N, dtype=np.int64), np.diff(R.indptr))\\n    users = R.indices\\n    flat_groups = item_ids * (2 * B) + half[users] * B + user_bin[users]\\n    n_ifb = np.bincount(flat_groups, minlength=N * 2 * B).reshape(N, 2, B)\\n    del item_ids, flat_groups\\n    inv_sizes = np.zeros((2, B), dtype=np.float64)\\n    np.divide(1.0, group_sizes, out=inv_sizes, where=group_sizes > 0)\\n    n_if = n_ifb.sum(axis=2).astype(np.float64)\\n    inv_half = np.zeros(2, dtype=np.float64)\\n    half_sizes = group_sizes.sum(axis=1)\\n    np.divide(1.0, half_sizes, out=inv_half, where=half_sizes > 0)\\n\\n    # NumPy searchsorted executes the posting-list search in compiled code. Python\\n    # dispatch is per eligible edge, never per user-history pair. Peak intersection\\n    # scratch is O(min(n_i,n_j)), independent of M*candidate_block or N*N.\\n    for start in range(0, E, candidate_block):\\n        stop = min(start + candidate_block, E)\\n        src, dst = E_sem[start:stop].T\\n        for offset, (i, j) in enumerate(zip(src, dst)):\\n            left = R.indices[R.indptr[i]:R.indptr[i + 1]]\\n            right = R.indices[R.indptr[j]:R.indptr[j + 1]]\\n            if len(left) > len(right):\\n                left, right = right, left\\n            if not len(left):\\n                continue\\n            positions = np.searchsorted(right, left)\\n            valid = positions < len(right)\\n            selected = left[valid]\\n            common = selected[right[positions[valid]] == selected]\\n            c[start + offset] = np.bincount(half[common], minlength=2)\\n        if mode == \\"unstratified_replication\\":\\n            mu[start:stop] = n_if[src] * n_if[dst] * inv_half\\n        else:\\n            # Cast each operand before multiplication; do not overflow int counters.\\n            left_counts = n_ifb[src].astype(np.float64)\\n            right_counts = n_ifb[dst].astype(np.float64)\\n            mu[start:stop] = np.sum(left_counts * right_counts * inv_sizes, axis=2)\\n        if np.any((c[start:stop] > 0) & (mu[start:stop] <= 0)):\\n            raise ArithmeticError(\\"Positive overlap must imply positive null expectation\\")\\n    return c, mu\\n\\n\\ndef replicated_excess_score(\\n    c: np.ndarray,\\n    mu: np.ndarray,\\n    tau: float = 2.0,\\n    mode: str = \\"replicated_excess\\",\\n    placebo_seed: int = 20261016,\\n    degrees: Optional[np.ndarray] = None,\\n    E_sem: Optional[np.ndarray] = None,\\n) -> np.ndarray:\\n    \\"\\"\\"Computes bounded replicated excess score r in [0, 1) (Eq 5) and scientific controls.\\n\\n    Primary formulation:\\n    - e_ijf = [c_ijf - mu_ijf]_+ / (c_ijf + mu_ijf + tau)\\n    - r_ij = min(e_ij0, e_ij1)\\n\\n    Guarantees:\\n    - 0 <= r_ij < 1.0\\n    - Primary replicated_excess is zero if either half has zero count.\\n    - Pooled/uniform controls intentionally have different neutrality semantics.\\n    - No division by zero, no NaN/Inf.\\n\\n    Ablation modes:\\n    - \'replicated_excess\': Primary replicated excess score (Eq 5).\\n    - \'v8_exact\': Exactly r = 0 (recovers native V8).\\n    - \'pooled_excess\': r = [c_pool - mu_pool]_+ / (c_pool + mu_pool + 2*tau).\\n    - \'unstratified_replication\': Unstratified null expectation replication.\\n    - \'replicated_count\': r = min(c0 / (c0 + tau), c1 / (c1 + tau)).\\n    - \'degree_stratified_shuffle\': Permute r among pairs stratified by endpoint degree.\\n    - \'uniform_multiplier\': Constant r across all candidate edges.\\n    \\"\\"\\"\\n    from stair_icaart_config import ALLOWED_EVIDENCE_MODES, FULL_ACTIVITY_MODES\\n    if mode not in ALLOWED_EVIDENCE_MODES:\\n        raise ValueError(f\\"Unknown evidence mode: {mode}\\")\\n    if isinstance(tau, bool) or tau <= 0 or not math.isfinite(tau):\\n        raise ValueError(f\\"tau must be finite and positive; got {tau}\\")\\n    if mode in FULL_ACTIVITY_MODES:\\n        raise ValueError(\\"Full-activity modes require pooled_activity_score, not half statistics\\")\\n    c, mu = np.asarray(c), np.asarray(mu)\\n    if c.ndim != 2 or c.shape[1] != 2 or c.dtype.kind not in \\"iu\\" or mu.shape != c.shape:\\n        raise ValueError(\\"c must be integer [E, 2] and mu must have the same shape\\")\\n    if np.any(c < 0) or not np.isfinite(mu).all() or np.any(mu < 0):\\n        raise ValueError(\\"Counts and expectations must be finite and non-negative\\")\\n    if np.any((c > 0) & (mu <= 0)):\\n        raise ArithmeticError(\\"Positive overlap must imply positive null expectation\\")\\n    E = c.shape[0]\\n    if E == 0:\\n        return np.empty(0, dtype=np.float64)\\n\\n    c0 = c[:, 0].astype(np.float64)\\n    c1 = c[:, 1].astype(np.float64)\\n    mu0 = mu[:, 0].astype(np.float64)\\n    mu1 = mu[:, 1].astype(np.float64)\\n\\n    if mode == \\"v8_exact\\":\\n        return np.zeros(E, dtype=np.float64)\\n\\n    elif mode in (\\"replicated_excess\\", \\"unstratified_replication\\"):\\n        excess0 = np.maximum(0.0, c0 - mu0)\\n        excess1 = np.maximum(0.0, c1 - mu1)\\n        denom0 = c0 + mu0 + tau\\n        denom1 = c1 + mu1 + tau\\n        e0 = excess0 / denom0\\n        e1 = excess1 / denom1\\n        r = np.minimum(e0, e1)\\n\\n    elif mode == \\"pooled_excess\\":\\n        c_pool = c0 + c1\\n        mu_pool = mu0 + mu1\\n        excess_pool = np.maximum(0.0, c_pool - mu_pool)\\n        denom_pool = c_pool + mu_pool + 2.0 * tau\\n        r = excess_pool / denom_pool\\n\\n    elif mode == \\"replicated_count\\":\\n        r0 = c0 / (c0 + tau)\\n        r1 = c1 / (c1 + tau)\\n        r = np.minimum(r0, r1)\\n\\n    elif mode == \\"degree_stratified_shuffle\\":\\n        # First compute standard replicated excess\\n        excess0 = np.maximum(0.0, c0 - mu0)\\n        excess1 = np.maximum(0.0, c1 - mu1)\\n        e0 = excess0 / (c0 + mu0 + tau)\\n        e1 = excess1 / (c1 + mu1 + tau)\\n        r_base = np.minimum(e0, e1)\\n\\n        if degrees is None or E_sem is None:\\n            raise ValueError(\\"Degree-stratified shuffle requires train item degrees and candidate pairs\\")\\n        degrees, E_sem = np.asarray(degrees), np.asarray(E_sem)\\n        if (degrees.ndim != 1 or not np.isfinite(degrees).all() or np.any(degrees < 0)\\n                or E_sem.shape != (E, 2) or E_sem.dtype.kind not in \\"iu\\"\\n                or np.any(E_sem < 0) or np.any(E_sem >= len(degrees))):\\n            raise ValueError(\\"Invalid train item degrees or candidate pairs for shuffle\\")\\n        cuts = np.unique(np.quantile(degrees, [0.2, 0.4, 0.6, 0.8]))\\n        strata = np.searchsorted(cuts, degrees, side=\\"right\\")\\n        left, right = strata[E_sem[:, 0]], strata[E_sem[:, 1]]\\n        groups = np.minimum(left, right) * 5 + np.maximum(left, right)\\n        rng = np.random.default_rng(placebo_seed)\\n        r = r_base.copy()\\n        for group in np.unique(groups):\\n            positions = np.flatnonzero(groups == group)\\n            if len(positions) > 1:\\n                r[positions] = rng.permutation(r_base[positions])\\n\\n    elif mode == \\"uniform_multiplier\\":\\n        excess0 = np.maximum(0.0, c0 - mu0)\\n        excess1 = np.maximum(0.0, c1 - mu1)\\n        e0 = excess0 / (c0 + mu0 + tau)\\n        e1 = excess1 / (c1 + mu1 + tau)\\n        r_base = np.minimum(e0, e1)\\n        # Set to mean of positive values, or 0.5\\n        pos = r_base[r_base > 0]\\n        val = float(np.mean(pos)) if len(pos) > 0 else 0.5\\n        r = np.full(E, val, dtype=np.float64)\\n\\n    else:\\n        raise ValueError(f\\"Unknown evidence mode: {mode}\\")\\n\\n    # Boundedness assertion: 0 <= r < 1\\n    if np.any(r < 0.0) or np.any(r >= 1.0) or not np.isfinite(r).all():\\n        raise ArithmeticError(f\\"Evidence score r outside [0, 1): min={r.min()}, max={r.max()}\\")\\n\\n    return r\\n\\n\\n# Public alias for backward compatibility and verification scripts\\ncompute_replicated_excess_evidence = replicated_excess_score\\n\\n"},"models/stair_icaart_graph.py":{"base_sha256":"33bd582c8cd3b298eed1ce47ad99837ec9f0cd220778d684ae3d96161a455aad","sha256":"aebe7c2a2e7f70c5ea9d7abef58e64f62537d0ad0a226f8f228f33265976a229","content":"\\"\\"\\"models/stair_icaart_graph.py — Pooled / Replicated Excess-Evidence Calibration (PEEC / REEC).\\n========================================================================================\\nImplements:\\n1. Recovery of exact native STAIR5-v8 raw semantic graph W^8 and degree normalization N(W).\\n2. Exponential calibration W^delta = W^8 * exp(delta * r) on undirected candidate support E_sem.\\n3. True actual-degree re-normalization S_delta = N(W^delta) and unchanged CF blend S_ICA.\\n4. Exact V8 bypass (at delta = 0 or when all r = 0).\\n5. Offline mechanism diagnostics: coverage, disagreement, endpoint contrast (Eq 10),\\n   operator dose, polynomial direction dose (Eq 11), and degree side-effect ratios.\\n6. Checkpoint manifest generation and graph fingerprinting.\\n\\"\\"\\"\\n\\nfrom dataclasses import dataclass\\nimport hashlib\\nimport json\\nimport math\\nimport time\\nfrom pathlib import Path\\nfrom typing import Any, Dict, List, Optional, Tuple\\n\\nimport numpy as np\\nimport scipy.sparse as sp\\nimport torch\\n\\nfrom models.stair5_v8_graph import (\\n    build_wmsg_semantic_graph,\\n)\\nfrom models.stair5_v4_graph import (\\n    normalize_cf_graph_with_fallback,\\n    binary_train_keys, CACHE_VERSION, _digest_array, build_candidate_support_graph,\\n    sparse_fingerprint,\\n    tensor_to_scipy,\\n    scipy_to_tensor,\\n)\\nfrom models.stair5_v4_utils import atomic_torch_save, file_sha256\\nfrom stair_icaart_config import (\\n    FULL_ACTIVITY_MODES, analytical_icaart_payload, evidence_identity, validate_icaart_config,\\n)\\nfrom models.stair_icaart_evidence import (\\n    build_activity_replication,\\n    candidate_excess_statistics,\\n    replicated_excess_score,\\n    build_activity_strata,\\n    candidate_pooled_statistics,\\n    pooled_activity_score,\\n)\\n\\n\\n@dataclass\\nclass GraphStateICAART:\\n    \\"\\"\\"Encapsulates the final calibrated S_ICA operator, audit manifest, and diagnostics.\\"\\"\\"\\n    operator: torch.Tensor\\n    metadata: Dict[str, Any]\\n    manifest: Dict[str, Any]\\n\\n\\ndef normalize_degree_isolated(W: sp.csr_matrix, dtype: Any = np.float32) -> sp.csr_matrix:\\n    \\"\\"\\"Computes exact symmetric degree normalization N(W) with identity isolated blocks.\\n\\n    Mathematical definition (Eq 2):\\n        N(W) = D^{-1/2} W D^{-1/2} + diag(1[d_i == 0]), where d = W 1.\\n\\n    Contracts:\\n    - Computed in FP64 for numerical precision.\\n    - Preserves exact symmetry and zero diagonal on connected nodes.\\n    - Returns CSR matrix with sorted indices and eliminated zeros in requested dtype (default float32).\\n    \\"\\"\\"\\n    if W.shape[0] != W.shape[1]:\\n        raise ValueError(\\"Graph matrix must be square\\")\\n\\n    W_64 = W.astype(np.float64).tocsr(copy=True)\\n    W_64.sum_duplicates()\\n    W_64.eliminate_zeros()\\n\\n    if not np.isfinite(W_64.data).all() or np.any(W_64.data < 0.0):\\n        raise ValueError(\\"Matrix entries must be finite and non-negative\\")\\n\\n    # Strict symmetry check\\n    diff = W_64 - W_64.T\\n    diff.eliminate_zeros()\\n    if diff.nnz > 0:\\n        raise ValueError(\\"Raw graph must be exactly symmetric before normalization\\")\\n\\n    n = W_64.shape[0]\\n    degrees = np.asarray(W_64.sum(axis=1)).ravel()\\n\\n    inv_sqrt = np.zeros(n, dtype=np.float64)\\n    pos_mask = degrees > 0.0\\n    inv_sqrt[pos_mask] = 1.0 / np.sqrt(degrees[pos_mask])\\n\\n    D_scale = sp.diags(inv_sqrt, format=\\"csr\\")\\n    normalized = (D_scale @ W_64 @ D_scale).astype(dtype)\\n\\n    # Isolated nodes receive identity diagonal\\n    isolated_mask = (degrees == 0.0).astype(dtype)\\n    if np.any(isolated_mask > 0.0):\\n        normalized = normalized + sp.diags(isolated_mask, format=\\"csr\\")\\n\\n    normalized = normalized.tocsr()\\n    normalized.sum_duplicates()\\n    normalized.eliminate_zeros()\\n    normalized.sort_indices()\\n    return normalized\\n\\n\\ndef recover_v8_raw_semantic_graph(\\n    mfeats: List[torch.Tensor],\\n    knn_edges_list: List[torch.Tensor],\\n    n_items: int,\\n) -> Tuple[sp.csr_matrix, np.ndarray, np.ndarray, Dict[str, Any]]:\\n    \\"\\"\\"Recovers the exact native STAIR5-v8 raw semantic graph W^8 (Eq 1).\\n\\n    Uses frozen text (5) and vision (1) candidate sets with f_p(a) = 0.05 + 0.95 * [max(0, a)]^2,\\n    B_ij = sum_m 1[j in K_m(i)] f_p(a_ij^m), W^8_ij = max(B_ij, B_ji), W^8_ii = 0.\\n\\n    Returns:\\n        W8_csr: Native V8 symmetric raw semantic matrix [N, N].\\n        E_sem: Unique undirected candidate pairs [E, 2] with i < j.\\n        w8_values: Raw semantic weights [E] corresponding to E_sem.\\n        audit: Modality and graph construction audit statistics.\\n    \\"\\"\\"\\n    W8_csr, audit = build_wmsg_semantic_graph(\\n        mfeats=mfeats,\\n        knn_edges_list=knn_edges_list,\\n        n_items=n_items,\\n        semantic_mode=\\"weighted\\",\\n        edge_power=2.0,\\n        edge_floor=0.05,\\n        relation_gate=\\"off\\",\\n        relation_strength=0.0,\\n    )\\n\\n    # Extract unique undirected pairs i < j\\n    upper = sp.triu(W8_csr, k=1).tocoo()\\n    E_sem = np.column_stack((upper.row, upper.col)).astype(np.int64)\\n    w8_values = upper.data.astype(np.float64)\\n\\n    return W8_csr, E_sem, w8_values, audit\\n\\n\\ndef compute_mechanism_diagnostics(\\n    W8_csr: sp.csr_matrix,\\n    E_sem: np.ndarray,\\n    w8_values: np.ndarray,\\n    r: np.ndarray,\\n    c: np.ndarray,\\n    mu: np.ndarray,\\n    S_8: sp.csr_matrix,\\n    S_ICA: sp.csr_matrix,\\n    d8: np.ndarray,\\n    d_delta: np.ndarray,\\n    gamma: float = 0.2,\\n    embedding_dim: int = 64,\\n    probe_dim: int = 8,\\n    probe_seed: int = 20261016,\\n    evidence_mode: str = \\"replicated_excess\\",\\n) -> Dict[str, Any]:\\n    \\"\\"\\"Computes comprehensive offline mechanism diagnostics before expensive GPU training.\\n\\n    Diagnostics computed:\\n    1. Replicated coverage: frac(r > 0) and weighted coverage sum(W8 * 1[r > 0]) / sum(W8).\\n    2. Single-half disagreement: frac((e0 > 0) xor (e1 > 0)).\\n    3. Endpoint contrast (Eq 10): r_ij - 0.5 * (rbar_i + rbar_j).\\n    4. Operator relative dose: ||S_ICA - S_8||_F / ||S_8||_F.\\n    5. Polynomial direction dose (Eq 11): ||[P(S_ICA) - P(S_8)] Z||_F / ||P(S_8) Z||_F.\\n    6. Degree side effect: d_delta / d8 on edges with r = 0.\\n    \\"\\"\\"\\n    E = len(E_sem)\\n    N = W8_csr.shape[0]\\n    diag: Dict[str, Any] = {}\\n\\n    if E == 0:\\n        empty = {\\"empty_support\\": True, \\"num_candidate_edges\\": 0,\\n                 \\"num_edges_with_positive_evidence\\": 0,\\n                 \\"frac_edges_with_positive_evidence\\": 0.0,\\n                 \\"weighted_evidence_coverage\\": 0.0,\\n                 \\"frac_single_half_disagreement\\": None,\\n                 \\"relative_operator_dose\\": 0.0, \\"relative_polynomial_dose\\": 0.0,\\n                 \\"polynomial_dose_by_coordinate\\": {}}\\n        if evidence_mode not in FULL_ACTIVITY_MODES:\\n            empty.update(num_edges_with_replicated_excess=0,\\n                         frac_edges_with_replicated_excess=0.0,\\n                         weighted_replicated_coverage=0.0)\\n        return empty\\n\\n    # 1. Replicated coverage\\n    r_pos = r > 0.0\\n    num_pos = int(np.count_nonzero(r_pos))\\n    frac_pos = float(num_pos / E)\\n    total_w8 = float(np.sum(w8_values))\\n    weighted_pos = float(np.sum(w8_values[r_pos]) / max(1e-12, total_w8))\\n\\n    diag[\\"num_candidate_edges\\"] = E\\n    diag[\\"num_edges_with_positive_evidence\\"] = num_pos\\n    diag[\\"frac_edges_with_positive_evidence\\"] = frac_pos\\n    diag[\\"weighted_evidence_coverage\\"] = weighted_pos\\n    if evidence_mode not in FULL_ACTIVITY_MODES:\\n        # Read-compatible aliases for legacy manifests only; PEEC is not replication.\\n        diag[\\"num_edges_with_replicated_excess\\"] = num_pos\\n        diag[\\"frac_edges_with_replicated_excess\\"] = frac_pos\\n        diag[\\"weighted_replicated_coverage\\"] = weighted_pos\\n    diag[\\"r_mean\\"] = float(np.mean(r))\\n    diag[\\"r_max\\"] = float(np.max(r)) if E > 0 else 0.0\\n    diag[\\"r_positive_mean\\"] = float(np.mean(r[r_pos])) if num_pos > 0 else 0.0\\n\\n    # 2. Single-half disagreement\\n    diag[\\"frac_single_half_disagreement\\"] = None\\n    diag[\\"num_single_half_disagreement\\"] = None\\n    diag[\\"num_both_halves_positive\\"] = None\\n    if c is not None and mu is not None and c.shape == mu.shape == (E, 2):\\n        e0_pos = (c[:, 0] - mu[:, 0]) > 0.0\\n        e1_pos = (c[:, 1] - mu[:, 1]) > 0.0\\n        disagreement = e0_pos ^ e1_pos\\n        diag[\\"frac_single_half_disagreement\\"] = float(np.mean(disagreement))\\n        diag[\\"num_single_half_disagreement\\"] = int(np.count_nonzero(disagreement))\\n        diag[\\"num_both_halves_positive\\"] = int(np.count_nonzero(e0_pos & e1_pos))\\n\\n    # 3. Endpoint contrast (Eq 10): r_ij - 0.5 * rbar_i - 0.5 * rbar_j\\n    # rbar_i = sum_j W8_ij * r_ij / d8_i\\n    weighted_scores = w8_values * r\\n    weighted_r_sum = (np.bincount(E_sem[:, 0], weights=weighted_scores, minlength=N)\\n                      + np.bincount(E_sem[:, 1], weights=weighted_scores, minlength=N))\\n\\n    rbar = np.zeros(N, dtype=np.float64)\\n    pos_deg = d8 > 0.0\\n    rbar[pos_deg] = weighted_r_sum[pos_deg] / d8[pos_deg]\\n\\n    u_idx = E_sem[:, 0]\\n    v_idx = E_sem[:, 1]\\n    contrast = r - 0.5 * (rbar[u_idx] + rbar[v_idx])\\n\\n    diag[\\"endpoint_contrast_mean\\"] = float(np.mean(contrast))\\n    diag[\\"endpoint_contrast_std\\"] = float(np.std(contrast))\\n    diag[\\"endpoint_contrast_min\\"] = float(np.min(contrast))\\n    diag[\\"endpoint_contrast_max\\"] = float(np.max(contrast))\\n    diag[\\"endpoint_contrast_p10\\"] = float(np.percentile(contrast, 10))\\n    diag[\\"endpoint_contrast_p50\\"] = float(np.percentile(contrast, 50))\\n    diag[\\"endpoint_contrast_p90\\"] = float(np.percentile(contrast, 90))\\n\\n    # 4. Operator relative dose: ||S_ICA - S_8||_F / ||S_8||_F\\n    diff_S = (S_ICA - S_8).astype(np.float64)\\n    fro_diff = float(np.sqrt(np.dot(diff_S.data, diff_S.data)))\\n    fro_s8 = float(np.sqrt(np.dot(S_8.data.astype(np.float64), S_8.data.astype(np.float64))))\\n    rel_operator_dose = fro_diff / max(1e-12, fro_s8)\\n    diag[\\"frobenius_operator_delta\\"] = fro_diff\\n    diag[\\"relative_operator_dose\\"] = rel_operator_dose\\n\\n    # 5. Polynomial direction dose (Eq 11): ||[P(S_ICA) - P(S_8)] Z||_F / ||P(S_8) Z||_F\\n    rng = np.random.default_rng(probe_seed)\\n    Z = rng.choice([-1.0, 1.0], size=(N, probe_dim)).astype(np.float64)\\n\\n    # Reuse three SpMM powers per operator across all five probe coordinates.\\n    coordinates = sorted({0, embedding_dim // 4, embedding_dim // 2,\\n                          3 * embedding_dim // 4, embedding_dim - 1})\\n    doses = {}\\n    if not diff_S.nnz:\\n        doses = {str(k): 0.0 for k in coordinates}\\n    else:\\n        powers8, powers_ica = [Z], [Z]\\n        for _ in range(3):\\n            powers8.append(S_8 @ powers8[-1])\\n            powers_ica.append(S_ICA @ powers_ica[-1])\\n        for coordinate in coordinates:\\n            beta = .1 + .9 * (coordinate / embedding_dim) ** gamma\\n            omega = beta ** np.arange(4, dtype=np.float64)\\n            omega /= omega.sum()\\n            reference = sum(w * x for w, x in zip(omega, powers8))\\n            calibrated = sum(w * x for w, x in zip(omega, powers_ica))\\n            doses[str(coordinate)] = float(\\n                np.linalg.norm(calibrated - reference) / max(1e-12, np.linalg.norm(reference))\\n            )\\n    diag[\\"polynomial_dose_by_coordinate\\"] = doses\\n    diag[\\"relative_polynomial_dose\\"] = doses[str(embedding_dim // 2)]\\n    diag[\\"polynomial_probe_coordinate\\"] = embedding_dim // 2\\n    diag[\\"polynomial_probe_dim\\"] = int(probe_dim)\\n    diag[\\"polynomial_probe_seed\\"] = int(probe_seed)\\n    diag[\\"polynomial_probe_scope\\"] = \\"fixed Rademacher L3 probes; per-coordinate ratios, not actual Adam directions\\"\\n\\n    row_delta = np.asarray(abs(diff_S).sum(axis=1)).ravel()\\n    row_base = np.asarray(abs(S_8).sum(axis=1)).ravel()\\n    relative_rows = np.divide(row_delta, row_base, out=np.zeros(N), where=row_base > 0)\\n    diag[\\"relative_row_l1_dose_quantiles\\"] = np.quantile(relative_rows, [0, .1, .5, .9, 1]).tolist()\\n\\n    # 6. Degree side effect: d_delta / d8 on edges where r = 0\\n    zero_r_mask = ~r_pos\\n    if np.any(zero_r_mask):\\n        u_zero = u_idx[zero_r_mask]\\n        v_zero = v_idx[zero_r_mask]\\n        items_in_zero_r = np.unique(np.concatenate([u_zero, v_zero]))\\n        valid_items = items_in_zero_r[d8[items_in_zero_r] > 0]\\n        if len(valid_items) > 0:\\n            deg_ratios = d_delta[valid_items] / d8[valid_items]\\n            diag[\\"degree_ratio_on_zero_r_mean\\"] = float(np.mean(deg_ratios))\\n            diag[\\"degree_ratio_on_zero_r_max\\"] = float(np.max(deg_ratios))\\n            diag[\\"degree_ratio_on_zero_r_min\\"] = float(np.min(deg_ratios))\\n        # With zero evidence on the edge, its normalized coefficient changes only\\n        # through endpoint degree growth. This is not a retained-weight guarantee.\\n        ratios = np.sqrt((d8[u_zero] / d_delta[u_zero]) * (d8[v_zero] / d_delta[v_zero]))\\n        diag[\\"zero_score_semantic_edge_coefficient_ratio_quantiles\\"] = np.quantile(\\n            ratios, [0, .1, .5, .9, 1],\\n        ).tolist()\\n\\n    return diag\\n\\n\\ndef calibrate_semantic_operator(\\n    W8_csr: sp.csr_matrix,\\n    E_sem: np.ndarray,\\n    r: np.ndarray,\\n    delta: float,\\n    SCF_csr: Optional[sp.csr_matrix] = None,\\n    eta: float = 0.1,\\n    V8_operator_cached: Optional[sp.csr_matrix] = None,\\n    gamma: float = 0.2,\\n    embedding_dim: int = 64,\\n    c_counts: Optional[np.ndarray] = None,\\n    mu_expectations: Optional[np.ndarray] = None,\\n    evidence_mode: str = \\"replicated_excess\\",\\n) -> Tuple[sp.csr_matrix, Dict[str, Any]]:\\n    \\"\\"\\"Calibrates semantic operator with exponential reweighting and returns S_ICA (Eq 6).\\n\\n    Mathematical formulation:\\n        W^delta_ij = W^8_ij * exp(delta * r_ij) on undirected candidate edges i < j,\\n        W^delta_ji = W^delta_ij, W^delta_ii = 0.\\n        S_delta = N(W^delta) with true actual new degrees d_delta = W^delta 1.\\n        S_ICA = (1 - eta) * S_delta + eta * S_CF (or S_delta if CF is empty/disabled).\\n\\n    Exact Bypass:\\n        - When delta == 0: returns exact V8 cached operator directly.\\n        - When all r == 0: returns exact V8 cached operator with inactive_reason=\\"no_replicated_excess\\".\\n    \\"\\"\\"\\n    if not math.isfinite(delta) or not 0 <= delta <= 2:\\n        raise ValueError(\\"delta must be finite in [0, 2]\\")\\n    if not math.isfinite(eta) or not 0 <= eta <= 1:\\n        raise ValueError(\\"eta must be finite in [0, 1]\\")\\n    W8_csr = W8_csr.tocsr(copy=True)\\n    W8_csr.sum_duplicates()\\n    W8_csr.eliminate_zeros()\\n    W8_csr.sort_indices()\\n    N = W8_csr.shape[0]\\n    if (W8_csr.shape != (N, N) or not np.isfinite(W8_csr.data).all()\\n            or np.any(W8_csr.data < 0) or np.any(W8_csr.diagonal() != 0)\\n            or (W8_csr - W8_csr.T).nnz):\\n        raise ValueError(\\"W8 must be square, symmetric, finite, nonnegative and zero-diagonal\\")\\n    E_sem, r = np.asarray(E_sem), np.asarray(r, dtype=np.float64)\\n    E = len(E_sem)\\n    if (E_sem.shape != (E, 2) or E_sem.dtype.kind not in \\"iu\\" or r.shape != (E,)\\n            or not np.isfinite(r).all() or np.any(r < 0) or np.any(r >= 1)\\n            or np.any(E_sem < 0) or np.any(E_sem >= N)\\n            or np.any(E_sem[:, 0] >= E_sem[:, 1])):\\n        raise ValueError(\\"Invalid candidate pairs or scores\\")\\n    upper = sp.triu(W8_csr, k=1, format=\\"coo\\")\\n    expected_keys = upper.row.astype(np.int64) * N + upper.col\\n    pair_keys = E_sem[:, 0].astype(np.int64) * N + E_sem[:, 1]\\n    if (len(pair_keys) != upper.nnz\\n            or not np.array_equal(np.sort(pair_keys), np.sort(expected_keys))):\\n        raise ValueError(\\"E_sem must contain each raw semantic edge exactly once\\")\\n    # Explicit indexing binds weights to caller pair order; triu storage order is\\n    # not an API contract and must never silently permute evidence assignments.\\n    if E == 0:\\n        w8_values = np.empty(0, dtype=np.float64)\\n    else:\\n        w8_values = np.asarray(W8_csr[E_sem[:, 0], E_sem[:, 1]]).ravel().astype(np.float64)\\n    d8 = np.asarray(W8_csr.astype(np.float64).sum(axis=1)).ravel()\\n    if SCF_csr is not None:\\n        if (SCF_csr.shape != (N, N) or not np.isfinite(SCF_csr.data).all()\\n                or np.any(SCF_csr.data < 0)):\\n            raise ValueError(\\"Invalid normalized CF graph\\")\\n        diff_cf = SCF_csr - SCF_csr.T\\n        if diff_cf.nnz and np.max(np.abs(diff_cf.data)) > 1e-7:\\n            raise ValueError(\\"Normalized CF graph must be symmetric within FP32 tolerance\\")\\n    effective_eta = float(eta if SCF_csr is not None and SCF_csr.nnz and eta > 0 else 0)\\n    if V8_operator_cached is not None:\\n        if (V8_operator_cached.shape != (N, N)\\n                or not np.isfinite(V8_operator_cached.data).all()\\n                or np.any(V8_operator_cached.data < 0)):\\n            raise ValueError(\\"Cached V8 operator must be finite, nonnegative and shape-compatible\\")\\n        S_8 = V8_operator_cached\\n    else:\\n        semantic = normalize_degree_isolated(W8_csr)\\n        S_8 = (((1 - effective_eta) * semantic + effective_eta * SCF_csr).tocsr()\\n               if effective_eta else semantic)\\n        S_8.eliminate_zeros()\\n        S_8.sort_indices()\\n\\n    audit = {\\"delta\\": float(delta), \\"effective_eta\\": effective_eta,\\n             \\"n_items\\": N, \\"n_candidate_edges\\": E,\\n             \\"exact_v8_bypass\\": False, \\"inactive_reason\\": None}\\n    d_delta = d8\\n    if delta == 0 or E == 0 or not np.any(r):\\n        reason = \\"delta_zero\\" if delta == 0 else (\\n            \\"no_positive_excess\\" if evidence_mode in FULL_ACTIVITY_MODES else \\"no_replicated_excess\\"\\n        )\\n        audit.update(exact_v8_bypass=True, inactive_reason=reason)\\n        S_ICA = S_8\\n    else:\\n        weighted = w8_values * np.exp(delta * r)\\n        directed = sp.csr_matrix((weighted, (E_sem[:, 0], E_sem[:, 1])), shape=(N, N))\\n        W_delta = (directed + directed.T).tocsr()\\n        d_delta = np.asarray(W_delta.sum(axis=1)).ravel()\\n        # Detect exact component-constant evidence without treating small real\\n        # contrasts as zero via np.allclose\'s default relative tolerance.\\n        _, components = sp.csgraph.connected_components(W8_csr, directed=False)\\n        labels = components[E_sem[:, 0]]\\n        lo = np.full(N, np.inf)\\n        hi = np.full(N, -np.inf)\\n        np.minimum.at(lo, labels, r)\\n        np.maximum.at(hi, labels, r)\\n        present = np.isfinite(lo)\\n        neutral = bool(np.all(lo[present] == hi[present]))\\n        audit[\\"neutral_calibration\\"] = neutral\\n        if neutral:\\n            S_ICA = S_8\\n            audit.update(exact_v8_bypass=True, inactive_reason=\\"component_constant_evidence\\")\\n        else:\\n            semantic = normalize_degree_isolated(W_delta)\\n            S_ICA = (((1 - effective_eta) * semantic + effective_eta * SCF_csr).tocsr()\\n                     if effective_eta else semantic)\\n            S_ICA.sum_duplicates()\\n            S_ICA.eliminate_zeros()\\n            S_ICA.sort_indices()\\n    audit.update(compute_mechanism_diagnostics(\\n        W8_csr, E_sem, w8_values, r,\\n        c_counts, mu_expectations,\\n        S_8, S_ICA, d8, d_delta, gamma, embedding_dim,\\n        evidence_mode=evidence_mode,\\n    ))\\n    audit[\\"s_ica_nnz\\"] = int(S_ICA.nnz)\\n    audit[\\"s_ica_fingerprint\\"] = sparse_fingerprint(S_ICA)\\n    return S_ICA, audit\\n\\n\\ndef _native_cf_operator(train_edges, n_users, n_items, cfg):\\n    \\"\\"\\"Reuse V8\'s raw-CF cache signature, checksum and exact native normalizer.\\"\\"\\"\\n    pairs = binary_train_keys(train_edges, n_users, n_items)\\n    digest = hashlib.sha256()\\n    _digest_array(digest, pairs)\\n    signature = {\\n        \\"cache_version\\": CACHE_VERSION, \\"train\\": digest.hexdigest(),\\n        \\"users\\": n_users, \\"items\\": n_items, \\"k\\": int(getattr(cfg, \\"k_cf\\", 5)),\\n        \\"cmin\\": int(getattr(cfg, \\"c_min\\", 2)), \\"t\\": float(getattr(cfg, \\"t_shrinkage\\", 5)),\\n        \\"source\\": file_sha256(Path(__file__).with_name(\\"stair5_v4_graph.py\\")),\\n    }\\n    key = hashlib.sha256(json.dumps(signature, sort_keys=True).encode()).hexdigest()\\n    cache_dir = getattr(cfg, \\"graph_cache_dir\\", \\"\\")\\n    cache = Path(cache_dir) / f\\"cf_{key}.pt\\" if cache_dir else None\\n    raw = None\\n    hit = False\\n    if cache is not None and cache.is_file():\\n        payload = torch.load(cache, map_location=\\"cpu\\", weights_only=True)\\n        raw = tensor_to_scipy(payload[\\"operator\\"])\\n        if (payload.get(\\"signature\\") != signature or raw.shape != (n_items, n_items)\\n                or payload.get(\\"fingerprint\\") != sparse_fingerprint(raw)):\\n            raise ValueError(\\"Native CF cache signature or checksum mismatch\\")\\n        hit = True\\n    if raw is None:\\n        raw = build_candidate_support_graph(\\n            train_edges, n_items, n_users, signature[\\"k\\"], signature[\\"cmin\\"], signature[\\"t\\"],\\n            int(getattr(cfg, \\"cf_block_size\\", 64)), float(getattr(cfg, \\"cf_memory_budget_mib\\", 128)),\\n        )\\n        if cache is not None:\\n            atomic_torch_save({\\"operator\\": scipy_to_tensor(raw), \\"signature\\": signature,\\n                               \\"fingerprint\\": sparse_fingerprint(raw)}, cache)\\n    raw_hash = sparse_fingerprint(raw)\\n    # A wholly empty raw graph must not become an identity CF contribution.\\n    normalized = normalize_cf_graph_with_fallback(raw) if raw.nnz and signature[\\"k\\"] else None\\n    return normalized, {\\"native_cf_cache_hit\\": hit, \\"raw_cf_fingerprint\\": raw_hash,\\n                        \\"normalized_cf_fingerprint\\": (sparse_fingerprint(normalized)\\n                                                      if normalized is not None else None),\\n                        \\"cf_cache_key\\": key}\\n\\n\\n@torch.no_grad()\\ndef build_stair_icaart_graph(\\n    mfeats: List[torch.Tensor],\\n    knn_edges_list: List[torch.Tensor],\\n    train_edges: torch.Tensor,\\n    n_users: int,\\n    n_items: int,\\n    cfg: Any,\\n    V8_operator_cached: Optional[sp.csr_matrix] = None,\\n    SCF_csr_cached: Optional[sp.csr_matrix] = None,\\n) -> GraphStateICAART:\\n    \\"\\"\\"End-to-end builder for STAIR-REEC (versionICAART) static item graph S_ICA.\\n\\n    Pipeline:\\n    1. Recover exact native V8 raw semantic graph W^8.\\n    2. Extract deduplicated binary train-presence matrix R.\\n    3. Full activity strata for PEEC, or original two-half strata for legacy controls.\\n    4. Candidate-only co-interaction counting c and activity-null expectation mu.\\n    5. Explicitly versioned pooled-full or replicated/pooled-half scoring.\\n    6. Exponential calibration and actual-degree normalization S_ICA.\\n    7. Generate comprehensive audit manifest.\\n    \\"\\"\\"\\n    started = time.perf_counter()\\n\\n    settings = validate_icaart_config(cfg)\\n    identity = evidence_identity(settings)\\n    evidence_mode = settings[\\"evidence_mode\\"]\\n    full = evidence_mode in FULL_ACTIVITY_MODES\\n    evidence_seed = settings[\\"evidence_seed\\"]\\n    evidence_tau = settings[\\"evidence_tau\\"]\\n    evidence_delta = settings[\\"evidence_delta\\"]\\n    candidate_block = settings[\\"evidence_candidate_block\\"]\\n    placebo_seed = settings[\\"evidence_placebo_seed\\"]\\n    eta = float(getattr(cfg, \\"eta\\", 0.1))\\n    gamma = float(getattr(cfg, \\"gamma\\", 0.2))\\n    embedding_dim = int(getattr(cfg, \\"embedding_dim\\", 64))\\n\\n    # 1. Recover exact native W^8\\n    W8_csr, E_sem, w8_values, w8_audit = recover_v8_raw_semantic_graph(\\n        mfeats=mfeats,\\n        knn_edges_list=knn_edges_list,\\n        n_items=n_items,\\n    )\\n\\n    # 2. Deduplicate train presence matrix R [M, N]\\n    unique_keys = binary_train_keys(train_edges, n_users, n_items)\\n    num_raw_interactions = int(train_edges.shape[1])\\n    num_unique_interactions = len(unique_keys)\\n\\n    u_dedup = unique_keys // np.int64(n_items)\\n    i_dedup = unique_keys % np.int64(n_items)\\n\\n    R_csr = sp.csr_matrix(\\n        (np.ones(num_unique_interactions, dtype=np.int8), (u_dedup, i_dedup)),\\n        shape=(n_users, n_items),\\n    )\\n    R_csc = R_csr.tocsc()\\n\\n    # Presence hash for provenance\\n    presence_digest = hashlib.sha256(str((n_users, n_items)).encode())\\n    _digest_array(presence_digest, unique_keys)\\n    presence_hash = presence_digest.hexdigest()\\n\\n    degrees = np.diff(R_csc.indptr).astype(np.int64)\\n    if full:\\n        # No half assignment or seeded hashing in the full-activity estimator.\\n        if evidence_mode == \\"pooled_global_excess\\":\\n            # Inactive users never enter M_active. This is not a two-half control.\\n            active_users = np.diff(R_csr.indptr) > 0\\n            user_bin = np.where(active_users, 0, -1).astype(np.int64)\\n            group_sizes = np.array([np.count_nonzero(active_users)], dtype=np.int64)\\n        else:\\n            user_bin, group_sizes = build_activity_strata(R_csr)\\n        half = None\\n        c, mu = candidate_pooled_statistics(\\n            R_csc, E_sem, user_bin, group_sizes, candidate_block,\\n            null_scope=identity[\\"null_scope\\"],\\n        )\\n        r = pooled_activity_score(\\n            c, mu, evidence_tau, evidence_mode, degrees=degrees,\\n            E_sem=E_sem, placebo_seed=placebo_seed,\\n        )\\n    else:\\n        # Preserve R1 and pooled-half arithmetic and assignment semantics.\\n        user_bin, half, group_sizes = build_activity_replication(R_csr, evidence_seed)\\n        c, mu = candidate_excess_statistics(\\n            R_csc, E_sem, half, user_bin, group_sizes, candidate_block, evidence_mode,\\n        )\\n        r = replicated_excess_score(\\n            c, mu, evidence_tau, evidence_mode, placebo_seed, degrees, E_sem,\\n        )\\n\\n    # 6. Load or build CF operator S_CF\\n    cf_audit = {}\\n    if SCF_csr_cached is None:\\n        SCF_csr, cf_audit = _native_cf_operator(train_edges, n_users, n_items, cfg)\\n    else:\\n        SCF_csr = SCF_csr_cached\\n        cf_audit[\\"normalized_cf_fingerprint\\"] = sparse_fingerprint(SCF_csr)\\n\\n    if V8_operator_cached is not None:\\n        semantic_reference = normalize_degree_isolated(W8_csr)\\n        native_eta = eta if SCF_csr is not None and SCF_csr.nnz else 0.0\\n        reference = (((1 - native_eta) * semantic_reference + native_eta * SCF_csr).tocsr()\\n                     if native_eta else semantic_reference)\\n        if sparse_fingerprint(reference) != sparse_fingerprint(V8_operator_cached):\\n            raise ValueError(\\"Reconstructed native graph differs from the cached V8 control\\")\\n\\n    # 7. Calibrate semantic operator\\n    S_ICA, cal_audit = calibrate_semantic_operator(\\n        W8_csr=W8_csr,\\n        E_sem=E_sem,\\n        r=r,\\n        delta=evidence_delta,\\n        SCF_csr=SCF_csr,\\n        eta=eta,\\n        V8_operator_cached=V8_operator_cached,\\n        gamma=gamma,\\n        embedding_dim=embedding_dim,\\n        c_counts=c,\\n        mu_expectations=mu,\\n        evidence_mode=evidence_mode,\\n    )\\n\\n    pooled_counts = c if full else c.sum(axis=1)\\n    pooled_null = mu if full else mu.sum(axis=1)\\n    quantiles = [0, .1, .5, .9, .95, .99, 1]\\n    cal_audit[\\"evidence_quantile_levels\\"] = quantiles\\n    cal_audit[\\"candidate_count_quantiles\\"] = np.quantile(pooled_counts, quantiles).tolist() if len(c) else []\\n    cal_audit[\\"candidate_null_quantiles\\"] = np.quantile(pooled_null, quantiles).tolist() if len(mu) else []\\n    cal_audit[\\"evidence_score_quantiles\\"] = np.quantile(r, quantiles).tolist() if len(r) else []\\n\\n    elapsed = time.perf_counter() - started\\n\\n    # 8. Compile comprehensive manifest\\n    manifest = {\\n        **identity,\\n        \\"evidence_mode\\": evidence_mode,\\n        \\"evidence_seed\\": evidence_seed,\\n        \\"evidence_tau\\": evidence_tau,\\n        \\"evidence_delta\\": evidence_delta,\\n        \\"evidence_candidate_block\\": candidate_block,\\n        \\"evidence_activity_bins\\": settings[\\"evidence_activity_bins\\"],\\n        \\"evidence_halves\\": identity[\\"active_halves\\"],\\n        \\"evidence_placebo_seed\\": placebo_seed,\\n        \\"training_seed\\": int(getattr(cfg, \\"seed\\", 1)),\\n        \\"evidence_dtype\\": \\"float64\\",\\n        \\"operator_dtype\\": \\"float32\\",\\n        \\"raw_semantic_fingerprint\\": sparse_fingerprint(W8_csr),\\n        \\"baseline_v8_fingerprint\\": sparse_fingerprint(V8_operator_cached) if V8_operator_cached is not None else None,\\n        \\"assignment_hash\\": hashlib.sha256(\\n            user_bin.astype(\\"<i8\\").tobytes()\\n            + (b\\"\\" if half is None else half.astype(\\"<i8\\").tobytes())\\n        ).hexdigest(),\\n        \\"group_sizes\\": group_sizes.tolist(),\\n        \\"evidence_score_hash\\": hashlib.sha256(r.astype(\\"<f8\\").tobytes()).hexdigest(),\\n        \\"evidence_config\\": settings,\\n        \\"counts_shape\\": list(c.shape),\\n        \\"null_expectation_shape\\": list(mu.shape),\\n        \\"counts_hash\\": hashlib.sha256(c.astype(\\"<i8\\").tobytes()).hexdigest(),\\n        \\"null_expectation_hash\\": hashlib.sha256(mu.astype(\\"<f8\\").tobytes()).hexdigest(),\\n        \\"cf\\": {k: v for k, v in cf_audit.items() if k != \\"native_cf_cache_hit\\"},\\n        \\"num_raw_train_interactions\\": num_raw_interactions,\\n        \\"num_deduplicated_train_interactions\\": num_unique_interactions,\\n        \\"train_presence_hash\\": presence_hash,\\n        \\"num_users\\": n_users,\\n        \\"num_items\\": n_items,\\n        \\"num_activity_strata\\": int(group_sizes.shape[-1]),\\n        \\"user_half_counts\\": None if half is None else [int(np.sum(half == 0)), int(np.sum(half == 1))],\\n        \\"graph_fingerprint\\": cal_audit[\\"s_ica_fingerprint\\"],\\n    }\\n\\n    if evidence_mode == \\"pooled_global_excess\\":\\n        m_active = int(group_sizes[0])\\n        max_null_error = 0.0\\n        for start in range(0, len(E_sem), candidate_block):\\n            stop = min(start + candidate_block, len(E_sem))\\n            left, right = E_sem[start:stop].T\\n            expected_global = (degrees[left].astype(np.float64)\\n                               * degrees[right].astype(np.float64))\\n            if m_active:\\n                expected_global /= m_active\\n            else:\\n                expected_global.fill(0.0)\\n            error = np.abs(mu[start:stop] - expected_global)\\n            max_null_error = max(max_null_error, float(error.max(initial=0.0)))\\n            if not np.allclose(mu[start:stop], expected_global, rtol=1e-12, atol=1e-12):\\n                raise ArithmeticError(\\"Global-full null disagrees with active-user marginal formula\\")\\n        manifest.update(\\n            activity_conditioning=False,\\n            num_activity_strata=0,\\n            num_null_groups=1,\\n            num_active_train_users=m_active,\\n            num_inactive_train_users=n_users - m_active,\\n            global_null_formula=\\"n_i*n_j/M_active; binary TRAIN item marginals\\",\\n            global_null_max_abs_formula_error=max_null_error,\\n            global_null_check_scope=\\"candidate-wise formula consistency, not statistical calibration\\",\\n        )\\n\\n    metadata = {**w8_audit, **cal_audit, **cf_audit, **manifest,\\n                \\"build_elapsed_seconds\\": elapsed,\\n                \\"analytical_candidate_payload\\": analytical_icaart_payload(n_items, len(E_sem), pooled=full)}\\n\\n    # Preparation is CPU-only regardless of where caller edge IDs were stored.\\n    # The native coach moves the finished model and its single final CSR graph.\\n    operator_tensor = scipy_to_tensor(S_ICA)\\n\\n    return GraphStateICAART(operator=operator_tensor, metadata=metadata, manifest=manifest)\\n\\n\\n# Public alias for backward compatibility and verification scripts\\ncalibrate_semantic_support = calibrate_semantic_operator\\n\\n"}}}')
GLOBAL_OVERLAY_ID='f62a74a254ef021fb3c064070c85ea8d6b6f0d8810870e1ae9293bb3f90a24c2'
EXPECTED_RUNTIME_SOURCES=dict(SEED1_REFERENCE['baby']['contract']['sources'])
DECLARED_SOURCE_CHANGES={}
for relative, record in GLOBAL_SOURCE_OVERLAY['files'].items():
    basename=Path(relative).name
    if EXPECTED_RUNTIME_SOURCES[basename]!=record['base_sha256']:
        raise RuntimeError('Overlay base differs from recorded Full reference: '+relative)
    EXPECTED_RUNTIME_SOURCES[basename]=record['sha256']
    DECLARED_SOURCE_CHANGES[relative]={'full_sha256':record['base_sha256'],'global_sha256':record['sha256']}
write_json(RUN_ROOT/'provenance'/'global_full_source_overlay.json',GLOBAL_SOURCE_OVERLAY)
write_json(RUN_ROOT/'provenance'/'declared_source_changes.json',{'base_commit':RUNTIME_COMMIT,
    'overlay_id':GLOBAL_OVERLAY_ID,'changes':DECLARED_SOURCE_CHANGES,
    'scope':'Global-full estimator/config wiring only; Full algorithm sources remain unchanged elsewhere.'})

NOTEBOOK_SOURCE_TEMPLATE=json.loads('{"cells":[{"cell_type":"markdown","id":"acf843e3","metadata":{},"source":["# ECL-global_full_null — ECLIPSE activity-conditioning control | training seed 1\\n","\\n","**Baby → Sports**, two fresh 64D/500-epoch runs. Only the null-conditioning rule changes from Full ECLIPSE:\\n","\\n","\\\\[\\n","M_{active}=|\\\\{u:h_u>0\\\\}|,\\\\quad \\\\mu^{global}_{ij}=n_i n_j/M_{active},\\\\quad\\n","r^{global}_{ij}=\\\\frac{[c_{ij}-\\\\mu^{global}_{ij}]_+}{c_{ij}+\\\\mu^{global}_{ij}+4},\\\\quad\\n","\\\\widetilde W_{ij}=W_{ij}\\\\exp(r^{global}_{ij}).\\n","\\\\]\\n","\\n","Full-user pooled excess, **no user halves and no activity conditioning**. Native mode `pooled_global_excess`, bins `none`, halves0, delta1/tau2; WMSG+CSE eta.1+LC.01+MI/FSC/BSC and per-dataset settings match Full. Inactive TRAIN users do not enter M_active. This is neither Count-only nor `unstratified_replication`.\\n","\\n","- Attach processed Baby/Sports inputs before Save & Run All; enable GPU/Internet for setup. RUN_MODE=`train` starts jobs only in the dataset sections. Jobs run sequentially on GPU_INDEX. Electronics is excluded from this plan.\\n","- Source is the recorded Full commit **plus three declared, SHA-locked estimator files embedded here**. A private checkout is used; existing Full/Count checkouts are not patched. No unpublished GitHub commit is required. All other recorded source hashes must still match Full.\\n","- `RUN_MODE=\'report_only\'` accepts only corresponding Global-full receipts; no clone/install/model/CUDA/training in that mode. Never relabel Full/Count receipts as this arm.\\n","- Native VALID NDCG@20 selects the checkpoint; report selected TEST. Keep 500epochs; no TEST-based tuning or automatic retries/resume/deletion.\\n","- PDFs/PNG300dpi, tables/source data/captions and ZIP+checksum are generated after measured runs. A session has one actual seed/dataset, no invented SD/p-values or missing-as-zero values.\\n"]},{"cell_type":"markdown","id":"258596b3","metadata":{},"source":["## 1. Session configuration — one source of truth\\n"]},{"cell_type":"code","id":"43c7162b","metadata":{},"source":["from pathlib import Path\\n","from datetime import datetime, timezone\\n","import hashlib\\n","import json\\n","import os\\n","import sys\\n","import uuid\\n","\\n","ARM = \'global_full_null\'\\n","SEED = 1\\n","RUN_MODE = \'train\'  # or \'report_only\'\\n","EXISTING_ARTIFACT_DIRS = {}  # report_only: {\'baby\':\'/path/native_run\', ...}\\n","EXISTING_STDOUT_LOGS = {}  # report_only: {\'baby\':\'/path/stdout.log\', ...}; missing log -> timing NOT_MEASURED\\n","NOTEBOOK_SOURCE_PATH = None  # optional exact submitted .ipynb to include\\n","SELECTED_DATASETS = (\'baby\', \'sports\')\\n","GPU_INDEX = 0\\n","INSTALL_MISSING_DEPENDENCIES = True\\n","AUTO_DOWNLOAD_DATA = False  # Optional outside Kaggle only; attach inputs before Save & Run All.\\n","DATASET_HANDLE = \'rainyle/stair-datasets-mmrec\'\\n","DATASET_SEARCH_ROOTS = []  # Optional parent folders; default input/cache paths are searched automatically.\\n","STRICT_RUNTIME = True  # Require the seed-1 Python major/minor, Torch and FreeRec versions.\\n","RUNTIME_COMMIT = \'5212263e87c5c1cfe6e48c91afa90201a48f5929\'\\n","REPO_URL = \'https://github.com/ThanhChuong12/STAIR-Enhanced.git\'\\n","REPO_OVERRIDE = None  # Disabled for this arm: use an isolated base-plus-overlay checkout.\\n","DATASET_SOURCE_OVERRIDES = {}  # Example: {\'baby\': \'/kaggle/input/.../baby\'}\\n","if RUN_MODE not in (\'train\',\'report_only\'): raise ValueError(\'Invalid RUN_MODE\')\\n","if RUN_MODE==\'report_only\' and set(SELECTED_DATASETS)-set(EXISTING_ARTIFACT_DIRS):\\n","    raise ValueError(\'Set EXISTING_ARTIFACT_DIRS for every selected dataset.\')\\n","\\n","if isinstance(SEED, bool) or SEED not in (1, 2, 3):\\n","    raise ValueError(\'SEED must be 1,2 or3; default job is Global-full null seed1.\')\\n","if not SELECTED_DATASETS or len(set(SELECTED_DATASETS)) != len(SELECTED_DATASETS):\\n","    raise ValueError(\'Select one or more datasets without duplicates.\')\\n","if set(SELECTED_DATASETS)-{\'baby\',\'sports\'}:\\n","    raise ValueError(\'This follow-up notebook supports Baby/Sports only; Electronics is outside the Global-full plan.\')\\n","if \'electronics\' in SELECTED_DATASETS:\\n","    raise ValueError(\'Global-full null plan: Baby/Sports only for seeds1/2/3.\')\\n","if isinstance(GPU_INDEX, bool) or not isinstance(GPU_INDEX, int) or GPU_INDEX < 0:\\n","    raise ValueError(\'GPU_INDEX must be a non-negative integer.\')\\n","\\n","WORK_DIR = Path(\'/kaggle/working\') if Path(\'/kaggle/working\').is_dir() else Path.cwd()\\n","SESSION_ID = datetime.now(timezone.utc).strftime(\'%Y%m%dT%H%M%SZ\') + \'_\' + uuid.uuid4().hex[:8]\\n","RUN_ROOT = WORK_DIR / \'eclipse_runs\' / f\'eclipse_{ARM}_seed{SEED}_{SESSION_ID}\'\\n","RUN_ROOT.mkdir(parents=True,exist_ok=False)\\n","for name in (\'logs\', \'artifacts\', \'configs\', \'figures\', \'tables\', \'provenance\',\'helpers\'):\\n","    (RUN_ROOT / name).mkdir(parents=True, exist_ok=True)\\n","DATA_ROOT = WORK_DIR / \'eclipse_data\'\\n","GRAPH_CACHE_ROOT = WORK_DIR / \'eclipse_graph_cache\' / RUNTIME_COMMIT[:12] / ARM / \'f62a74a254ef\'\\n","GRAPH_CACHE_ROOT.mkdir(parents=True, exist_ok=True)\\n","\\n","PRIMARY_METRICS = (\'RECALL@10\', \'RECALL@20\', \'NDCG@10\', \'NDCG@20\')\\n","DATASET_TITLES = {\'baby\': \'Amazon Baby\', \'sports\': \'Amazon Sports\', \'electronics\': \'Amazon Electronics\'}\\n","DATASET_FOLDERS = {k: f\'Amazon2014{v}_550_MMRec\' for k, v in\\n","                   [(\'baby\', \'Baby\'), (\'sports\', \'Sports\'), (\'electronics\', \'Electronics\')]}\\n","print(f\'ECLIPSE Global-full null | seed={SEED} | datasets={SELECTED_DATASETS}\')\\n","print(f\'Run directory: {RUN_ROOT}\')\\n","print(\'Training starts only in the dataset training cells below.\')\\n","\\n","SESSION_STEM=f\'eclipse_{ARM}_seed{SEED}\'\\n","def stem_for(key):\\n","    if key not in DATASET_FOLDERS:raise ValueError(f\'Unknown dataset: {key}\')\\n","    return f\'eclipse_{key}_{ARM}_seed{SEED}\'\\n","ARTIFACT_PATHS={key:(Path(EXISTING_ARTIFACT_DIRS[key]).expanduser().resolve() if RUN_MODE==\'report_only\'\\n","                   else RUN_ROOT/\'artifacts\'/stem_for(key)) for key in SELECTED_DATASETS}\\n","LOG_PATHS={key:((Path(EXISTING_STDOUT_LOGS[key]).expanduser().resolve() if EXISTING_STDOUT_LOGS.get(key) else None)\\n","               if RUN_MODE==\'report_only\' else RUN_ROOT/\'logs\'/f\'{stem_for(key)}.log\') for key in SELECTED_DATASETS}\\n","if ARM!=\'global_full_null\':raise ValueError(\'This notebook is locked to Global-full null.\')\\n"],"execution_count":null,"outputs":[]},{"cell_type":"markdown","id":"48e80be6","metadata":{},"source":["## 2. Frozen Full protocol and declared Global-full extension\\n","Input/candidate/count/raw semantic/CF hashes remain fixed. Null expectation, one-pool assignment, scores and calibrated final graph may differ. Full source references remain immutable; only three declared estimator files have new source digests.\\n"]},{"cell_type":"code","id":"308c869a","metadata":{},"source":["import math, re, time, shutil\\n","def file_sha256(path):\\n","    digest=hashlib.sha256()\\n","    with Path(path).open(\'rb\') as handle:\\n","        for chunk in iter(lambda:handle.read(1024*1024),b\'\'):digest.update(chunk)\\n","    return digest.hexdigest()\\n","\\n","def write_json(path,payload):\\n","    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)\\n","    temporary=path.with_name(path.name+\'.tmp\')\\n","    temporary.write_text(json.dumps(payload,indent=2,ensure_ascii=False,allow_nan=False)+\'\\\\n\',encoding=\'utf-8\')\\n","    temporary.replace(path)\\n","\\n","SEED1_REFERENCE=json.loads(\'{\\"baby\\":{\\"commit\\":\\"5212263e87c5c1cfe6e48c91afa90201a48f5929\\",\\"python\\":\\"3.13.15\\",\\"torch\\":\\"2.11.0+cu128\\",\\"freerec\\":\\"0.9.7\\",\\"gpu\\":\\"Tesla T4\\",\\"v8_config\\":{\\"embedding_dim\\":64,\\"num_layers\\":3,\\"gamma\\":0.1,\\"lr\\":0.001,\\"weight_decay\\":0.3,\\"batch_size\\":1024,\\"epochs\\":500,\\"beta1\\":0.9,\\"beta2\\":0.999,\\"mfiles\\":[\\"textual_modality.pkl\\",\\"visual_modality.pkl\\"],\\"num_neighbors\\":[5,1],\\"eval_freq\\":5,\\"lambda_nlgcl\\":0.01,\\"nlgcl_tau\\":0.2,\\"nlgcl_G\\":1,\\"nlgcl_alpha\\":0.5,\\"eta\\":0.1,\\"k_cf\\":5,\\"c_min\\":2,\\"t_shrinkage\\":5.0,\\"v8_arm\\":\\"WMSG-core\\",\\"semantic_mode\\":\\"weighted\\",\\"edge_power\\":2.0,\\"edge_floor\\":0.05,\\"semantic_mix\\":1.0,\\"cl_chunk_size\\":1024,\\"knn_chunk_size\\":1024,\\"knn_device\\":\\"cpu\\",\\"cf_block_size\\":64,\\"cf_memory_budget_mib\\":128.0},\\"icaart_config\\":{\\"evidence_mode\\":\\"pooled_activity_excess\\",\\"evidence_activity_bins\\":\\"log2\\",\\"evidence_halves\\":0,\\"evidence_seed\\":null,\\"evidence_tau\\":2.0,\\"evidence_delta\\":1.0,\\"evidence_candidate_block\\":8192,\\"evidence_placebo_seed\\":20261016},\\"features\\":[{\\"file\\":\\"textual_modality.pkl\\",\\"sha256\\":\\"ceda3582c04f4e276e8d4bdcc2146f2634f6ca171e1f9dfafeb219dc5cfe96eb\\"},{\\"file\\":\\"visual_modality.pkl\\",\\"sha256\\":\\"763bf9510c74fd5c6a45ee8164ca9f1ababc5c355c1757fc3acbdcc5d46af0a5\\"}],\\"knn\\":[{\\"features\\":\\"ceda3582c04f4e276e8d4bdcc2146f2634f6ca171e1f9dfafeb219dc5cfe96eb\\",\\"k\\":5,\\"chunk\\":1024,\\"algorithm\\":\\"cosine_fp32_diagonal_minus10_torch_topk_v1\\",\\"torch\\":\\"2.11.0+cu128\\",\\"device\\":\\"cpu\\",\\"tf32\\":false,\\"source\\":\\"640f26b80b5b5eabf9bdcd44bac7a9c0544571168b31eefb7cbf96e59c166038\\",\\"fingerprint\\":\\"2cb2ad16db4996b65a179bbb221bbf09ac3eb390cf2cacf2cb3b26fd471efa40\\",\\"cache_hit\\":false,\\"candidate_sha256\\":\\"5bfeae7c9bb0a96be4ca20803e20d0d84a57068aeb322749bd132f0ce2691827\\"},{\\"features\\":\\"763bf9510c74fd5c6a45ee8164ca9f1ababc5c355c1757fc3acbdcc5d46af0a5\\",\\"k\\":1,\\"chunk\\":1024,\\"algorithm\\":\\"cosine_fp32_diagonal_minus10_torch_topk_v1\\",\\"torch\\":\\"2.11.0+cu128\\",\\"device\\":\\"cpu\\",\\"tf32\\":false,\\"source\\":\\"640f26b80b5b5eabf9bdcd44bac7a9c0544571168b31eefb7cbf96e59c166038\\",\\"fingerprint\\":\\"a50d5b1b5a586f38a4a5aec66d128556d43046d899d631a8756b476d47e04586\\",\\"cache_hit\\":false,\\"candidate_sha256\\":\\"ce97e02fbb82f94038c40fd2ae8e911c387c951b7710ee29eb2b144fc3cd7b08\\"}],\\"contract\\":{\\"config\\":{\\"embedding_dim\\":64,\\"num_layers\\":3,\\"gamma\\":0.1,\\"num_neighbors\\":[5,1],\\"mfiles\\":[\\"textual_modality.pkl\\",\\"visual_modality.pkl\\"],\\"lr\\":0.001,\\"weight_decay\\":0.3,\\"beta1\\":0.9,\\"beta2\\":0.999,\\"batch_size\\":1024,\\"seed\\":1,\\"eval_freq\\":5,\\"ranking\\":\\"full\\",\\"which4best\\":\\"NDCG@20\\",\\"num_workers\\":4,\\"nlgcl_G\\":1,\\"nlgcl_tau\\":0.2,\\"nlgcl_alpha\\":0.5,\\"eta\\":0.1,\\"k_cf\\":5,\\"c_min\\":2,\\"t_shrinkage\\":5.0,\\"v8_arm\\":\\"WMSG-core\\",\\"semantic_mode\\":\\"weighted\\",\\"edge_power\\":2.0,\\"edge_floor\\":0.05,\\"semantic_mix\\":1.0,\\"alignment\\":\\"off\\",\\"anchor_seed\\":2718,\\"anchor_per_stratum\\":1024,\\"relation_gate\\":\\"off\\",\\"relation_strength\\":0.0,\\"relation_threshold\\":0.7,\\"relation_temperature\\":0.1,\\"bsc_residual\\":0.0,\\"lambda_dirichlet\\":0.0,\\"placebo_seed\\":1,\\"knn_device\\":\\"cpu\\",\\"knn_chunk_size\\":1024,\\"cl_chunk_size\\":1024,\\"metadata_fingerprint\\":null},\\"splits\\":{\\"test.txt\\":\\"307162309a9fae52172dabbbb0f373e845af5e73740d987b70bb0f406d5ac411\\",\\"train.txt\\":\\"bcced50dea427734d76b031f1de140f219d67ba6bb6bc5eac85d7b3be51645bb\\",\\"valid.txt\\":\\"c44687959888d29e2d931aa3de1210f7f708e204d169b7680e5adfcf0a183a3a\\"},\\"candidate_hashes\\":[\\"5bfeae7c9bb0a96be4ca20803e20d0d84a57068aeb322749bd132f0ce2691827\\",\\"ce97e02fbb82f94038c40fd2ae8e911c387c951b7710ee29eb2b144fc3cd7b08\\"],\\"sources\\":{\\"stair5_v8.py\\":\\"dea612a122c82f7cc6c4c1ba069510fa1242fa3819f00b6cbfe0d37b9ed9c502\\",\\"stair5_v8_alignment.py\\":\\"8d6bb9eb25dfe2426d2a358ec82268d27c9b4913eef0472b6fda83fbd231607d\\",\\"stair5_v8_graph.py\\":\\"2a7c931740ceeac177ce3fc6dc2cdae479eb75673a8ef1b115122ba4a0826ae7\\",\\"stair5_v4_graph.py\\":\\"eb5d2b2355871711931d597bbe2d01f520b174852430e922be268826c7252241\\",\\"stair5_v4.py\\":\\"640f26b80b5b5eabf9bdcd44bac7a9c0544571168b31eefb7cbf96e59c166038\\",\\"stair5_v4_objectives.py\\":\\"1afd078a22aad8635778cfa61f05628beaecd180cd2c6c17ed56f72a6a94182e\\",\\"stair5_v4_utils.py\\":\\"39930cb54af3f5cdb65983f05b3b03cabe502673b9d70889b0ac7c1d22274b1a\\",\\"freerec_compat.py\\":\\"de46f3dd9bc11bb69acd5f151d175968e73d980be37ad36d5afc636b0b9c5e90\\",\\"stair5_v8_smoother.py\\":\\"c4f1ebab6008ffa5f7720007a17854040f0a7211fbd6087fc9dedd26d10d7663\\",\\"AdamW.py\\":\\"43108b6264cd5b1fa723d0a731faff0269d05fb91bcf525445c9bd0406443715\\",\\"main_stair5_v8.py\\":\\"036b45594a8c0b17e32c5401c5b84a93e85992fadae6303aa0db9eeaea8331b8\\",\\"stair_icaart.py\\":\\"77b862894f9aa91eba8fa96ad54a38eb6e29a25d9e134b7718a58f4bb5851014\\",\\"stair_icaart_evidence.py\\":\\"11487e0fef87077cde7863b58847bcf22adeebef6e5378ae32f71f366d88c41a\\",\\"stair_icaart_graph.py\\":\\"33bd582c8cd3b298eed1ce47ad99837ec9f0cd220778d684ae3d96161a455aad\\",\\"stair_icaart_config.py\\":\\"fe5bcaa711ba64fd40d431db27ae6278e7c937ce05f57786b6f7262668061a51\\",\\"main_stair_icaart.py\\":\\"b95eb5423ac8947da290bd13562e0aeb433adb8742ca0f0246139b4880aeec3f\\",\\"stair_icaart_reporting.py\\":\\"6902360e9ac92dd511ea87e988ad622cae51886eb0ac183b170ce7df1b5065cc\\"},\\"data_fingerprint\\":\\"de38b811bc05e4838c2f52d025ee3464447a1af4e1a1844747635cddafd13a9e\\",\\"graph_fingerprint\\":\\"b8013795d439064965dcd6946a44751d9a275ca0cefdc5231192066582d757a8\\"},\\"provenance\\":\\"logs/GD6/STAIR-PEEC/stair_icaart/baby_STAIR_ICAART_pooled_activity_excess_d1.0_seed1\\",\\"evidence_invariants\\":{\\"raw_semantic_fingerprint\\":\\"6c9010ff253ecd73e031c024db783e2ac3922be074560d5f1827ec30bcd32064\\",\\"assignment_hash\\":\\"9dd175d6e15c8d0bf6733797390191bea57263f06d6e99b57b3eb3555aba32a4\\",\\"counts_shape\\":[29926],\\"null_expectation_shape\\":[29926],\\"counts_hash\\":\\"a2b9aa0811aaf132948b4478ead0aadbada027b3c8d80c77d3e020a6c563d8eb\\",\\"null_expectation_hash\\":\\"1c56431d4ab4f2179e7ef1efeba35924f3f6b881ddd6b15864df8dc6468e7c35\\",\\"train_presence_hash\\":\\"5ac41072b49e5a21d02fc06e5c5f87f9a215561bd010e5c8f58eebdbb109344b\\",\\"num_raw_train_interactions\\":118551,\\"num_deduplicated_train_interactions\\":118551,\\"num_users\\":19445,\\"num_items\\":7050,\\"group_sizes\\":[0,5998,9065,3545,746,86,5],\\"baseline_v8_fingerprint\\":\\"eab753510c9d830425ba0f1f44f6d4964aa0143c574d4707fb2e7c9d93532041\\",\\"cf\\":{\\"raw_cf_fingerprint\\":\\"f93a45f8fd1e3db84017cd47c5a4293f26a7a48a537db8cc4a99c0fa79df44a7\\",\\"normalized_cf_fingerprint\\":\\"213d645c5f3b302a7ffc2539849238967edc409bc495f7ac87317cc179ba7314\\",\\"cf_cache_key\\":\\"8d678b20109a0b4f099ad7fee46585d5f59aec9fda227f55b5e4ece87c9fafa0\\"}}},\\"sports\\":{\\"commit\\":\\"5212263e87c5c1cfe6e48c91afa90201a48f5929\\",\\"python\\":\\"3.13.15\\",\\"torch\\":\\"2.11.0+cu128\\",\\"freerec\\":\\"0.9.7\\",\\"gpu\\":\\"Tesla T4\\",\\"v8_config\\":{\\"embedding_dim\\":64,\\"num_layers\\":3,\\"gamma\\":0.2,\\"lr\\":0.001,\\"weight_decay\\":0.1,\\"batch_size\\":1024,\\"epochs\\":500,\\"beta1\\":0.9,\\"beta2\\":0.999,\\"mfiles\\":[\\"textual_modality.pkl\\",\\"visual_modality.pkl\\"],\\"num_neighbors\\":[5,1],\\"eval_freq\\":5,\\"lambda_nlgcl\\":0.01,\\"nlgcl_tau\\":0.2,\\"nlgcl_G\\":1,\\"nlgcl_alpha\\":0.5,\\"eta\\":0.1,\\"k_cf\\":5,\\"c_min\\":2,\\"t_shrinkage\\":5.0,\\"v8_arm\\":\\"WMSG-core\\",\\"semantic_mode\\":\\"weighted\\",\\"edge_power\\":2.0,\\"edge_floor\\":0.05,\\"semantic_mix\\":1.0,\\"cl_chunk_size\\":1024,\\"knn_chunk_size\\":1024,\\"knn_device\\":\\"cpu\\",\\"cf_block_size\\":64,\\"cf_memory_budget_mib\\":128.0},\\"icaart_config\\":{\\"evidence_mode\\":\\"pooled_activity_excess\\",\\"evidence_activity_bins\\":\\"log2\\",\\"evidence_halves\\":0,\\"evidence_seed\\":null,\\"evidence_tau\\":2.0,\\"evidence_delta\\":1.0,\\"evidence_candidate_block\\":8192,\\"evidence_placebo_seed\\":20261016},\\"features\\":[{\\"file\\":\\"textual_modality.pkl\\",\\"sha256\\":\\"ca99972f3a2c7a9ead14e9a0d649351411722bbd98e920b4c5fe9f8ffa1564a2\\"},{\\"file\\":\\"visual_modality.pkl\\",\\"sha256\\":\\"5d2ea8187bbc52f806616404815930061acefe6cb22f353e3f02f286c95b9959\\"}],\\"knn\\":[{\\"features\\":\\"ca99972f3a2c7a9ead14e9a0d649351411722bbd98e920b4c5fe9f8ffa1564a2\\",\\"k\\":5,\\"chunk\\":609,\\"algorithm\\":\\"cosine_fp32_diagonal_minus10_torch_topk_v1\\",\\"torch\\":\\"2.11.0+cu128\\",\\"device\\":\\"cpu\\",\\"tf32\\":false,\\"source\\":\\"640f26b80b5b5eabf9bdcd44bac7a9c0544571168b31eefb7cbf96e59c166038\\",\\"fingerprint\\":\\"9dacd55f8d2be69d2bebff457c2c83345135e05f90eb44c1c4f0b8d62b4347e7\\",\\"cache_hit\\":false,\\"candidate_sha256\\":\\"bbe9e918209a03b5186a74cbd78ba474f420dfaab4eb6c2095c7c578ac0ca5af\\"},{\\"features\\":\\"5d2ea8187bbc52f806616404815930061acefe6cb22f353e3f02f286c95b9959\\",\\"k\\":1,\\"chunk\\":609,\\"algorithm\\":\\"cosine_fp32_diagonal_minus10_torch_topk_v1\\",\\"torch\\":\\"2.11.0+cu128\\",\\"device\\":\\"cpu\\",\\"tf32\\":false,\\"source\\":\\"640f26b80b5b5eabf9bdcd44bac7a9c0544571168b31eefb7cbf96e59c166038\\",\\"fingerprint\\":\\"9a875c92e35fbe67a4e43ac1298d898fed91f7160d449e4d9e8d5b86899889e5\\",\\"cache_hit\\":false,\\"candidate_sha256\\":\\"e3270692a18ae7d11d50a83d30b7f071f4a095e630714c01b2a0f6b4cff4e8a5\\"}],\\"contract\\":{\\"config\\":{\\"embedding_dim\\":64,\\"num_layers\\":3,\\"gamma\\":0.2,\\"num_neighbors\\":[5,1],\\"mfiles\\":[\\"textual_modality.pkl\\",\\"visual_modality.pkl\\"],\\"lr\\":0.001,\\"weight_decay\\":0.1,\\"beta1\\":0.9,\\"beta2\\":0.999,\\"batch_size\\":1024,\\"seed\\":1,\\"eval_freq\\":5,\\"ranking\\":\\"full\\",\\"which4best\\":\\"NDCG@20\\",\\"num_workers\\":4,\\"nlgcl_G\\":1,\\"nlgcl_tau\\":0.2,\\"nlgcl_alpha\\":0.5,\\"eta\\":0.1,\\"k_cf\\":5,\\"c_min\\":2,\\"t_shrinkage\\":5.0,\\"v8_arm\\":\\"WMSG-core\\",\\"semantic_mode\\":\\"weighted\\",\\"edge_power\\":2.0,\\"edge_floor\\":0.05,\\"semantic_mix\\":1.0,\\"alignment\\":\\"off\\",\\"anchor_seed\\":2718,\\"anchor_per_stratum\\":1024,\\"relation_gate\\":\\"off\\",\\"relation_strength\\":0.0,\\"relation_threshold\\":0.7,\\"relation_temperature\\":0.1,\\"bsc_residual\\":0.0,\\"lambda_dirichlet\\":0.0,\\"placebo_seed\\":1,\\"knn_device\\":\\"cpu\\",\\"knn_chunk_size\\":1024,\\"cl_chunk_size\\":1024,\\"metadata_fingerprint\\":null},\\"splits\\":{\\"test.txt\\":\\"218501613d897422d24d2035e268af027f4ca77688a3f836080b197621f5610a\\",\\"train.txt\\":\\"2491704e8543dbdeb9046cec0928c6734aa80521b1026afd53e0d296f4f7427c\\",\\"valid.txt\\":\\"6dabc7efcc14de50909c6b1995f7bf9862d6233a3440d593d389e1bf91fef56e\\"},\\"candidate_hashes\\":[\\"bbe9e918209a03b5186a74cbd78ba474f420dfaab4eb6c2095c7c578ac0ca5af\\",\\"e3270692a18ae7d11d50a83d30b7f071f4a095e630714c01b2a0f6b4cff4e8a5\\"],\\"sources\\":{\\"stair5_v8.py\\":\\"dea612a122c82f7cc6c4c1ba069510fa1242fa3819f00b6cbfe0d37b9ed9c502\\",\\"stair5_v8_alignment.py\\":\\"8d6bb9eb25dfe2426d2a358ec82268d27c9b4913eef0472b6fda83fbd231607d\\",\\"stair5_v8_graph.py\\":\\"2a7c931740ceeac177ce3fc6dc2cdae479eb75673a8ef1b115122ba4a0826ae7\\",\\"stair5_v4_graph.py\\":\\"eb5d2b2355871711931d597bbe2d01f520b174852430e922be268826c7252241\\",\\"stair5_v4.py\\":\\"640f26b80b5b5eabf9bdcd44bac7a9c0544571168b31eefb7cbf96e59c166038\\",\\"stair5_v4_objectives.py\\":\\"1afd078a22aad8635778cfa61f05628beaecd180cd2c6c17ed56f72a6a94182e\\",\\"stair5_v4_utils.py\\":\\"39930cb54af3f5cdb65983f05b3b03cabe502673b9d70889b0ac7c1d22274b1a\\",\\"freerec_compat.py\\":\\"de46f3dd9bc11bb69acd5f151d175968e73d980be37ad36d5afc636b0b9c5e90\\",\\"stair5_v8_smoother.py\\":\\"c4f1ebab6008ffa5f7720007a17854040f0a7211fbd6087fc9dedd26d10d7663\\",\\"AdamW.py\\":\\"43108b6264cd5b1fa723d0a731faff0269d05fb91bcf525445c9bd0406443715\\",\\"main_stair5_v8.py\\":\\"036b45594a8c0b17e32c5401c5b84a93e85992fadae6303aa0db9eeaea8331b8\\",\\"stair_icaart.py\\":\\"77b862894f9aa91eba8fa96ad54a38eb6e29a25d9e134b7718a58f4bb5851014\\",\\"stair_icaart_evidence.py\\":\\"11487e0fef87077cde7863b58847bcf22adeebef6e5378ae32f71f366d88c41a\\",\\"stair_icaart_graph.py\\":\\"33bd582c8cd3b298eed1ce47ad99837ec9f0cd220778d684ae3d96161a455aad\\",\\"stair_icaart_config.py\\":\\"fe5bcaa711ba64fd40d431db27ae6278e7c937ce05f57786b6f7262668061a51\\",\\"main_stair_icaart.py\\":\\"b95eb5423ac8947da290bd13562e0aeb433adb8742ca0f0246139b4880aeec3f\\",\\"stair_icaart_reporting.py\\":\\"6902360e9ac92dd511ea87e988ad622cae51886eb0ac183b170ce7df1b5065cc\\"},\\"data_fingerprint\\":\\"383929cee0a814ae8211bde6c15ef0abb6ecd4499822bdf5f3579cb87fa04643\\",\\"graph_fingerprint\\":\\"6b7be789b969cd17a17459deea25bfdf5bfcf518a008b02b7c2925a2d3f88f6c\\"},\\"provenance\\":\\"logs/GD6/STAIR-PEEC/stair_icaart/sports_STAIR_ICAART_pooled_activity_excess_d1.0_seed1\\",\\"evidence_invariants\\":{\\"raw_semantic_fingerprint\\":\\"7be2cf2f50ab128cb070eef6f853982741f6ff52e8229c45d227d352c114c2ce\\",\\"assignment_hash\\":\\"68fc8c356709aa20dca248289216a90d298c182672e9438b5b888b1bf27281bd\\",\\"counts_shape\\":[78872],\\"null_expectation_shape\\":[78872],\\"counts_hash\\":\\"fa866fab24c551f62ef527c0ea90b2c5f87886a3186ac7387f0a1000e5deaec2\\",\\"null_expectation_hash\\":\\"0cb8ef4d164d8c28ada6f5b897df36c44a60c8bd3fcb23b396ca367ca346a42a\\",\\"train_presence_hash\\":\\"3823a2e457063833e1ee27cef6622c24d20880c706a5f078ff8474f05dad20f7\\",\\"num_raw_train_interactions\\":218409,\\"num_deduplicated_train_interactions\\":218409,\\"num_users\\":35598,\\"num_items\\":18357,\\"group_sizes\\":[0,11416,16324,6216,1430,190,21,1],\\"baseline_v8_fingerprint\\":\\"cc0ff8deb6bbdac1c9295e27781dbb96de422731779794d79b4e1fd355ee75ea\\",\\"cf\\":{\\"raw_cf_fingerprint\\":\\"aa9eb264792a316ef7f6caf6015cf260ab2e0544846f3ce6f2e6d633c1c4c4dd\\",\\"normalized_cf_fingerprint\\":\\"b72c22e849dd477b641135cd2196e090472d56b8396ab37adff9b6137c7c949b\\",\\"cf_cache_key\\":\\"20b7e1854aac1bf2b3cafa9d06a1b7265da6d19c326e037a617d1c14ef919195\\"}}}}\')\\n","write_json(RUN_ROOT/\'provenance\'/\'protocol_reference.json\',SEED1_REFERENCE)\\n","write_json(RUN_ROOT/\'run_manifest.json\',{\'method\':\'ECLIPSE\',\'arm_id\':ARM,\'training_seed\':SEED,\'datasets\':list(SELECTED_DATASETS),\'mode\':RUN_MODE,\'status\':\'PREPARED\',\'session_id\':SESSION_ID,\'expected_algorithm_diff\':{\'evidence_mode\':[\'pooled_activity_excess\',\'pooled_global_excess\']}})\\n","\\n","GLOBAL_SOURCE_OVERLAY=json.loads(\'{\\"base_commit\\":\\"5212263e87c5c1cfe6e48c91afa90201a48f5929\\",\\"arm\\":\\"global_full_null\\",\\"files\\":{\\"stair_icaart_config.py\\":{\\"base_sha256\\":\\"fe5bcaa711ba64fd40d431db27ae6278e7c937ce05f57786b6f7262668061a51\\",\\"sha256\\":\\"b1dd36f1247753930c3785ae48e91ee91bfba3c3c85854500a840cbaf5288098\\",\\"content\\":\\"\\\\\\\\\\"\\\\\\\\\\"\\\\\\\\\\"Versioned, dependency-free contracts for PEEC and legacy REEC controls.\\\\\\\\n\\\\\\\\nPlatform: STAIR5-v8 / WMSG-core (64-dimensional primary).\\\\\\\\nCalibrates pre-BSC semantic graph weights using pooled-full evidence or historical\\\\\\\\nhalf-conditioned controls; all use binary TRAIN presence.\\\\\\\\n\\\\\\\\\\"\\\\\\\\\\"\\\\\\\\\\"\\\\\\\\n\\\\\\\\nfrom collections.abc import Mapping\\\\\\\\nimport math\\\\\\\\nfrom typing import Any, Dict, Tuple\\\\\\\\n\\\\\\\\n\\\\\\\\nICAART_OPTIONS: Tuple[Tuple[str, type, Any], ...] = (\\\\\\\\n    (\\\\\\\\\\"evidence-mode\\\\\\\\\\", str, \\\\\\\\\\"replicated_excess\\\\\\\\\\"),\\\\\\\\n    (\\\\\\\\\\"evidence-activity-bins\\\\\\\\\\", str, \\\\\\\\\\"log2\\\\\\\\\\"),\\\\\\\\n    (\\\\\\\\\\"evidence-halves\\\\\\\\\\", int, 2),\\\\\\\\n    (\\\\\\\\\\"evidence-seed\\\\\\\\\\", int, 20261016),\\\\\\\\n    (\\\\\\\\\\"evidence-tau\\\\\\\\\\", float, 2.0),\\\\\\\\n    (\\\\\\\\\\"evidence-delta\\\\\\\\\\", float, 1.0),\\\\\\\\n    (\\\\\\\\\\"evidence-candidate-block\\\\\\\\\\", int, 8192),\\\\\\\\n    (\\\\\\\\\\"evidence-placebo-seed\\\\\\\\\\", int, 20261016),\\\\\\\\n)\\\\\\\\n\\\\\\\\nICAART_NAMES: Tuple[str, ...] = tuple(flag.replace(\\\\\\\\\\"-\\\\\\\\\\", \\\\\\\\\\"_\\\\\\\\\\") for flag, _, _ in ICAART_OPTIONS)\\\\\\\\nDEFAULTS: Dict[str, Any] = {flag.replace(\\\\\\\\\\"-\\\\\\\\\\", \\\\\\\\\\"_\\\\\\\\\\"): value for flag, _, value in ICAART_OPTIONS}\\\\\\\\n\\\\\\\\nALLOWED_EVIDENCE_MODES: Tuple[str, ...] = (\\\\\\\\n    \\\\\\\\\\"replicated_excess\\\\\\\\\\",\\\\\\\\n    \\\\\\\\\\"v8_exact\\\\\\\\\\",\\\\\\\\n    \\\\\\\\\\"pooled_excess\\\\\\\\\\",\\\\\\\\n    \\\\\\\\\\"unstratified_replication\\\\\\\\\\",\\\\\\\\n    \\\\\\\\\\"replicated_count\\\\\\\\\\",\\\\\\\\n    \\\\\\\\\\"degree_stratified_shuffle\\\\\\\\\\",\\\\\\\\n    \\\\\\\\\\"uniform_multiplier\\\\\\\\\\",\\\\\\\\n    \\\\\\\\\\"pooled_activity_excess\\\\\\\\\\",\\\\\\\\n    \\\\\\\\\\"pooled_activity_count\\\\\\\\\\",\\\\\\\\n    \\\\\\\\\\"pooled_activity_shuffle\\\\\\\\\\",\\\\\\\\n    \\\\\\\\\\"pooled_global_excess\\\\\\\\\\",\\\\\\\\n)\\\\\\\\n\\\\\\\\nFULL_ACTIVITY_MODES = frozenset((\\\\\\\\n    \\\\\\\\\\"pooled_activity_excess\\\\\\\\\\", \\\\\\\\\\"pooled_activity_count\\\\\\\\\\", \\\\\\\\\\"pooled_activity_shuffle\\\\\\\\\\",\\\\\\\\n    \\\\\\\\\\"pooled_global_excess\\\\\\\\\\",  # Full-user pooling, without activity conditioning.\\\\\\\\n))\\\\\\\\nEVIDENCE_CONTRACT_VERSION = 2\\\\\\\\n\\\\\\\\n\\\\\\\\ndef evidence_identity(values: Mapping) -> Dict[str, Any]:\\\\\\\\n    \\\\\\\\\\"\\\\\\\\\\"\\\\\\\\\\"Describe the active estimator, without mislabeling pooled scores as replication.\\\\\\\\\\"\\\\\\\\\\"\\\\\\\\\\"\\\\\\\\n    mode = values[\\\\\\\\\\"evidence_mode\\\\\\\\\\"]\\\\\\\\n    full = mode in FULL_ACTIVITY_MODES\\\\\\\\n    return {\\\\\\\\n        \\\\\\\\\\"contract_version\\\\\\\\\\": EVIDENCE_CONTRACT_VERSION,\\\\\\\\n        \\\\\\\\\\"architecture\\\\\\\\\\": \\\\\\\\\\"STAIR-PEEC-versionICAART-R2\\\\\\\\\\" if full else \\\\\\\\\\"STAIR-REEC-versionICAART\\\\\\\\\\",\\\\\\\\n        \\\\\\\\\\"null_scope\\\\\\\\\\": \\\\\\\\\\"global_full\\\\\\\\\\" if mode == \\\\\\\\\\"pooled_global_excess\\\\\\\\\\" else \\\\\\\\\\"full_activity\\\\\\\\\\" if full else (\\\\\\\\n            \\\\\\\\\\"unstratified_halves\\\\\\\\\\" if mode == \\\\\\\\\\"unstratified_replication\\\\\\\\\\" else \\\\\\\\\\"half_activity\\\\\\\\\\"\\\\\\\\n        ),\\\\\\\\n        \\\\\\\\\\"active_partition\\\\\\\\\\": not full,\\\\\\\\n        \\\\\\\\\\"active_halves\\\\\\\\\\": 0 if full else 2,\\\\\\\\n        \\\\\\\\\\"score_kind\\\\\\\\\\": mode,\\\\\\\\n        \\\\\\\\\\"pooled_regularizer\\\\\\\\\\": 2.0 * values[\\\\\\\\\\"evidence_tau\\\\\\\\\\"] if full or mode == \\\\\\\\\\"pooled_excess\\\\\\\\\\" else None,\\\\\\\\n        \\\\\\\\\\"normalization\\\\\\\\\\": \\\\\\\\\\"actual_degree\\\\\\\\\\",\\\\\\\\n    }\\\\\\\\n\\\\\\\\n\\\\\\\\ndef _get(config: Any, name: str, default: Any = None) -> Any:\\\\\\\\n    \\\\\\\\\\"\\\\\\\\\\"\\\\\\\\\\"Retrieve attribute or dict key uniformly.\\\\\\\\\\"\\\\\\\\\\"\\\\\\\\\\"\\\\\\\\n    return config.get(name, default) if isinstance(config, Mapping) else getattr(config, name, default)\\\\\\\\n\\\\\\\\n\\\\\\\\ndef validate_icaart_config(config: Any) -> Dict[str, Any]:\\\\\\\\n    \\\\\\\\\\"\\\\\\\\\\"\\\\\\\\\\"Validate PEEC and legacy-control contracts before model execution.\\\\\\\\n\\\\\\\\n    Enforces strict mathematical bounds on evidence estimation and preserves\\\\\\\\n    the unchanged STAIR5-v8 WMSG-core backbone contracts.\\\\\\\\n    \\\\\\\\\\"\\\\\\\\\\"\\\\\\\\\\"\\\\\\\\n    values = {name: _get(config, name, default) for name, default in DEFAULTS.items()}\\\\\\\\n\\\\\\\\n    mode = values[\\\\\\\\\\"evidence_mode\\\\\\\\\\"]\\\\\\\\n    if mode not in ALLOWED_EVIDENCE_MODES:\\\\\\\\n        raise ValueError(f\\\\\\\\\\"evidence_mode must be one of {ALLOWED_EVIDENCE_MODES}; got {mode!r}\\\\\\\\\\")\\\\\\\\n\\\\\\\\n    expected_bins = \\\\\\\\\\"none\\\\\\\\\\" if mode == \\\\\\\\\\"pooled_global_excess\\\\\\\\\\" else \\\\\\\\\\"log2\\\\\\\\\\"\\\\\\\\n    if values[\\\\\\\\\\"evidence_activity_bins\\\\\\\\\\"] != expected_bins:\\\\\\\\n        raise ValueError(f\\\\\\\\\\"{mode} requires evidence_activity_bins={expected_bins!r}\\\\\\\\\\")\\\\\\\\n\\\\\\\\n    halves = values[\\\\\\\\\\"evidence_halves\\\\\\\\\\"]\\\\\\\\n    full = mode in FULL_ACTIVITY_MODES\\\\\\\\n    expected_halves = 0 if full else 2\\\\\\\\n    if halves is None:\\\\\\\\n        values[\\\\\\\\\\"evidence_halves\\\\\\\\\\"] = expected_halves\\\\\\\\n        halves = expected_halves\\\\\\\\n    if isinstance(halves, bool) or not isinstance(halves, int) or halves != expected_halves:\\\\\\\\n        raise ValueError(f\\\\\\\\\\"{mode} requires evidence_halves={expected_halves}\\\\\\\\\\")\\\\\\\\n\\\\\\\\n    for int_field in (\\\\\\\\\\"evidence_seed\\\\\\\\\\", \\\\\\\\\\"evidence_placebo_seed\\\\\\\\\\", \\\\\\\\\\"evidence_candidate_block\\\\\\\\\\"):\\\\\\\\n        if int_field == \\\\\\\\\\"evidence_seed\\\\\\\\\\" and full:\\\\\\\\n            # No hashing, partition or graph identity depends on this inactive seed.\\\\\\\\n            values[int_field] = None\\\\\\\\n            continue\\\\\\\\n        val = values[int_field]\\\\\\\\n        if isinstance(val, bool) or not isinstance(val, int) or val < 0:\\\\\\\\n            raise ValueError(f\\\\\\\\\\"{int_field} must be a non-negative integer\\\\\\\\\\")\\\\\\\\n    if values[\\\\\\\\\\"evidence_candidate_block\\\\\\\\\\"] < 1:\\\\\\\\n        raise ValueError(\\\\\\\\\\"evidence_candidate_block must be at least 1\\\\\\\\\\")\\\\\\\\n\\\\\\\\n    if isinstance(values[\\\\\\\\\\"evidence_tau\\\\\\\\\\"], bool) or isinstance(values[\\\\\\\\\\"evidence_delta\\\\\\\\\\"], bool):\\\\\\\\n        raise ValueError(\\\\\\\\\\"evidence_tau and evidence_delta must be numeric, not boolean\\\\\\\\\\")\\\\\\\\n    tau = float(values[\\\\\\\\\\"evidence_tau\\\\\\\\\\"])\\\\\\\\n    if not math.isfinite(tau) or not math.isfinite(2.0 * tau) or tau <= 0:\\\\\\\\n        raise ValueError(f\\\\\\\\\\"evidence_tau must be finite and > 0; got {tau}\\\\\\\\\\")\\\\\\\\n    values[\\\\\\\\\\"evidence_tau\\\\\\\\\\"] = tau\\\\\\\\n\\\\\\\\n    delta = float(values[\\\\\\\\\\"evidence_delta\\\\\\\\\\"])\\\\\\\\n    if not math.isfinite(delta) or not 0.0 <= delta <= 2.0:\\\\\\\\n        raise ValueError(f\\\\\\\\\\"evidence_delta must be finite and in [0.0, 2.0]; got {delta}\\\\\\\\\\")\\\\\\\\n    values[\\\\\\\\\\"evidence_delta\\\\\\\\\\"] = delta\\\\\\\\n\\\\\\\\n    # Backbone contract validations: STAIR5-v8 WMSG-core constraints\\\\\\\\n    required_backbone = {\\\\\\\\n        \\\\\\\\\\"embedding_dim\\\\\\\\\\": (64, 256),\\\\\\\\n        \\\\\\\\\\"num_layers\\\\\\\\\\": (3,),\\\\\\\\n        \\\\\\\\\\"nlgcl_G\\\\\\\\\\": (1,),\\\\\\\\n        \\\\\\\\\\"lambda_nlgcl\\\\\\\\\\": (0.01,),\\\\\\\\n        \\\\\\\\\\"nlgcl_tau\\\\\\\\\\": (0.2,),\\\\\\\\n        \\\\\\\\\\"nlgcl_alpha\\\\\\\\\\": (0.5,),\\\\\\\\n        \\\\\\\\\\"v8_arm\\\\\\\\\\": (\\\\\\\\\\"WMSG-core\\\\\\\\\\",),\\\\\\\\n        \\\\\\\\\\"alignment\\\\\\\\\\": (\\\\\\\\\\"off\\\\\\\\\\",),\\\\\\\\n        \\\\\\\\\\"relation_gate\\\\\\\\\\": (\\\\\\\\\\"off\\\\\\\\\\",),\\\\\\\\n        \\\\\\\\\\"relation_strength\\\\\\\\\\": (0.0,),\\\\\\\\n        \\\\\\\\\\"bsc_residual\\\\\\\\\\": (0.0,),\\\\\\\\n        \\\\\\\\\\"lambda_dirichlet\\\\\\\\\\": (0.0,),\\\\\\\\n        \\\\\\\\\\"semantic_mode\\\\\\\\\\": (\\\\\\\\\\"weighted\\\\\\\\\\",),\\\\\\\\n        \\\\\\\\\\"semantic_mix\\\\\\\\\\": (1.0,),\\\\\\\\n        \\\\\\\\\\"edge_power\\\\\\\\\\": (2.0,),\\\\\\\\n        \\\\\\\\\\"edge_floor\\\\\\\\\\": (0.05,),\\\\\\\\n        \\\\\\\\\\"knn_device\\\\\\\\\\": (\\\\\\\\\\"cpu\\\\\\\\\\",),\\\\\\\\n    }\\\\\\\\n\\\\\\\\n    for name, allowed in required_backbone.items():\\\\\\\\n        val = _get(config, name, allowed[0])\\\\\\\\n        if isinstance(val, bool) or val not in allowed:\\\\\\\\n            raise ValueError(f\\\\\\\\\\"STAIR-REEC requires {name} in {allowed}; got {val!r}\\\\\\\\\\")\\\\\\\\n\\\\\\\\n    neighbors = _get(config, \\\\\\\\\\"num_neighbors\\\\\\\\\\", [5, 1])\\\\\\\\n    if isinstance(neighbors, str):\\\\\\\\n        neighbors = list(map(int, neighbors.split(\\\\\\\\\\"-\\\\\\\\\\")))\\\\\\\\n    modalities = _get(config, \\\\\\\\\\"mfiles\\\\\\\\\\", [\\\\\\\\\\"textual_modality.pkl\\\\\\\\\\", \\\\\\\\\\"visual_modality.pkl\\\\\\\\\\"])\\\\\\\\n    if isinstance(modalities, str):\\\\\\\\n        modalities = modalities.split(\\\\\\\\\\",\\\\\\\\\\")\\\\\\\\n    if list(neighbors) != [5, 1] or len(modalities) != 2:\\\\\\\\n        raise ValueError(\\\\\\\\\\"STAIR-REEC requires two modalities [text, vision] with num_neighbors=[5, 1]\\\\\\\\\\")\\\\\\\\n    if \\\\\\\\\\"visual\\\\\\\\\\" in str(modalities[0]).lower() or \\\\\\\\\\"textual\\\\\\\\\\" in str(modalities[1]).lower():\\\\\\\\n        raise ValueError(\\\\\\\\\\"Modality order must be [text, vision], not [vision, text]\\\\\\\\\\")\\\\\\\\n\\\\\\\\n    return values\\\\\\\\n\\\\\\\\n\\\\\\\\ndef analytical_icaart_payload(n_items: int, n_edges: int, *, pooled: bool = False) -> Dict[str, Any]:\\\\\\\\n    \\\\\\\\\\"\\\\\\\\\\"\\\\\\\\\\"Analytical memory footprint of graph and temporary evidence structures.\\\\\\\\\\"\\\\\\\\\\"\\\\\\\\\\"\\\\\\\\n    if n_items <= 0 or n_edges < 0:\\\\\\\\n        raise ValueError(\\\\\\\\\\"Require positive item count and non-negative edge count\\\\\\\\\\")\\\\\\\\n    return {\\\\\\\\n        \\\\\\\\\\"semantic_directed_csr_bytes\\\\\\\\\\": 8 * (n_items + 1) + 24 * n_edges,\\\\\\\\n        \\\\\\\\\\"edge_count_convention\\\\\\\\\\": \\\\\\\\\\"n_edges counts undirected semantic pairs; final CF union is additional\\\\\\\\\\",\\\\\\\\n        \\\\\\\\\\"evidence_pairs_int64_bytes\\\\\\\\\\": 16 * n_edges,\\\\\\\\n        \\\\\\\\\\"evidence_counts_int64_bytes\\\\\\\\\\": (8 if pooled else 16) * n_edges,\\\\\\\\n        \\\\\\\\\\"evidence_mu_fp64_bytes\\\\\\\\\\": (8 if pooled else 16) * n_edges,\\\\\\\\n        \\\\\\\\\\"evidence_score_fp64_bytes\\\\\\\\\\": 8 * n_edges,\\\\\\\\n        \\\\\\\\\\"evidence_weight_fp64_bytes\\\\\\\\\\": 8 * n_edges,\\\\\\\\n        \\\\\\\\\\"additional_trainable_parameters\\\\\\\\\\": 0,\\\\\\\\n        \\\\\\\\\\"scope\\\\\\\\\\": \\\\\\\\\\"All evidence structures reside transiently on CPU and are freed before GPU training.\\\\\\\\\\",\\\\\\\\n    }\\\\\\\\n\\"},\\"models/stair_icaart_evidence.py\\":{\\"base_sha256\\":\\"11487e0fef87077cde7863b58847bcf22adeebef6e5378ae32f71f366d88c41a\\",\\"sha256\\":\\"8c0d2a22b32a7a4c827fa522e91e40c66997713327b6c57866155563c1ab15f2\\",\\"content\\":\\"\\\\\\\\\\"\\\\\\\\\\"\\\\\\\\\\"models/stair_icaart_evidence.py — Full-activity pooled evidence and legacy replicated controls.\\\\\\\\n====================================================================================================\\\\\\\\nImplements:\\\\\\\\n1. Deterministic activity stratification b(u) = floor(log2(h_u)) on deduplicated TRAIN data.\\\\\\\\n2. Disjoint user splitting f in {0, 1} with deterministic seeded hash (no Python randomized hash).\\\\\\\\n3. Candidate-only observed co-interactions c_ijf and activity-null expectations mu_ijf.\\\\\\\\n4. Full-activity pooled excess with kappa=2*tau, preserving all legacy half-based controls.\\\\\\\\n\\\\\\\\nContracts:\\\\\\\\n- Runs strictly on CPU in bounded blocks (default candidate_block = 8192).\\\\\\\\n- Never allocates dense N x N matrices or full R.T @ R.\\\\\\\\n- Zero gradients, zero learned parameters.\\\\\\\\n\\\\\\\\\\"\\\\\\\\\\"\\\\\\\\\\"\\\\\\\\n\\\\\\\\nimport hashlib\\\\\\\\nimport math\\\\\\\\nfrom typing import Dict, Optional, Tuple\\\\\\\\n\\\\\\\\nimport numpy as np\\\\\\\\nimport scipy.sparse as sp\\\\\\\\n\\\\\\\\n\\\\\\\\ndef deterministic_user_hash(evidence_seed: int, user_id: int) -> int:\\\\\\\\n    \\\\\\\\\\"\\\\\\\\\\"\\\\\\\\\\"Compute a deterministic 64-bit unsigned integer hash for (evidence_seed, user_id).\\\\\\\\n\\\\\\\\n    Uses SHA-256 to guarantee bitwise reproducibility across platforms and Python runs,\\\\\\\\n    completely independent of Python\\\\\'s randomized hash() seed or PyTorch global RNG.\\\\\\\\n    \\\\\\\\\\"\\\\\\\\\\"\\\\\\\\\\"\\\\\\\\n    token = f\\\\\\\\\\"{evidence_seed}:{user_id}\\\\\\\\\\".encode(\\\\\\\\\\"ascii\\\\\\\\\\")\\\\\\\\n    return int.from_bytes(hashlib.sha256(token).digest()[:8], \\\\\\\\\\"little\\\\\\\\\\")\\\\\\\\n\\\\\\\\n\\\\\\\\ndef build_activity_replication(\\\\\\\\n    R_csr: sp.csr_matrix,\\\\\\\\n    evidence_seed: int = 20261016,\\\\\\\\n) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:\\\\\\\\n    \\\\\\\\\\"\\\\\\\\\\"\\\\\\\\\\"Stratifies users by train activity and partitions active users into disjoint halves.\\\\\\\\n\\\\\\\\n    Mathematical formulation (Eq 3-4):\\\\\\\\n    - h_u = sum_i R_{ui} (train activity only)\\\\\\\\n    - Zero-history users (h_u == 0): excluded from strata, assigned user_bin = -1, half = -1.\\\\\\\\n    - Active users (h_u >= 1): b(u) = floor(log2(h_u)).\\\\\\\\n    - Within each non-empty stratum b: sort active users by (deterministic_hash, user_id),\\\\\\\\n      then alternate assignment f in {0, 1}.\\\\\\\\n    - Singleton strata (|U_b| == 1): deterministically assigned to half 0.\\\\\\\\n\\\\\\\\n    Args:\\\\\\\\n        R_csr: Deduplicated binary interaction matrix [M, N] in CSR format (CPU).\\\\\\\\n        evidence_seed: Integer seed for user hashing. Default 20261016.\\\\\\\\n\\\\\\\\n    Returns:\\\\\\\\n        user_bin: int64 array [M] of activity strata (-1 for inactive).\\\\\\\\n        half: int64 array [M] of user halves {0, 1} (-1 for inactive).\\\\\\\\n        group_sizes: int64 array [2, B] of user counts per (half, stratum).\\\\\\\\n    \\\\\\\\\\"\\\\\\\\\\"\\\\\\\\\\"\\\\\\\\n    if not sp.isspmatrix_csr(R_csr):\\\\\\\\n        raise TypeError(\\\\\\\\\\"R_csr must be a scipy.sparse.csr_matrix on CPU\\\\\\\\\\")\\\\\\\\n    if isinstance(evidence_seed, bool) or not isinstance(evidence_seed, (int, np.integer)) or evidence_seed < 0:\\\\\\\\n        raise ValueError(\\\\\\\\\\"evidence_seed must be a non-negative integer\\\\\\\\\\")\\\\\\\\n    R_csr = _canonical_presence(R_csr, \\\\\\\\\\"csr\\\\\\\\\\")\\\\\\\\n\\\\\\\\n    M, N = R_csr.shape\\\\\\\\n    if M <= 0 or N <= 0:\\\\\\\\n        raise ValueError(f\\\\\\\\\\"Invalid matrix shape: ({M}, {N})\\\\\\\\\\")\\\\\\\\n\\\\\\\\n    # h_u: user interaction count (train activity)\\\\\\\\n    h_u = np.diff(R_csr.indptr).astype(np.int64)\\\\\\\\n\\\\\\\\n    user_bin = np.full(M, -1, dtype=np.int64)\\\\\\\\n    half = np.full(M, -1, dtype=np.int64)\\\\\\\\n\\\\\\\\n    active_mask = h_u > 0\\\\\\\\n    active_users = np.flatnonzero(active_mask)\\\\\\\\n\\\\\\\\n    if len(active_users) == 0:\\\\\\\\n        return user_bin, half, np.zeros((2, 0), dtype=np.int64)\\\\\\\\n\\\\\\\\n    # b(u) = floor(log2(max(1, h_u)))\\\\\\\\n    active_bins = np.floor(np.log2(h_u[active_users])).astype(np.int64)\\\\\\\\n    user_bin[active_users] = active_bins\\\\\\\\n\\\\\\\\n    B = int(np.max(active_bins)) + 1\\\\\\\\n    group_sizes = np.zeros((2, B), dtype=np.int64)\\\\\\\\n\\\\\\\\n    for b in range(B):\\\\\\\\n        u_in_b = active_users[active_bins == b]\\\\\\\\n        num_u = len(u_in_b)\\\\\\\\n        if num_u == 0:\\\\\\\\n            continue\\\\\\\\n        elif num_u == 1:\\\\\\\\n            u_single = u_in_b[0]\\\\\\\\n            half[u_single] = 0\\\\\\\\n            group_sizes[0, b] += 1\\\\\\\\n        else:\\\\\\\\n            # Deterministic seeded hash with user_id tie-breaker\\\\\\\\n            hashes = np.fromiter(\\\\\\\\n                (deterministic_user_hash(evidence_seed, int(u)) for u in u_in_b),\\\\\\\\n                dtype=np.uint64, count=num_u,\\\\\\\\n            )\\\\\\\\n            order = np.lexsort((u_in_b, hashes))\\\\\\\\n            sorted_u = u_in_b[order]\\\\\\\\n            half[sorted_u] = np.arange(num_u, dtype=np.int64) % 2\\\\\\\\n            group_sizes[:, b] = ((num_u + 1) // 2, num_u // 2)\\\\\\\\n\\\\\\\\n    return user_bin, half, group_sizes\\\\\\\\n\\\\\\\\n\\\\\\\\ndef _canonical_presence(matrix: sp.spmatrix, format: str) -> sp.spmatrix:\\\\\\\\n    \\\\\\\\\\"\\\\\\\\\\"\\\\\\\\\\"Return private binary, sorted storage without changing caller training data.\\\\\\\\\\"\\\\\\\\\\"\\\\\\\\\\"\\\\\\\\n    if not np.isfinite(matrix.data).all() or np.any(matrix.data < 0):\\\\\\\\n        raise ValueError(\\\\\\\\\\"Interaction counts must be finite and non-negative\\\\\\\\\\")\\\\\\\\n    # Boolean duplicate reduction avoids int8 count overflow inside this conversion.\\\\\\\\n    result = matrix.astype(bool).asformat(format, copy=True)\\\\\\\\n    result.sum_duplicates()\\\\\\\\n    result.eliminate_zeros()\\\\\\\\n    result.sort_indices()\\\\\\\\n    return result\\\\\\\\n\\\\\\\\n\\\\\\\\ndef build_activity_strata(R_csr: sp.csr_matrix) -> Tuple[np.ndarray, np.ndarray]:\\\\\\\\n    \\\\\\\\\\"\\\\\\\\\\"\\\\\\\\\\"Full-user log2 activity strata; inactive users are excluded, with no RNG or split.\\\\\\\\\\"\\\\\\\\\\"\\\\\\\\\\"\\\\\\\\n    if not sp.isspmatrix_csr(R_csr):\\\\\\\\n        raise TypeError(\\\\\\\\\\"R_csr must be a scipy.sparse.csr_matrix\\\\\\\\\\")\\\\\\\\n    if min(R_csr.shape) <= 0:\\\\\\\\n        raise ValueError(\\\\\\\\\\"Interaction dimensions must be positive\\\\\\\\\\")\\\\\\\\n    matrix = _canonical_presence(R_csr, \\\\\\\\\\"csr\\\\\\\\\\")\\\\\\\\n    activity = np.diff(matrix.indptr).astype(np.int64)\\\\\\\\n    bins = np.full(matrix.shape[0], -1, dtype=np.int64)\\\\\\\\n    active = activity > 0\\\\\\\\n    bins[active] = np.floor(np.log2(activity[active])).astype(np.int64)\\\\\\\\n    sizes = np.bincount(bins[active]).astype(np.int64)\\\\\\\\n    return bins, sizes\\\\\\\\n\\\\\\\\n\\\\\\\\ndef candidate_pooled_statistics(\\\\\\\\n    R_csc: sp.csc_matrix,\\\\\\\\n    E_sem: np.ndarray,\\\\\\\\n    user_bin: np.ndarray,\\\\\\\\n    group_sizes: np.ndarray,\\\\\\\\n    candidate_block: int = 8192,\\\\\\\\n    *, null_scope: str = \\\\\\\\\\"full_activity\\\\\\\\\\",\\\\\\\\n) -> Tuple[np.ndarray, np.ndarray]:\\\\\\\\n    \\\\\\\\\\"\\\\\\\\\\"\\\\\\\\\\"Compute full-user c[E], mu[E] on semantic candidates, without half-conditioned nulls.\\\\\\\\n\\\\\\\\n    mu_ij = sum_b n_ib*n_jb/M_b, using binary TRAIN presence. With\\\\\\\\n    null_scope=\\\\\'global_full\\\\\', one pool contains all and only active TRAIN users,\\\\\\\\n    giving mu_ij=n_i*n_j/M_active. Observed overlap counting is identical.\\\\\\\\n    Integer counters\\\\\\\\n    are promoted before multiplication. Temporary expectation arrays are block-by-bin,\\\\\\\\n    never item-by-item. Posting searches execute in NumPy, not per-history pair loops.\\\\\\\\n    \\\\\\\\\\"\\\\\\\\\\"\\\\\\\\\\"\\\\\\\\n    if not sp.isspmatrix_csc(R_csc):\\\\\\\\n        raise TypeError(\\\\\\\\\\"R_csc must be a scipy.sparse.csc_matrix\\\\\\\\\\")\\\\\\\\n    if (isinstance(candidate_block, bool)\\\\\\\\n            or not isinstance(candidate_block, (int, np.integer)) or candidate_block < 1):\\\\\\\\n        raise ValueError(\\\\\\\\\\"candidate_block must be a positive integer\\\\\\\\\\")\\\\\\\\n    matrix = _canonical_presence(R_csc, \\\\\\\\\\"csc\\\\\\\\\\")\\\\\\\\n    n_users, n_items = matrix.shape\\\\\\\\n    if min(matrix.shape) <= 0:\\\\\\\\n        raise ValueError(\\\\\\\\\\"Interaction dimensions must be positive\\\\\\\\\\")\\\\\\\\n    pairs = np.asarray(E_sem)\\\\\\\\n    if (pairs.ndim != 2 or pairs.shape[1] != 2 or pairs.dtype.kind not in \\\\\\\\\\"iu\\\\\\\\\\"\\\\\\\\n            or np.any(pairs < 0) or np.any(pairs >= n_items)\\\\\\\\n            or np.any(pairs[:, 0] >= pairs[:, 1])):\\\\\\\\n        raise ValueError(\\\\\\\\\\"Candidate pairs must be integer [E,2], in catalog, with i < j\\\\\\\\\\")\\\\\\\\n    pairs = pairs.astype(np.int64, copy=False)\\\\\\\\n    if len(np.unique(pairs[:, 0] * n_items + pairs[:, 1])) != len(pairs):\\\\\\\\n        raise ValueError(\\\\\\\\\\"Candidate pairs must be unique\\\\\\\\\\")\\\\\\\\n    if null_scope not in (\\\\\\\\\\"full_activity\\\\\\\\\\", \\\\\\\\\\"global_full\\\\\\\\\\"):\\\\\\\\n        raise ValueError(\\\\\\\\\\"Unknown full-user null scope\\\\\\\\\\")\\\\\\\\n    bins, sizes = np.asarray(user_bin), np.asarray(group_sizes)\\\\\\\\n    activity = np.bincount(matrix.indices, minlength=n_users)\\\\\\\\n    active = activity > 0\\\\\\\\n    expected_bins = np.full(n_users, -1, dtype=np.int64)\\\\\\\\n    if null_scope == \\\\\\\\\\"global_full\\\\\\\\\\":\\\\\\\\n        expected_bins[active] = 0\\\\\\\\n        expected_sizes = np.array([np.count_nonzero(active)], dtype=np.int64)\\\\\\\\n    else:\\\\\\\\n        expected_bins[active] = np.floor(np.log2(activity[active])).astype(np.int64)\\\\\\\\n        expected_sizes = np.bincount(expected_bins[active])\\\\\\\\n    if (bins.shape != (n_users,) or bins.dtype.kind not in \\\\\\\\\\"iu\\\\\\\\\\"\\\\\\\\n            or sizes.ndim != 1 or sizes.dtype.kind not in \\\\\\\\\\"iu\\\\\\\\\\"\\\\\\\\n            or not np.array_equal(bins, expected_bins)\\\\\\\\n            or not np.array_equal(sizes, expected_sizes)):\\\\\\\\n        raise ValueError(\\\\\\\\\\"Null groups must match the selected full-user TRAIN scope exactly\\\\\\\\\\")\\\\\\\\n\\\\\\\\n    counts = np.zeros(len(pairs), dtype=np.int64)\\\\\\\\n    expected = np.zeros(len(pairs), dtype=np.float64)\\\\\\\\n    if not len(pairs) or not active.any():\\\\\\\\n        return counts, expected\\\\\\\\n    n_bins = len(sizes)\\\\\\\\n    item_ids = np.repeat(np.arange(n_items, dtype=np.int64), np.diff(matrix.indptr))\\\\\\\\n    flat = item_ids * n_bins + bins[matrix.indices]\\\\\\\\n    item_counts = np.bincount(flat, minlength=n_items * n_bins).reshape(n_items, n_bins)\\\\\\\\n    del flat, item_ids\\\\\\\\n    inverse = np.zeros(n_bins, dtype=np.float64)\\\\\\\\n    np.divide(1.0, sizes, out=inverse, where=sizes > 0)\\\\\\\\n    for start in range(0, len(pairs), candidate_block):\\\\\\\\n        stop = min(start + candidate_block, len(pairs))\\\\\\\\n        left_ids, right_ids = pairs[start:stop].T\\\\\\\\n        for offset, (i, j) in enumerate(zip(left_ids, right_ids)):\\\\\\\\n            left = matrix.indices[matrix.indptr[i]:matrix.indptr[i + 1]]\\\\\\\\n            right = matrix.indices[matrix.indptr[j]:matrix.indptr[j + 1]]\\\\\\\\n            if len(left) > len(right):\\\\\\\\n                left, right = right, left\\\\\\\\n            positions = np.searchsorted(right, left)\\\\\\\\n            valid = positions < len(right)\\\\\\\\n            selected = left[valid]\\\\\\\\n            counts[start + offset] = np.count_nonzero(right[positions[valid]] == selected)\\\\\\\\n        products = item_counts[left_ids].astype(np.float64)\\\\\\\\n        products *= item_counts[right_ids].astype(np.float64)\\\\\\\\n        expected[start:stop] = products @ inverse\\\\\\\\n    if np.any((counts > 0) & (expected <= 0)):\\\\\\\\n        raise ArithmeticError(\\\\\\\\\\"Positive overlap must imply positive null expectation\\\\\\\\\\")\\\\\\\\n    return counts, expected\\\\\\\\n\\\\\\\\n\\\\\\\\ndef degree_stratified_permutation(\\\\\\\\n    scores: np.ndarray, degrees: np.ndarray, pairs: np.ndarray, seed: int,\\\\\\\\n) -> np.ndarray:\\\\\\\\n    \\\\\\\\\\"\\\\\\\\\\"\\\\\\\\\\"Shuffle the supplied score family on unordered endpoint-degree quintile pairs.\\\\\\\\\\"\\\\\\\\\\"\\\\\\\\\\"\\\\\\\\n    scores, degrees, pairs = map(np.asarray, (scores, degrees, pairs))\\\\\\\\n    if (scores.ndim != 1 or not np.isfinite(scores).all()\\\\\\\\n            or degrees.ndim != 1 or not np.isfinite(degrees).all() or np.any(degrees < 0)\\\\\\\\n            or pairs.shape != (len(scores), 2) or pairs.dtype.kind not in \\\\\\\\\\"iu\\\\\\\\\\"\\\\\\\\n            or np.any(pairs < 0) or np.any(pairs >= len(degrees))\\\\\\\\n            or np.any(pairs[:, 0] >= pairs[:, 1])):\\\\\\\\n        raise ValueError(\\\\\\\\\\"Invalid scores, train degrees or unordered candidate pairs\\\\\\\\\\")\\\\\\\\n    if isinstance(seed, bool) or not isinstance(seed, (int, np.integer)) or seed < 0:\\\\\\\\n        raise ValueError(\\\\\\\\\\"Permutation seed must be a non-negative integer\\\\\\\\\\")\\\\\\\\n    if not len(scores):\\\\\\\\n        return scores.astype(np.float64, copy=True)\\\\\\\\n    cuts = np.unique(np.quantile(degrees, [.2, .4, .6, .8]))\\\\\\\\n    strata = np.searchsorted(cuts, degrees, side=\\\\\\\\\\"right\\\\\\\\\\")\\\\\\\\n    left, right = strata[pairs[:, 0]], strata[pairs[:, 1]]\\\\\\\\n    groups = np.minimum(left, right) * 5 + np.maximum(left, right)\\\\\\\\n    rng = np.random.default_rng(seed)\\\\\\\\n    result = scores.astype(np.float64, copy=True)\\\\\\\\n    for group in np.unique(groups):\\\\\\\\n        indices = np.flatnonzero(groups == group)\\\\\\\\n        result[indices] = rng.permutation(scores[indices])\\\\\\\\n    return result\\\\\\\\n\\\\\\\\n\\\\\\\\ndef pooled_activity_score(\\\\\\\\n    c: np.ndarray, mu: np.ndarray, tau: float = 2.0,\\\\\\\\n    mode: str = \\\\\\\\\\"pooled_activity_excess\\\\\\\\\\", *, degrees: Optional[np.ndarray] = None,\\\\\\\\n    E_sem: Optional[np.ndarray] = None, placebo_seed: int = 20261016,\\\\\\\\n) -> np.ndarray:\\\\\\\\n    \\\\\\\\\\"\\\\\\\\\\"\\\\\\\\\\"PEEC-full: [c-mu]+/(c+mu+2*tau), or its explicitly labeled count/placebo controls.\\\\\\\\n\\\\\\\\n    The vectors are full-stratum statistics, not sums of half-specific expectations.\\\\\\\\n    tau retains its legacy count-scale convention; pooled kappa is exactly 2*tau.\\\\\\\\n    \\\\\\\\\\"\\\\\\\\\\"\\\\\\\\\\"\\\\\\\\n    from stair_icaart_config import FULL_ACTIVITY_MODES\\\\\\\\n    if mode not in FULL_ACTIVITY_MODES:\\\\\\\\n        raise ValueError(f\\\\\\\\\\"Unknown full-activity evidence mode: {mode}\\\\\\\\\\")\\\\\\\\n    if isinstance(tau, bool) or not math.isfinite(tau) or tau <= 0 or not math.isfinite(2 * tau):\\\\\\\\n        raise ValueError(\\\\\\\\\\"tau and pooled kappa must be finite and positive\\\\\\\\\\")\\\\\\\\n    c, mu = np.asarray(c), np.asarray(mu)\\\\\\\\n    if (c.ndim != 1 or c.dtype.kind not in \\\\\\\\\\"iu\\\\\\\\\\" or mu.shape != c.shape\\\\\\\\n            or np.any(c < 0) or not np.isfinite(mu).all() or np.any(mu < 0)):\\\\\\\\n        raise ValueError(\\\\\\\\\\"Full counts must be integer [E]; expectations must be finite nonnegative [E]\\\\\\\\\\")\\\\\\\\n    if np.any((c > 0) & (mu <= 0)):\\\\\\\\n        raise ArithmeticError(\\\\\\\\\\"Positive overlap must imply positive null expectation\\\\\\\\\\")\\\\\\\\n    counts = c.astype(np.float64)\\\\\\\\n    expected = mu.astype(np.float64)\\\\\\\\n    if mode == \\\\\\\\\\"pooled_activity_count\\\\\\\\\\":\\\\\\\\n        scores = counts / (counts + 2.0 * tau)\\\\\\\\n    else:\\\\\\\\n        scores = np.maximum(0.0, counts - expected) / (counts + expected + 2.0 * tau)\\\\\\\\n    if not np.isfinite(scores).all() or np.any(scores < 0) or np.any(scores >= 1):\\\\\\\\n        raise ArithmeticError(\\\\\\\\\\"Pooled evidence score must lie in [0,1)\\\\\\\\\\")\\\\\\\\n    if mode == \\\\\\\\\\"pooled_activity_shuffle\\\\\\\\\\":\\\\\\\\n        if degrees is None or E_sem is None:\\\\\\\\n            raise ValueError(\\\\\\\\\\"Pooled placebo requires TRAIN item degrees and candidate pairs\\\\\\\\\\")\\\\\\\\n        scores = degree_stratified_permutation(scores, degrees, E_sem, placebo_seed)\\\\\\\\n    return scores\\\\\\\\n\\\\\\\\n\\\\\\\\ndef candidate_excess_statistics(\\\\\\\\n    R_csc: sp.csc_matrix,\\\\\\\\n    E_sem: np.ndarray,\\\\\\\\n    half: np.ndarray,\\\\\\\\n    user_bin: np.ndarray,\\\\\\\\n    group_sizes: np.ndarray,\\\\\\\\n    candidate_block: int = 8192,\\\\\\\\n    mode: str = \\\\\\\\\\"replicated_excess\\\\\\\\\\",\\\\\\\\n) -> Tuple[np.ndarray, np.ndarray]:\\\\\\\\n    \\\\\\\\\\"\\\\\\\\\\"\\\\\\\\\\"Computes observed co-interaction counts c_ijf and conditional null expectations mu_ijf.\\\\\\\\n\\\\\\\\n    Only evaluates unique undirected candidate pairs (i, j) in E_sem with i < j.\\\\\\\\n    Uses bounded block processing (default candidate_block = 8192) to strictly limit memory.\\\\\\\\n\\\\\\\\n    Mathematical formulation (Eq 4):\\\\\\\\n    - M_fb = |{u : f(u) = f, b(u) = b}|\\\\\\\\n    - n_ifb = sum_{u: f(u)=f, b(u)=b} R_{ui}\\\\\\\\n    - c_ijf = sum_{u: f(u)=f} R_{ui} R_{uj}\\\\\\\\n    - mu_ijf = sum_{b: M_fb > 0} (n_ifb * n_jfb) / M_fb\\\\\\\\n\\\\\\\\n    Args:\\\\\\\\n        R_csc: Binary interaction matrix [M, N] in CSC format (CPU).\\\\\\\\n        E_sem: int64 array [E, 2] of candidate item pairs (i, j) with i < j.\\\\\\\\n        half: int64 array [M] of user halves {0, 1} (-1 for inactive).\\\\\\\\n        user_bin: int64 array [M] of user strata (-1 for inactive).\\\\\\\\n        group_sizes: int64 array [2, B] of user group sizes.\\\\\\\\n        candidate_block: Maximum candidate edges processed per chunk.\\\\\\\\n        mode: Evidence mode (supports \\\\\'unstratified_replication\\\\\' ablation).\\\\\\\\n\\\\\\\\n    Returns:\\\\\\\\n        c: int64 array [E, 2] of observed co-occurrence counts.\\\\\\\\n        mu: float64 array [E, 2] of null expectation values.\\\\\\\\n    \\\\\\\\\\"\\\\\\\\\\"\\\\\\\\\\"\\\\\\\\n    if not sp.isspmatrix_csc(R_csc):\\\\\\\\n        raise TypeError(\\\\\\\\\\"R_csc must be a scipy.sparse.csc_matrix on CPU\\\\\\\\\\")\\\\\\\\n    if (isinstance(candidate_block, bool)\\\\\\\\n            or not isinstance(candidate_block, (int, np.integer)) or candidate_block < 1):\\\\\\\\n        raise ValueError(\\\\\\\\\\"candidate_block must be a positive integer\\\\\\\\\\")\\\\\\\\n    if mode not in (\\\\\\\\\\"replicated_excess\\\\\\\\\\", \\\\\\\\\\"v8_exact\\\\\\\\\\", \\\\\\\\\\"pooled_excess\\\\\\\\\\",\\\\\\\\n                    \\\\\\\\\\"unstratified_replication\\\\\\\\\\", \\\\\\\\\\"replicated_count\\\\\\\\\\",\\\\\\\\n                    \\\\\\\\\\"degree_stratified_shuffle\\\\\\\\\\", \\\\\\\\\\"uniform_multiplier\\\\\\\\\\"):\\\\\\\\n        raise ValueError(f\\\\\\\\\\"Unknown evidence mode: {mode}\\\\\\\\\\")\\\\\\\\n    R = _canonical_presence(R_csc, \\\\\\\\\\"csc\\\\\\\\\\")\\\\\\\\n    M, N = R.shape\\\\\\\\n    E_sem = np.asarray(E_sem)\\\\\\\\n    if E_sem.ndim != 2 or E_sem.shape[1] != 2 or E_sem.dtype.kind not in \\\\\\\\\\"iu\\\\\\\\\\":\\\\\\\\n        raise ValueError(\\\\\\\\\\"E_sem must be an integer array of shape [E, 2]\\\\\\\\\\")\\\\\\\\n    if np.any(E_sem < 0) or np.any(E_sem >= N) or np.any(E_sem[:, 0] >= E_sem[:, 1]):\\\\\\\\n        raise ValueError(\\\\\\\\\\"Candidate pairs must be in catalog and strictly satisfy i < j\\\\\\\\\\")\\\\\\\\n    if N and len(np.unique(E_sem[:, 0].astype(np.int64) * N + E_sem[:, 1])) != len(E_sem):\\\\\\\\n        raise ValueError(\\\\\\\\\\"Candidate pairs must be unique\\\\\\\\\\")\\\\\\\\n    half, user_bin, group_sizes = map(np.asarray, (half, user_bin, group_sizes))\\\\\\\\n    if (half.shape != (M,) or user_bin.shape != (M,)\\\\\\\\n            or half.dtype.kind not in \\\\\\\\\\"iu\\\\\\\\\\" or user_bin.dtype.kind not in \\\\\\\\\\"iu\\\\\\\\\\"\\\\\\\\n            or group_sizes.ndim != 2 or group_sizes.shape[0] != 2\\\\\\\\n            or group_sizes.dtype.kind not in \\\\\\\\\\"iu\\\\\\\\\\" or np.any(group_sizes < 0)):\\\\\\\\n        raise ValueError(\\\\\\\\\\"Invalid user assignments or group_sizes shape/dtype\\\\\\\\\\")\\\\\\\\n    B = group_sizes.shape[1]\\\\\\\\n    active = np.bincount(R.indices, minlength=M) > 0\\\\\\\\n    if (np.any(~np.isin(half[active], (0, 1))) or np.any(user_bin[active] < 0)\\\\\\\\n            or np.any(user_bin[active] >= B)\\\\\\\\n            or np.any(half[~active] != -1) or np.any(user_bin[~active] != -1)):\\\\\\\\n        raise ValueError(\\\\\\\\\\"Active users require valid groups; inactive users require group ID -1\\\\\\\\\\")\\\\\\\\n    actual_sizes = np.bincount(\\\\\\\\n        half[active] * B + user_bin[active], minlength=2 * B,\\\\\\\\n    ).reshape(2, B)\\\\\\\\n    if not np.array_equal(actual_sizes, group_sizes):\\\\\\\\n        raise ArithmeticError(\\\\\\\\\\"group_sizes do not match the disjoint active-user assignments\\\\\\\\\\")\\\\\\\\n\\\\\\\\n    E = len(E_sem)\\\\\\\\n    c = np.zeros((E, 2), dtype=np.int64)\\\\\\\\n    mu = np.zeros((E, 2), dtype=np.float64)\\\\\\\\n    if E == 0:\\\\\\\\n        return c, mu\\\\\\\\n    if not active.any():\\\\\\\\n        return c, mu\\\\\\\\n\\\\\\\\n    # One streaming incidence pass; no split R0/R1 matrices and no M x block product.\\\\\\\\n    item_ids = np.repeat(np.arange(N, dtype=np.int64), np.diff(R.indptr))\\\\\\\\n    users = R.indices\\\\\\\\n    flat_groups = item_ids * (2 * B) + half[users] * B + user_bin[users]\\\\\\\\n    n_ifb = np.bincount(flat_groups, minlength=N * 2 * B).reshape(N, 2, B)\\\\\\\\n    del item_ids, flat_groups\\\\\\\\n    inv_sizes = np.zeros((2, B), dtype=np.float64)\\\\\\\\n    np.divide(1.0, group_sizes, out=inv_sizes, where=group_sizes > 0)\\\\\\\\n    n_if = n_ifb.sum(axis=2).astype(np.float64)\\\\\\\\n    inv_half = np.zeros(2, dtype=np.float64)\\\\\\\\n    half_sizes = group_sizes.sum(axis=1)\\\\\\\\n    np.divide(1.0, half_sizes, out=inv_half, where=half_sizes > 0)\\\\\\\\n\\\\\\\\n    # NumPy searchsorted executes the posting-list search in compiled code. Python\\\\\\\\n    # dispatch is per eligible edge, never per user-history pair. Peak intersection\\\\\\\\n    # scratch is O(min(n_i,n_j)), independent of M*candidate_block or N*N.\\\\\\\\n    for start in range(0, E, candidate_block):\\\\\\\\n        stop = min(start + candidate_block, E)\\\\\\\\n        src, dst = E_sem[start:stop].T\\\\\\\\n        for offset, (i, j) in enumerate(zip(src, dst)):\\\\\\\\n            left = R.indices[R.indptr[i]:R.indptr[i + 1]]\\\\\\\\n            right = R.indices[R.indptr[j]:R.indptr[j + 1]]\\\\\\\\n            if len(left) > len(right):\\\\\\\\n                left, right = right, left\\\\\\\\n            if not len(left):\\\\\\\\n                continue\\\\\\\\n            positions = np.searchsorted(right, left)\\\\\\\\n            valid = positions < len(right)\\\\\\\\n            selected = left[valid]\\\\\\\\n            common = selected[right[positions[valid]] == selected]\\\\\\\\n            c[start + offset] = np.bincount(half[common], minlength=2)\\\\\\\\n        if mode == \\\\\\\\\\"unstratified_replication\\\\\\\\\\":\\\\\\\\n            mu[start:stop] = n_if[src] * n_if[dst] * inv_half\\\\\\\\n        else:\\\\\\\\n            # Cast each operand before multiplication; do not overflow int counters.\\\\\\\\n            left_counts = n_ifb[src].astype(np.float64)\\\\\\\\n            right_counts = n_ifb[dst].astype(np.float64)\\\\\\\\n            mu[start:stop] = np.sum(left_counts * right_counts * inv_sizes, axis=2)\\\\\\\\n        if np.any((c[start:stop] > 0) & (mu[start:stop] <= 0)):\\\\\\\\n            raise ArithmeticError(\\\\\\\\\\"Positive overlap must imply positive null expectation\\\\\\\\\\")\\\\\\\\n    return c, mu\\\\\\\\n\\\\\\\\n\\\\\\\\ndef replicated_excess_score(\\\\\\\\n    c: np.ndarray,\\\\\\\\n    mu: np.ndarray,\\\\\\\\n    tau: float = 2.0,\\\\\\\\n    mode: str = \\\\\\\\\\"replicated_excess\\\\\\\\\\",\\\\\\\\n    placebo_seed: int = 20261016,\\\\\\\\n    degrees: Optional[np.ndarray] = None,\\\\\\\\n    E_sem: Optional[np.ndarray] = None,\\\\\\\\n) -> np.ndarray:\\\\\\\\n    \\\\\\\\\\"\\\\\\\\\\"\\\\\\\\\\"Computes bounded replicated excess score r in [0, 1) (Eq 5) and scientific controls.\\\\\\\\n\\\\\\\\n    Primary formulation:\\\\\\\\n    - e_ijf = [c_ijf - mu_ijf]_+ / (c_ijf + mu_ijf + tau)\\\\\\\\n    - r_ij = min(e_ij0, e_ij1)\\\\\\\\n\\\\\\\\n    Guarantees:\\\\\\\\n    - 0 <= r_ij < 1.0\\\\\\\\n    - Primary replicated_excess is zero if either half has zero count.\\\\\\\\n    - Pooled/uniform controls intentionally have different neutrality semantics.\\\\\\\\n    - No division by zero, no NaN/Inf.\\\\\\\\n\\\\\\\\n    Ablation modes:\\\\\\\\n    - \\\\\'replicated_excess\\\\\': Primary replicated excess score (Eq 5).\\\\\\\\n    - \\\\\'v8_exact\\\\\': Exactly r = 0 (recovers native V8).\\\\\\\\n    - \\\\\'pooled_excess\\\\\': r = [c_pool - mu_pool]_+ / (c_pool + mu_pool + 2*tau).\\\\\\\\n    - \\\\\'unstratified_replication\\\\\': Unstratified null expectation replication.\\\\\\\\n    - \\\\\'replicated_count\\\\\': r = min(c0 / (c0 + tau), c1 / (c1 + tau)).\\\\\\\\n    - \\\\\'degree_stratified_shuffle\\\\\': Permute r among pairs stratified by endpoint degree.\\\\\\\\n    - \\\\\'uniform_multiplier\\\\\': Constant r across all candidate edges.\\\\\\\\n    \\\\\\\\\\"\\\\\\\\\\"\\\\\\\\\\"\\\\\\\\n    from stair_icaart_config import ALLOWED_EVIDENCE_MODES, FULL_ACTIVITY_MODES\\\\\\\\n    if mode not in ALLOWED_EVIDENCE_MODES:\\\\\\\\n        raise ValueError(f\\\\\\\\\\"Unknown evidence mode: {mode}\\\\\\\\\\")\\\\\\\\n    if isinstance(tau, bool) or tau <= 0 or not math.isfinite(tau):\\\\\\\\n        raise ValueError(f\\\\\\\\\\"tau must be finite and positive; got {tau}\\\\\\\\\\")\\\\\\\\n    if mode in FULL_ACTIVITY_MODES:\\\\\\\\n        raise ValueError(\\\\\\\\\\"Full-activity modes require pooled_activity_score, not half statistics\\\\\\\\\\")\\\\\\\\n    c, mu = np.asarray(c), np.asarray(mu)\\\\\\\\n    if c.ndim != 2 or c.shape[1] != 2 or c.dtype.kind not in \\\\\\\\\\"iu\\\\\\\\\\" or mu.shape != c.shape:\\\\\\\\n        raise ValueError(\\\\\\\\\\"c must be integer [E, 2] and mu must have the same shape\\\\\\\\\\")\\\\\\\\n    if np.any(c < 0) or not np.isfinite(mu).all() or np.any(mu < 0):\\\\\\\\n        raise ValueError(\\\\\\\\\\"Counts and expectations must be finite and non-negative\\\\\\\\\\")\\\\\\\\n    if np.any((c > 0) & (mu <= 0)):\\\\\\\\n        raise ArithmeticError(\\\\\\\\\\"Positive overlap must imply positive null expectation\\\\\\\\\\")\\\\\\\\n    E = c.shape[0]\\\\\\\\n    if E == 0:\\\\\\\\n        return np.empty(0, dtype=np.float64)\\\\\\\\n\\\\\\\\n    c0 = c[:, 0].astype(np.float64)\\\\\\\\n    c1 = c[:, 1].astype(np.float64)\\\\\\\\n    mu0 = mu[:, 0].astype(np.float64)\\\\\\\\n    mu1 = mu[:, 1].astype(np.float64)\\\\\\\\n\\\\\\\\n    if mode == \\\\\\\\\\"v8_exact\\\\\\\\\\":\\\\\\\\n        return np.zeros(E, dtype=np.float64)\\\\\\\\n\\\\\\\\n    elif mode in (\\\\\\\\\\"replicated_excess\\\\\\\\\\", \\\\\\\\\\"unstratified_replication\\\\\\\\\\"):\\\\\\\\n        excess0 = np.maximum(0.0, c0 - mu0)\\\\\\\\n        excess1 = np.maximum(0.0, c1 - mu1)\\\\\\\\n        denom0 = c0 + mu0 + tau\\\\\\\\n        denom1 = c1 + mu1 + tau\\\\\\\\n        e0 = excess0 / denom0\\\\\\\\n        e1 = excess1 / denom1\\\\\\\\n        r = np.minimum(e0, e1)\\\\\\\\n\\\\\\\\n    elif mode == \\\\\\\\\\"pooled_excess\\\\\\\\\\":\\\\\\\\n        c_pool = c0 + c1\\\\\\\\n        mu_pool = mu0 + mu1\\\\\\\\n        excess_pool = np.maximum(0.0, c_pool - mu_pool)\\\\\\\\n        denom_pool = c_pool + mu_pool + 2.0 * tau\\\\\\\\n        r = excess_pool / denom_pool\\\\\\\\n\\\\\\\\n    elif mode == \\\\\\\\\\"replicated_count\\\\\\\\\\":\\\\\\\\n        r0 = c0 / (c0 + tau)\\\\\\\\n        r1 = c1 / (c1 + tau)\\\\\\\\n        r = np.minimum(r0, r1)\\\\\\\\n\\\\\\\\n    elif mode == \\\\\\\\\\"degree_stratified_shuffle\\\\\\\\\\":\\\\\\\\n        # First compute standard replicated excess\\\\\\\\n        excess0 = np.maximum(0.0, c0 - mu0)\\\\\\\\n        excess1 = np.maximum(0.0, c1 - mu1)\\\\\\\\n        e0 = excess0 / (c0 + mu0 + tau)\\\\\\\\n        e1 = excess1 / (c1 + mu1 + tau)\\\\\\\\n        r_base = np.minimum(e0, e1)\\\\\\\\n\\\\\\\\n        if degrees is None or E_sem is None:\\\\\\\\n            raise ValueError(\\\\\\\\\\"Degree-stratified shuffle requires train item degrees and candidate pairs\\\\\\\\\\")\\\\\\\\n        degrees, E_sem = np.asarray(degrees), np.asarray(E_sem)\\\\\\\\n        if (degrees.ndim != 1 or not np.isfinite(degrees).all() or np.any(degrees < 0)\\\\\\\\n                or E_sem.shape != (E, 2) or E_sem.dtype.kind not in \\\\\\\\\\"iu\\\\\\\\\\"\\\\\\\\n                or np.any(E_sem < 0) or np.any(E_sem >= len(degrees))):\\\\\\\\n            raise ValueError(\\\\\\\\\\"Invalid train item degrees or candidate pairs for shuffle\\\\\\\\\\")\\\\\\\\n        cuts = np.unique(np.quantile(degrees, [0.2, 0.4, 0.6, 0.8]))\\\\\\\\n        strata = np.searchsorted(cuts, degrees, side=\\\\\\\\\\"right\\\\\\\\\\")\\\\\\\\n        left, right = strata[E_sem[:, 0]], strata[E_sem[:, 1]]\\\\\\\\n        groups = np.minimum(left, right) * 5 + np.maximum(left, right)\\\\\\\\n        rng = np.random.default_rng(placebo_seed)\\\\\\\\n        r = r_base.copy()\\\\\\\\n        for group in np.unique(groups):\\\\\\\\n            positions = np.flatnonzero(groups == group)\\\\\\\\n            if len(positions) > 1:\\\\\\\\n                r[positions] = rng.permutation(r_base[positions])\\\\\\\\n\\\\\\\\n    elif mode == \\\\\\\\\\"uniform_multiplier\\\\\\\\\\":\\\\\\\\n        excess0 = np.maximum(0.0, c0 - mu0)\\\\\\\\n        excess1 = np.maximum(0.0, c1 - mu1)\\\\\\\\n        e0 = excess0 / (c0 + mu0 + tau)\\\\\\\\n        e1 = excess1 / (c1 + mu1 + tau)\\\\\\\\n        r_base = np.minimum(e0, e1)\\\\\\\\n        # Set to mean of positive values, or 0.5\\\\\\\\n        pos = r_base[r_base > 0]\\\\\\\\n        val = float(np.mean(pos)) if len(pos) > 0 else 0.5\\\\\\\\n        r = np.full(E, val, dtype=np.float64)\\\\\\\\n\\\\\\\\n    else:\\\\\\\\n        raise ValueError(f\\\\\\\\\\"Unknown evidence mode: {mode}\\\\\\\\\\")\\\\\\\\n\\\\\\\\n    # Boundedness assertion: 0 <= r < 1\\\\\\\\n    if np.any(r < 0.0) or np.any(r >= 1.0) or not np.isfinite(r).all():\\\\\\\\n        raise ArithmeticError(f\\\\\\\\\\"Evidence score r outside [0, 1): min={r.min()}, max={r.max()}\\\\\\\\\\")\\\\\\\\n\\\\\\\\n    return r\\\\\\\\n\\\\\\\\n\\\\\\\\n# Public alias for backward compatibility and verification scripts\\\\\\\\ncompute_replicated_excess_evidence = replicated_excess_score\\\\\\\\n\\\\\\\\n\\"},\\"models/stair_icaart_graph.py\\":{\\"base_sha256\\":\\"33bd582c8cd3b298eed1ce47ad99837ec9f0cd220778d684ae3d96161a455aad\\",\\"sha256\\":\\"aebe7c2a2e7f70c5ea9d7abef58e64f62537d0ad0a226f8f228f33265976a229\\",\\"content\\":\\"\\\\\\\\\\"\\\\\\\\\\"\\\\\\\\\\"models/stair_icaart_graph.py — Pooled / Replicated Excess-Evidence Calibration (PEEC / REEC).\\\\\\\\n========================================================================================\\\\\\\\nImplements:\\\\\\\\n1. Recovery of exact native STAIR5-v8 raw semantic graph W^8 and degree normalization N(W).\\\\\\\\n2. Exponential calibration W^delta = W^8 * exp(delta * r) on undirected candidate support E_sem.\\\\\\\\n3. True actual-degree re-normalization S_delta = N(W^delta) and unchanged CF blend S_ICA.\\\\\\\\n4. Exact V8 bypass (at delta = 0 or when all r = 0).\\\\\\\\n5. Offline mechanism diagnostics: coverage, disagreement, endpoint contrast (Eq 10),\\\\\\\\n   operator dose, polynomial direction dose (Eq 11), and degree side-effect ratios.\\\\\\\\n6. Checkpoint manifest generation and graph fingerprinting.\\\\\\\\n\\\\\\\\\\"\\\\\\\\\\"\\\\\\\\\\"\\\\\\\\n\\\\\\\\nfrom dataclasses import dataclass\\\\\\\\nimport hashlib\\\\\\\\nimport json\\\\\\\\nimport math\\\\\\\\nimport time\\\\\\\\nfrom pathlib import Path\\\\\\\\nfrom typing import Any, Dict, List, Optional, Tuple\\\\\\\\n\\\\\\\\nimport numpy as np\\\\\\\\nimport scipy.sparse as sp\\\\\\\\nimport torch\\\\\\\\n\\\\\\\\nfrom models.stair5_v8_graph import (\\\\\\\\n    build_wmsg_semantic_graph,\\\\\\\\n)\\\\\\\\nfrom models.stair5_v4_graph import (\\\\\\\\n    normalize_cf_graph_with_fallback,\\\\\\\\n    binary_train_keys, CACHE_VERSION, _digest_array, build_candidate_support_graph,\\\\\\\\n    sparse_fingerprint,\\\\\\\\n    tensor_to_scipy,\\\\\\\\n    scipy_to_tensor,\\\\\\\\n)\\\\\\\\nfrom models.stair5_v4_utils import atomic_torch_save, file_sha256\\\\\\\\nfrom stair_icaart_config import (\\\\\\\\n    FULL_ACTIVITY_MODES, analytical_icaart_payload, evidence_identity, validate_icaart_config,\\\\\\\\n)\\\\\\\\nfrom models.stair_icaart_evidence import (\\\\\\\\n    build_activity_replication,\\\\\\\\n    candidate_excess_statistics,\\\\\\\\n    replicated_excess_score,\\\\\\\\n    build_activity_strata,\\\\\\\\n    candidate_pooled_statistics,\\\\\\\\n    pooled_activity_score,\\\\\\\\n)\\\\\\\\n\\\\\\\\n\\\\\\\\n@dataclass\\\\\\\\nclass GraphStateICAART:\\\\\\\\n    \\\\\\\\\\"\\\\\\\\\\"\\\\\\\\\\"Encapsulates the final calibrated S_ICA operator, audit manifest, and diagnostics.\\\\\\\\\\"\\\\\\\\\\"\\\\\\\\\\"\\\\\\\\n    operator: torch.Tensor\\\\\\\\n    metadata: Dict[str, Any]\\\\\\\\n    manifest: Dict[str, Any]\\\\\\\\n\\\\\\\\n\\\\\\\\ndef normalize_degree_isolated(W: sp.csr_matrix, dtype: Any = np.float32) -> sp.csr_matrix:\\\\\\\\n    \\\\\\\\\\"\\\\\\\\\\"\\\\\\\\\\"Computes exact symmetric degree normalization N(W) with identity isolated blocks.\\\\\\\\n\\\\\\\\n    Mathematical definition (Eq 2):\\\\\\\\n        N(W) = D^{-1/2} W D^{-1/2} + diag(1[d_i == 0]), where d = W 1.\\\\\\\\n\\\\\\\\n    Contracts:\\\\\\\\n    - Computed in FP64 for numerical precision.\\\\\\\\n    - Preserves exact symmetry and zero diagonal on connected nodes.\\\\\\\\n    - Returns CSR matrix with sorted indices and eliminated zeros in requested dtype (default float32).\\\\\\\\n    \\\\\\\\\\"\\\\\\\\\\"\\\\\\\\\\"\\\\\\\\n    if W.shape[0] != W.shape[1]:\\\\\\\\n        raise ValueError(\\\\\\\\\\"Graph matrix must be square\\\\\\\\\\")\\\\\\\\n\\\\\\\\n    W_64 = W.astype(np.float64).tocsr(copy=True)\\\\\\\\n    W_64.sum_duplicates()\\\\\\\\n    W_64.eliminate_zeros()\\\\\\\\n\\\\\\\\n    if not np.isfinite(W_64.data).all() or np.any(W_64.data < 0.0):\\\\\\\\n        raise ValueError(\\\\\\\\\\"Matrix entries must be finite and non-negative\\\\\\\\\\")\\\\\\\\n\\\\\\\\n    # Strict symmetry check\\\\\\\\n    diff = W_64 - W_64.T\\\\\\\\n    diff.eliminate_zeros()\\\\\\\\n    if diff.nnz > 0:\\\\\\\\n        raise ValueError(\\\\\\\\\\"Raw graph must be exactly symmetric before normalization\\\\\\\\\\")\\\\\\\\n\\\\\\\\n    n = W_64.shape[0]\\\\\\\\n    degrees = np.asarray(W_64.sum(axis=1)).ravel()\\\\\\\\n\\\\\\\\n    inv_sqrt = np.zeros(n, dtype=np.float64)\\\\\\\\n    pos_mask = degrees > 0.0\\\\\\\\n    inv_sqrt[pos_mask] = 1.0 / np.sqrt(degrees[pos_mask])\\\\\\\\n\\\\\\\\n    D_scale = sp.diags(inv_sqrt, format=\\\\\\\\\\"csr\\\\\\\\\\")\\\\\\\\n    normalized = (D_scale @ W_64 @ D_scale).astype(dtype)\\\\\\\\n\\\\\\\\n    # Isolated nodes receive identity diagonal\\\\\\\\n    isolated_mask = (degrees == 0.0).astype(dtype)\\\\\\\\n    if np.any(isolated_mask > 0.0):\\\\\\\\n        normalized = normalized + sp.diags(isolated_mask, format=\\\\\\\\\\"csr\\\\\\\\\\")\\\\\\\\n\\\\\\\\n    normalized = normalized.tocsr()\\\\\\\\n    normalized.sum_duplicates()\\\\\\\\n    normalized.eliminate_zeros()\\\\\\\\n    normalized.sort_indices()\\\\\\\\n    return normalized\\\\\\\\n\\\\\\\\n\\\\\\\\ndef recover_v8_raw_semantic_graph(\\\\\\\\n    mfeats: List[torch.Tensor],\\\\\\\\n    knn_edges_list: List[torch.Tensor],\\\\\\\\n    n_items: int,\\\\\\\\n) -> Tuple[sp.csr_matrix, np.ndarray, np.ndarray, Dict[str, Any]]:\\\\\\\\n    \\\\\\\\\\"\\\\\\\\\\"\\\\\\\\\\"Recovers the exact native STAIR5-v8 raw semantic graph W^8 (Eq 1).\\\\\\\\n\\\\\\\\n    Uses frozen text (5) and vision (1) candidate sets with f_p(a) = 0.05 + 0.95 * [max(0, a)]^2,\\\\\\\\n    B_ij = sum_m 1[j in K_m(i)] f_p(a_ij^m), W^8_ij = max(B_ij, B_ji), W^8_ii = 0.\\\\\\\\n\\\\\\\\n    Returns:\\\\\\\\n        W8_csr: Native V8 symmetric raw semantic matrix [N, N].\\\\\\\\n        E_sem: Unique undirected candidate pairs [E, 2] with i < j.\\\\\\\\n        w8_values: Raw semantic weights [E] corresponding to E_sem.\\\\\\\\n        audit: Modality and graph construction audit statistics.\\\\\\\\n    \\\\\\\\\\"\\\\\\\\\\"\\\\\\\\\\"\\\\\\\\n    W8_csr, audit = build_wmsg_semantic_graph(\\\\\\\\n        mfeats=mfeats,\\\\\\\\n        knn_edges_list=knn_edges_list,\\\\\\\\n        n_items=n_items,\\\\\\\\n        semantic_mode=\\\\\\\\\\"weighted\\\\\\\\\\",\\\\\\\\n        edge_power=2.0,\\\\\\\\n        edge_floor=0.05,\\\\\\\\n        relation_gate=\\\\\\\\\\"off\\\\\\\\\\",\\\\\\\\n        relation_strength=0.0,\\\\\\\\n    )\\\\\\\\n\\\\\\\\n    # Extract unique undirected pairs i < j\\\\\\\\n    upper = sp.triu(W8_csr, k=1).tocoo()\\\\\\\\n    E_sem = np.column_stack((upper.row, upper.col)).astype(np.int64)\\\\\\\\n    w8_values = upper.data.astype(np.float64)\\\\\\\\n\\\\\\\\n    return W8_csr, E_sem, w8_values, audit\\\\\\\\n\\\\\\\\n\\\\\\\\ndef compute_mechanism_diagnostics(\\\\\\\\n    W8_csr: sp.csr_matrix,\\\\\\\\n    E_sem: np.ndarray,\\\\\\\\n    w8_values: np.ndarray,\\\\\\\\n    r: np.ndarray,\\\\\\\\n    c: np.ndarray,\\\\\\\\n    mu: np.ndarray,\\\\\\\\n    S_8: sp.csr_matrix,\\\\\\\\n    S_ICA: sp.csr_matrix,\\\\\\\\n    d8: np.ndarray,\\\\\\\\n    d_delta: np.ndarray,\\\\\\\\n    gamma: float = 0.2,\\\\\\\\n    embedding_dim: int = 64,\\\\\\\\n    probe_dim: int = 8,\\\\\\\\n    probe_seed: int = 20261016,\\\\\\\\n    evidence_mode: str = \\\\\\\\\\"replicated_excess\\\\\\\\\\",\\\\\\\\n) -> Dict[str, Any]:\\\\\\\\n    \\\\\\\\\\"\\\\\\\\\\"\\\\\\\\\\"Computes comprehensive offline mechanism diagnostics before expensive GPU training.\\\\\\\\n\\\\\\\\n    Diagnostics computed:\\\\\\\\n    1. Replicated coverage: frac(r > 0) and weighted coverage sum(W8 * 1[r > 0]) / sum(W8).\\\\\\\\n    2. Single-half disagreement: frac((e0 > 0) xor (e1 > 0)).\\\\\\\\n    3. Endpoint contrast (Eq 10): r_ij - 0.5 * (rbar_i + rbar_j).\\\\\\\\n    4. Operator relative dose: ||S_ICA - S_8||_F / ||S_8||_F.\\\\\\\\n    5. Polynomial direction dose (Eq 11): ||[P(S_ICA) - P(S_8)] Z||_F / ||P(S_8) Z||_F.\\\\\\\\n    6. Degree side effect: d_delta / d8 on edges with r = 0.\\\\\\\\n    \\\\\\\\\\"\\\\\\\\\\"\\\\\\\\\\"\\\\\\\\n    E = len(E_sem)\\\\\\\\n    N = W8_csr.shape[0]\\\\\\\\n    diag: Dict[str, Any] = {}\\\\\\\\n\\\\\\\\n    if E == 0:\\\\\\\\n        empty = {\\\\\\\\\\"empty_support\\\\\\\\\\": True, \\\\\\\\\\"num_candidate_edges\\\\\\\\\\": 0,\\\\\\\\n                 \\\\\\\\\\"num_edges_with_positive_evidence\\\\\\\\\\": 0,\\\\\\\\n                 \\\\\\\\\\"frac_edges_with_positive_evidence\\\\\\\\\\": 0.0,\\\\\\\\n                 \\\\\\\\\\"weighted_evidence_coverage\\\\\\\\\\": 0.0,\\\\\\\\n                 \\\\\\\\\\"frac_single_half_disagreement\\\\\\\\\\": None,\\\\\\\\n                 \\\\\\\\\\"relative_operator_dose\\\\\\\\\\": 0.0, \\\\\\\\\\"relative_polynomial_dose\\\\\\\\\\": 0.0,\\\\\\\\n                 \\\\\\\\\\"polynomial_dose_by_coordinate\\\\\\\\\\": {}}\\\\\\\\n        if evidence_mode not in FULL_ACTIVITY_MODES:\\\\\\\\n            empty.update(num_edges_with_replicated_excess=0,\\\\\\\\n                         frac_edges_with_replicated_excess=0.0,\\\\\\\\n                         weighted_replicated_coverage=0.0)\\\\\\\\n        return empty\\\\\\\\n\\\\\\\\n    # 1. Replicated coverage\\\\\\\\n    r_pos = r > 0.0\\\\\\\\n    num_pos = int(np.count_nonzero(r_pos))\\\\\\\\n    frac_pos = float(num_pos / E)\\\\\\\\n    total_w8 = float(np.sum(w8_values))\\\\\\\\n    weighted_pos = float(np.sum(w8_values[r_pos]) / max(1e-12, total_w8))\\\\\\\\n\\\\\\\\n    diag[\\\\\\\\\\"num_candidate_edges\\\\\\\\\\"] = E\\\\\\\\n    diag[\\\\\\\\\\"num_edges_with_positive_evidence\\\\\\\\\\"] = num_pos\\\\\\\\n    diag[\\\\\\\\\\"frac_edges_with_positive_evidence\\\\\\\\\\"] = frac_pos\\\\\\\\n    diag[\\\\\\\\\\"weighted_evidence_coverage\\\\\\\\\\"] = weighted_pos\\\\\\\\n    if evidence_mode not in FULL_ACTIVITY_MODES:\\\\\\\\n        # Read-compatible aliases for legacy manifests only; PEEC is not replication.\\\\\\\\n        diag[\\\\\\\\\\"num_edges_with_replicated_excess\\\\\\\\\\"] = num_pos\\\\\\\\n        diag[\\\\\\\\\\"frac_edges_with_replicated_excess\\\\\\\\\\"] = frac_pos\\\\\\\\n        diag[\\\\\\\\\\"weighted_replicated_coverage\\\\\\\\\\"] = weighted_pos\\\\\\\\n    diag[\\\\\\\\\\"r_mean\\\\\\\\\\"] = float(np.mean(r))\\\\\\\\n    diag[\\\\\\\\\\"r_max\\\\\\\\\\"] = float(np.max(r)) if E > 0 else 0.0\\\\\\\\n    diag[\\\\\\\\\\"r_positive_mean\\\\\\\\\\"] = float(np.mean(r[r_pos])) if num_pos > 0 else 0.0\\\\\\\\n\\\\\\\\n    # 2. Single-half disagreement\\\\\\\\n    diag[\\\\\\\\\\"frac_single_half_disagreement\\\\\\\\\\"] = None\\\\\\\\n    diag[\\\\\\\\\\"num_single_half_disagreement\\\\\\\\\\"] = None\\\\\\\\n    diag[\\\\\\\\\\"num_both_halves_positive\\\\\\\\\\"] = None\\\\\\\\n    if c is not None and mu is not None and c.shape == mu.shape == (E, 2):\\\\\\\\n        e0_pos = (c[:, 0] - mu[:, 0]) > 0.0\\\\\\\\n        e1_pos = (c[:, 1] - mu[:, 1]) > 0.0\\\\\\\\n        disagreement = e0_pos ^ e1_pos\\\\\\\\n        diag[\\\\\\\\\\"frac_single_half_disagreement\\\\\\\\\\"] = float(np.mean(disagreement))\\\\\\\\n        diag[\\\\\\\\\\"num_single_half_disagreement\\\\\\\\\\"] = int(np.count_nonzero(disagreement))\\\\\\\\n        diag[\\\\\\\\\\"num_both_halves_positive\\\\\\\\\\"] = int(np.count_nonzero(e0_pos & e1_pos))\\\\\\\\n\\\\\\\\n    # 3. Endpoint contrast (Eq 10): r_ij - 0.5 * rbar_i - 0.5 * rbar_j\\\\\\\\n    # rbar_i = sum_j W8_ij * r_ij / d8_i\\\\\\\\n    weighted_scores = w8_values * r\\\\\\\\n    weighted_r_sum = (np.bincount(E_sem[:, 0], weights=weighted_scores, minlength=N)\\\\\\\\n                      + np.bincount(E_sem[:, 1], weights=weighted_scores, minlength=N))\\\\\\\\n\\\\\\\\n    rbar = np.zeros(N, dtype=np.float64)\\\\\\\\n    pos_deg = d8 > 0.0\\\\\\\\n    rbar[pos_deg] = weighted_r_sum[pos_deg] / d8[pos_deg]\\\\\\\\n\\\\\\\\n    u_idx = E_sem[:, 0]\\\\\\\\n    v_idx = E_sem[:, 1]\\\\\\\\n    contrast = r - 0.5 * (rbar[u_idx] + rbar[v_idx])\\\\\\\\n\\\\\\\\n    diag[\\\\\\\\\\"endpoint_contrast_mean\\\\\\\\\\"] = float(np.mean(contrast))\\\\\\\\n    diag[\\\\\\\\\\"endpoint_contrast_std\\\\\\\\\\"] = float(np.std(contrast))\\\\\\\\n    diag[\\\\\\\\\\"endpoint_contrast_min\\\\\\\\\\"] = float(np.min(contrast))\\\\\\\\n    diag[\\\\\\\\\\"endpoint_contrast_max\\\\\\\\\\"] = float(np.max(contrast))\\\\\\\\n    diag[\\\\\\\\\\"endpoint_contrast_p10\\\\\\\\\\"] = float(np.percentile(contrast, 10))\\\\\\\\n    diag[\\\\\\\\\\"endpoint_contrast_p50\\\\\\\\\\"] = float(np.percentile(contrast, 50))\\\\\\\\n    diag[\\\\\\\\\\"endpoint_contrast_p90\\\\\\\\\\"] = float(np.percentile(contrast, 90))\\\\\\\\n\\\\\\\\n    # 4. Operator relative dose: ||S_ICA - S_8||_F / ||S_8||_F\\\\\\\\n    diff_S = (S_ICA - S_8).astype(np.float64)\\\\\\\\n    fro_diff = float(np.sqrt(np.dot(diff_S.data, diff_S.data)))\\\\\\\\n    fro_s8 = float(np.sqrt(np.dot(S_8.data.astype(np.float64), S_8.data.astype(np.float64))))\\\\\\\\n    rel_operator_dose = fro_diff / max(1e-12, fro_s8)\\\\\\\\n    diag[\\\\\\\\\\"frobenius_operator_delta\\\\\\\\\\"] = fro_diff\\\\\\\\n    diag[\\\\\\\\\\"relative_operator_dose\\\\\\\\\\"] = rel_operator_dose\\\\\\\\n\\\\\\\\n    # 5. Polynomial direction dose (Eq 11): ||[P(S_ICA) - P(S_8)] Z||_F / ||P(S_8) Z||_F\\\\\\\\n    rng = np.random.default_rng(probe_seed)\\\\\\\\n    Z = rng.choice([-1.0, 1.0], size=(N, probe_dim)).astype(np.float64)\\\\\\\\n\\\\\\\\n    # Reuse three SpMM powers per operator across all five probe coordinates.\\\\\\\\n    coordinates = sorted({0, embedding_dim // 4, embedding_dim // 2,\\\\\\\\n                          3 * embedding_dim // 4, embedding_dim - 1})\\\\\\\\n    doses = {}\\\\\\\\n    if not diff_S.nnz:\\\\\\\\n        doses = {str(k): 0.0 for k in coordinates}\\\\\\\\n    else:\\\\\\\\n        powers8, powers_ica = [Z], [Z]\\\\\\\\n        for _ in range(3):\\\\\\\\n            powers8.append(S_8 @ powers8[-1])\\\\\\\\n            powers_ica.append(S_ICA @ powers_ica[-1])\\\\\\\\n        for coordinate in coordinates:\\\\\\\\n            beta = .1 + .9 * (coordinate / embedding_dim) ** gamma\\\\\\\\n            omega = beta ** np.arange(4, dtype=np.float64)\\\\\\\\n            omega /= omega.sum()\\\\\\\\n            reference = sum(w * x for w, x in zip(omega, powers8))\\\\\\\\n            calibrated = sum(w * x for w, x in zip(omega, powers_ica))\\\\\\\\n            doses[str(coordinate)] = float(\\\\\\\\n                np.linalg.norm(calibrated - reference) / max(1e-12, np.linalg.norm(reference))\\\\\\\\n            )\\\\\\\\n    diag[\\\\\\\\\\"polynomial_dose_by_coordinate\\\\\\\\\\"] = doses\\\\\\\\n    diag[\\\\\\\\\\"relative_polynomial_dose\\\\\\\\\\"] = doses[str(embedding_dim // 2)]\\\\\\\\n    diag[\\\\\\\\\\"polynomial_probe_coordinate\\\\\\\\\\"] = embedding_dim // 2\\\\\\\\n    diag[\\\\\\\\\\"polynomial_probe_dim\\\\\\\\\\"] = int(probe_dim)\\\\\\\\n    diag[\\\\\\\\\\"polynomial_probe_seed\\\\\\\\\\"] = int(probe_seed)\\\\\\\\n    diag[\\\\\\\\\\"polynomial_probe_scope\\\\\\\\\\"] = \\\\\\\\\\"fixed Rademacher L3 probes; per-coordinate ratios, not actual Adam directions\\\\\\\\\\"\\\\\\\\n\\\\\\\\n    row_delta = np.asarray(abs(diff_S).sum(axis=1)).ravel()\\\\\\\\n    row_base = np.asarray(abs(S_8).sum(axis=1)).ravel()\\\\\\\\n    relative_rows = np.divide(row_delta, row_base, out=np.zeros(N), where=row_base > 0)\\\\\\\\n    diag[\\\\\\\\\\"relative_row_l1_dose_quantiles\\\\\\\\\\"] = np.quantile(relative_rows, [0, .1, .5, .9, 1]).tolist()\\\\\\\\n\\\\\\\\n    # 6. Degree side effect: d_delta / d8 on edges where r = 0\\\\\\\\n    zero_r_mask = ~r_pos\\\\\\\\n    if np.any(zero_r_mask):\\\\\\\\n        u_zero = u_idx[zero_r_mask]\\\\\\\\n        v_zero = v_idx[zero_r_mask]\\\\\\\\n        items_in_zero_r = np.unique(np.concatenate([u_zero, v_zero]))\\\\\\\\n        valid_items = items_in_zero_r[d8[items_in_zero_r] > 0]\\\\\\\\n        if len(valid_items) > 0:\\\\\\\\n            deg_ratios = d_delta[valid_items] / d8[valid_items]\\\\\\\\n            diag[\\\\\\\\\\"degree_ratio_on_zero_r_mean\\\\\\\\\\"] = float(np.mean(deg_ratios))\\\\\\\\n            diag[\\\\\\\\\\"degree_ratio_on_zero_r_max\\\\\\\\\\"] = float(np.max(deg_ratios))\\\\\\\\n            diag[\\\\\\\\\\"degree_ratio_on_zero_r_min\\\\\\\\\\"] = float(np.min(deg_ratios))\\\\\\\\n        # With zero evidence on the edge, its normalized coefficient changes only\\\\\\\\n        # through endpoint degree growth. This is not a retained-weight guarantee.\\\\\\\\n        ratios = np.sqrt((d8[u_zero] / d_delta[u_zero]) * (d8[v_zero] / d_delta[v_zero]))\\\\\\\\n        diag[\\\\\\\\\\"zero_score_semantic_edge_coefficient_ratio_quantiles\\\\\\\\\\"] = np.quantile(\\\\\\\\n            ratios, [0, .1, .5, .9, 1],\\\\\\\\n        ).tolist()\\\\\\\\n\\\\\\\\n    return diag\\\\\\\\n\\\\\\\\n\\\\\\\\ndef calibrate_semantic_operator(\\\\\\\\n    W8_csr: sp.csr_matrix,\\\\\\\\n    E_sem: np.ndarray,\\\\\\\\n    r: np.ndarray,\\\\\\\\n    delta: float,\\\\\\\\n    SCF_csr: Optional[sp.csr_matrix] = None,\\\\\\\\n    eta: float = 0.1,\\\\\\\\n    V8_operator_cached: Optional[sp.csr_matrix] = None,\\\\\\\\n    gamma: float = 0.2,\\\\\\\\n    embedding_dim: int = 64,\\\\\\\\n    c_counts: Optional[np.ndarray] = None,\\\\\\\\n    mu_expectations: Optional[np.ndarray] = None,\\\\\\\\n    evidence_mode: str = \\\\\\\\\\"replicated_excess\\\\\\\\\\",\\\\\\\\n) -> Tuple[sp.csr_matrix, Dict[str, Any]]:\\\\\\\\n    \\\\\\\\\\"\\\\\\\\\\"\\\\\\\\\\"Calibrates semantic operator with exponential reweighting and returns S_ICA (Eq 6).\\\\\\\\n\\\\\\\\n    Mathematical formulation:\\\\\\\\n        W^delta_ij = W^8_ij * exp(delta * r_ij) on undirected candidate edges i < j,\\\\\\\\n        W^delta_ji = W^delta_ij, W^delta_ii = 0.\\\\\\\\n        S_delta = N(W^delta) with true actual new degrees d_delta = W^delta 1.\\\\\\\\n        S_ICA = (1 - eta) * S_delta + eta * S_CF (or S_delta if CF is empty/disabled).\\\\\\\\n\\\\\\\\n    Exact Bypass:\\\\\\\\n        - When delta == 0: returns exact V8 cached operator directly.\\\\\\\\n        - When all r == 0: returns exact V8 cached operator with inactive_reason=\\\\\\\\\\"no_replicated_excess\\\\\\\\\\".\\\\\\\\n    \\\\\\\\\\"\\\\\\\\\\"\\\\\\\\\\"\\\\\\\\n    if not math.isfinite(delta) or not 0 <= delta <= 2:\\\\\\\\n        raise ValueError(\\\\\\\\\\"delta must be finite in [0, 2]\\\\\\\\\\")\\\\\\\\n    if not math.isfinite(eta) or not 0 <= eta <= 1:\\\\\\\\n        raise ValueError(\\\\\\\\\\"eta must be finite in [0, 1]\\\\\\\\\\")\\\\\\\\n    W8_csr = W8_csr.tocsr(copy=True)\\\\\\\\n    W8_csr.sum_duplicates()\\\\\\\\n    W8_csr.eliminate_zeros()\\\\\\\\n    W8_csr.sort_indices()\\\\\\\\n    N = W8_csr.shape[0]\\\\\\\\n    if (W8_csr.shape != (N, N) or not np.isfinite(W8_csr.data).all()\\\\\\\\n            or np.any(W8_csr.data < 0) or np.any(W8_csr.diagonal() != 0)\\\\\\\\n            or (W8_csr - W8_csr.T).nnz):\\\\\\\\n        raise ValueError(\\\\\\\\\\"W8 must be square, symmetric, finite, nonnegative and zero-diagonal\\\\\\\\\\")\\\\\\\\n    E_sem, r = np.asarray(E_sem), np.asarray(r, dtype=np.float64)\\\\\\\\n    E = len(E_sem)\\\\\\\\n    if (E_sem.shape != (E, 2) or E_sem.dtype.kind not in \\\\\\\\\\"iu\\\\\\\\\\" or r.shape != (E,)\\\\\\\\n            or not np.isfinite(r).all() or np.any(r < 0) or np.any(r >= 1)\\\\\\\\n            or np.any(E_sem < 0) or np.any(E_sem >= N)\\\\\\\\n            or np.any(E_sem[:, 0] >= E_sem[:, 1])):\\\\\\\\n        raise ValueError(\\\\\\\\\\"Invalid candidate pairs or scores\\\\\\\\\\")\\\\\\\\n    upper = sp.triu(W8_csr, k=1, format=\\\\\\\\\\"coo\\\\\\\\\\")\\\\\\\\n    expected_keys = upper.row.astype(np.int64) * N + upper.col\\\\\\\\n    pair_keys = E_sem[:, 0].astype(np.int64) * N + E_sem[:, 1]\\\\\\\\n    if (len(pair_keys) != upper.nnz\\\\\\\\n            or not np.array_equal(np.sort(pair_keys), np.sort(expected_keys))):\\\\\\\\n        raise ValueError(\\\\\\\\\\"E_sem must contain each raw semantic edge exactly once\\\\\\\\\\")\\\\\\\\n    # Explicit indexing binds weights to caller pair order; triu storage order is\\\\\\\\n    # not an API contract and must never silently permute evidence assignments.\\\\\\\\n    if E == 0:\\\\\\\\n        w8_values = np.empty(0, dtype=np.float64)\\\\\\\\n    else:\\\\\\\\n        w8_values = np.asarray(W8_csr[E_sem[:, 0], E_sem[:, 1]]).ravel().astype(np.float64)\\\\\\\\n    d8 = np.asarray(W8_csr.astype(np.float64).sum(axis=1)).ravel()\\\\\\\\n    if SCF_csr is not None:\\\\\\\\n        if (SCF_csr.shape != (N, N) or not np.isfinite(SCF_csr.data).all()\\\\\\\\n                or np.any(SCF_csr.data < 0)):\\\\\\\\n            raise ValueError(\\\\\\\\\\"Invalid normalized CF graph\\\\\\\\\\")\\\\\\\\n        diff_cf = SCF_csr - SCF_csr.T\\\\\\\\n        if diff_cf.nnz and np.max(np.abs(diff_cf.data)) > 1e-7:\\\\\\\\n            raise ValueError(\\\\\\\\\\"Normalized CF graph must be symmetric within FP32 tolerance\\\\\\\\\\")\\\\\\\\n    effective_eta = float(eta if SCF_csr is not None and SCF_csr.nnz and eta > 0 else 0)\\\\\\\\n    if V8_operator_cached is not None:\\\\\\\\n        if (V8_operator_cached.shape != (N, N)\\\\\\\\n                or not np.isfinite(V8_operator_cached.data).all()\\\\\\\\n                or np.any(V8_operator_cached.data < 0)):\\\\\\\\n            raise ValueError(\\\\\\\\\\"Cached V8 operator must be finite, nonnegative and shape-compatible\\\\\\\\\\")\\\\\\\\n        S_8 = V8_operator_cached\\\\\\\\n    else:\\\\\\\\n        semantic = normalize_degree_isolated(W8_csr)\\\\\\\\n        S_8 = (((1 - effective_eta) * semantic + effective_eta * SCF_csr).tocsr()\\\\\\\\n               if effective_eta else semantic)\\\\\\\\n        S_8.eliminate_zeros()\\\\\\\\n        S_8.sort_indices()\\\\\\\\n\\\\\\\\n    audit = {\\\\\\\\\\"delta\\\\\\\\\\": float(delta), \\\\\\\\\\"effective_eta\\\\\\\\\\": effective_eta,\\\\\\\\n             \\\\\\\\\\"n_items\\\\\\\\\\": N, \\\\\\\\\\"n_candidate_edges\\\\\\\\\\": E,\\\\\\\\n             \\\\\\\\\\"exact_v8_bypass\\\\\\\\\\": False, \\\\\\\\\\"inactive_reason\\\\\\\\\\": None}\\\\\\\\n    d_delta = d8\\\\\\\\n    if delta == 0 or E == 0 or not np.any(r):\\\\\\\\n        reason = \\\\\\\\\\"delta_zero\\\\\\\\\\" if delta == 0 else (\\\\\\\\n            \\\\\\\\\\"no_positive_excess\\\\\\\\\\" if evidence_mode in FULL_ACTIVITY_MODES else \\\\\\\\\\"no_replicated_excess\\\\\\\\\\"\\\\\\\\n        )\\\\\\\\n        audit.update(exact_v8_bypass=True, inactive_reason=reason)\\\\\\\\n        S_ICA = S_8\\\\\\\\n    else:\\\\\\\\n        weighted = w8_values * np.exp(delta * r)\\\\\\\\n        directed = sp.csr_matrix((weighted, (E_sem[:, 0], E_sem[:, 1])), shape=(N, N))\\\\\\\\n        W_delta = (directed + directed.T).tocsr()\\\\\\\\n        d_delta = np.asarray(W_delta.sum(axis=1)).ravel()\\\\\\\\n        # Detect exact component-constant evidence without treating small real\\\\\\\\n        # contrasts as zero via np.allclose\\\\\'s default relative tolerance.\\\\\\\\n        _, components = sp.csgraph.connected_components(W8_csr, directed=False)\\\\\\\\n        labels = components[E_sem[:, 0]]\\\\\\\\n        lo = np.full(N, np.inf)\\\\\\\\n        hi = np.full(N, -np.inf)\\\\\\\\n        np.minimum.at(lo, labels, r)\\\\\\\\n        np.maximum.at(hi, labels, r)\\\\\\\\n        present = np.isfinite(lo)\\\\\\\\n        neutral = bool(np.all(lo[present] == hi[present]))\\\\\\\\n        audit[\\\\\\\\\\"neutral_calibration\\\\\\\\\\"] = neutral\\\\\\\\n        if neutral:\\\\\\\\n            S_ICA = S_8\\\\\\\\n            audit.update(exact_v8_bypass=True, inactive_reason=\\\\\\\\\\"component_constant_evidence\\\\\\\\\\")\\\\\\\\n        else:\\\\\\\\n            semantic = normalize_degree_isolated(W_delta)\\\\\\\\n            S_ICA = (((1 - effective_eta) * semantic + effective_eta * SCF_csr).tocsr()\\\\\\\\n                     if effective_eta else semantic)\\\\\\\\n            S_ICA.sum_duplicates()\\\\\\\\n            S_ICA.eliminate_zeros()\\\\\\\\n            S_ICA.sort_indices()\\\\\\\\n    audit.update(compute_mechanism_diagnostics(\\\\\\\\n        W8_csr, E_sem, w8_values, r,\\\\\\\\n        c_counts, mu_expectations,\\\\\\\\n        S_8, S_ICA, d8, d_delta, gamma, embedding_dim,\\\\\\\\n        evidence_mode=evidence_mode,\\\\\\\\n    ))\\\\\\\\n    audit[\\\\\\\\\\"s_ica_nnz\\\\\\\\\\"] = int(S_ICA.nnz)\\\\\\\\n    audit[\\\\\\\\\\"s_ica_fingerprint\\\\\\\\\\"] = sparse_fingerprint(S_ICA)\\\\\\\\n    return S_ICA, audit\\\\\\\\n\\\\\\\\n\\\\\\\\ndef _native_cf_operator(train_edges, n_users, n_items, cfg):\\\\\\\\n    \\\\\\\\\\"\\\\\\\\\\"\\\\\\\\\\"Reuse V8\\\\\'s raw-CF cache signature, checksum and exact native normalizer.\\\\\\\\\\"\\\\\\\\\\"\\\\\\\\\\"\\\\\\\\n    pairs = binary_train_keys(train_edges, n_users, n_items)\\\\\\\\n    digest = hashlib.sha256()\\\\\\\\n    _digest_array(digest, pairs)\\\\\\\\n    signature = {\\\\\\\\n        \\\\\\\\\\"cache_version\\\\\\\\\\": CACHE_VERSION, \\\\\\\\\\"train\\\\\\\\\\": digest.hexdigest(),\\\\\\\\n        \\\\\\\\\\"users\\\\\\\\\\": n_users, \\\\\\\\\\"items\\\\\\\\\\": n_items, \\\\\\\\\\"k\\\\\\\\\\": int(getattr(cfg, \\\\\\\\\\"k_cf\\\\\\\\\\", 5)),\\\\\\\\n        \\\\\\\\\\"cmin\\\\\\\\\\": int(getattr(cfg, \\\\\\\\\\"c_min\\\\\\\\\\", 2)), \\\\\\\\\\"t\\\\\\\\\\": float(getattr(cfg, \\\\\\\\\\"t_shrinkage\\\\\\\\\\", 5)),\\\\\\\\n        \\\\\\\\\\"source\\\\\\\\\\": file_sha256(Path(__file__).with_name(\\\\\\\\\\"stair5_v4_graph.py\\\\\\\\\\")),\\\\\\\\n    }\\\\\\\\n    key = hashlib.sha256(json.dumps(signature, sort_keys=True).encode()).hexdigest()\\\\\\\\n    cache_dir = getattr(cfg, \\\\\\\\\\"graph_cache_dir\\\\\\\\\\", \\\\\\\\\\"\\\\\\\\\\")\\\\\\\\n    cache = Path(cache_dir) / f\\\\\\\\\\"cf_{key}.pt\\\\\\\\\\" if cache_dir else None\\\\\\\\n    raw = None\\\\\\\\n    hit = False\\\\\\\\n    if cache is not None and cache.is_file():\\\\\\\\n        payload = torch.load(cache, map_location=\\\\\\\\\\"cpu\\\\\\\\\\", weights_only=True)\\\\\\\\n        raw = tensor_to_scipy(payload[\\\\\\\\\\"operator\\\\\\\\\\"])\\\\\\\\n        if (payload.get(\\\\\\\\\\"signature\\\\\\\\\\") != signature or raw.shape != (n_items, n_items)\\\\\\\\n                or payload.get(\\\\\\\\\\"fingerprint\\\\\\\\\\") != sparse_fingerprint(raw)):\\\\\\\\n            raise ValueError(\\\\\\\\\\"Native CF cache signature or checksum mismatch\\\\\\\\\\")\\\\\\\\n        hit = True\\\\\\\\n    if raw is None:\\\\\\\\n        raw = build_candidate_support_graph(\\\\\\\\n            train_edges, n_items, n_users, signature[\\\\\\\\\\"k\\\\\\\\\\"], signature[\\\\\\\\\\"cmin\\\\\\\\\\"], signature[\\\\\\\\\\"t\\\\\\\\\\"],\\\\\\\\n            int(getattr(cfg, \\\\\\\\\\"cf_block_size\\\\\\\\\\", 64)), float(getattr(cfg, \\\\\\\\\\"cf_memory_budget_mib\\\\\\\\\\", 128)),\\\\\\\\n        )\\\\\\\\n        if cache is not None:\\\\\\\\n            atomic_torch_save({\\\\\\\\\\"operator\\\\\\\\\\": scipy_to_tensor(raw), \\\\\\\\\\"signature\\\\\\\\\\": signature,\\\\\\\\n                               \\\\\\\\\\"fingerprint\\\\\\\\\\": sparse_fingerprint(raw)}, cache)\\\\\\\\n    raw_hash = sparse_fingerprint(raw)\\\\\\\\n    # A wholly empty raw graph must not become an identity CF contribution.\\\\\\\\n    normalized = normalize_cf_graph_with_fallback(raw) if raw.nnz and signature[\\\\\\\\\\"k\\\\\\\\\\"] else None\\\\\\\\n    return normalized, {\\\\\\\\\\"native_cf_cache_hit\\\\\\\\\\": hit, \\\\\\\\\\"raw_cf_fingerprint\\\\\\\\\\": raw_hash,\\\\\\\\n                        \\\\\\\\\\"normalized_cf_fingerprint\\\\\\\\\\": (sparse_fingerprint(normalized)\\\\\\\\n                                                      if normalized is not None else None),\\\\\\\\n                        \\\\\\\\\\"cf_cache_key\\\\\\\\\\": key}\\\\\\\\n\\\\\\\\n\\\\\\\\n@torch.no_grad()\\\\\\\\ndef build_stair_icaart_graph(\\\\\\\\n    mfeats: List[torch.Tensor],\\\\\\\\n    knn_edges_list: List[torch.Tensor],\\\\\\\\n    train_edges: torch.Tensor,\\\\\\\\n    n_users: int,\\\\\\\\n    n_items: int,\\\\\\\\n    cfg: Any,\\\\\\\\n    V8_operator_cached: Optional[sp.csr_matrix] = None,\\\\\\\\n    SCF_csr_cached: Optional[sp.csr_matrix] = None,\\\\\\\\n) -> GraphStateICAART:\\\\\\\\n    \\\\\\\\\\"\\\\\\\\\\"\\\\\\\\\\"End-to-end builder for STAIR-REEC (versionICAART) static item graph S_ICA.\\\\\\\\n\\\\\\\\n    Pipeline:\\\\\\\\n    1. Recover exact native V8 raw semantic graph W^8.\\\\\\\\n    2. Extract deduplicated binary train-presence matrix R.\\\\\\\\n    3. Full activity strata for PEEC, or original two-half strata for legacy controls.\\\\\\\\n    4. Candidate-only co-interaction counting c and activity-null expectation mu.\\\\\\\\n    5. Explicitly versioned pooled-full or replicated/pooled-half scoring.\\\\\\\\n    6. Exponential calibration and actual-degree normalization S_ICA.\\\\\\\\n    7. Generate comprehensive audit manifest.\\\\\\\\n    \\\\\\\\\\"\\\\\\\\\\"\\\\\\\\\\"\\\\\\\\n    started = time.perf_counter()\\\\\\\\n\\\\\\\\n    settings = validate_icaart_config(cfg)\\\\\\\\n    identity = evidence_identity(settings)\\\\\\\\n    evidence_mode = settings[\\\\\\\\\\"evidence_mode\\\\\\\\\\"]\\\\\\\\n    full = evidence_mode in FULL_ACTIVITY_MODES\\\\\\\\n    evidence_seed = settings[\\\\\\\\\\"evidence_seed\\\\\\\\\\"]\\\\\\\\n    evidence_tau = settings[\\\\\\\\\\"evidence_tau\\\\\\\\\\"]\\\\\\\\n    evidence_delta = settings[\\\\\\\\\\"evidence_delta\\\\\\\\\\"]\\\\\\\\n    candidate_block = settings[\\\\\\\\\\"evidence_candidate_block\\\\\\\\\\"]\\\\\\\\n    placebo_seed = settings[\\\\\\\\\\"evidence_placebo_seed\\\\\\\\\\"]\\\\\\\\n    eta = float(getattr(cfg, \\\\\\\\\\"eta\\\\\\\\\\", 0.1))\\\\\\\\n    gamma = float(getattr(cfg, \\\\\\\\\\"gamma\\\\\\\\\\", 0.2))\\\\\\\\n    embedding_dim = int(getattr(cfg, \\\\\\\\\\"embedding_dim\\\\\\\\\\", 64))\\\\\\\\n\\\\\\\\n    # 1. Recover exact native W^8\\\\\\\\n    W8_csr, E_sem, w8_values, w8_audit = recover_v8_raw_semantic_graph(\\\\\\\\n        mfeats=mfeats,\\\\\\\\n        knn_edges_list=knn_edges_list,\\\\\\\\n        n_items=n_items,\\\\\\\\n    )\\\\\\\\n\\\\\\\\n    # 2. Deduplicate train presence matrix R [M, N]\\\\\\\\n    unique_keys = binary_train_keys(train_edges, n_users, n_items)\\\\\\\\n    num_raw_interactions = int(train_edges.shape[1])\\\\\\\\n    num_unique_interactions = len(unique_keys)\\\\\\\\n\\\\\\\\n    u_dedup = unique_keys // np.int64(n_items)\\\\\\\\n    i_dedup = unique_keys % np.int64(n_items)\\\\\\\\n\\\\\\\\n    R_csr = sp.csr_matrix(\\\\\\\\n        (np.ones(num_unique_interactions, dtype=np.int8), (u_dedup, i_dedup)),\\\\\\\\n        shape=(n_users, n_items),\\\\\\\\n    )\\\\\\\\n    R_csc = R_csr.tocsc()\\\\\\\\n\\\\\\\\n    # Presence hash for provenance\\\\\\\\n    presence_digest = hashlib.sha256(str((n_users, n_items)).encode())\\\\\\\\n    _digest_array(presence_digest, unique_keys)\\\\\\\\n    presence_hash = presence_digest.hexdigest()\\\\\\\\n\\\\\\\\n    degrees = np.diff(R_csc.indptr).astype(np.int64)\\\\\\\\n    if full:\\\\\\\\n        # No half assignment or seeded hashing in the full-activity estimator.\\\\\\\\n        if evidence_mode == \\\\\\\\\\"pooled_global_excess\\\\\\\\\\":\\\\\\\\n            # Inactive users never enter M_active. This is not a two-half control.\\\\\\\\n            active_users = np.diff(R_csr.indptr) > 0\\\\\\\\n            user_bin = np.where(active_users, 0, -1).astype(np.int64)\\\\\\\\n            group_sizes = np.array([np.count_nonzero(active_users)], dtype=np.int64)\\\\\\\\n        else:\\\\\\\\n            user_bin, group_sizes = build_activity_strata(R_csr)\\\\\\\\n        half = None\\\\\\\\n        c, mu = candidate_pooled_statistics(\\\\\\\\n            R_csc, E_sem, user_bin, group_sizes, candidate_block,\\\\\\\\n            null_scope=identity[\\\\\\\\\\"null_scope\\\\\\\\\\"],\\\\\\\\n        )\\\\\\\\n        r = pooled_activity_score(\\\\\\\\n            c, mu, evidence_tau, evidence_mode, degrees=degrees,\\\\\\\\n            E_sem=E_sem, placebo_seed=placebo_seed,\\\\\\\\n        )\\\\\\\\n    else:\\\\\\\\n        # Preserve R1 and pooled-half arithmetic and assignment semantics.\\\\\\\\n        user_bin, half, group_sizes = build_activity_replication(R_csr, evidence_seed)\\\\\\\\n        c, mu = candidate_excess_statistics(\\\\\\\\n            R_csc, E_sem, half, user_bin, group_sizes, candidate_block, evidence_mode,\\\\\\\\n        )\\\\\\\\n        r = replicated_excess_score(\\\\\\\\n            c, mu, evidence_tau, evidence_mode, placebo_seed, degrees, E_sem,\\\\\\\\n        )\\\\\\\\n\\\\\\\\n    # 6. Load or build CF operator S_CF\\\\\\\\n    cf_audit = {}\\\\\\\\n    if SCF_csr_cached is None:\\\\\\\\n        SCF_csr, cf_audit = _native_cf_operator(train_edges, n_users, n_items, cfg)\\\\\\\\n    else:\\\\\\\\n        SCF_csr = SCF_csr_cached\\\\\\\\n        cf_audit[\\\\\\\\\\"normalized_cf_fingerprint\\\\\\\\\\"] = sparse_fingerprint(SCF_csr)\\\\\\\\n\\\\\\\\n    if V8_operator_cached is not None:\\\\\\\\n        semantic_reference = normalize_degree_isolated(W8_csr)\\\\\\\\n        native_eta = eta if SCF_csr is not None and SCF_csr.nnz else 0.0\\\\\\\\n        reference = (((1 - native_eta) * semantic_reference + native_eta * SCF_csr).tocsr()\\\\\\\\n                     if native_eta else semantic_reference)\\\\\\\\n        if sparse_fingerprint(reference) != sparse_fingerprint(V8_operator_cached):\\\\\\\\n            raise ValueError(\\\\\\\\\\"Reconstructed native graph differs from the cached V8 control\\\\\\\\\\")\\\\\\\\n\\\\\\\\n    # 7. Calibrate semantic operator\\\\\\\\n    S_ICA, cal_audit = calibrate_semantic_operator(\\\\\\\\n        W8_csr=W8_csr,\\\\\\\\n        E_sem=E_sem,\\\\\\\\n        r=r,\\\\\\\\n        delta=evidence_delta,\\\\\\\\n        SCF_csr=SCF_csr,\\\\\\\\n        eta=eta,\\\\\\\\n        V8_operator_cached=V8_operator_cached,\\\\\\\\n        gamma=gamma,\\\\\\\\n        embedding_dim=embedding_dim,\\\\\\\\n        c_counts=c,\\\\\\\\n        mu_expectations=mu,\\\\\\\\n        evidence_mode=evidence_mode,\\\\\\\\n    )\\\\\\\\n\\\\\\\\n    pooled_counts = c if full else c.sum(axis=1)\\\\\\\\n    pooled_null = mu if full else mu.sum(axis=1)\\\\\\\\n    quantiles = [0, .1, .5, .9, .95, .99, 1]\\\\\\\\n    cal_audit[\\\\\\\\\\"evidence_quantile_levels\\\\\\\\\\"] = quantiles\\\\\\\\n    cal_audit[\\\\\\\\\\"candidate_count_quantiles\\\\\\\\\\"] = np.quantile(pooled_counts, quantiles).tolist() if len(c) else []\\\\\\\\n    cal_audit[\\\\\\\\\\"candidate_null_quantiles\\\\\\\\\\"] = np.quantile(pooled_null, quantiles).tolist() if len(mu) else []\\\\\\\\n    cal_audit[\\\\\\\\\\"evidence_score_quantiles\\\\\\\\\\"] = np.quantile(r, quantiles).tolist() if len(r) else []\\\\\\\\n\\\\\\\\n    elapsed = time.perf_counter() - started\\\\\\\\n\\\\\\\\n    # 8. Compile comprehensive manifest\\\\\\\\n    manifest = {\\\\\\\\n        **identity,\\\\\\\\n        \\\\\\\\\\"evidence_mode\\\\\\\\\\": evidence_mode,\\\\\\\\n        \\\\\\\\\\"evidence_seed\\\\\\\\\\": evidence_seed,\\\\\\\\n        \\\\\\\\\\"evidence_tau\\\\\\\\\\": evidence_tau,\\\\\\\\n        \\\\\\\\\\"evidence_delta\\\\\\\\\\": evidence_delta,\\\\\\\\n        \\\\\\\\\\"evidence_candidate_block\\\\\\\\\\": candidate_block,\\\\\\\\n        \\\\\\\\\\"evidence_activity_bins\\\\\\\\\\": settings[\\\\\\\\\\"evidence_activity_bins\\\\\\\\\\"],\\\\\\\\n        \\\\\\\\\\"evidence_halves\\\\\\\\\\": identity[\\\\\\\\\\"active_halves\\\\\\\\\\"],\\\\\\\\n        \\\\\\\\\\"evidence_placebo_seed\\\\\\\\\\": placebo_seed,\\\\\\\\n        \\\\\\\\\\"training_seed\\\\\\\\\\": int(getattr(cfg, \\\\\\\\\\"seed\\\\\\\\\\", 1)),\\\\\\\\n        \\\\\\\\\\"evidence_dtype\\\\\\\\\\": \\\\\\\\\\"float64\\\\\\\\\\",\\\\\\\\n        \\\\\\\\\\"operator_dtype\\\\\\\\\\": \\\\\\\\\\"float32\\\\\\\\\\",\\\\\\\\n        \\\\\\\\\\"raw_semantic_fingerprint\\\\\\\\\\": sparse_fingerprint(W8_csr),\\\\\\\\n        \\\\\\\\\\"baseline_v8_fingerprint\\\\\\\\\\": sparse_fingerprint(V8_operator_cached) if V8_operator_cached is not None else None,\\\\\\\\n        \\\\\\\\\\"assignment_hash\\\\\\\\\\": hashlib.sha256(\\\\\\\\n            user_bin.astype(\\\\\\\\\\"<i8\\\\\\\\\\").tobytes()\\\\\\\\n            + (b\\\\\\\\\\"\\\\\\\\\\" if half is None else half.astype(\\\\\\\\\\"<i8\\\\\\\\\\").tobytes())\\\\\\\\n        ).hexdigest(),\\\\\\\\n        \\\\\\\\\\"group_sizes\\\\\\\\\\": group_sizes.tolist(),\\\\\\\\n        \\\\\\\\\\"evidence_score_hash\\\\\\\\\\": hashlib.sha256(r.astype(\\\\\\\\\\"<f8\\\\\\\\\\").tobytes()).hexdigest(),\\\\\\\\n        \\\\\\\\\\"evidence_config\\\\\\\\\\": settings,\\\\\\\\n        \\\\\\\\\\"counts_shape\\\\\\\\\\": list(c.shape),\\\\\\\\n        \\\\\\\\\\"null_expectation_shape\\\\\\\\\\": list(mu.shape),\\\\\\\\n        \\\\\\\\\\"counts_hash\\\\\\\\\\": hashlib.sha256(c.astype(\\\\\\\\\\"<i8\\\\\\\\\\").tobytes()).hexdigest(),\\\\\\\\n        \\\\\\\\\\"null_expectation_hash\\\\\\\\\\": hashlib.sha256(mu.astype(\\\\\\\\\\"<f8\\\\\\\\\\").tobytes()).hexdigest(),\\\\\\\\n        \\\\\\\\\\"cf\\\\\\\\\\": {k: v for k, v in cf_audit.items() if k != \\\\\\\\\\"native_cf_cache_hit\\\\\\\\\\"},\\\\\\\\n        \\\\\\\\\\"num_raw_train_interactions\\\\\\\\\\": num_raw_interactions,\\\\\\\\n        \\\\\\\\\\"num_deduplicated_train_interactions\\\\\\\\\\": num_unique_interactions,\\\\\\\\n        \\\\\\\\\\"train_presence_hash\\\\\\\\\\": presence_hash,\\\\\\\\n        \\\\\\\\\\"num_users\\\\\\\\\\": n_users,\\\\\\\\n        \\\\\\\\\\"num_items\\\\\\\\\\": n_items,\\\\\\\\n        \\\\\\\\\\"num_activity_strata\\\\\\\\\\": int(group_sizes.shape[-1]),\\\\\\\\n        \\\\\\\\\\"user_half_counts\\\\\\\\\\": None if half is None else [int(np.sum(half == 0)), int(np.sum(half == 1))],\\\\\\\\n        \\\\\\\\\\"graph_fingerprint\\\\\\\\\\": cal_audit[\\\\\\\\\\"s_ica_fingerprint\\\\\\\\\\"],\\\\\\\\n    }\\\\\\\\n\\\\\\\\n    if evidence_mode == \\\\\\\\\\"pooled_global_excess\\\\\\\\\\":\\\\\\\\n        m_active = int(group_sizes[0])\\\\\\\\n        max_null_error = 0.0\\\\\\\\n        for start in range(0, len(E_sem), candidate_block):\\\\\\\\n            stop = min(start + candidate_block, len(E_sem))\\\\\\\\n            left, right = E_sem[start:stop].T\\\\\\\\n            expected_global = (degrees[left].astype(np.float64)\\\\\\\\n                               * degrees[right].astype(np.float64))\\\\\\\\n            if m_active:\\\\\\\\n                expected_global /= m_active\\\\\\\\n            else:\\\\\\\\n                expected_global.fill(0.0)\\\\\\\\n            error = np.abs(mu[start:stop] - expected_global)\\\\\\\\n            max_null_error = max(max_null_error, float(error.max(initial=0.0)))\\\\\\\\n            if not np.allclose(mu[start:stop], expected_global, rtol=1e-12, atol=1e-12):\\\\\\\\n                raise ArithmeticError(\\\\\\\\\\"Global-full null disagrees with active-user marginal formula\\\\\\\\\\")\\\\\\\\n        manifest.update(\\\\\\\\n            activity_conditioning=False,\\\\\\\\n            num_activity_strata=0,\\\\\\\\n            num_null_groups=1,\\\\\\\\n            num_active_train_users=m_active,\\\\\\\\n            num_inactive_train_users=n_users - m_active,\\\\\\\\n            global_null_formula=\\\\\\\\\\"n_i*n_j/M_active; binary TRAIN item marginals\\\\\\\\\\",\\\\\\\\n            global_null_max_abs_formula_error=max_null_error,\\\\\\\\n            global_null_check_scope=\\\\\\\\\\"candidate-wise formula consistency, not statistical calibration\\\\\\\\\\",\\\\\\\\n        )\\\\\\\\n\\\\\\\\n    metadata = {**w8_audit, **cal_audit, **cf_audit, **manifest,\\\\\\\\n                \\\\\\\\\\"build_elapsed_seconds\\\\\\\\\\": elapsed,\\\\\\\\n                \\\\\\\\\\"analytical_candidate_payload\\\\\\\\\\": analytical_icaart_payload(n_items, len(E_sem), pooled=full)}\\\\\\\\n\\\\\\\\n    # Preparation is CPU-only regardless of where caller edge IDs were stored.\\\\\\\\n    # The native coach moves the finished model and its single final CSR graph.\\\\\\\\n    operator_tensor = scipy_to_tensor(S_ICA)\\\\\\\\n\\\\\\\\n    return GraphStateICAART(operator=operator_tensor, metadata=metadata, manifest=manifest)\\\\\\\\n\\\\\\\\n\\\\\\\\n# Public alias for backward compatibility and verification scripts\\\\\\\\ncalibrate_semantic_support = calibrate_semantic_operator\\\\\\\\n\\\\\\\\n\\"}}}\')\\n","GLOBAL_OVERLAY_ID=\'f62a74a254ef021fb3c064070c85ea8d6b6f0d8810870e1ae9293bb3f90a24c2\'\\n","EXPECTED_RUNTIME_SOURCES=dict(SEED1_REFERENCE[\'baby\'][\'contract\'][\'sources\'])\\n","DECLARED_SOURCE_CHANGES={}\\n","for relative, record in GLOBAL_SOURCE_OVERLAY[\'files\'].items():\\n","    basename=Path(relative).name\\n","    if EXPECTED_RUNTIME_SOURCES[basename]!=record[\'base_sha256\']:\\n","        raise RuntimeError(\'Overlay base differs from recorded Full reference: \'+relative)\\n","    EXPECTED_RUNTIME_SOURCES[basename]=record[\'sha256\']\\n","    DECLARED_SOURCE_CHANGES[relative]={\'full_sha256\':record[\'base_sha256\'],\'global_sha256\':record[\'sha256\']}\\n","write_json(RUN_ROOT/\'provenance\'/\'global_full_source_overlay.json\',GLOBAL_SOURCE_OVERLAY)\\n","write_json(RUN_ROOT/\'provenance\'/\'declared_source_changes.json\',{\'base_commit\':RUNTIME_COMMIT,\\n","    \'overlay_id\':GLOBAL_OVERLAY_ID,\'changes\':DECLARED_SOURCE_CHANGES,\\n","    \'scope\':\'Global-full estimator/config wiring only; Full algorithm sources remain unchanged elsewhere.\'})\\n"],"execution_count":null,"outputs":[]},{"cell_type":"markdown","id":"eb0b3cd5","metadata":{},"source":["## 3. Private pinned source + declared overlay and matched runtime\\n","Clone the recorded Full revision into a private session directory, apply the embedded three-file extension after base-hash checks, and save source hashes/full diff. No existing training checkout is modified. No fallback HEAD, anonymous monkey-patch or Torch upgrade. All unmodified source digests must match Full.\\n"]},{"cell_type":"code","id":"8e1894e2","metadata":{},"source":["if RUN_MODE == \'train\':\\n","    import importlib.metadata\\n","    import importlib.util\\n","    import subprocess\\n","\\n","    if REPO_OVERRIDE is not None:\\n","        raise ValueError(\'REPO_OVERRIDE is disabled: this control uses its own private checkout.\')\\n","    else:\\n","        REPO_ROOT=WORK_DIR/\'eclipse_sources\'/(\'global_full_null_\'+GLOBAL_OVERLAY_ID[:12])/SESSION_ID\\n","        if not REPO_ROOT.exists():\\n","            REPO_ROOT.mkdir(parents=True,exist_ok=False)\\n","            subprocess.run([\'git\',\'init\',str(REPO_ROOT)],check=True)\\n","            subprocess.run([\'git\',\'-C\',str(REPO_ROOT),\'remote\',\'add\',\'origin\',REPO_URL],check=True)\\n","            subprocess.run([\'git\',\'-C\',str(REPO_ROOT),\'fetch\',\'--depth\',\'1\',\'origin\',RUNTIME_COMMIT],check=True)\\n","            subprocess.run([\'git\',\'-C\',str(REPO_ROOT),\'checkout\',\'--detach\',RUNTIME_COMMIT],check=True)\\n","    actual=subprocess.run([\'git\',\'-C\',str(REPO_ROOT),\'rev-parse\',\'HEAD\'],text=True,capture_output=True,check=True).stdout.strip()\\n","    if actual!=RUNTIME_COMMIT:raise RuntimeError(\'Use the pinned checkout; existing source is never reset.\')\\n","    # Apply only the explicitly declared source extension before any imports/probes.\\n","    overlay_receipt={}\\n","    for relative, record in GLOBAL_SOURCE_OVERLAY[\'files\'].items():\\n","        path=REPO_ROOT/relative\\n","        if path.is_symlink() or not path.is_file():raise ValueError(\'Invalid overlay target: \'+relative)\\n","        current=file_sha256(path)\\n","        current_lf=hashlib.sha256(path.read_bytes().replace(b\'\\\\r\\\\n\',b\'\\\\n\')).hexdigest()\\n","        if not ({current,current_lf}&{record[\'base_sha256\'],record[\'sha256\']}):\\n","            raise RuntimeError(\'Unexpected private-checkout source; refusing overwrite: \'+relative)\\n","        payload=record[\'content\'].encode(\'utf-8\')\\n","        if hashlib.sha256(payload).hexdigest()!=record[\'sha256\']:raise RuntimeError(\'Overlay payload hash mismatch\')\\n","        path.write_bytes(payload)\\n","        if file_sha256(path)!=record[\'sha256\']:raise RuntimeError(\'Overlay write mismatch\')\\n","        overlay_receipt[relative]={\'base_sha256\':record[\'base_sha256\'],\'sha256\':file_sha256(path)}\\n","    write_json(RUN_ROOT/\'provenance\'/\'applied_global_full_overlay.json\',{\\n","        \'base_commit\':actual,\'overlay_id\':GLOBAL_OVERLAY_ID,\'files\':overlay_receipt,\'private_checkout\':str(REPO_ROOT)})\\n","    for args,name in (([\'status\',\'--porcelain\'],\'git_status.txt\'),([\'diff\',\'HEAD\',\'--\'],\'source_diff.patch\'),([\'remote\',\'-v\'],\'git_remotes.txt\')):\\n","        result=subprocess.run([\'git\',\'-C\',str(REPO_ROOT),*args],text=True,encoding=\'utf-8\',capture_output=True,check=True)\\n","        (RUN_ROOT/\'provenance\'/name).write_text(result.stdout,encoding=\'utf-8\')\\n","    if not (REPO_ROOT / \'main_stair_icaart.py\').is_file():\\n","        raise FileNotFoundError(REPO_ROOT / \'main_stair_icaart.py\')\\n","\\n","    ENV = os.environ.copy()\\n","    ENV[\'PYTHONPATH\'] = os.pathsep.join(filter(None, [str(REPO_ROOT), ENV.get(\'PYTHONPATH\', \'\')]))\\n","    ENV[\'PYTHONUTF8\'] = \'1\'\\n","    ENV[\'PYTHONIOENCODING\'] = \'utf-8\'\\n","    ENV[\'PYTHONHASHSEED\'] = str(SEED)\\n","    ENV[\'CUDA_VISIBLE_DEVICES\'] = str(GPU_INDEX)\\n","    ENV.setdefault(\'CUBLAS_WORKSPACE_CONFIG\', \':4096:8\')\\n","\\n","    # Do not import Torch into the notebook kernel: all CUDA work belongs to the training subprocess.\\n","    if INSTALL_MISSING_DEPENDENCIES:\\n","        packages = {\\n","            \'freerec\': (\'freerec==0.9.7\', True),\\n","            \'torch_geometric\': (\'torch-geometric==2.8.1\', True),\\n","            \'numpy\': (\'numpy==2.1.3\', True), \'scipy\': (\'scipy==1.16.3\', True),\\n","            \'yaml\': (\'PyYAML==6.0.3\', True), \'pandas\': (\'pandas==2.3.3\', True),\\n","            \'matplotlib\': (\'matplotlib==3.10.0\', True), \'prettytable\': (\'prettytable==3.18.0\', True),\\n","            \'sklearn\': (\'scikit-learn==1.6.1\', True), \'psutil\': (\'psutil==5.9.5\', True),\\n","        }\\n","        for module, (requirement, no_deps) in packages.items():\\n","            missing = importlib.util.find_spec(module) is None\\n","            if module == \'freerec\' and not missing:\\n","                missing = importlib.metadata.version(\'freerec\') != \'0.9.7\'\\n","            if missing:\\n","                command = [sys.executable, \'-m\', \'pip\', \'install\', \'--quiet\']\\n","                if no_deps:\\n","                    command.append(\'--no-deps\')\\n","                subprocess.run(command + [requirement], check=True)\\n","\\n","    expected_sources = EXPECTED_RUNTIME_SOURCES\\n","    source_report = {}\\n","    for name, expected in expected_sources.items():\\n","        choices = [REPO_ROOT / name, REPO_ROOT / \'models\' / name, REPO_ROOT / \'optimizers\' / name]\\n","        path = next((p for p in choices if p.is_file()), None)\\n","        if path is None:\\n","            raise FileNotFoundError(f\'Missing seed-1 source: {name}\')\\n","        payload = path.read_bytes()\\n","        actual = hashlib.sha256(payload).hexdigest()\\n","        lf_digest = hashlib.sha256(payload.replace(b\'\\\\r\\\\n\', b\'\\\\n\')).hexdigest()\\n","        source_report[name] = {\'path\': str(path.relative_to(REPO_ROOT)), \'sha256\': actual,\\n","                               \'line_endings_only\': actual != expected and lf_digest == expected}\\n","        if expected not in (actual, lf_digest):\\n","            raise RuntimeError(f\'Source differs from declared Full-plus-Global overlay: {path}. Use the pinned runtime commit.\')\\n","    write_json(RUN_ROOT / \'provenance\' / \'source_check.json\', source_report)\\n","\\n","    probe = r\\"\\"\\"\\n","    import json, platform, torch\\n","    import models.freerec_compat\\n","    import freerec\\n","    from main_stair_icaart import build_config\\n","    from stair_icaart_reporting import validate_selected_receipt\\n","    print(\'ECLIPSE_ENV=\' + json.dumps({\'python\': platform.python_version(), \'torch\': str(torch.__version__),\\n","     \'freerec\': str(freerec.__version__), \'torch_geometric\': __import__(\'importlib.metadata\',fromlist=[\'version\']).version(\'torch-geometric\'), \'cuda\': torch.cuda.is_available(),\\n","     \'gpu\': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}))\\n","    \\"\\"\\"\\n","    probe = __import__(\'textwrap\').dedent(probe)\\n","    checked = subprocess.run([sys.executable, \'-c\', probe], cwd=REPO_ROOT, env=ENV,\\n","                             text=True, encoding=\'utf-8\', capture_output=True)\\n","    if checked.stdout:\\n","        print(checked.stdout)\\n","    if checked.stderr:\\n","        print(checked.stderr)\\n","    checked.check_returncode()\\n","    line = next(line for line in checked.stdout.splitlines() if line.startswith(\'ECLIPSE_ENV=\'))\\n","    RUNTIME_ENV = json.loads(line.split(\'=\', 1)[1])\\n","    if not RUNTIME_ENV[\'cuda\']:\\n","        raise RuntimeError(\'Enable a Kaggle GPU accelerator before training.\')\\n","    reference = SEED1_REFERENCE[SELECTED_DATASETS[0]]\\n","    deviations = {key: {\'reference\': reference[key], \'current\': RUNTIME_ENV[key]}\\n","                  for key in (\'torch\', \'freerec\',\'gpu\') if RUNTIME_ENV[key] != reference[key]}\\n","    if RUNTIME_ENV[\'python\'].split(\'.\')[:2] != reference[\'python\'].split(\'.\')[:2]:\\n","        deviations[\'python\'] = {\'reference\': reference[\'python\'], \'current\': RUNTIME_ENV[\'python\']}\\n","    if RUNTIME_ENV.get(\'torch_geometric\') != \'2.8.1\':\\n","        deviations[\'torch_geometric\'] = {\'reference\':\'2.8.1\',\'current\':RUNTIME_ENV.get(\'torch_geometric\')}\\n","    write_json(RUN_ROOT / \'provenance\' / \'environment.json\', {\'runtime\': RUNTIME_ENV, \'deviations\': deviations,\\n","               \'strict_runtime\': STRICT_RUNTIME, \'gpu_index\': GPU_INDEX})\\n","    if deviations and STRICT_RUNTIME:\\n","        raise RuntimeError(f\'Runtime differs from seed 1: {deviations}. Restore the seed-1 environment; \'\\n","                           \'setting STRICT_RUNTIME=False explicitly records a protocol deviation.\')\\n","    if deviations:\\n","        print(\'PROTOCOL DEVIATION:\', deviations)\\n","    if RUNTIME_ENV[\'gpu\'] != reference[\'gpu\']:\\n","        print(\'HARDWARE DEVIATION:\', RUNTIME_ENV[\'gpu\'], \'vs\', reference[\'gpu\'],\\n","              \'- runtime measurements are not a matched-hardware comparison.\')\\n","    freeze = subprocess.run([sys.executable, \'-m\', \'pip\', \'freeze\'], text=True, capture_output=True, check=True)\\n","    (RUN_ROOT / \'provenance\' / \'pip_freeze.txt\').write_text(freeze.stdout, encoding=\'utf-8\')\\n","    commit = subprocess.run([\'git\', \'-C\', str(REPO_ROOT), \'rev-parse\', \'HEAD\'], text=True, capture_output=True)\\n","    ACTUAL_COMMIT = commit.stdout.strip() if commit.returncode == 0 else \'unavailable\'\\n","    print(\'Runtime ready:\', REPO_ROOT, \'| commit:\', ACTUAL_COMMIT)\\n","    print(\'Import/environment checks only; no model test or training has been executed.\')\\n","\\n","    driver=subprocess.run([\'nvidia-smi\',\'--query-gpu=name,driver_version,memory.total\',\'--format=csv\'],text=True,capture_output=True)\\n","    (RUN_ROOT/\'provenance\'/\'nvidia_smi.txt\').write_text(driver.stdout+driver.stderr,encoding=\'utf-8\')\\n","\\n","else:\\n","    print(\'report_only: skipped training setup; no model/network/GPU invocation.\')\\n"],"execution_count":null,"outputs":[]},{"cell_type":"markdown","id":"d223f897","metadata":{},"source":["## 4. Processed inputs — Attach before Kaggle batch\\n","Five split/feature files per selected dataset must match reference SHA256. Writable bridge/chunks; no KaggleHub attachment in batch. Report-only skips this entire stage.\\n"]},{"cell_type":"code","id":"c88d2aae","metadata":{},"source":["if RUN_MODE == \'train\':\\n","    import shutil\\n","\\n","    REQUIRED_FILES = (\'train.txt\', \'valid.txt\', \'test.txt\', \'textual_modality.pkl\', \'visual_modality.pkl\')\\n","\\n","\\n","    def dataset_search_roots(extra_roots=()):\\n","        # Include writable bridges and caches created by earlier interactive sessions.\\n","        return [REPO_ROOT / \'data\', DATA_ROOT, WORK_DIR / \'data\',\\n","                Path(\'/kaggle/data\'), Path(\'/kaggle/input\'),\\n","                Path.home() / \'.cache\' / \'kagglehub\' / \'datasets\',\\n","                *map(Path, DATASET_SEARCH_ROOTS), *map(Path, extra_roots)]\\n","\\n","\\n","    def discover_datasets(extra_roots=()):\\n","        found = {}\\n","        unknown = set(DATASET_SOURCE_OVERRIDES) - set(DATASET_FOLDERS)\\n","        if unknown:\\n","            raise ValueError(f\'Unknown DATASET_SOURCE_OVERRIDES keys: {sorted(unknown)}\')\\n","        for key in SELECTED_DATASETS:\\n","            if key not in DATASET_SOURCE_OVERRIDES:\\n","                continue\\n","            path = Path(DATASET_SOURCE_OVERRIDES[key]).expanduser()\\n","            absent = [name for name in REQUIRED_FILES if not (path / name).is_file()]\\n","            if absent:\\n","                raise FileNotFoundError(f\'Incomplete explicit dataset source for {key}: {path}; missing {absent}\')\\n","            found[key] = path\\n","        aliases = {\'baby\': (\'baby\',), \'sports\': (\'sport\',), \'electronics\': (\'electronic\',)}\\n","        visited = set()\\n","        for root in dataset_search_roots(extra_roots):\\n","            root = root.expanduser()\\n","            if not root.is_dir() or str(root.resolve()) in visited:\\n","                continue\\n","            visited.add(str(root.resolve()))\\n","            for current, directories, files in os.walk(root, followlinks=False):\\n","                directories[:] = sorted(d for d in directories if d not in\\n","                                        {\'.git\', \'chunks\', \'__pycache__\', \'.stair_icaart_graph_cache\'})\\n","                if not set(REQUIRED_FILES).issubset(files):\\n","                    continue\\n","                path = Path(current)\\n","                for key in SELECTED_DATASETS:\\n","                    if any(alias in path.name.lower() for alias in aliases[key]):\\n","                        found.setdefault(key, path)\\n","        return found\\n","\\n","\\n","    def missing_dataset_message(missing):\\n","        return (\\n","            f\'Missing processed datasets: {sorted(missing)}.\\\\n\'\\n","            f\'On Kaggle, attach {DATASET_HANDLE!r} with Add Input BEFORE Save & Run All \'\\n","            \'(or attach an equivalent processed dataset matching the seed-1 fingerprints). \'\\n","            \'Save a new notebook version after attaching it. Internet access alone does not \'\\n","            \'attach a datasource, and kagglehub cannot attach a new one in a non-interactive run.\\\\n\'\\n","            \'Alternatively set DATASET_SOURCE_OVERRIDES to the existing per-dataset folders, \'\\n","            \'or DATASET_SEARCH_ROOTS to their parent directory. Each dataset folder must contain: \'\\n","            + \', \'.join(REQUIRED_FILES) + \'.\\\\n\'\\n","            \'Searched roots: \' + \', \'.join(str(p.expanduser()) for p in dataset_search_roots())\\n","            + f\'\\\\nDiscovery receipt: {RUN_ROOT / \\"provenance\\" / \\"data_discovery.json\\"}\'\\n","        )\\n","\\n","\\n","    def resolve_dataset_sources():\\n","        sources = discover_datasets()\\n","        missing = set(SELECTED_DATASETS) - sources.keys()\\n","        on_kaggle = Path(\'/kaggle/working\').is_dir()\\n","        discovery = {\'dataset_handle\': DATASET_HANDLE, \'selected_datasets\': list(SELECTED_DATASETS),\\n","                     \'searched_roots\': [str(p.expanduser()) for p in dataset_search_roots()],\\n","                     \'on_kaggle\': on_kaggle, \'auto_download_requested\': AUTO_DOWNLOAD_DATA,\\n","                     \'download_attempted\': False}\\n","\\n","        def record(status):\\n","            discovery.update(status=status, missing=sorted(missing),\\n","                             sources={key: str(path.resolve()) for key, path in sources.items()})\\n","            write_json(RUN_ROOT / \'provenance\' / \'data_discovery.json\', discovery)\\n","\\n","        if missing and on_kaggle:\\n","            # Never try to attach a datasource during a Kaggle batch job. The caller\\n","            # must attach inputs before execution; this also makes batch runs repeatable.\\n","            record(\'MISSING_ATTACHED_INPUT\')\\n","            raise FileNotFoundError(missing_dataset_message(missing))\\n","        if missing and AUTO_DOWNLOAD_DATA:\\n","            discovery[\'download_attempted\'] = True\\n","            try:\\n","                import kagglehub\\n","                downloaded = kagglehub.dataset_download(DATASET_HANDLE)\\n","                discovery[\'downloaded_root\'] = str(downloaded)\\n","                sources.update(discover_datasets([downloaded]))\\n","                missing = set(SELECTED_DATASETS) - sources.keys()\\n","            except Exception as exc:\\n","                discovery[\'download_error\'] = {\'type\': type(exc).__name__, \'message\': str(exc)}\\n","                record(\'DOWNLOAD_FAILED\')\\n","                raise FileNotFoundError(missing_dataset_message(missing)) from exc\\n","        if missing:\\n","            record(\'MISSING_PROCESSED_DATA\')\\n","            raise FileNotFoundError(missing_dataset_message(missing))\\n","        record(\'FOUND_PENDING_FINGERPRINT_CHECK\')\\n","        return sources\\n","\\n","\\n","    sources = resolve_dataset_sources()\\n","\\n","    DATA_RECEIPTS = {}\\n","    for key in SELECTED_DATASETS:\\n","        source = sources[key].resolve()\\n","        reference = SEED1_REFERENCE[key]\\n","        expected = dict(reference[\'contract\'][\'splits\'])\\n","        expected.update({record[\'file\']: record[\'sha256\'] for record in reference[\'features\']})\\n","        destination = DATA_ROOT / \'Processed\' / DATASET_FOLDERS[key]\\n","        if destination.is_symlink():\\n","            raise RuntimeError(f\'Processed destination must be a writable directory, not a symlink: {destination}\')\\n","        destination.mkdir(parents=True, exist_ok=True)\\n","        for name, expected_hash in expected.items():\\n","            original = source / name\\n","            if not original.is_file():\\n","                raise FileNotFoundError(original)\\n","            actual_hash = file_sha256(original)\\n","            if actual_hash != expected_hash:\\n","                raise RuntimeError(f\'Dataset differs from seed 1: {original}\')\\n","            target = destination / name\\n","            if target.is_symlink() and not target.exists():\\n","                raise FileExistsError(f\'Broken input link preserved; choose a fresh DATA_ROOT: {target}\')\\n","            if target.exists():\\n","                if file_sha256(target) != expected_hash:\\n","                    raise RuntimeError(f\'Existing linked/copied input has a different fingerprint: {target}\')\\n","            else:\\n","                try:\\n","                    target.symlink_to(original)\\n","                except OSError:\\n","                    shutil.copy2(original, target)\\n","        if (destination / \'chunks\').is_symlink():\\n","            raise RuntimeError(f\'FreeRec chunks must be writable, not an input symlink: {destination / \\"chunks\\"}\')\\n","        (destination / \'chunks\').mkdir(exist_ok=True)\\n","        DATA_RECEIPTS[key] = {\'source\': str(source), \'processed\': str(destination), \'fingerprints\': expected}\\n","        print(f\'{DATASET_TITLES[key]}: split and feature fingerprints match seed 1.\')\\n","    write_json(RUN_ROOT / \'provenance\' / \'data_check.json\', DATA_RECEIPTS)\\n","\\n","else:\\n","    print(\'report_only: skipped training setup; no model/network/GPU invocation.\')\\n"],"execution_count":null,"outputs":[]},{"cell_type":"markdown","id":"7d979928","metadata":{},"source":["## 5. Resolved configuration and native CLI preflight\\n","Mode=pooled_global_excess, bins=none, halves0, delta1/tau2; eta.1/LC.01 and per-dataset training hyperparameters remain Full. Native resolved settings are checked before training.\\n"]},{"cell_type":"code","id":"41ad41ac","metadata":{},"source":["if RUN_MODE == \'train\':\\n","    import yaml\\n","\\n","    CONFIG_PATHS = {}\\n","    ARTIFACT_PATHS = {}\\n","    LOG_PATHS = {}\\n","    for key in SELECTED_DATASETS:\\n","        stem = stem_for(key)\\n","        title = key.capitalize()\\n","        original = REPO_ROOT / \'configs\' / f\'Amazon2014{title}_STAIR_PEEC.yaml\'\\n","        if not original.is_file():\\n","            raise FileNotFoundError(original)\\n","        config = yaml.safe_load(original.read_text(encoding=\'utf-8\'))\\n","        config.update(SEED1_REFERENCE[key][\'v8_config\'])\\n","        config.update(SEED1_REFERENCE[key][\'icaart_config\'])\\n","        config[\'eta\'] = 0.1\\n","        config[\'evidence_mode\'] = \'pooled_global_excess\'\\n","        config[\'evidence_activity_bins\'] = \'none\'\\n","        config[\'evidence_delta\'] = 1.0\\n","        config.pop(\'evidence_seed\', None)  # Inactive; do not confuse the evidence partition with training RNG.\\n","        config.update(seed=SEED, root=str(DATA_ROOT), dataset=DATASET_FOLDERS[key],\\n","                      num_workers=SEED1_REFERENCE[key][\'contract\'][\'config\'][\'num_workers\'],\\n","                      eval_valid=True, eval_test=False, ranking=\'full\', which4best=\'NDCG@20\',\\n","                      resume=False, resume_from=\'\')\\n","        config[\'mfiles\'] = \',\'.join(config[\'mfiles\'])\\n","        config[\'num_neighbors\'] = \'-\'.join(map(str, config[\'num_neighbors\']))\\n","        artifact = RUN_ROOT / \'artifacts\' / stem\\n","        config[\'artifact_dir\'] = str(artifact)\\n","        config[\'graph_cache_dir\'] = str(GRAPH_CACHE_ROOT / key)\\n","        target = RUN_ROOT / \'configs\' / f\'{stem}.yaml\'\\n","        target.write_text(yaml.safe_dump(config, sort_keys=False), encoding=\'utf-8\')\\n","        shutil.copy2(original,RUN_ROOT/\\"configs\\"/f\\"{stem}_original.yaml\\")\\n","        CONFIG_PATHS[key], ARTIFACT_PATHS[key] = target, artifact\\n","        LOG_PATHS[key] = RUN_ROOT / \'logs\' / f\'{stem}.log\'\\n","\\n","\\n","    def build_command(key):\\n","        if key not in SELECTED_DATASETS:\\n","            raise ValueError(f\'{key} is not selected; edit SELECTED_DATASETS and rerun setup.\')\\n","        return [sys.executable, \'-u\', str(REPO_ROOT / \'main_stair_icaart.py\'),\\n","                \'--config\', str(CONFIG_PATHS[key]), \'--seed\', str(SEED),\\n","                \'--root\', str(DATA_ROOT), \'--dataset\', DATASET_FOLDERS[key],\\n","                \'--device\', \'0\', \'--ranking\', \'full\',\\n","                \'--artifact-dir\', str(ARTIFACT_PATHS[key]),\\n","                \'--graph-cache-dir\', str(GRAPH_CACHE_ROOT / key)]\\n","\\n","\\n","    config_probe = r\\"\\"\\"\\n","    import json, sys\\n","    from main_stair_icaart import build_config\\n","    cfg = build_config(sys.argv[1:])\\n","    keys = json.loads(__import__(\'os\').environ[\'ECLIPSE_CHECK_KEYS\'])\\n","    receipt = {key: getattr(cfg, key) for key in keys}\\n","    print(\'ECLIPSE_CONFIG=\' + json.dumps(receipt))\\n","    \\"\\"\\"\\n","    config_probe = __import__(\'textwrap\').dedent(config_probe)\\n","    for key in SELECTED_DATASETS:\\n","        expected = dict(SEED1_REFERENCE[key][\'v8_config\'])\\n","        expected.update(SEED1_REFERENCE[key][\'icaart_config\'])\\n","        expected[\'eta\'] = 0.1\\n","        expected[\'evidence_mode\'] = \'pooled_global_excess\'\\n","        expected[\'evidence_activity_bins\'] = \'none\'\\n","        expected[\'evidence_delta\'] = 1.0\\n","        expected.update(seed=SEED, dataset=DATASET_FOLDERS[key], ranking=\'full\', which4best=\'NDCG@20\',\\n","                        eval_valid=True, eval_test=False, resume=False, resume_from=\'\',\\n","                        num_workers=SEED1_REFERENCE[key][\'contract\'][\'config\'][\'num_workers\'])\\n","        probe_env = ENV.copy()\\n","        probe_env[\'ECLIPSE_CHECK_KEYS\'] = json.dumps(list(expected))\\n","        result = subprocess.run([sys.executable, \'-c\', config_probe, *build_command(key)[3:]],\\n","                                cwd=REPO_ROOT, env=probe_env, text=True, encoding=\'utf-8\', capture_output=True)\\n","        if result.returncode:\\n","            print(result.stdout, result.stderr)\\n","            result.check_returncode()\\n","        payload = next(line.split(\'=\', 1)[1] for line in result.stdout.splitlines()\\n","                       if line.startswith(\'ECLIPSE_CONFIG=\'))\\n","        resolved = json.loads(payload)\\n","        changes = {name: {\'expected\': value, \'actual\': resolved.get(name)}\\n","                   for name, value in expected.items() if resolved.get(name) != value}\\n","        if changes:\\n","            raise RuntimeError(f\'{key}: resolved CLI/YAML protocol mismatch: {changes}\')\\n","        write_json(RUN_ROOT / \'provenance\' / f\'{key}_resolved_config.json\', resolved)\\n","        print(f\'{key}: seed={resolved[\\"seed\\"]}, epochs={resolved[\\"epochs\\"]}, \'\\n","              f\'batch={resolved[\\"batch_size\\"]}, gamma={resolved[\\"gamma\\"]}, \'\\n","              f\'weight_decay={resolved[\\"weight_decay\\"]}, cl_chunk={resolved[\\"cl_chunk_size\\"]}\')\\n","    write_json(RUN_ROOT / \'run_manifest.json\', {\\n","        \'method\': \'ECLIPSE\', \'arm\': ARM, \'arm_id\': ARM, \'status\':\'READY\', \'training_seed\': SEED,\\n","        \'datasets\': list(SELECTED_DATASETS), \'session_id\': SESSION_ID,\\n","        \'reference_commit\': RUNTIME_COMMIT, \'actual_commit\': ACTUAL_COMMIT,\\n","        \'source_overlay_id\': GLOBAL_OVERLAY_ID, \'declared_source_changes\': DECLARED_SOURCE_CHANGES,\\n","        \'runtime\': RUNTIME_ENV, \'runtime_deviations\': deviations,\\n","        \'selection\': \'validation NDCG@20\', \'test_policy\': \'report only VALID-selected TEST; native terminal TEST may also be evaluated\',\\n","        \'configs\': {k: {\'path\': str(p), \'sha256\': file_sha256(p)} for k, p in CONFIG_PATHS.items()},\\n","        \'dataset_paths\': DATA_RECEIPTS,\\n","        \'note\': \'Global-full null: pooled_global_excess, delta1/tau2/halves0; eta.1/LC.01 retained; no resume.\',\\n","    })\\n","\\n","else:\\n","    print(\'report_only: skipped training setup; no model/network/GPU invocation.\')\\n"],"execution_count":null,"outputs":[]},{"cell_type":"markdown","id":"223ae2df","metadata":{},"source":["## 6. Global-full guards and output engine\\n","Checks selected TEST/seed/config/status, 500epochs, objective identity, source extension, invariant candidates/counts/CF, one active-user pool, null formula audit and score range. Null/assignment/score/final operator are allowed to differ from activity-conditioned Full.\\n"]},{"cell_type":"code","id":"167558fb","metadata":{},"source":["import math\\n","import re\\n","import signal\\n","import time\\n","import pandas as pd\\n","import numpy as np\\n","from IPython.display import display\\n","\\n","RESULTS = {}\\n","\\n","\\n","def read_jsonl(path):\\n","    path = Path(path)\\n","    if not path.is_file():\\n","        return []\\n","    result = []\\n","    with path.open(encoding=\'utf-8\') as handle:\\n","        for number, line in enumerate(handle, 1):\\n","            if line.strip():\\n","                try:\\n","                    result.append(json.loads(line))\\n","                except json.JSONDecodeError as exc:\\n","                    raise ValueError(f\'Malformed JSONL: {path}, line {number}\') from exc\\n","    return result\\n","\\n","\\n","def compare_manifest(key, manifest, expected_seed):\\n","    ref=SEED1_REFERENCE[key];problems={}\\n","    if manifest.get(\'seed\')!=expected_seed:problems[\'seed\']=manifest.get(\'seed\')\\n","    if manifest.get(\'selection\')!=\'validation NDCG@20\':problems[\'selection\']=manifest.get(\'selection\')\\n","    for name,expected in dict(ref[\'v8_config\'],eta=0.1).items():\\n","        if manifest.get(\'v8_config\',{}).get(name)!=expected:problems[\'v8_config.\'+name]=manifest.get(\'v8_config\',{}).get(name)\\n","    if manifest.get(\'icaart_config\')!=dict(ref[\'icaart_config\'],evidence_mode=\'pooled_global_excess\',evidence_activity_bins=\'none\'):problems[\'evidence_config\']=manifest.get(\'icaart_config\')\\n","    contract=manifest.get(\'checkpoint_contract\',{})\\n","    for name in (\'splits\',\'candidate_hashes\',\'data_fingerprint\'):\\n","        if contract.get(name)!=ref[\'contract\'][name]:problems[name]=contract.get(name)\\n","    for name,expected in ref[\'contract\'][\'config\'].items():\\n","        if name==\'seed\':expected=expected_seed\\n","        if name==\'eta\':expected=0.1\\n","        if contract.get(\'config\',{}).get(name)!=expected:problems[\'config.\'+name]=contract.get(\'config\',{}).get(name)\\n","    if manifest.get(\'features\')!=ref[\'features\']:problems[\'features\']=manifest.get(\'features\')\\n","    if manifest.get(\'precision\')!={\'tf32\':False,\'amp\':False}:problems[\'precision\']=manifest.get(\'precision\')\\n","    e=manifest.get(\'icaart_manifest\',{});g=manifest.get(\'graph\',{});d=g.get(\'icaart\',{})\\n","    expected_identity={\'null_scope\':\'global_full\',\'score_kind\':\'pooled_global_excess\',\\n","        \'active_partition\':False,\'active_halves\':0,\'normalization\':\'actual_degree\'}\\n","    checkpoint_estimator=contract.get(\'stair_icaart\',{})\\n","    for label, snapshot in ((\'root\',manifest),(\'checkpoint\',checkpoint_estimator),\\n","                             (\'checkpoint_manifest\',checkpoint_estimator.get(\'manifest\',{})),(\'diagnostics\',d)):\\n","        for name,expected in expected_identity.items():\\n","            if snapshot.get(name)!=expected:problems[label+\'.\'+name]=snapshot.get(name)\\n","    for name in (\'num_active_train_users\',\'num_inactive_train_users\',\'num_null_groups\',\'num_activity_strata\',\\n","                 \'activity_conditioning\',\'evidence_activity_bins\',\'counts_hash\',\'null_expectation_hash\',\\n","                 \'global_null_formula\',\'global_null_max_abs_formula_error\'):\\n","        if d.get(name)!=e.get(name):problems[\'diagnostic_consistency.\'+name]=d.get(name)\\n","    for name,expected in ref[\'evidence_invariants\'].items():\\n","        if name in (\'assignment_hash\',\'group_sizes\',\'null_expectation_hash\'):continue\\n","        if e.get(name)!=expected:problems[\'evidence.\'+name]=e.get(name)\\n","    if e.get(\'training_seed\')!=expected_seed:problems[\'evidence.training_seed\']=e.get(\'training_seed\')\\n","    for name,actual in ((\'graph.eta\',g.get(\'eta\')),(\'graph.effective_eta\',g.get(\'effective_eta\')),(\'icaart.effective_eta\',d.get(\'effective_eta\'))):\\n","        if actual!=0.1:problems[name]=actual\\n","    if e.get(\'null_scope\')!=\'global_full\' or e.get(\'normalization\')!=\'actual_degree\':problems[\'null_normalization\']=e\\n","    if e.get(\'evidence_mode\')!=\'pooled_global_excess\' or e.get(\'score_kind\')!=\'pooled_global_excess\':problems[\'score_identity\']=e.get(\'score_kind\')\\n","    if e.get(\'pooled_regularizer\')!=4.0 or e.get(\'active_partition\') is not False or e.get(\'active_halves\')!=0:problems[\'pooling_identity\']=e\\n","    if d.get(\'delta\')!=1.0:problems[\'diagnostic_delta\']=d.get(\'delta\')\\n","    score_hash=e.get(\'evidence_score_hash\')\\n","    if not isinstance(score_hash,str) or re.fullmatch(r\'[0-9a-f]{64}\',score_hash) is None:problems[\'global_score_hash\']=score_hash\\n","    active_expected=sum(ref[\'evidence_invariants\'][\'group_sizes\'])\\n","    if e.get(\'num_active_train_users\')!=active_expected:problems[\'M_active\']=e.get(\'num_active_train_users\')\\n","    if e.get(\'num_inactive_train_users\')!=e.get(\'num_users\',0)-active_expected:problems[\'inactive_users\']=e.get(\'num_inactive_train_users\')\\n","    if e.get(\'group_sizes\')!=[active_expected]:problems[\'global_pool\']=e.get(\'group_sizes\')\\n","    if e.get(\'num_null_groups\')!=1 or e.get(\'num_activity_strata\')!=0 or e.get(\'activity_conditioning\') is not False:\\n","        problems[\'activity_conditioning\']=\'Global-full must contain one non-stratified active-user pool\'\\n","    if e.get(\'evidence_activity_bins\')!=\'none\':problems[\'activity_bins\']=e.get(\'evidence_activity_bins\')\\n","    if e.get(\'global_null_formula\')!=\'n_i*n_j/M_active; binary TRAIN item marginals\':problems[\'null_formula\']=e.get(\'global_null_formula\')\\n","    error=e.get(\'global_null_max_abs_formula_error\')\\n","    if isinstance(error,bool) or not isinstance(error,(int,float)) or not math.isfinite(error) or error<0:\\n","        problems[\'global_null_formula_check\']=\'missing/invalid candidate-wise formula check\'\\n","    for name in (\'null_expectation_hash\',\'assignment_hash\'):\\n","        if not isinstance(e.get(name),str) or re.fullmatch(r\'[0-9a-f]{64}\',e[name]) is None:problems[name]=e.get(name)\\n","    sq=d.get(\'evidence_score_quantiles\')\\n","    if not sq or any(isinstance(v,bool) or not isinstance(v,(int,float)) or not math.isfinite(v) or not 0<=v<1 for v in sq):\\n","        problems[\'global_excess_score_range\']=sq\\n","    fp=contract.get(\'graph_fingerprint\')\\n","    if not isinstance(fp,str) or len(fp)!=64:problems[\'final_graph_fingerprint\']=fp\\n","    for name,actual in ((\'graph\',g.get(\'graph_fingerprint\')),(\'icaart_manifest\',e.get(\'graph_fingerprint\')),(\'diagnostics\',d.get(\'s_ica_fingerprint\'))):\\n","        if actual!=fp:problems[\'final_graph_consistency.\'+name]=actual\\n","    if problems:raise RuntimeError(f\'{key}: Global-full null comparability failed: {problems}\')\\n","    source_diff={k:{\'reference\':v,\'actual\':contract.get(\'sources\',{}).get(k)} for k,v in EXPECTED_RUNTIME_SOURCES.items() if contract.get(\'sources\',{}).get(k)!=v}\\n","    runtime_diff={k:{\'reference\':ref[k],\'actual\':manifest.get(k)} for k in (\'torch\',\'freerec\',\'gpu\') if manifest.get(k)!=ref[k]}\\n","    if str(manifest.get(\'python\',\'\')).split(\'.\')[:2]!=ref[\'python\'].split(\'.\')[:2]:runtime_diff[\'python\']=manifest.get(\'python\')\\n","    if RUN_MODE==\'train\' and source_diff:raise RuntimeError(f\'Source mismatch: {source_diff}\')\\n","    if RUN_MODE==\'train\' and STRICT_RUNTIME and runtime_diff:raise RuntimeError(f\'Runtime mismatch: {runtime_diff}\')\\n","    return {\'status\':\'MATCH_WITH_REVIEW_REQUIRED\' if source_diff or runtime_diff else \'MATCH\',\\n","        \'arm\':ARM,\'training_seed\':expected_seed,\'model_intervention\':{\'null_scope\':[\'full_activity\',\'global_full\'],\'evidence_mode\':[\'pooled_activity_excess\',\'pooled_global_excess\']},\'declared_source_changes\':DECLARED_SOURCE_CHANGES,\'source_overlay_id\':GLOBAL_OVERLAY_ID,\\n","        \'same_candidates_raw_weights_counts_cf\':True,\'same_activity_null\':\'NOT_REQUIRED\',\'global_scores_may_differ\':True,\'same_final_graph_as_full\':\'NOT_REQUIRED\',\\n","        \'dose_reference\':\'uncalibrated weighted semantic+CSE eta0.1\',\'source_deviations\':source_diff,\'runtime_deviations\':runtime_diff,\\n","        \'operator_check_scope\':\'manifest and pinned source contract; not an independent sparse-tensor reconstruction\',\\n","        \'cf_propagation_retained\':True}\\n","\\n","\\n","def validate_history(telemetry, evaluations, receipt):\\n","    if [row.get(\'epoch\') for row in telemetry]!=list(range(1,501)):raise ValueError(\'Require exactly ordered epochs1..500; no duplicate/missing epochs or implicit resume\')\\n","    for row in telemetry:\\n","        for name in (\'loss\',\'bpr_loss\',\'nlgcl_loss\',\'weighted_nlgcl_loss\',\'seconds\',\'peak_allocated_gib\',\'peak_reserved_gib\'):\\n","            value=row.get(name)\\n","            if isinstance(value,bool) or not isinstance(value,(int,float)) or not math.isfinite(value) or value<0:raise ValueError(f\'Invalid {name} at epoch {row[\\"epoch\\"]}\')\\n","        if row.get(\'lambda_nlgcl\')!=.01:raise ValueError(\'LC must stay enabled\')\\n","        if not math.isclose(row[\'weighted_nlgcl_loss\'],.01*row[\'nlgcl_loss\'],rel_tol=1e-6,abs_tol=1e-7):raise ValueError(\'Weighted LC mismatch\')\\n","        if not math.isclose(row[\'loss\'],row[\'bpr_loss\']+row[\'weighted_nlgcl_loss\'],rel_tol=1e-6,abs_tol=1e-7):raise ValueError(\'Objective mismatch\')\\n","    valid=[r for r in evaluations if r.get(\'mode\')==\'valid\' and not r.get(\'selected_checkpoint\')]\\n","    if not valid:raise ValueError(\'Missing VALID history\')\\n","    seen=set()\\n","    for row in valid:\\n","        epoch=row.get(\'epoch\');v=row.get(\'metrics\',{}).get(\'NDCG@20\')\\n","        if isinstance(epoch,bool) or not isinstance(epoch,int) or not 0<=epoch<=500 or epoch in seen:raise ValueError(\'Invalid/duplicate VALID epoch\')\\n","        seen.add(epoch)\\n","        if isinstance(v,bool) or not isinstance(v,(int,float)) or not math.isfinite(v):raise ValueError(\'Invalid VALID NDCG@20\')\\n","    maximum=max(r[\'metrics\'][\'NDCG@20\'] for r in valid)\\n","    tied=[r[\'epoch\'] for r in valid if r[\'metrics\'][\'NDCG@20\']==maximum]\\n","    if receipt[\'epoch\'] not in tied:raise ValueError(\'Selected epoch is not a VALID maximum\')\\n","    selected=[r for r in evaluations if r.get(\'mode\')==\'test\' and r.get(\'selected_checkpoint\') is True]\\n","    if len(selected)!=1 or selected[0]!=receipt:raise ValueError(\'Selected receipt/history mismatch\')\\n","    return {\'valid_max_epochs\':tied,\'tie_rule\':\'native receipt retained; inspect native log if tied\' if len(tied)>1 else \'unique maximum\',\\n","        \'test_events\':[{\'epoch\':r.get(\'epoch\'),\'selected\':r.get(\'selected_checkpoint\')} for r in evaluations if r.get(\'mode\')==\'test\']}\\n","\\n","\\n","def validate_receipt(key, receipt, manifest, expected_seed):\\n","    if (receipt.get(\'mode\') != \'test\' or receipt.get(\'selected_checkpoint\') is not True\\n","            or receipt.get(\'selection\') != \'validation NDCG@20\'\\n","            or receipt.get(\'seed\') != expected_seed\\n","            or receipt.get(\'evidence_mode\') != \'pooled_global_excess\'\\n","            or receipt.get(\'evidence_delta\') != 1.0):\\n","        raise ValueError(f\'{key}: not a selected-checkpoint ECLIPSE Global-full null TEST receipt\')\\n","    if receipt.get(\'graph_fingerprint\') != manifest.get(\'checkpoint_contract\', {}).get(\'graph_fingerprint\'):\\n","        raise ValueError(f\'{key}: metric receipt belongs to another operator\')\\n","    epoch = receipt.get(\'epoch\')\\n","    if isinstance(epoch, bool) or not isinstance(epoch, int) or not 1 <= epoch <= 500:\\n","        raise ValueError(f\'{key}: invalid selected epoch\')\\n","    normalized = receipt.get(\'metrics\', {})\\n","    for metric in PRIMARY_METRICS:\\n","        value = normalized.get(metric)\\n","        if isinstance(value, bool) or not isinstance(value, (float, int)) or not math.isfinite(value) or not 0 <= value <= 1:\\n","            raise ValueError(f\'{key}: missing/invalid measured TEST metric {metric}\')\\n","    return normalized\\n","\\n","\\n","def export_run_tables(key):\\n","    stem=stem_for(key)\\n","    artifact = ARTIFACT_PATHS[key]\\n","    manifest = json.loads((artifact / \'manifest.json\').read_text(encoding=\'utf-8\'))\\n","    status = json.loads((artifact / \'run_status.json\').read_text(encoding=\'utf-8\'))\\n","    if status.get(\'status\') != \'COMPLETED\' or status.get(\'seed\') != SEED:\\n","        raise RuntimeError(f\'{key}: run is not completed for training seed {SEED}\')\\n","    comparability = compare_manifest(key, manifest, SEED)\\n","    receipt = json.loads((artifact / \'selected_metrics.json\').read_text(encoding=\'utf-8\'))\\n","    metrics = validate_receipt(key, receipt, manifest, SEED)\\n","    telemetry = pd.DataFrame(read_jsonl(artifact / \'training_telemetry.jsonl\'))\\n","    evaluations = read_jsonl(artifact / \'evaluation_metrics.jsonl\')\\n","    history = validate_history(telemetry.to_dict(\'records\'), evaluations, receipt)\\n","    checkpoint = artifact / \'best_model.pt\'\\n","    if not checkpoint.is_file(): raise FileNotFoundError(checkpoint)\\n","    telemetry.to_csv(RUN_ROOT / \'tables\' / f\'{stem}_training_telemetry.csv\', index=False, na_rep=\'NOT_MEASURED\')\\n","    evaluation_rows = [{k: v for k, v in record.items() if k != \'metrics\'} |\\n","                       {str(k).upper(): v for k, v in record.get(\'metrics\', {}).items()}\\n","                       for record in evaluations]\\n","    pd.DataFrame(evaluation_rows).to_csv(RUN_ROOT / \'tables\' / f\'{stem}_evaluations.csv\',\\n","                                       index=False, na_rep=\'NOT_MEASURED\')\\n","    log_text = LOG_PATHS[key].read_text(encoding=\'utf-8\', errors=\'replace\') if LOG_PATHS[key] and LOG_PATHS[key].is_file() else \'\'\\n","    fit_matches = re.findall(r\'Coach\\\\.fit takes\\\\s+([0-9.eE+-]+)\\\\s+seconds\', log_text)\\n","    summary = {\'method\': \'ECLIPSE\', \'arm\': ARM, \'dataset\': key, \'seed\': SEED,\\n","               \'selected_epoch\': receipt[\'epoch\'], \'split\': \'TEST\', **metrics,\\n","               \'fit_seconds\': float(fit_matches[-1]) if fit_matches else None,\\n","               \'train_epoch_seconds_sum\': float(telemetry[\'seconds\'].sum()),\\n","               \'preprocessing_seconds\': manifest.get(\'preprocessing_seconds\'),\\n","               \'peak_training_allocated_mib\': float(telemetry[\'peak_allocated_gib\'].max() * 1024),\\n","               \'peak_training_reserved_mib\': float(telemetry[\'peak_reserved_gib\'].max() * 1024),\\n","               \'memory_scope\': \'child-process per-training-epoch CUDA allocator peaks; excludes evaluation peaks\',\\n","               \'commit\': manifest.get(\'commit\'), \'graph_fingerprint\': receipt[\'graph_fingerprint\'],\\n","               \'artifact_dir\': str(artifact), \'checkpoint_path\': str(artifact / \'best_model.pt\'),\\n","               \'runtime_deviations\': comparability[\'runtime_deviations\'], \'comparability\': comparability}\\n","    write_json(RUN_ROOT / \'tables\' / f\'{stem}_selected_test.json\', summary)\\n","    write_json(RUN_ROOT / \'tables\' / f\'{stem}_mechanism.json\', manifest[\'graph\'][\'icaart\'])\\n","    execution_path=RUN_ROOT/\'provenance\'/f\'{stem}_notebook_execution.json\'\\n","    execution=json.loads(execution_path.read_text(encoding=\'utf-8\')) if execution_path.is_file() else {}\\n","    summary.update(status=\'COMPLETED_PENDING_HUMAN_REVIEW\', attempt_id=SESSION_ID,\\n","       checkpoint_sha256=file_sha256(checkpoint), history=history,\\n","       subprocess_wall_seconds=execution.get(\'subprocess_wall_seconds\'),\\n","       preprocessing_cache_hit=manifest.get(\'graph\',{}).get(\'cache_hit\'), reviewed_by=None)\\n","    pd.DataFrame([{k:v for k,v in summary.items() if not isinstance(v,(dict,list))}]).to_csv(\\n","        RUN_ROOT/\'tables\'/f\'{stem}_selected_test.csv\',index=False,na_rep=\'NOT_MEASURED\')\\n","    write_json(RUN_ROOT/\'tables\'/f\'{stem}_selected_test.json\',summary)\\n","    write_json(RUN_ROOT/\'provenance\'/f\'{stem}_comparability.json\',comparability)\\n","    write_json(RUN_ROOT/\'provenance\'/f\'{stem}_input_receipts.json\',{\\n","       name:file_sha256(artifact/name) for name in (\'manifest.json\',\'run_status.json\',\'selected_metrics.json\',\'training_telemetry.jsonl\',\'evaluation_metrics.jsonl\',\'best_model.pt\')})\\n","    write_json(RUN_ROOT/\'tables\'/f\'{stem}_diagnostics.json\',{\'score_kind\':\'pooled_global_excess\',\'score_formula\':\'[c-mu_global]+/(c+mu_global+4)\',\'reference\':\'uncalibrated weighted semantic+CSE eta0.1\',\'diagnostics\':manifest[\'graph\'][\'icaart\']})\\n","    write_json(RUN_ROOT/\'tables\'/f\'{stem}_global_null_audit.json\',\\n","        {name:manifest[\'icaart_manifest\'].get(name) for name in (\'null_scope\',\'activity_conditioning\',\\n","          \'num_active_train_users\',\'num_inactive_train_users\',\'num_null_groups\',\'num_activity_strata\',\\n","          \'global_null_formula\',\'global_null_max_abs_formula_error\',\'counts_hash\',\'null_expectation_hash\')})\\n","    RESULTS[key] = summary\\n","    return summary\\n","\\n","\\n","def train_dataset(key):\\n","    stem=stem_for(key)\\n","    if key not in SELECTED_DATASETS:\\n","        print(f\'Skipped {key}: not in SELECTED_DATASETS.\')\\n","        return None\\n","    if RUN_MODE == \'report_only\': return export_run_tables(key)\\n","    artifact = ARTIFACT_PATHS[key]\\n","    status_path = artifact / \'run_status.json\'\\n","    if status_path.is_file():\\n","        status = json.loads(status_path.read_text(encoding=\'utf-8\'))\\n","        if status.get(\'status\') == \'COMPLETED\':\\n","            print(f\'{key}: reading the completed attempt; no retraining.\')\\n","            return export_run_tables(key)\\n","    # build_config() creates artifact_dir during CLI preflight, before training.\\n","    # An empty real directory with no stdout log is not a previous run.\\n","    log_path = LOG_PATHS[key]\\n","    invalid_directory = artifact.is_symlink() or (artifact.exists() and not artifact.is_dir())\\n","    has_payload = artifact.is_dir() and any(artifact.iterdir())\\n","    has_log = log_path.exists() or log_path.is_symlink()\\n","    if invalid_directory or has_payload or has_log:\\n","        raise FileExistsError(f\'Existing attempt payload/log preserved at {artifact}; \'\\n","                              \'use report_only for completed runs, or create a new SESSION_ID for a fresh attempt. \'\\n","                              \'No automatic resume/delete/retry.\')\\n","    artifact.mkdir(parents=True, exist_ok=True)\\n","    command = build_command(key)\\n","    write_json(RUN_ROOT / \'provenance\' / f\'{stem}_command.json\', {\\n","        \'argv\': command, \'training_seed\': SEED, \'arm_id\': ARM, \'dataset\': key, \'purpose\':\'evidence_control_global_full_null\', \'pythonhashseed\': ENV[\'PYTHONHASHSEED\'],\\n","        \'cuda_visible_devices\': ENV[\'CUDA_VISIBLE_DEVICES\'], \'cwd\': str(REPO_ROOT)})\\n","    print(\'Running:\', \' \'.join(command))\\n","    start = time.perf_counter()\\n","    manifest_checked = False\\n","    process = None\\n","    try:\\n","        with LOG_PATHS[key].open(\'x\', encoding=\'utf-8\') as log:\\n","            process = subprocess.Popen(command, cwd=REPO_ROOT, env=ENV, stdout=subprocess.PIPE,\\n","                                       stderr=subprocess.STDOUT, text=True, encoding=\'utf-8\',\\n","                                       errors=\'replace\', bufsize=1)\\n","            for line in process.stdout:\\n","                log.write(line)\\n","                log.flush()\\n","                print(line, end=\'\', flush=True)\\n","                manifest_path = artifact / \'manifest.json\'\\n","                if not manifest_checked and manifest_path.is_file():\\n","                    comparability = compare_manifest(key, json.loads(manifest_path.read_text(encoding=\'utf-8\')), SEED)\\n","                    write_json(RUN_ROOT / \'provenance\' / f\'{key}_comparability.json\', comparability)\\n","                    manifest_checked = True\\n","            process.wait()\\n","            process.stdout.close()\\n","        if process.returncode != 0:\\n","            raise RuntimeError(f\'{key}: training failed with exit code {process.returncode}; see {LOG_PATHS[key]}\')\\n","        child_wall_seconds=time.perf_counter()-start\\n","        write_json(RUN_ROOT/\'provenance\'/f\'{stem}_notebook_execution.json\',{\'status\':\'CHILD_COMPLETED_PENDING_EXPORT\',\'subprocess_wall_seconds\':child_wall_seconds})\\n","        result = export_run_tables(key)\\n","        write_json(RUN_ROOT / \'provenance\' / f\'{stem}_notebook_execution.json\', {\\n","            \'status\': \'COMPLETED\', \'subprocess_wall_seconds\': child_wall_seconds,\\n","            \'exit_code\': process.returncode, \'training_seed\': SEED})\\n","        print(f\'\\\\nCompleted {DATASET_TITLES[key]} | selected TEST NDCG@20={result[\\"NDCG@20\\"]:.10f}\')\\n","        return result\\n","    except BaseException as exc:\\n","        if process is not None and process.poll() is None:\\n","            if isinstance(exc, KeyboardInterrupt):\\n","                process.send_signal(signal.SIGINT)\\n","            else:\\n","                process.terminate()\\n","            try:\\n","                process.wait(timeout=20)\\n","            except subprocess.TimeoutExpired:\\n","                process.kill()\\n","                process.wait(timeout=10)\\n","        write_json(RUN_ROOT / \'provenance\' / f\'{stem}_notebook_execution.json\', {\\n","            \'status\': \'INTERRUPTED\' if isinstance(exc, KeyboardInterrupt) else \'FAILED\',\\n","            \'error_type\': type(exc).__name__, \'error\': str(exc), \'training_seed\': SEED,\\n","            \'subprocess_wall_seconds\': time.perf_counter() - start})\\n","        raise\\n"],"execution_count":null,"outputs":[]},{"cell_type":"markdown","id":"72f2942a","metadata":{},"source":["## 7. Figure utilities — PDF vector / PNG300dpi / measured data / captions\\n","Training objective, VALID ranking, training allocator/time and Global-full excess-score diagnostics. No TEST learning-curve selection or phantom error bars. Positive coverage means c exceeds the global active-user null, not merely c>0. Missing diagnostics remain NOT_MEASURED.\\n"]},{"cell_type":"code","id":"3d75ad7a","metadata":{},"source":["import matplotlib.pyplot as plt\\n","from matplotlib.patches import FancyBboxPatch\\n","\\n","plt.rcParams.update({\'font.family\': \'DejaVu Sans\', \'font.size\': 9, \'axes.titlesize\': 10,\\n","                     \'axes.labelsize\': 9, \'legend.fontsize\': 8, \'xtick.labelsize\': 8,\\n","                     \'ytick.labelsize\': 8, \'lines.linewidth\': 1.5,\\n","                     \'pdf.fonttype\': 42, \'ps.fonttype\': 42, \'savefig.dpi\': 300,\\n","                     \'axes.spines.top\': False, \'axes.spines.right\': False})\\n","COLORS = {\'blue\': \'#0072B2\', \'orange\': \'#D55E00\', \'green\': \'#009E73\',\\n","          \'purple\': \'#CC79A7\', \'gray\': \'#666666\'}\\n","_existing_figure_manifest = RUN_ROOT / \'figures\' / \'figure_manifest.json\'\\n","FIGURE_RECORDS = json.loads(_existing_figure_manifest.read_text(encoding=\'utf-8\')) if _existing_figure_manifest.is_file() else []\\n","\\n","\\n","def save_figure(fig, stem, caption, sources):\\n","    for extension in (\'pdf\', \'png\'):\\n","        fig.savefig(RUN_ROOT / \'figures\' / f\'{stem}.{extension}\', bbox_inches=\'tight\', facecolor=\'white\')\\n","    plt.show()\\n","    plt.close(fig)\\n","    FIGURE_RECORDS[:] = [record for record in FIGURE_RECORDS if record[\'stem\'] != stem]\\n","    FIGURE_RECORDS.append({\'stem\': stem, \'caption\': caption, \'sources\': list(map(str, sources))})\\n","    write_json(RUN_ROOT / \'figures\' / \'figure_manifest.json\', FIGURE_RECORDS)\\n","    (RUN_ROOT/\'figures\'/\'FIGURE_CAPTIONS.md\').write_text(\'# Figure captions\\\\n\\\\n\'+\'\\\\n\\\\n\'.join(\'## \'+r[\'stem\']+\'\\\\n\\\\n\'+r[\'caption\'] for r in FIGURE_RECORDS),encoding=\'utf-8\')\\n","    print(caption)\\n","\\n","\\n","def completed_artifact(key):\\n","    if key not in SELECTED_DATASETS:\\n","        print(f\'{key}: not selected; no figure generated.\')\\n","        return None\\n","    path = ARTIFACT_PATHS[key]\\n","    status = path / \'run_status.json\'\\n","    if not status.is_file() or json.loads(status.read_text(encoding=\'utf-8\')).get(\'status\') != \'COMPLETED\':\\n","        print(f\'{key}: no completed run available; no figure generated.\')\\n","        return None\\n","    return path\\n","\\n","\\n","def plot_learning_curves(key):\\n","    stem=stem_for(key)\\n","    artifact = completed_artifact(key)\\n","    if artifact is None:\\n","        return\\n","    telemetry = pd.DataFrame(read_jsonl(artifact / \'training_telemetry.jsonl\'))\\n","    records = [record for record in read_jsonl(artifact / \'evaluation_metrics.jsonl\')\\n","               if record.get(\'mode\') == \'valid\' and not record.get(\'selected_checkpoint\')]\\n","    selected = json.loads((artifact / \'selected_metrics.json\').read_text(encoding=\'utf-8\'))[\'epoch\']\\n","    valid = pd.DataFrame([{\'epoch\': record[\'epoch\'],\\n","                           **{str(k).upper(): v for k, v in record[\'metrics\'].items()}}\\n","                          for record in records]).sort_values(\'epoch\')\\n","    if telemetry.empty or valid.empty:\\n","        print(f\'{key}: missing measured curves; no substitute data generated.\')\\n","        return\\n","    fig, axes = plt.subplots(1, 3, figsize=(7.2, 3.0), constrained_layout=True)\\n","    for column, label, color in [(\'loss\', \'Total\', COLORS[\'blue\']),\\n","                                  (\'bpr_loss\', \'BPR\', COLORS[\'orange\']),\\n","                                  (\'weighted_nlgcl_loss\', \'Weighted LC\', COLORS[\'green\'])]:\\n","        if column in telemetry:\\n","            axes[0].plot(telemetry[\'epoch\'], telemetry[column], label=label, color=color)\\n","    axes[0].set(xlabel=\'Training epoch\', ylabel=\'Mean training loss\', title=\'Training objective\')\\n","    axes[0].legend(frameon=False)\\n","    for ax, metric in zip(axes[1:], (\'NDCG@20\', \'RECALL@20\')):\\n","        if metric in valid:\\n","            ax.plot(valid[\'epoch\'], valid[metric], color=COLORS[\'blue\'], label=f\'VALID {metric}\')\\n","            point = valid.loc[valid[\'epoch\'] == selected, metric]\\n","            if len(point):\\n","                ax.scatter([selected], [point.iloc[-1]], color=COLORS[\'orange\'], s=22, zorder=3)\\n","        ax.axvline(selected, color=COLORS[\'gray\'], linestyle=\'--\', linewidth=1, label=f\'Selected epoch {selected}\')\\n","        ax.set(xlabel=\'Training epoch\', ylabel=f\'Validation {metric}\', title=f\'Validation {metric}\')\\n","        ax.legend(frameon=False)\\n","    for ax in axes:\\n","        ax.grid(alpha=.2)\\n","    fig.suptitle(f\'ECLIPSE Global-full null | {DATASET_TITLES[key]} | seed {SEED}\', fontsize=11)\\n","    save_figure(fig, f\'{stem}_learning_curves\',\\n","                f\'{DATASET_TITLES[key]}, Global-full null, training seed {SEED}. Training objective and validation ranking curves. The checkpoint is selected by validation NDCG@20; \'\\n","                \'TEST is evaluated at the selected checkpoint and is not a learning-curve selection signal.\',\\n","                [artifact / \'training_telemetry.jsonl\', artifact / \'evaluation_metrics.jsonl\', artifact / \'selected_metrics.json\'])\\n","\\n","\\n","def plot_training_memory(key):\\n","    stem=stem_for(key)\\n","    artifact = completed_artifact(key)\\n","    if artifact is None:\\n","        return\\n","    telemetry = pd.DataFrame(read_jsonl(artifact / \'training_telemetry.jsonl\'))\\n","    if telemetry.empty:\\n","        return\\n","    fig, axes = plt.subplots(1, 2, figsize=(7.2, 2.8), constrained_layout=True)\\n","    for column, label, color in [(\'peak_allocated_gib\', \'Peak allocated\', COLORS[\'blue\']),\\n","                                  (\'peak_reserved_gib\', \'Peak reserved\', COLORS[\'orange\'])]:\\n","        if column in telemetry:\\n","            axes[0].plot(telemetry[\'epoch\'], telemetry[column] * 1024, label=label, color=color)\\n","    axes[0].set(xlabel=\'Training epoch\', ylabel=\'CUDA allocator memory (MiB)\', title=\'Training-epoch allocator peaks\')\\n","    axes[0].legend(frameon=False)\\n","    axes[1].plot(telemetry[\'epoch\'], telemetry[\'seconds\'], color=COLORS[\'green\'])\\n","    axes[1].set(xlabel=\'Training epoch\', ylabel=\'Training time (s/epoch)\', title=\'Training time per epoch\')\\n","    for ax in axes:\\n","        ax.grid(alpha=.2)\\n","    fig.suptitle(f\'ECLIPSE Global-full null | {DATASET_TITLES[key]} | seed {SEED}\', fontsize=11)\\n","    save_figure(fig, f\'{stem}_training_memory\',\\n","                f\'{DATASET_TITLES[key]}, Global-full null, training seed {SEED}. CUDA allocator peaks are measured in the training subprocess for each training epoch. \'\\n","                \'They exclude evaluation peaks, other GPU processes and driver memory. No arbitrary hardware ceiling is drawn.\',\\n","                [artifact / \'training_telemetry.jsonl\'])\\n","\\n","\\n","def plot_calibration_diagnostics(key):\\n","    stem=stem_for(key)\\n","    artifact = completed_artifact(key)\\n","    if artifact is None:\\n","        return\\n","    manifest = json.loads((artifact / \'manifest.json\').read_text(encoding=\'utf-8\'))\\n","    diagnostics = manifest[\'graph\'][\'icaart\']\\n","    coverage = [diagnostics.get(\'frac_edges_with_positive_evidence\'), diagnostics.get(\'weighted_evidence_coverage\')]\\n","    if any(value is None for value in coverage):\\n","        print(f\'{key}: coverage was not measured.\')\\n","        return\\n","    fig, axes = plt.subplots(1, 3, figsize=(7.2, 3.0), constrained_layout=True)\\n","    bars = axes[0].bar([\'Candidate\\\\nedges\', \'Raw semantic\\\\nweight\'], np.asarray(coverage) * 100,\\n","                       color=[COLORS[\'blue\'], COLORS[\'green\']], width=.6)\\n","    axes[0].bar_label(bars, fmt=\'%.2f%%\', padding=3, fontsize=8)\\n","    axes[0].set(ylabel=\'Coverage with positive excess score (%)\', title=\'Positive excess-score coverage\')\\n","    axes[0].set_ylim(0, max(coverage) * 125 + 1)\\n","    polynomial = diagnostics.get(\'polynomial_dose_by_coordinate\', {})\\n","    if polynomial:\\n","        coordinates = sorted(map(int, polynomial))\\n","        values = [100 * polynomial[str(coordinate)] for coordinate in coordinates]\\n","        axes[1].plot(coordinates, values, \'o-\', color=COLORS[\'blue\'], label=\'Polynomial probe dose\')\\n","    operator_dose = diagnostics.get(\'relative_operator_dose\')\\n","    if operator_dose is not None:\\n","        axes[1].axhline(100 * operator_dose, color=COLORS[\'orange\'], linestyle=\'--\', label=\'Operator Frobenius dose\')\\n","    axes[1].set(xlabel=\'Embedding coordinate\', ylabel=\'Relative dose (%)\', title=\'Operator / polynomial\\\\nprobe dose\')\\n","    axes[1].legend(frameon=False)\\n","    levels, values = diagnostics.get(\'evidence_quantile_levels\'), diagnostics.get(\'evidence_score_quantiles\')\\n","    if levels is not None and values is not None and len(levels) == len(values):\\n","        axes[2].plot(np.asarray(levels) * 100, values, \'o-\', color=COLORS[\'green\'])\\n","        pd.DataFrame({\'quantile\': levels, \'evidence_score\': values}).to_csv(\\n","            RUN_ROOT / \'tables\' / f\'{stem}_evidence_quantiles.csv\', index=False)\\n","    else:\\n","        axes[2].text(.5, .5, \'Score quantiles not measured\', ha=\'center\', va=\'center\', transform=axes[2].transAxes)\\n","    axes[2].set(xlabel=\'Quantile (%)\', ylabel=\'Excess score r\', ylim=(0, 1), title=\'Excess-score quantile profile\')\\n","    for ax in axes:\\n","        ax.grid(axis=\'y\', alpha=.2)\\n","    write_json(RUN_ROOT/\'tables\'/f\'{stem}_calibration_figure_data.json\',{\\n","       \'coverage_fraction\':coverage,\'operator_relative_dose\':operator_dose,\'polynomial_dose_by_coordinate\':polynomial,\\n","       \'quantile_levels\':levels,\'score_quantiles\':values,\'reference\':\'uncalibrated weighted semantic+CSE eta0.1\',\\n","       \'probe_scope\':diagnostics.get(\'polynomial_probe_scope\'),\'null_scope\':\'global_full\',\'num_active_train_users\':diagnostics.get(\'num_active_train_users\')})\\n","    fig.suptitle(f\'ECLIPSE Global-full null | {DATASET_TITLES[key]} | seed {SEED}\', fontsize=11)\\n","    save_figure(fig, f\'{stem}_calibration_diagnostics\',\\n","                f\'{DATASET_TITLES[key]}, training seed {SEED}. Global-full null uses positive excess over n_i*n_j/M_active with denominator c+mu_global+4; reference is the uncalibrated weighted semantic+CSE eta0.1 operator. \'\\n","                \'Polynomial dose uses fixed Rademacher probes, not actual Adam directions. \'\\n","                \'The score panel displays measured quantiles, not a reconstructed histogram.\', [artifact / \'manifest.json\'])\\n"],"execution_count":null,"outputs":[]},{"cell_type":"markdown","id":"0fbd3acd","metadata":{},"source":["## 8. Training / report-only — Amazon Baby seed 1\\n","Runs only if `baby` is in SELECTED_DATASETS. A completed attempt is read for export; an incomplete attempt is never overwritten. Checkpoints are initialized fresh, not from other seeds/arms.\\n"]},{"cell_type":"code","id":"05c7fbcf","metadata":{},"source":["baby_result=train_dataset(\'baby\')\\n","if baby_result is not None:\\n","    display(pd.DataFrame([{k:baby_result[k] for k in (\'arm\',\'dataset\',\'seed\',\'selected_epoch\',*PRIMARY_METRICS)}]))\\n"],"execution_count":null,"outputs":[]},{"cell_type":"markdown","id":"043b0a73","metadata":{},"source":["### VALID learning curves\\n"]},{"cell_type":"code","id":"7779de13","metadata":{},"source":["plot_learning_curves(\'baby\')\\n"],"execution_count":null,"outputs":[]},{"cell_type":"markdown","id":"72cd7f80","metadata":{},"source":["### Training memory and epoch time\\n"]},{"cell_type":"code","id":"78c612cb","metadata":{},"source":["plot_training_memory(\'baby\')\\n"],"execution_count":null,"outputs":[]},{"cell_type":"markdown","id":"22c07702","metadata":{},"source":["### Global-full null calibration diagnostics\\n"]},{"cell_type":"code","id":"8208b9be","metadata":{},"source":["plot_calibration_diagnostics(\'baby\')\\n"],"execution_count":null,"outputs":[]},{"cell_type":"markdown","id":"03a22728","metadata":{},"source":["## 9. Training / report-only — Amazon Sports seed 1\\n","Runs only if `sports` is in SELECTED_DATASETS. A completed attempt is read for export; an incomplete attempt is never overwritten. Checkpoints are initialized fresh, not from other seeds/arms.\\n"]},{"cell_type":"code","id":"6829223d","metadata":{},"source":["sports_result=train_dataset(\'sports\')\\n","if sports_result is not None:\\n","    display(pd.DataFrame([{k:sports_result[k] for k in (\'arm\',\'dataset\',\'seed\',\'selected_epoch\',*PRIMARY_METRICS)}]))\\n"],"execution_count":null,"outputs":[]},{"cell_type":"markdown","id":"9e639304","metadata":{},"source":["### VALID learning curves\\n"]},{"cell_type":"code","id":"f1e93e8c","metadata":{},"source":["plot_learning_curves(\'sports\')\\n"],"execution_count":null,"outputs":[]},{"cell_type":"markdown","id":"5388d8b6","metadata":{},"source":["### Training memory and epoch time\\n"]},{"cell_type":"code","id":"bf7e89ab","metadata":{},"source":["plot_training_memory(\'sports\')\\n"],"execution_count":null,"outputs":[]},{"cell_type":"markdown","id":"e16d9c0e","metadata":{},"source":["### Global-full null calibration diagnostics\\n"]},{"cell_type":"code","id":"87c9b864","metadata":{},"source":["plot_calibration_diagnostics(\'sports\')\\n"],"execution_count":null,"outputs":[]},{"cell_type":"markdown","id":"94640327","metadata":{},"source":["## 10. Current-session selected TEST table and resource profile\\n","No Full/other-arm results are mixed into this session. Each dataset currently has n=1. All seeds/attempts must be catalogued before a later matched multi-seed comparison; there is no SD or significance from epochs. Missing run/timing fields are NOT_MEASURED. Resource axes use distinct measurement scopes.\\n"]},{"cell_type":"code","id":"54a1cb52","metadata":{},"source":["summary_rows = []\\n","for key in SELECTED_DATASETS:\\n","    if completed_artifact(key) is None:\\n","        summary_rows.append({\'method\': \'ECLIPSE\', \'dataset\': key, \'seed\': SEED, \'status\': \'NOT_MEASURED\'})\\n","        continue\\n","    summary = export_run_tables(key)\\n","    names = [\'method\', \'arm\', \'dataset\', \'seed\', \'selected_epoch\', *PRIMARY_METRICS,\\n","             \'preprocessing_seconds\', \'fit_seconds\', \'train_epoch_seconds_sum\',\\n","             \'peak_training_allocated_mib\', \'peak_training_reserved_mib\',\\n","             \'checkpoint_path\', \'commit\', \'graph_fingerprint\']\\n","    summary_rows.append({name: summary.get(name) for name in names} | {\'status\': \'COMPLETED\'})\\n","SUMMARY = pd.DataFrame(summary_rows)\\n","for column in [\'selected_epoch\', *PRIMARY_METRICS, \'preprocessing_seconds\', \'fit_seconds\',\\n","               \'peak_training_allocated_mib\', \'peak_training_reserved_mib\']:\\n","    if column not in SUMMARY:\\n","        SUMMARY[column] = np.nan\\n","SUMMARY.to_csv(RUN_ROOT / \'tables\' / f\'{SESSION_STEM}_selected_test_overview.csv\', index=False, na_rep=\'NOT_MEASURED\')\\n","display(SUMMARY[[\'dataset\', \'seed\', \'status\', \'selected_epoch\', *PRIMARY_METRICS]].fillna(\'NOT_MEASURED\'))\\n","measured = SUMMARY.loc[SUMMARY[\'status\'] == \'COMPLETED\'].copy()\\n","if len(measured):\\n","    labels = [DATASET_TITLES[key].replace(\'Amazon \', \'\') for key in measured[\'dataset\']]\\n","    x = np.arange(len(measured))\\n","    fig, axes = plt.subplots(1, 3, figsize=(7.2, 3.0), constrained_layout=True)\\n","    if measured[\'fit_seconds\'].notna().any():\\n","        axes[0].bar(x, pd.to_numeric(measured[\'fit_seconds\'],errors=\'coerce\') / 3600, color=COLORS[\'blue\'])\\n","    else:\\n","        axes[0].text(.5,.5,\'NOT_MEASURED\',ha=\'center\',va=\'center\',transform=axes[0].transAxes)\\n","    axes[0].set(ylabel=\'Coach.fit wall time (h)\', title=\'Training + native evaluation / I/O\')\\n","    axes[1].bar(x, measured[\'preprocessing_seconds\'], color=COLORS[\'green\'])\\n","    axes[1].set(ylabel=\'Preprocessing wall time (s)\', title=\'Total model preprocessing\')\\n","    axes[2].bar(x - .18, measured[\'peak_training_allocated_mib\'], width=.36, color=COLORS[\'blue\'], label=\'Allocated\')\\n","    axes[2].bar(x + .18, measured[\'peak_training_reserved_mib\'], width=.36, color=COLORS[\'orange\'], label=\'Reserved\')\\n","    axes[2].set(ylabel=\'Training allocator peak (MiB)\', title=\'Training-epoch CUDA memory\')\\n","    axes[2].legend(frameon=False)\\n","    for ax in axes:\\n","        ax.set_xticks(x, labels)\\n","        ax.grid(axis=\'y\', alpha=.2)\\n","    fig.suptitle(f\'ECLIPSE Global-full null resource profile | seed {SEED}\', fontsize=11)\\n","    save_figure(fig, f\'{SESSION_STEM}_resource_profile\',\\n","                \'ECLIPSE Global-full null resource measurements, not a comparison against another model. \'\\n","                \'Coach.fit includes native evaluation and I/O; preprocessing is separate. \'\\n","                \'Memory excludes evaluation peaks. Cache state and runtime remain in manifests.\',\\n","                [RUN_ROOT / \'tables\' / f\'{SESSION_STEM}_selected_test_overview.csv\'])\\n","\\n","resource_fields=[\'preprocessing_seconds\',\'fit_seconds\',\'train_epoch_seconds_sum\',\'subprocess_wall_seconds\',\'peak_training_allocated_mib\',\'peak_training_reserved_mib\',\'memory_scope\',\'preprocessing_cache_hit\']\\n","for key in SELECTED_DATASETS:\\n","    if key in RESULTS:\\n","        pd.DataFrame([{k:RESULTS[key].get(k) for k in resource_fields}]).to_csv(RUN_ROOT/\'tables\'/f\'{stem_for(key)}_resource_summary.csv\',index=False,na_rep=\'NOT_MEASURED\')\\n","status_rows=[]\\n","for key in SELECTED_DATASETS:\\n","    status_rows.append({\'dataset\':key,\'arm\':ARM,\'training_seed\':SEED,\'actual_n\':1 if key in RESULTS else 0,\\n","       \'state\':\'COMPLETED_PENDING_HUMAN_REVIEW\' if key in RESULTS else \'NOT_MEASURED\'})\\n","pd.DataFrame(status_rows).to_csv(RUN_ROOT/\'tables\'/f\'{SESSION_STEM}_seed_status.csv\',index=False)\\n"],"execution_count":null,"outputs":[]},{"cell_type":"markdown","id":"e3a907bf","metadata":{},"source":["## 11. Complete handoff ZIP / inventory / SHA256\\n","Includes all completed dataset artifacts, stdout logs outside native directories, resolved/original config and commands, source/runtime/input provenance, tables, English captions and PDF/PNG/source data. Default run order is Baby→Sports; do not leave archiving to quota expiry. If an interrupted job stops Run All, preserve it and manually execute reporting/packaging for completed datasets; missing datasets remain unmeasured.\\n","\\n","Report-only copies originals into a **new** export root. The exact submitted notebook is included when NOTEBOOK_SOURCE_PATH is provided; otherwise a nonrecursive source template is included and the executed Kaggle notebook should be downloaded separately. No checkpoint tensors are loaded. Human cross-read remains pending.\\n"]},{"cell_type":"code","id":"9ba4c684","metadata":{},"source":["import zipfile\\n","write_json(RUN_ROOT/\'provenance\'/\'notebook_source_template.ipynb\',NOTEBOOK_SOURCE_TEMPLATE)\\n","script=\'\\\\n\\\\n\'.join(\'# CELL \'+str(i)+\'\\\\n\'+\'\'.join(c[\'source\']) for i,c in enumerate(NOTEBOOK_SOURCE_TEMPLATE[\'cells\']) if c[\'cell_type\']==\'code\')\\n","(RUN_ROOT/\'helpers\'/\'orchestration_source.py\').write_text(script,encoding=\'utf-8\')\\n","if NOTEBOOK_SOURCE_PATH:\\n","    original=Path(NOTEBOOK_SOURCE_PATH).expanduser().resolve()\\n","    if not original.is_file():raise FileNotFoundError(original)\\n","    shutil.copy2(original,RUN_ROOT/\'provenance\'/\'submitted_notebook.ipynb\')\\n","    write_json(RUN_ROOT/\'provenance\'/\'submitted_notebook_receipt.json\',{\'path\':str(original),\'sha256\':file_sha256(original)})\\n","for key in SELECTED_DATASETS:\\n","    if key not in RESULTS:continue\\n","    stem=stem_for(key);native=ARTIFACT_PATHS[key]\\n","    if RUN_MODE==\'report_only\':\\n","        copied=RUN_ROOT/\'artifacts\'/stem\\n","        shutil.copytree(native,copied,dirs_exist_ok=False)\\n","        log=LOG_PATHS[key]\\n","        if log and log.is_file():shutil.copy2(log,RUN_ROOT/\'logs\'/f\'{stem}.log\')\\n","        m=json.loads((native/\'manifest.json\').read_text(encoding=\'utf-8\'))\\n","        write_json(RUN_ROOT/\'provenance\'/f\'{stem}_report_only_origin.json\',{\'artifact\':str(native),\'copied_to\':str(copied),\\n","          \'stdout_log\':str(log) if log else None,\'raw_artifacts_modified\':False,\'recovered_from_native_manifest\':True,\\n","          \'original_yaml_available\':False,\'original_dirty_diff_available\':False,\'note\':\'Do not invent unavailable historical provenance.\'})\\n","        write_json(RUN_ROOT/\'configs\'/f\'{stem}_recorded_native_config.json\',{\'v8_config\':m.get(\'v8_config\'),\'icaart_config\':m.get(\'icaart_config\'),\\n","           \'checkpoint_config\':m.get(\'checkpoint_contract\',{}).get(\'config\'),\'argv\':m.get(\'argv\')})\\n","        write_json(RUN_ROOT/\'provenance\'/f\'{stem}_recorded_native_environment.json\',{k:m.get(k) for k in (\'python\',\'torch\',\'freerec\',\'gpu\',\'commit\',\'precision\')})\\n","    for kind in (\'learning_curves\',\'training_memory\',\'calibration_diagnostics\'):\\n","        for extension in (\'pdf\',\'png\'):\\n","            path=RUN_ROOT/\'figures\'/f\'{stem}_{kind}.{extension}\'\\n","            if not path.is_file():raise FileNotFoundError(f\'Missing measured figure {path}; generate plotting cells, never fabricate values.\')\\n","entries=json.loads((RUN_ROOT/\'figures\'/\'figure_manifest.json\').read_text(encoding=\'utf-8\')) if (RUN_ROOT/\'figures\'/\'figure_manifest.json\').is_file() else []\\n","caption_text=\'# ECLIPSE Global-full null figure captions\\\\n\\\\n\'\\n","for entry in entries:\\n","    caption_text+=\'## \'+entry[\'stem\']+\'\\\\n\\\\n\'+entry[\'caption\']+\'\\\\n\\\\nSources:\\\\n\'+\'\\\\n\'.join(\'- \'+p for p in entry[\'sources\'])+\'\\\\n\\\\n\'\\n","(RUN_ROOT/\'figures\'/\'FIGURE_CAPTIONS.md\').write_text(caption_text,encoding=\'utf-8\')\\n","run=json.loads((RUN_ROOT/\'run_manifest.json\').read_text(encoding=\'utf-8\'))\\n","run.update(status=\'COMPLETED_PENDING_HUMAN_REVIEW\' if len(RESULTS)==len(SELECTED_DATASETS) else \'PARTIAL_PENDING_REVIEW\',\\n","    datasets={key:{\'state\':\'COMPLETED_PENDING_HUMAN_REVIEW\' if key in RESULTS else \'NOT_MEASURED\',\'actual_n\':1 if key in RESULTS else 0,\\n","      \'selected_epoch\':RESULTS.get(key,{}).get(\'selected_epoch\'),\'comparability\':RESULTS.get(key,{}).get(\'comparability\',{}).get(\'status\')} for key in SELECTED_DATASETS},\\n","    reviewed_by=None,matched_full_comparison=\'NOT_COMPUTED\')\\n","write_json(RUN_ROOT/\'run_manifest.json\',run)\\n","(RUN_ROOT/\'README.md\').write_text(f\\"\\"\\"# ECLIPSE Global-full null | training seed {SEED}\\n","Selected datasets: {\', \'.join(SELECTED_DATASETS)}. Mode: {RUN_MODE}; attempt/export: {SESSION_ID}.\\n","Protocol: 64D/500epochs; pooled_global_excess, delta1/tau2/halves0; eta.1/LC.01 retained.\\n","Global excess score: [c-mu_global]+/(c+mu_global+4), mu_global=n_i*n_j/M_active; no user halves/activity strata.\\n","Native selected TEST: artifacts/eclipse_<dataset>_global_full_null_seed{SEED}/selected_metrics.json.\\n","Tables/ contains selected TEST/resource/diagnostic data; figures/ has PDF vector/PNG300dpi + captions.\\n","Source/input/runtime/config/commands under provenance/ and configs/; stdout in logs/.\\n","Algorithm source: base {RUNTIME_COMMIT} + declared Global-full overlay {GLOBAL_OVERLAY_ID}. See source digests and diff. Source template excludes recursive self-embedding.\\n","Human review pending. Actual current seed only per dataset: no seed SD/p-values or phantom completed runs.\\n","Coach.fit, preprocessing, subprocess wall and epoch sums have distinct scopes; missing data NOT_MEASURED.\\n","Memory is child training-epoch allocator, not total-VRAM/whole-run/evaluation peak.\\n","\\"\\"\\",encoding=\'utf-8\')\\n","inventory=[]\\n","for path in sorted(RUN_ROOT.rglob(\'*\')):\\n","    if path.is_file() and path.name!=\'file_inventory.json\':inventory.append({\'path\':path.relative_to(RUN_ROOT).as_posix(),\'bytes\':path.stat().st_size,\'sha256\':file_sha256(path)})\\n","write_json(RUN_ROOT/\'file_inventory.json\',inventory)\\n","archive=RUN_ROOT.parent/f\'{RUN_ROOT.name}.zip\'\\n","with zipfile.ZipFile(archive,\'x\',compression=zipfile.ZIP_DEFLATED,compresslevel=1) as z:\\n","    for path in sorted(RUN_ROOT.rglob(\'*\')):\\n","        if path.is_file():z.write(path,arcname=str(Path(RUN_ROOT.name)/path.relative_to(RUN_ROOT)))\\n","with zipfile.ZipFile(archive) as z:\\n","    if z.testzip() is not None:raise RuntimeError(\'Archive CRC failed\')\\n","archive_receipt={\'archive\':str(archive),\'bytes\':archive.stat().st_size,\'sha256\':file_sha256(archive),\'file_count\':len(inventory)+1,\\n","    \'status\':\'PACKAGED_PENDING_HUMAN_REVIEW\',\'run_root\':str(RUN_ROOT)}\\n","write_json(archive.with_suffix(\'.archive_receipt.json\'),archive_receipt)\\n","print(json.dumps(archive_receipt,indent=2))\\n","try:\\n","    from IPython.display import FileLink\\n","    display(FileLink(str(archive)));display(FileLink(str(archive.with_suffix(\'.archive_receipt.json\'))))\\n","except ImportError:pass\\n"],"execution_count":null,"outputs":[]}],"metadata":{"kernelspec":{"display_name":"Python 3","language":"python","name":"python3"},"language_info":{"name":"python","file_extension":".py"},"eclipse":{"arm":"global_full_null","default_seed":1,"datasets":["baby","sports"],"runtime_commit":"5212263e87c5c1cfe6e48c91afa90201a48f5929","created":"2026-10-20","derived_from":"notebook/P6/eclipse_count_only_seed2.ipynb","derived_from_sha256":"a430f96f59229096e6c9e5de2c50d72a2f0a5d707e25804e273c40962261246e","electronics_scope":"not_in_plan","source_overlay_id":"f62a74a254ef021fb3c064070c85ea8d6b6f0d8810870e1ae9293bb3f90a24c2","source_base_commit":"5212263e87c5c1cfe6e48c91afa90201a48f5929"}},"nbformat":4,"nbformat_minor":5}')


## 3. Private pinned source + declared overlay and matched runtime
Clone the recorded Full revision into a private session directory, apply the embedded three-file extension after base-hash checks, and save source hashes/full diff. No existing training checkout is modified. No fallback HEAD, anonymous monkey-patch or Torch upgrade. All unmodified source digests must match Full.


In [ ]:
if RUN_MODE == 'train':
    import importlib.metadata
    import importlib.util
    import subprocess

    if REPO_OVERRIDE is not None:
        raise ValueError('REPO_OVERRIDE is disabled: this control uses its own private checkout.')
    else:
        REPO_ROOT=WORK_DIR/'eclipse_sources'/('global_full_null_'+GLOBAL_OVERLAY_ID[:12])/SESSION_ID
        if not REPO_ROOT.exists():
            REPO_ROOT.mkdir(parents=True,exist_ok=False)
            subprocess.run(['git','init',str(REPO_ROOT)],check=True)
            subprocess.run(['git','-C',str(REPO_ROOT),'remote','add','origin',REPO_URL],check=True)
            subprocess.run(['git','-C',str(REPO_ROOT),'fetch','--depth','1','origin',RUNTIME_COMMIT],check=True)
            subprocess.run(['git','-C',str(REPO_ROOT),'checkout','--detach',RUNTIME_COMMIT],check=True)
    actual=subprocess.run(['git','-C',str(REPO_ROOT),'rev-parse','HEAD'],text=True,capture_output=True,check=True).stdout.strip()
    if actual!=RUNTIME_COMMIT:raise RuntimeError('Use the pinned checkout; existing source is never reset.')
    # Apply only the explicitly declared source extension before any imports/probes.
    overlay_receipt={}
    for relative, record in GLOBAL_SOURCE_OVERLAY['files'].items():
        path=REPO_ROOT/relative
        if path.is_symlink() or not path.is_file():raise ValueError('Invalid overlay target: '+relative)
        current=file_sha256(path)
        current_lf=hashlib.sha256(path.read_bytes().replace(b'\r\n',b'\n')).hexdigest()
        if not ({current,current_lf}&{record['base_sha256'],record['sha256']}):
            raise RuntimeError('Unexpected private-checkout source; refusing overwrite: '+relative)
        payload=record['content'].encode('utf-8')
        if hashlib.sha256(payload).hexdigest()!=record['sha256']:raise RuntimeError('Overlay payload hash mismatch')
        path.write_bytes(payload)
        if file_sha256(path)!=record['sha256']:raise RuntimeError('Overlay write mismatch')
        overlay_receipt[relative]={'base_sha256':record['base_sha256'],'sha256':file_sha256(path)}
    write_json(RUN_ROOT/'provenance'/'applied_global_full_overlay.json',{
        'base_commit':actual,'overlay_id':GLOBAL_OVERLAY_ID,'files':overlay_receipt,'private_checkout':str(REPO_ROOT)})
    for args,name in ((['status','--porcelain'],'git_status.txt'),(['diff','HEAD','--'],'source_diff.patch'),(['remote','-v'],'git_remotes.txt')):
        result=subprocess.run(['git','-C',str(REPO_ROOT),*args],text=True,encoding='utf-8',capture_output=True,check=True)
        (RUN_ROOT/'provenance'/name).write_text(result.stdout,encoding='utf-8')
    if not (REPO_ROOT / 'main_stair_icaart.py').is_file():
        raise FileNotFoundError(REPO_ROOT / 'main_stair_icaart.py')

    ENV = os.environ.copy()
    ENV['PYTHONPATH'] = os.pathsep.join(filter(None, [str(REPO_ROOT), ENV.get('PYTHONPATH', '')]))
    ENV['PYTHONUTF8'] = '1'
    ENV['PYTHONIOENCODING'] = 'utf-8'
    ENV['PYTHONHASHSEED'] = str(SEED)
    ENV['CUDA_VISIBLE_DEVICES'] = str(GPU_INDEX)
    ENV.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')

    # Do not import Torch into the notebook kernel: all CUDA work belongs to the training subprocess.
    if INSTALL_MISSING_DEPENDENCIES:
        packages = {
            'freerec': ('freerec==0.9.7', True),
            'torch_geometric': ('torch-geometric==2.8.1', True),
            'numpy': ('numpy==2.1.3', True), 'scipy': ('scipy==1.16.3', True),
            'yaml': ('PyYAML==6.0.3', True), 'pandas': ('pandas==2.3.3', True),
            'matplotlib': ('matplotlib==3.10.0', True), 'prettytable': ('prettytable==3.18.0', True),
            'sklearn': ('scikit-learn==1.6.1', True), 'psutil': ('psutil==5.9.5', True),
        }
        for module, (requirement, no_deps) in packages.items():
            missing = importlib.util.find_spec(module) is None
            if module == 'freerec' and not missing:
                missing = importlib.metadata.version('freerec') != '0.9.7'
            if missing:
                command = [sys.executable, '-m', 'pip', 'install', '--quiet']
                if no_deps:
                    command.append('--no-deps')
                subprocess.run(command + [requirement], check=True)

    expected_sources = EXPECTED_RUNTIME_SOURCES
    source_report = {}
    for name, expected in expected_sources.items():
        choices = [REPO_ROOT / name, REPO_ROOT / 'models' / name, REPO_ROOT / 'optimizers' / name]
        path = next((p for p in choices if p.is_file()), None)
        if path is None:
            raise FileNotFoundError(f'Missing seed-1 source: {name}')
        payload = path.read_bytes()
        actual = hashlib.sha256(payload).hexdigest()
        lf_digest = hashlib.sha256(payload.replace(b'\r\n', b'\n')).hexdigest()
        source_report[name] = {'path': str(path.relative_to(REPO_ROOT)), 'sha256': actual,
                               'line_endings_only': actual != expected and lf_digest == expected}
        if expected not in (actual, lf_digest):
            raise RuntimeError(f'Source differs from declared Full-plus-Global overlay: {path}. Use the pinned runtime commit.')
    write_json(RUN_ROOT / 'provenance' / 'source_check.json', source_report)

    probe = r"""
    import json, platform, torch
    import models.freerec_compat
    import freerec
    from main_stair_icaart import build_config
    from stair_icaart_reporting import validate_selected_receipt
    print('ECLIPSE_ENV=' + json.dumps({'python': platform.python_version(), 'torch': str(torch.__version__),
     'freerec': str(freerec.__version__), 'torch_geometric': __import__('importlib.metadata',fromlist=['version']).version('torch-geometric'), 'cuda': torch.cuda.is_available(),
     'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}))
    """
    probe = __import__('textwrap').dedent(probe)
    checked = subprocess.run([sys.executable, '-c', probe], cwd=REPO_ROOT, env=ENV,
                             text=True, encoding='utf-8', capture_output=True)
    if checked.stdout:
        print(checked.stdout)
    if checked.stderr:
        print(checked.stderr)
    checked.check_returncode()
    line = next(line for line in checked.stdout.splitlines() if line.startswith('ECLIPSE_ENV='))
    RUNTIME_ENV = json.loads(line.split('=', 1)[1])
    if not RUNTIME_ENV['cuda']:
        raise RuntimeError('Enable a Kaggle GPU accelerator before training.')
    reference = SEED1_REFERENCE[SELECTED_DATASETS[0]]
    deviations = {key: {'reference': reference[key], 'current': RUNTIME_ENV[key]}
                  for key in ('torch', 'freerec','gpu') if RUNTIME_ENV[key] != reference[key]}
    if RUNTIME_ENV['python'].split('.')[:2] != reference['python'].split('.')[:2]:
        deviations['python'] = {'reference': reference['python'], 'current': RUNTIME_ENV['python']}
    if RUNTIME_ENV.get('torch_geometric') != '2.8.1':
        deviations['torch_geometric'] = {'reference':'2.8.1','current':RUNTIME_ENV.get('torch_geometric')}
    write_json(RUN_ROOT / 'provenance' / 'environment.json', {'runtime': RUNTIME_ENV, 'deviations': deviations,
               'strict_runtime': STRICT_RUNTIME, 'gpu_index': GPU_INDEX})
    if deviations and STRICT_RUNTIME:
        raise RuntimeError(f'Runtime differs from seed 1: {deviations}. Restore the seed-1 environment; '
                           'setting STRICT_RUNTIME=False explicitly records a protocol deviation.')
    if deviations:
        print('PROTOCOL DEVIATION:', deviations)
    if RUNTIME_ENV['gpu'] != reference['gpu']:
        print('HARDWARE DEVIATION:', RUNTIME_ENV['gpu'], 'vs', reference['gpu'],
              '- runtime measurements are not a matched-hardware comparison.')
    freeze = subprocess.run([sys.executable, '-m', 'pip', 'freeze'], text=True, capture_output=True, check=True)
    (RUN_ROOT / 'provenance' / 'pip_freeze.txt').write_text(freeze.stdout, encoding='utf-8')
    commit = subprocess.run(['git', '-C', str(REPO_ROOT), 'rev-parse', 'HEAD'], text=True, capture_output=True)
    ACTUAL_COMMIT = commit.stdout.strip() if commit.returncode == 0 else 'unavailable'
    print('Runtime ready:', REPO_ROOT, '| commit:', ACTUAL_COMMIT)
    print('Import/environment checks only; no model test or training has been executed.')

    driver=subprocess.run(['nvidia-smi','--query-gpu=name,driver_version,memory.total','--format=csv'],text=True,capture_output=True)
    (RUN_ROOT/'provenance'/'nvidia_smi.txt').write_text(driver.stdout+driver.stderr,encoding='utf-8')

else:
    print('report_only: skipped training setup; no model/network/GPU invocation.')


## 4. Processed inputs — Attach before Kaggle batch
Five split/feature files per selected dataset must match reference SHA256. Writable bridge/chunks; no KaggleHub attachment in batch. Report-only skips this entire stage.


In [ ]:
if RUN_MODE == 'train':
    import shutil

    REQUIRED_FILES = ('train.txt', 'valid.txt', 'test.txt', 'textual_modality.pkl', 'visual_modality.pkl')


    def dataset_search_roots(extra_roots=()):
        # Include writable bridges and caches created by earlier interactive sessions.
        return [REPO_ROOT / 'data', DATA_ROOT, WORK_DIR / 'data',
                Path('/kaggle/data'), Path('/kaggle/input'),
                Path.home() / '.cache' / 'kagglehub' / 'datasets',
                *map(Path, DATASET_SEARCH_ROOTS), *map(Path, extra_roots)]


    def discover_datasets(extra_roots=()):
        found = {}
        unknown = set(DATASET_SOURCE_OVERRIDES) - set(DATASET_FOLDERS)
        if unknown:
            raise ValueError(f'Unknown DATASET_SOURCE_OVERRIDES keys: {sorted(unknown)}')
        for key in SELECTED_DATASETS:
            if key not in DATASET_SOURCE_OVERRIDES:
                continue
            path = Path(DATASET_SOURCE_OVERRIDES[key]).expanduser()
            absent = [name for name in REQUIRED_FILES if not (path / name).is_file()]
            if absent:
                raise FileNotFoundError(f'Incomplete explicit dataset source for {key}: {path}; missing {absent}')
            found[key] = path
        aliases = {'baby': ('baby',), 'sports': ('sport',), 'electronics': ('electronic',)}
        visited = set()
        for root in dataset_search_roots(extra_roots):
            root = root.expanduser()
            if not root.is_dir() or str(root.resolve()) in visited:
                continue
            visited.add(str(root.resolve()))
            for current, directories, files in os.walk(root, followlinks=False):
                directories[:] = sorted(d for d in directories if d not in
                                        {'.git', 'chunks', '__pycache__', '.stair_icaart_graph_cache'})
                if not set(REQUIRED_FILES).issubset(files):
                    continue
                path = Path(current)
                for key in SELECTED_DATASETS:
                    if any(alias in path.name.lower() for alias in aliases[key]):
                        found.setdefault(key, path)
        return found


    def missing_dataset_message(missing):
        return (
            f'Missing processed datasets: {sorted(missing)}.\n'
            f'On Kaggle, attach {DATASET_HANDLE!r} with Add Input BEFORE Save & Run All '
            '(or attach an equivalent processed dataset matching the seed-1 fingerprints). '
            'Save a new notebook version after attaching it. Internet access alone does not '
            'attach a datasource, and kagglehub cannot attach a new one in a non-interactive run.\n'
            'Alternatively set DATASET_SOURCE_OVERRIDES to the existing per-dataset folders, '
            'or DATASET_SEARCH_ROOTS to their parent directory. Each dataset folder must contain: '
            + ', '.join(REQUIRED_FILES) + '.\n'
            'Searched roots: ' + ', '.join(str(p.expanduser()) for p in dataset_search_roots())
            + f'\nDiscovery receipt: {RUN_ROOT / "provenance" / "data_discovery.json"}'
        )


    def resolve_dataset_sources():
        sources = discover_datasets()
        missing = set(SELECTED_DATASETS) - sources.keys()
        on_kaggle = Path('/kaggle/working').is_dir()
        discovery = {'dataset_handle': DATASET_HANDLE, 'selected_datasets': list(SELECTED_DATASETS),
                     'searched_roots': [str(p.expanduser()) for p in dataset_search_roots()],
                     'on_kaggle': on_kaggle, 'auto_download_requested': AUTO_DOWNLOAD_DATA,
                     'download_attempted': False}

        def record(status):
            discovery.update(status=status, missing=sorted(missing),
                             sources={key: str(path.resolve()) for key, path in sources.items()})
            write_json(RUN_ROOT / 'provenance' / 'data_discovery.json', discovery)

        if missing and on_kaggle:
            # Never try to attach a datasource during a Kaggle batch job. The caller
            # must attach inputs before execution; this also makes batch runs repeatable.
            record('MISSING_ATTACHED_INPUT')
            raise FileNotFoundError(missing_dataset_message(missing))
        if missing and AUTO_DOWNLOAD_DATA:
            discovery['download_attempted'] = True
            try:
                import kagglehub
                downloaded = kagglehub.dataset_download(DATASET_HANDLE)
                discovery['downloaded_root'] = str(downloaded)
                sources.update(discover_datasets([downloaded]))
                missing = set(SELECTED_DATASETS) - sources.keys()
            except Exception as exc:
                discovery['download_error'] = {'type': type(exc).__name__, 'message': str(exc)}
                record('DOWNLOAD_FAILED')
                raise FileNotFoundError(missing_dataset_message(missing)) from exc
        if missing:
            record('MISSING_PROCESSED_DATA')
            raise FileNotFoundError(missing_dataset_message(missing))
        record('FOUND_PENDING_FINGERPRINT_CHECK')
        return sources


    sources = resolve_dataset_sources()

    DATA_RECEIPTS = {}
    for key in SELECTED_DATASETS:
        source = sources[key].resolve()
        reference = SEED1_REFERENCE[key]
        expected = dict(reference['contract']['splits'])
        expected.update({record['file']: record['sha256'] for record in reference['features']})
        destination = DATA_ROOT / 'Processed' / DATASET_FOLDERS[key]
        if destination.is_symlink():
            raise RuntimeError(f'Processed destination must be a writable directory, not a symlink: {destination}')
        destination.mkdir(parents=True, exist_ok=True)
        for name, expected_hash in expected.items():
            original = source / name
            if not original.is_file():
                raise FileNotFoundError(original)
            actual_hash = file_sha256(original)
            if actual_hash != expected_hash:
                raise RuntimeError(f'Dataset differs from seed 1: {original}')
            target = destination / name
            if target.is_symlink() and not target.exists():
                raise FileExistsError(f'Broken input link preserved; choose a fresh DATA_ROOT: {target}')
            if target.exists():
                if file_sha256(target) != expected_hash:
                    raise RuntimeError(f'Existing linked/copied input has a different fingerprint: {target}')
            else:
                try:
                    target.symlink_to(original)
                except OSError:
                    shutil.copy2(original, target)
        if (destination / 'chunks').is_symlink():
            raise RuntimeError(f'FreeRec chunks must be writable, not an input symlink: {destination / "chunks"}')
        (destination / 'chunks').mkdir(exist_ok=True)
        DATA_RECEIPTS[key] = {'source': str(source), 'processed': str(destination), 'fingerprints': expected}
        print(f'{DATASET_TITLES[key]}: split and feature fingerprints match seed 1.')
    write_json(RUN_ROOT / 'provenance' / 'data_check.json', DATA_RECEIPTS)

else:
    print('report_only: skipped training setup; no model/network/GPU invocation.')


## 5. Resolved configuration and native CLI preflight
Mode=pooled_global_excess, bins=none, halves0, delta1/tau2; eta.1/LC.01 and per-dataset training hyperparameters remain Full. Native resolved settings are checked before training.


In [ ]:
if RUN_MODE == 'train':
    import yaml

    CONFIG_PATHS = {}
    ARTIFACT_PATHS = {}
    LOG_PATHS = {}
    for key in SELECTED_DATASETS:
        stem = stem_for(key)
        title = key.capitalize()
        original = REPO_ROOT / 'configs' / f'Amazon2014{title}_STAIR_PEEC.yaml'
        if not original.is_file():
            raise FileNotFoundError(original)
        config = yaml.safe_load(original.read_text(encoding='utf-8'))
        config.update(SEED1_REFERENCE[key]['v8_config'])
        config.update(SEED1_REFERENCE[key]['icaart_config'])
        config['eta'] = 0.1
        config['evidence_mode'] = 'pooled_global_excess'
        config['evidence_activity_bins'] = 'none'
        config['evidence_delta'] = 1.0
        config.pop('evidence_seed', None)  # Inactive; do not confuse the evidence partition with training RNG.
        config.update(seed=SEED, root=str(DATA_ROOT), dataset=DATASET_FOLDERS[key],
                      num_workers=SEED1_REFERENCE[key]['contract']['config']['num_workers'],
                      eval_valid=True, eval_test=False, ranking='full', which4best='NDCG@20',
                      resume=False, resume_from='')
        config['mfiles'] = ','.join(config['mfiles'])
        config['num_neighbors'] = '-'.join(map(str, config['num_neighbors']))
        artifact = RUN_ROOT / 'artifacts' / stem
        config['artifact_dir'] = str(artifact)
        config['graph_cache_dir'] = str(GRAPH_CACHE_ROOT / key)
        target = RUN_ROOT / 'configs' / f'{stem}.yaml'
        target.write_text(yaml.safe_dump(config, sort_keys=False), encoding='utf-8')
        shutil.copy2(original,RUN_ROOT/"configs"/f"{stem}_original.yaml")
        CONFIG_PATHS[key], ARTIFACT_PATHS[key] = target, artifact
        LOG_PATHS[key] = RUN_ROOT / 'logs' / f'{stem}.log'


    def build_command(key):
        if key not in SELECTED_DATASETS:
            raise ValueError(f'{key} is not selected; edit SELECTED_DATASETS and rerun setup.')
        return [sys.executable, '-u', str(REPO_ROOT / 'main_stair_icaart.py'),
                '--config', str(CONFIG_PATHS[key]), '--seed', str(SEED),
                '--root', str(DATA_ROOT), '--dataset', DATASET_FOLDERS[key],
                '--device', '0', '--ranking', 'full',
                '--artifact-dir', str(ARTIFACT_PATHS[key]),
                '--graph-cache-dir', str(GRAPH_CACHE_ROOT / key)]


    config_probe = r"""
    import json, sys
    from main_stair_icaart import build_config
    cfg = build_config(sys.argv[1:])
    keys = json.loads(__import__('os').environ['ECLIPSE_CHECK_KEYS'])
    receipt = {key: getattr(cfg, key) for key in keys}
    print('ECLIPSE_CONFIG=' + json.dumps(receipt))
    """
    config_probe = __import__('textwrap').dedent(config_probe)
    for key in SELECTED_DATASETS:
        expected = dict(SEED1_REFERENCE[key]['v8_config'])
        expected.update(SEED1_REFERENCE[key]['icaart_config'])
        expected['eta'] = 0.1
        expected['evidence_mode'] = 'pooled_global_excess'
        expected['evidence_activity_bins'] = 'none'
        expected['evidence_delta'] = 1.0
        expected.update(seed=SEED, dataset=DATASET_FOLDERS[key], ranking='full', which4best='NDCG@20',
                        eval_valid=True, eval_test=False, resume=False, resume_from='',
                        num_workers=SEED1_REFERENCE[key]['contract']['config']['num_workers'])
        probe_env = ENV.copy()
        probe_env['ECLIPSE_CHECK_KEYS'] = json.dumps(list(expected))
        result = subprocess.run([sys.executable, '-c', config_probe, *build_command(key)[3:]],
                                cwd=REPO_ROOT, env=probe_env, text=True, encoding='utf-8', capture_output=True)
        if result.returncode:
            print(result.stdout, result.stderr)
            result.check_returncode()
        payload = next(line.split('=', 1)[1] for line in result.stdout.splitlines()
                       if line.startswith('ECLIPSE_CONFIG='))
        resolved = json.loads(payload)
        changes = {name: {'expected': value, 'actual': resolved.get(name)}
                   for name, value in expected.items() if resolved.get(name) != value}
        if changes:
            raise RuntimeError(f'{key}: resolved CLI/YAML protocol mismatch: {changes}')
        write_json(RUN_ROOT / 'provenance' / f'{key}_resolved_config.json', resolved)
        print(f'{key}: seed={resolved["seed"]}, epochs={resolved["epochs"]}, '
              f'batch={resolved["batch_size"]}, gamma={resolved["gamma"]}, '
              f'weight_decay={resolved["weight_decay"]}, cl_chunk={resolved["cl_chunk_size"]}')
    write_json(RUN_ROOT / 'run_manifest.json', {
        'method': 'ECLIPSE', 'arm': ARM, 'arm_id': ARM, 'status':'READY', 'training_seed': SEED,
        'datasets': list(SELECTED_DATASETS), 'session_id': SESSION_ID,
        'reference_commit': RUNTIME_COMMIT, 'actual_commit': ACTUAL_COMMIT,
        'source_overlay_id': GLOBAL_OVERLAY_ID, 'declared_source_changes': DECLARED_SOURCE_CHANGES,
        'runtime': RUNTIME_ENV, 'runtime_deviations': deviations,
        'selection': 'validation NDCG@20', 'test_policy': 'report only VALID-selected TEST; native terminal TEST may also be evaluated',
        'configs': {k: {'path': str(p), 'sha256': file_sha256(p)} for k, p in CONFIG_PATHS.items()},
        'dataset_paths': DATA_RECEIPTS,
        'note': 'Global-full null: pooled_global_excess, delta1/tau2/halves0; eta.1/LC.01 retained; no resume.',
    })

else:
    print('report_only: skipped training setup; no model/network/GPU invocation.')


## 6. Global-full guards and output engine
Checks selected TEST/seed/config/status, 500epochs, objective identity, source extension, invariant candidates/counts/CF, one active-user pool, null formula audit and score range. Null/assignment/score/final operator are allowed to differ from activity-conditioned Full.


In [ ]:
import math
import re
import signal
import time
import pandas as pd
import numpy as np
from IPython.display import display

RESULTS = {}


def read_jsonl(path):
    path = Path(path)
    if not path.is_file():
        return []
    result = []
    with path.open(encoding='utf-8') as handle:
        for number, line in enumerate(handle, 1):
            if line.strip():
                try:
                    result.append(json.loads(line))
                except json.JSONDecodeError as exc:
                    raise ValueError(f'Malformed JSONL: {path}, line {number}') from exc
    return result


def compare_manifest(key, manifest, expected_seed):
    ref=SEED1_REFERENCE[key];problems={}
    if manifest.get('seed')!=expected_seed:problems['seed']=manifest.get('seed')
    if manifest.get('selection')!='validation NDCG@20':problems['selection']=manifest.get('selection')
    for name,expected in dict(ref['v8_config'],eta=0.1).items():
        if manifest.get('v8_config',{}).get(name)!=expected:problems['v8_config.'+name]=manifest.get('v8_config',{}).get(name)
    if manifest.get('icaart_config')!=dict(ref['icaart_config'],evidence_mode='pooled_global_excess',evidence_activity_bins='none'):problems['evidence_config']=manifest.get('icaart_config')
    contract=manifest.get('checkpoint_contract',{})
    for name in ('splits','candidate_hashes','data_fingerprint'):
        if contract.get(name)!=ref['contract'][name]:problems[name]=contract.get(name)
    for name,expected in ref['contract']['config'].items():
        if name=='seed':expected=expected_seed
        if name=='eta':expected=0.1
        if contract.get('config',{}).get(name)!=expected:problems['config.'+name]=contract.get('config',{}).get(name)
    if manifest.get('features')!=ref['features']:problems['features']=manifest.get('features')
    if manifest.get('precision')!={'tf32':False,'amp':False}:problems['precision']=manifest.get('precision')
    e=manifest.get('icaart_manifest',{});g=manifest.get('graph',{});d=g.get('icaart',{})
    expected_identity={'null_scope':'global_full','score_kind':'pooled_global_excess',
        'active_partition':False,'active_halves':0,'normalization':'actual_degree'}
    checkpoint_estimator=contract.get('stair_icaart',{})
    for label, snapshot in (('root',manifest),('checkpoint',checkpoint_estimator),
                             ('checkpoint_manifest',checkpoint_estimator.get('manifest',{})),('diagnostics',d)):
        for name,expected in expected_identity.items():
            if snapshot.get(name)!=expected:problems[label+'.'+name]=snapshot.get(name)
    for name in ('num_active_train_users','num_inactive_train_users','num_null_groups','num_activity_strata',
                 'activity_conditioning','evidence_activity_bins','counts_hash','null_expectation_hash',
                 'global_null_formula','global_null_max_abs_formula_error'):
        if d.get(name)!=e.get(name):problems['diagnostic_consistency.'+name]=d.get(name)
    for name,expected in ref['evidence_invariants'].items():
        if name in ('assignment_hash','group_sizes','null_expectation_hash'):continue
        if e.get(name)!=expected:problems['evidence.'+name]=e.get(name)
    if e.get('training_seed')!=expected_seed:problems['evidence.training_seed']=e.get('training_seed')
    for name,actual in (('graph.eta',g.get('eta')),('graph.effective_eta',g.get('effective_eta')),('icaart.effective_eta',d.get('effective_eta'))):
        if actual!=0.1:problems[name]=actual
    if e.get('null_scope')!='global_full' or e.get('normalization')!='actual_degree':problems['null_normalization']=e
    if e.get('evidence_mode')!='pooled_global_excess' or e.get('score_kind')!='pooled_global_excess':problems['score_identity']=e.get('score_kind')
    if e.get('pooled_regularizer')!=4.0 or e.get('active_partition') is not False or e.get('active_halves')!=0:problems['pooling_identity']=e
    if d.get('delta')!=1.0:problems['diagnostic_delta']=d.get('delta')
    score_hash=e.get('evidence_score_hash')
    if not isinstance(score_hash,str) or re.fullmatch(r'[0-9a-f]{64}',score_hash) is None:problems['global_score_hash']=score_hash
    active_expected=sum(ref['evidence_invariants']['group_sizes'])
    if e.get('num_active_train_users')!=active_expected:problems['M_active']=e.get('num_active_train_users')
    if e.get('num_inactive_train_users')!=e.get('num_users',0)-active_expected:problems['inactive_users']=e.get('num_inactive_train_users')
    if e.get('group_sizes')!=[active_expected]:problems['global_pool']=e.get('group_sizes')
    if e.get('num_null_groups')!=1 or e.get('num_activity_strata')!=0 or e.get('activity_conditioning') is not False:
        problems['activity_conditioning']='Global-full must contain one non-stratified active-user pool'
    if e.get('evidence_activity_bins')!='none':problems['activity_bins']=e.get('evidence_activity_bins')
    if e.get('global_null_formula')!='n_i*n_j/M_active; binary TRAIN item marginals':problems['null_formula']=e.get('global_null_formula')
    error=e.get('global_null_max_abs_formula_error')
    if isinstance(error,bool) or not isinstance(error,(int,float)) or not math.isfinite(error) or error<0:
        problems['global_null_formula_check']='missing/invalid candidate-wise formula check'
    for name in ('null_expectation_hash','assignment_hash'):
        if not isinstance(e.get(name),str) or re.fullmatch(r'[0-9a-f]{64}',e[name]) is None:problems[name]=e.get(name)
    sq=d.get('evidence_score_quantiles')
    if not sq or any(isinstance(v,bool) or not isinstance(v,(int,float)) or not math.isfinite(v) or not 0<=v<1 for v in sq):
        problems['global_excess_score_range']=sq
    fp=contract.get('graph_fingerprint')
    if not isinstance(fp,str) or len(fp)!=64:problems['final_graph_fingerprint']=fp
    for name,actual in (('graph',g.get('graph_fingerprint')),('icaart_manifest',e.get('graph_fingerprint')),('diagnostics',d.get('s_ica_fingerprint'))):
        if actual!=fp:problems['final_graph_consistency.'+name]=actual
    if problems:raise RuntimeError(f'{key}: Global-full null comparability failed: {problems}')
    source_diff={k:{'reference':v,'actual':contract.get('sources',{}).get(k)} for k,v in EXPECTED_RUNTIME_SOURCES.items() if contract.get('sources',{}).get(k)!=v}
    runtime_diff={k:{'reference':ref[k],'actual':manifest.get(k)} for k in ('torch','freerec','gpu') if manifest.get(k)!=ref[k]}
    if str(manifest.get('python','')).split('.')[:2]!=ref['python'].split('.')[:2]:runtime_diff['python']=manifest.get('python')
    if RUN_MODE=='train' and source_diff:raise RuntimeError(f'Source mismatch: {source_diff}')
    if RUN_MODE=='train' and STRICT_RUNTIME and runtime_diff:raise RuntimeError(f'Runtime mismatch: {runtime_diff}')
    return {'status':'MATCH_WITH_REVIEW_REQUIRED' if source_diff or runtime_diff else 'MATCH',
        'arm':ARM,'training_seed':expected_seed,'model_intervention':{'null_scope':['full_activity','global_full'],'evidence_mode':['pooled_activity_excess','pooled_global_excess']},'declared_source_changes':DECLARED_SOURCE_CHANGES,'source_overlay_id':GLOBAL_OVERLAY_ID,
        'same_candidates_raw_weights_counts_cf':True,'same_activity_null':'NOT_REQUIRED','global_scores_may_differ':True,'same_final_graph_as_full':'NOT_REQUIRED',
        'dose_reference':'uncalibrated weighted semantic+CSE eta0.1','source_deviations':source_diff,'runtime_deviations':runtime_diff,
        'operator_check_scope':'manifest and pinned source contract; not an independent sparse-tensor reconstruction',
        'cf_propagation_retained':True}


def validate_history(telemetry, evaluations, receipt):
    if [row.get('epoch') for row in telemetry]!=list(range(1,501)):raise ValueError('Require exactly ordered epochs1..500; no duplicate/missing epochs or implicit resume')
    for row in telemetry:
        for name in ('loss','bpr_loss','nlgcl_loss','weighted_nlgcl_loss','seconds','peak_allocated_gib','peak_reserved_gib'):
            value=row.get(name)
            if isinstance(value,bool) or not isinstance(value,(int,float)) or not math.isfinite(value) or value<0:raise ValueError(f'Invalid {name} at epoch {row["epoch"]}')
        if row.get('lambda_nlgcl')!=.01:raise ValueError('LC must stay enabled')
        if not math.isclose(row['weighted_nlgcl_loss'],.01*row['nlgcl_loss'],rel_tol=1e-6,abs_tol=1e-7):raise ValueError('Weighted LC mismatch')
        if not math.isclose(row['loss'],row['bpr_loss']+row['weighted_nlgcl_loss'],rel_tol=1e-6,abs_tol=1e-7):raise ValueError('Objective mismatch')
    valid=[r for r in evaluations if r.get('mode')=='valid' and not r.get('selected_checkpoint')]
    if not valid:raise ValueError('Missing VALID history')
    seen=set()
    for row in valid:
        epoch=row.get('epoch');v=row.get('metrics',{}).get('NDCG@20')
        if isinstance(epoch,bool) or not isinstance(epoch,int) or not 0<=epoch<=500 or epoch in seen:raise ValueError('Invalid/duplicate VALID epoch')
        seen.add(epoch)
        if isinstance(v,bool) or not isinstance(v,(int,float)) or not math.isfinite(v):raise ValueError('Invalid VALID NDCG@20')
    maximum=max(r['metrics']['NDCG@20'] for r in valid)
    tied=[r['epoch'] for r in valid if r['metrics']['NDCG@20']==maximum]
    if receipt['epoch'] not in tied:raise ValueError('Selected epoch is not a VALID maximum')
    selected=[r for r in evaluations if r.get('mode')=='test' and r.get('selected_checkpoint') is True]
    if len(selected)!=1 or selected[0]!=receipt:raise ValueError('Selected receipt/history mismatch')
    return {'valid_max_epochs':tied,'tie_rule':'native receipt retained; inspect native log if tied' if len(tied)>1 else 'unique maximum',
        'test_events':[{'epoch':r.get('epoch'),'selected':r.get('selected_checkpoint')} for r in evaluations if r.get('mode')=='test']}


def validate_receipt(key, receipt, manifest, expected_seed):
    if (receipt.get('mode') != 'test' or receipt.get('selected_checkpoint') is not True
            or receipt.get('selection') != 'validation NDCG@20'
            or receipt.get('seed') != expected_seed
            or receipt.get('evidence_mode') != 'pooled_global_excess'
            or receipt.get('evidence_delta') != 1.0):
        raise ValueError(f'{key}: not a selected-checkpoint ECLIPSE Global-full null TEST receipt')
    if receipt.get('graph_fingerprint') != manifest.get('checkpoint_contract', {}).get('graph_fingerprint'):
        raise ValueError(f'{key}: metric receipt belongs to another operator')
    epoch = receipt.get('epoch')
    if isinstance(epoch, bool) or not isinstance(epoch, int) or not 1 <= epoch <= 500:
        raise ValueError(f'{key}: invalid selected epoch')
    normalized = receipt.get('metrics', {})
    for metric in PRIMARY_METRICS:
        value = normalized.get(metric)
        if isinstance(value, bool) or not isinstance(value, (float, int)) or not math.isfinite(value) or not 0 <= value <= 1:
            raise ValueError(f'{key}: missing/invalid measured TEST metric {metric}')
    return normalized


def export_run_tables(key):
    stem=stem_for(key)
    artifact = ARTIFACT_PATHS[key]
    manifest = json.loads((artifact / 'manifest.json').read_text(encoding='utf-8'))
    status = json.loads((artifact / 'run_status.json').read_text(encoding='utf-8'))
    if status.get('status') != 'COMPLETED' or status.get('seed') != SEED:
        raise RuntimeError(f'{key}: run is not completed for training seed {SEED}')
    comparability = compare_manifest(key, manifest, SEED)
    receipt = json.loads((artifact / 'selected_metrics.json').read_text(encoding='utf-8'))
    metrics = validate_receipt(key, receipt, manifest, SEED)
    telemetry = pd.DataFrame(read_jsonl(artifact / 'training_telemetry.jsonl'))
    evaluations = read_jsonl(artifact / 'evaluation_metrics.jsonl')
    history = validate_history(telemetry.to_dict('records'), evaluations, receipt)
    checkpoint = artifact / 'best_model.pt'
    if not checkpoint.is_file(): raise FileNotFoundError(checkpoint)
    telemetry.to_csv(RUN_ROOT / 'tables' / f'{stem}_training_telemetry.csv', index=False, na_rep='NOT_MEASURED')
    evaluation_rows = [{k: v for k, v in record.items() if k != 'metrics'} |
                       {str(k).upper(): v for k, v in record.get('metrics', {}).items()}
                       for record in evaluations]
    pd.DataFrame(evaluation_rows).to_csv(RUN_ROOT / 'tables' / f'{stem}_evaluations.csv',
                                       index=False, na_rep='NOT_MEASURED')
    log_text = LOG_PATHS[key].read_text(encoding='utf-8', errors='replace') if LOG_PATHS[key] and LOG_PATHS[key].is_file() else ''
    fit_matches = re.findall(r'Coach\.fit takes\s+([0-9.eE+-]+)\s+seconds', log_text)
    summary = {'method': 'ECLIPSE', 'arm': ARM, 'dataset': key, 'seed': SEED,
               'selected_epoch': receipt['epoch'], 'split': 'TEST', **metrics,
               'fit_seconds': float(fit_matches[-1]) if fit_matches else None,
               'train_epoch_seconds_sum': float(telemetry['seconds'].sum()),
               'preprocessing_seconds': manifest.get('preprocessing_seconds'),
               'peak_training_allocated_mib': float(telemetry['peak_allocated_gib'].max() * 1024),
               'peak_training_reserved_mib': float(telemetry['peak_reserved_gib'].max() * 1024),
               'memory_scope': 'child-process per-training-epoch CUDA allocator peaks; excludes evaluation peaks',
               'commit': manifest.get('commit'), 'graph_fingerprint': receipt['graph_fingerprint'],
               'artifact_dir': str(artifact), 'checkpoint_path': str(artifact / 'best_model.pt'),
               'runtime_deviations': comparability['runtime_deviations'], 'comparability': comparability}
    write_json(RUN_ROOT / 'tables' / f'{stem}_selected_test.json', summary)
    write_json(RUN_ROOT / 'tables' / f'{stem}_mechanism.json', manifest['graph']['icaart'])
    execution_path=RUN_ROOT/'provenance'/f'{stem}_notebook_execution.json'
    execution=json.loads(execution_path.read_text(encoding='utf-8')) if execution_path.is_file() else {}
    summary.update(status='COMPLETED_PENDING_HUMAN_REVIEW', attempt_id=SESSION_ID,
       checkpoint_sha256=file_sha256(checkpoint), history=history,
       subprocess_wall_seconds=execution.get('subprocess_wall_seconds'),
       preprocessing_cache_hit=manifest.get('graph',{}).get('cache_hit'), reviewed_by=None)
    pd.DataFrame([{k:v for k,v in summary.items() if not isinstance(v,(dict,list))}]).to_csv(
        RUN_ROOT/'tables'/f'{stem}_selected_test.csv',index=False,na_rep='NOT_MEASURED')
    write_json(RUN_ROOT/'tables'/f'{stem}_selected_test.json',summary)
    write_json(RUN_ROOT/'provenance'/f'{stem}_comparability.json',comparability)
    write_json(RUN_ROOT/'provenance'/f'{stem}_input_receipts.json',{
       name:file_sha256(artifact/name) for name in ('manifest.json','run_status.json','selected_metrics.json','training_telemetry.jsonl','evaluation_metrics.jsonl','best_model.pt')})
    write_json(RUN_ROOT/'tables'/f'{stem}_diagnostics.json',{'score_kind':'pooled_global_excess','score_formula':'[c-mu_global]+/(c+mu_global+4)','reference':'uncalibrated weighted semantic+CSE eta0.1','diagnostics':manifest['graph']['icaart']})
    write_json(RUN_ROOT/'tables'/f'{stem}_global_null_audit.json',
        {name:manifest['icaart_manifest'].get(name) for name in ('null_scope','activity_conditioning',
          'num_active_train_users','num_inactive_train_users','num_null_groups','num_activity_strata',
          'global_null_formula','global_null_max_abs_formula_error','counts_hash','null_expectation_hash')})
    RESULTS[key] = summary
    return summary


def train_dataset(key):
    stem=stem_for(key)
    if key not in SELECTED_DATASETS:
        print(f'Skipped {key}: not in SELECTED_DATASETS.')
        return None
    if RUN_MODE == 'report_only': return export_run_tables(key)
    artifact = ARTIFACT_PATHS[key]
    status_path = artifact / 'run_status.json'
    if status_path.is_file():
        status = json.loads(status_path.read_text(encoding='utf-8'))
        if status.get('status') == 'COMPLETED':
            print(f'{key}: reading the completed attempt; no retraining.')
            return export_run_tables(key)
    # build_config() creates artifact_dir during CLI preflight, before training.
    # An empty real directory with no stdout log is not a previous run.
    log_path = LOG_PATHS[key]
    invalid_directory = artifact.is_symlink() or (artifact.exists() and not artifact.is_dir())
    has_payload = artifact.is_dir() and any(artifact.iterdir())
    has_log = log_path.exists() or log_path.is_symlink()
    if invalid_directory or has_payload or has_log:
        raise FileExistsError(f'Existing attempt payload/log preserved at {artifact}; '
                              'use report_only for completed runs, or create a new SESSION_ID for a fresh attempt. '
                              'No automatic resume/delete/retry.')
    artifact.mkdir(parents=True, exist_ok=True)
    command = build_command(key)
    write_json(RUN_ROOT / 'provenance' / f'{stem}_command.json', {
        'argv': command, 'training_seed': SEED, 'arm_id': ARM, 'dataset': key, 'purpose':'evidence_control_global_full_null', 'pythonhashseed': ENV['PYTHONHASHSEED'],
        'cuda_visible_devices': ENV['CUDA_VISIBLE_DEVICES'], 'cwd': str(REPO_ROOT)})
    print('Running:', ' '.join(command))
    start = time.perf_counter()
    manifest_checked = False
    process = None
    try:
        with LOG_PATHS[key].open('x', encoding='utf-8') as log:
            process = subprocess.Popen(command, cwd=REPO_ROOT, env=ENV, stdout=subprocess.PIPE,
                                       stderr=subprocess.STDOUT, text=True, encoding='utf-8',
                                       errors='replace', bufsize=1)
            for line in process.stdout:
                log.write(line)
                log.flush()
                print(line, end='', flush=True)
                manifest_path = artifact / 'manifest.json'
                if not manifest_checked and manifest_path.is_file():
                    comparability = compare_manifest(key, json.loads(manifest_path.read_text(encoding='utf-8')), SEED)
                    write_json(RUN_ROOT / 'provenance' / f'{key}_comparability.json', comparability)
                    manifest_checked = True
            process.wait()
            process.stdout.close()
        if process.returncode != 0:
            raise RuntimeError(f'{key}: training failed with exit code {process.returncode}; see {LOG_PATHS[key]}')
        child_wall_seconds=time.perf_counter()-start
        write_json(RUN_ROOT/'provenance'/f'{stem}_notebook_execution.json',{'status':'CHILD_COMPLETED_PENDING_EXPORT','subprocess_wall_seconds':child_wall_seconds})
        result = export_run_tables(key)
        write_json(RUN_ROOT / 'provenance' / f'{stem}_notebook_execution.json', {
            'status': 'COMPLETED', 'subprocess_wall_seconds': child_wall_seconds,
            'exit_code': process.returncode, 'training_seed': SEED})
        print(f'\nCompleted {DATASET_TITLES[key]} | selected TEST NDCG@20={result["NDCG@20"]:.10f}')
        return result
    except BaseException as exc:
        if process is not None and process.poll() is None:
            if isinstance(exc, KeyboardInterrupt):
                process.send_signal(signal.SIGINT)
            else:
                process.terminate()
            try:
                process.wait(timeout=20)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait(timeout=10)
        write_json(RUN_ROOT / 'provenance' / f'{stem}_notebook_execution.json', {
            'status': 'INTERRUPTED' if isinstance(exc, KeyboardInterrupt) else 'FAILED',
            'error_type': type(exc).__name__, 'error': str(exc), 'training_seed': SEED,
            'subprocess_wall_seconds': time.perf_counter() - start})
        raise


## 7. Figure utilities — PDF vector / PNG300dpi / measured data / captions
Training objective, VALID ranking, training allocator/time and Global-full excess-score diagnostics. No TEST learning-curve selection or phantom error bars. Positive coverage means c exceeds the global active-user null, not merely c>0. Missing diagnostics remain NOT_MEASURED.


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 9, 'axes.titlesize': 10,
                     'axes.labelsize': 9, 'legend.fontsize': 8, 'xtick.labelsize': 8,
                     'ytick.labelsize': 8, 'lines.linewidth': 1.5,
                     'pdf.fonttype': 42, 'ps.fonttype': 42, 'savefig.dpi': 300,
                     'axes.spines.top': False, 'axes.spines.right': False})
COLORS = {'blue': '#0072B2', 'orange': '#D55E00', 'green': '#009E73',
          'purple': '#CC79A7', 'gray': '#666666'}
_existing_figure_manifest = RUN_ROOT / 'figures' / 'figure_manifest.json'
FIGURE_RECORDS = json.loads(_existing_figure_manifest.read_text(encoding='utf-8')) if _existing_figure_manifest.is_file() else []


def save_figure(fig, stem, caption, sources):
    for extension in ('pdf', 'png'):
        fig.savefig(RUN_ROOT / 'figures' / f'{stem}.{extension}', bbox_inches='tight', facecolor='white')
    plt.show()
    plt.close(fig)
    FIGURE_RECORDS[:] = [record for record in FIGURE_RECORDS if record['stem'] != stem]
    FIGURE_RECORDS.append({'stem': stem, 'caption': caption, 'sources': list(map(str, sources))})
    write_json(RUN_ROOT / 'figures' / 'figure_manifest.json', FIGURE_RECORDS)
    (RUN_ROOT/'figures'/'FIGURE_CAPTIONS.md').write_text('# Figure captions\n\n'+'\n\n'.join('## '+r['stem']+'\n\n'+r['caption'] for r in FIGURE_RECORDS),encoding='utf-8')
    print(caption)


def completed_artifact(key):
    if key not in SELECTED_DATASETS:
        print(f'{key}: not selected; no figure generated.')
        return None
    path = ARTIFACT_PATHS[key]
    status = path / 'run_status.json'
    if not status.is_file() or json.loads(status.read_text(encoding='utf-8')).get('status') != 'COMPLETED':
        print(f'{key}: no completed run available; no figure generated.')
        return None
    return path


def plot_learning_curves(key):
    stem=stem_for(key)
    artifact = completed_artifact(key)
    if artifact is None:
        return
    telemetry = pd.DataFrame(read_jsonl(artifact / 'training_telemetry.jsonl'))
    records = [record for record in read_jsonl(artifact / 'evaluation_metrics.jsonl')
               if record.get('mode') == 'valid' and not record.get('selected_checkpoint')]
    selected = json.loads((artifact / 'selected_metrics.json').read_text(encoding='utf-8'))['epoch']
    valid = pd.DataFrame([{'epoch': record['epoch'],
                           **{str(k).upper(): v for k, v in record['metrics'].items()}}
                          for record in records]).sort_values('epoch')
    if telemetry.empty or valid.empty:
        print(f'{key}: missing measured curves; no substitute data generated.')
        return
    fig, axes = plt.subplots(1, 3, figsize=(7.2, 3.0), constrained_layout=True)
    for column, label, color in [('loss', 'Total', COLORS['blue']),
                                  ('bpr_loss', 'BPR', COLORS['orange']),
                                  ('weighted_nlgcl_loss', 'Weighted LC', COLORS['green'])]:
        if column in telemetry:
            axes[0].plot(telemetry['epoch'], telemetry[column], label=label, color=color)
    axes[0].set(xlabel='Training epoch', ylabel='Mean training loss', title='Training objective')
    axes[0].legend(frameon=False)
    for ax, metric in zip(axes[1:], ('NDCG@20', 'RECALL@20')):
        if metric in valid:
            ax.plot(valid['epoch'], valid[metric], color=COLORS['blue'], label=f'VALID {metric}')
            point = valid.loc[valid['epoch'] == selected, metric]
            if len(point):
                ax.scatter([selected], [point.iloc[-1]], color=COLORS['orange'], s=22, zorder=3)
        ax.axvline(selected, color=COLORS['gray'], linestyle='--', linewidth=1, label=f'Selected epoch {selected}')
        ax.set(xlabel='Training epoch', ylabel=f'Validation {metric}', title=f'Validation {metric}')
        ax.legend(frameon=False)
    for ax in axes:
        ax.grid(alpha=.2)
    fig.suptitle(f'ECLIPSE Global-full null | {DATASET_TITLES[key]} | seed {SEED}', fontsize=11)
    save_figure(fig, f'{stem}_learning_curves',
                f'{DATASET_TITLES[key]}, Global-full null, training seed {SEED}. Training objective and validation ranking curves. The checkpoint is selected by validation NDCG@20; '
                'TEST is evaluated at the selected checkpoint and is not a learning-curve selection signal.',
                [artifact / 'training_telemetry.jsonl', artifact / 'evaluation_metrics.jsonl', artifact / 'selected_metrics.json'])


def plot_training_memory(key):
    stem=stem_for(key)
    artifact = completed_artifact(key)
    if artifact is None:
        return
    telemetry = pd.DataFrame(read_jsonl(artifact / 'training_telemetry.jsonl'))
    if telemetry.empty:
        return
    fig, axes = plt.subplots(1, 2, figsize=(7.2, 2.8), constrained_layout=True)
    for column, label, color in [('peak_allocated_gib', 'Peak allocated', COLORS['blue']),
                                  ('peak_reserved_gib', 'Peak reserved', COLORS['orange'])]:
        if column in telemetry:
            axes[0].plot(telemetry['epoch'], telemetry[column] * 1024, label=label, color=color)
    axes[0].set(xlabel='Training epoch', ylabel='CUDA allocator memory (MiB)', title='Training-epoch allocator peaks')
    axes[0].legend(frameon=False)
    axes[1].plot(telemetry['epoch'], telemetry['seconds'], color=COLORS['green'])
    axes[1].set(xlabel='Training epoch', ylabel='Training time (s/epoch)', title='Training time per epoch')
    for ax in axes:
        ax.grid(alpha=.2)
    fig.suptitle(f'ECLIPSE Global-full null | {DATASET_TITLES[key]} | seed {SEED}', fontsize=11)
    save_figure(fig, f'{stem}_training_memory',
                f'{DATASET_TITLES[key]}, Global-full null, training seed {SEED}. CUDA allocator peaks are measured in the training subprocess for each training epoch. '
                'They exclude evaluation peaks, other GPU processes and driver memory. No arbitrary hardware ceiling is drawn.',
                [artifact / 'training_telemetry.jsonl'])


def plot_calibration_diagnostics(key):
    stem=stem_for(key)
    artifact = completed_artifact(key)
    if artifact is None:
        return
    manifest = json.loads((artifact / 'manifest.json').read_text(encoding='utf-8'))
    diagnostics = manifest['graph']['icaart']
    coverage = [diagnostics.get('frac_edges_with_positive_evidence'), diagnostics.get('weighted_evidence_coverage')]
    if any(value is None for value in coverage):
        print(f'{key}: coverage was not measured.')
        return
    fig, axes = plt.subplots(1, 3, figsize=(7.2, 3.0), constrained_layout=True)
    bars = axes[0].bar(['Candidate\nedges', 'Raw semantic\nweight'], np.asarray(coverage) * 100,
                       color=[COLORS['blue'], COLORS['green']], width=.6)
    axes[0].bar_label(bars, fmt='%.2f%%', padding=3, fontsize=8)
    axes[0].set(ylabel='Coverage with positive excess score (%)', title='Positive excess-score coverage')
    axes[0].set_ylim(0, max(coverage) * 125 + 1)
    polynomial = diagnostics.get('polynomial_dose_by_coordinate', {})
    if polynomial:
        coordinates = sorted(map(int, polynomial))
        values = [100 * polynomial[str(coordinate)] for coordinate in coordinates]
        axes[1].plot(coordinates, values, 'o-', color=COLORS['blue'], label='Polynomial probe dose')
    operator_dose = diagnostics.get('relative_operator_dose')
    if operator_dose is not None:
        axes[1].axhline(100 * operator_dose, color=COLORS['orange'], linestyle='--', label='Operator Frobenius dose')
    axes[1].set(xlabel='Embedding coordinate', ylabel='Relative dose (%)', title='Operator / polynomial\nprobe dose')
    axes[1].legend(frameon=False)
    levels, values = diagnostics.get('evidence_quantile_levels'), diagnostics.get('evidence_score_quantiles')
    if levels is not None and values is not None and len(levels) == len(values):
        axes[2].plot(np.asarray(levels) * 100, values, 'o-', color=COLORS['green'])
        pd.DataFrame({'quantile': levels, 'evidence_score': values}).to_csv(
            RUN_ROOT / 'tables' / f'{stem}_evidence_quantiles.csv', index=False)
    else:
        axes[2].text(.5, .5, 'Score quantiles not measured', ha='center', va='center', transform=axes[2].transAxes)
    axes[2].set(xlabel='Quantile (%)', ylabel='Excess score r', ylim=(0, 1), title='Excess-score quantile profile')
    for ax in axes:
        ax.grid(axis='y', alpha=.2)
    write_json(RUN_ROOT/'tables'/f'{stem}_calibration_figure_data.json',{
       'coverage_fraction':coverage,'operator_relative_dose':operator_dose,'polynomial_dose_by_coordinate':polynomial,
       'quantile_levels':levels,'score_quantiles':values,'reference':'uncalibrated weighted semantic+CSE eta0.1',
       'probe_scope':diagnostics.get('polynomial_probe_scope'),'null_scope':'global_full','num_active_train_users':diagnostics.get('num_active_train_users')})
    fig.suptitle(f'ECLIPSE Global-full null | {DATASET_TITLES[key]} | seed {SEED}', fontsize=11)
    save_figure(fig, f'{stem}_calibration_diagnostics',
                f'{DATASET_TITLES[key]}, training seed {SEED}. Global-full null uses positive excess over n_i*n_j/M_active with denominator c+mu_global+4; reference is the uncalibrated weighted semantic+CSE eta0.1 operator. '
                'Polynomial dose uses fixed Rademacher probes, not actual Adam directions. '
                'The score panel displays measured quantiles, not a reconstructed histogram.', [artifact / 'manifest.json'])


## 8. Training / report-only — Amazon Baby seed 1
Runs only if `baby` is in SELECTED_DATASETS. A completed attempt is read for export; an incomplete attempt is never overwritten. Checkpoints are initialized fresh, not from other seeds/arms.


In [ ]:
baby_result=train_dataset('baby')
if baby_result is not None:
    display(pd.DataFrame([{k:baby_result[k] for k in ('arm','dataset','seed','selected_epoch',*PRIMARY_METRICS)}]))


### VALID learning curves


In [ ]:
plot_learning_curves('baby')


### Training memory and epoch time


In [ ]:
plot_training_memory('baby')


### Global-full null calibration diagnostics


In [ ]:
plot_calibration_diagnostics('baby')


## 9. Training / report-only — Amazon Sports seed 1
Runs only if `sports` is in SELECTED_DATASETS. A completed attempt is read for export; an incomplete attempt is never overwritten. Checkpoints are initialized fresh, not from other seeds/arms.


In [ ]:
sports_result=train_dataset('sports')
if sports_result is not None:
    display(pd.DataFrame([{k:sports_result[k] for k in ('arm','dataset','seed','selected_epoch',*PRIMARY_METRICS)}]))


### VALID learning curves


In [ ]:
plot_learning_curves('sports')


### Training memory and epoch time


In [ ]:
plot_training_memory('sports')


### Global-full null calibration diagnostics


In [ ]:
plot_calibration_diagnostics('sports')


## 10. Current-session selected TEST table and resource profile
No Full/other-arm results are mixed into this session. Each dataset currently has n=1. All seeds/attempts must be catalogued before a later matched multi-seed comparison; there is no SD or significance from epochs. Missing run/timing fields are NOT_MEASURED. Resource axes use distinct measurement scopes.


In [ ]:
summary_rows = []
for key in SELECTED_DATASETS:
    if completed_artifact(key) is None:
        summary_rows.append({'method': 'ECLIPSE', 'dataset': key, 'seed': SEED, 'status': 'NOT_MEASURED'})
        continue
    summary = export_run_tables(key)
    names = ['method', 'arm', 'dataset', 'seed', 'selected_epoch', *PRIMARY_METRICS,
             'preprocessing_seconds', 'fit_seconds', 'train_epoch_seconds_sum',
             'peak_training_allocated_mib', 'peak_training_reserved_mib',
             'checkpoint_path', 'commit', 'graph_fingerprint']
    summary_rows.append({name: summary.get(name) for name in names} | {'status': 'COMPLETED'})
SUMMARY = pd.DataFrame(summary_rows)
for column in ['selected_epoch', *PRIMARY_METRICS, 'preprocessing_seconds', 'fit_seconds',
               'peak_training_allocated_mib', 'peak_training_reserved_mib']:
    if column not in SUMMARY:
        SUMMARY[column] = np.nan
SUMMARY.to_csv(RUN_ROOT / 'tables' / f'{SESSION_STEM}_selected_test_overview.csv', index=False, na_rep='NOT_MEASURED')
display(SUMMARY[['dataset', 'seed', 'status', 'selected_epoch', *PRIMARY_METRICS]].fillna('NOT_MEASURED'))
measured = SUMMARY.loc[SUMMARY['status'] == 'COMPLETED'].copy()
if len(measured):
    labels = [DATASET_TITLES[key].replace('Amazon ', '') for key in measured['dataset']]
    x = np.arange(len(measured))
    fig, axes = plt.subplots(1, 3, figsize=(7.2, 3.0), constrained_layout=True)
    if measured['fit_seconds'].notna().any():
        axes[0].bar(x, pd.to_numeric(measured['fit_seconds'],errors='coerce') / 3600, color=COLORS['blue'])
    else:
        axes[0].text(.5,.5,'NOT_MEASURED',ha='center',va='center',transform=axes[0].transAxes)
    axes[0].set(ylabel='Coach.fit wall time (h)', title='Training + native evaluation / I/O')
    axes[1].bar(x, measured['preprocessing_seconds'], color=COLORS['green'])
    axes[1].set(ylabel='Preprocessing wall time (s)', title='Total model preprocessing')
    axes[2].bar(x - .18, measured['peak_training_allocated_mib'], width=.36, color=COLORS['blue'], label='Allocated')
    axes[2].bar(x + .18, measured['peak_training_reserved_mib'], width=.36, color=COLORS['orange'], label='Reserved')
    axes[2].set(ylabel='Training allocator peak (MiB)', title='Training-epoch CUDA memory')
    axes[2].legend(frameon=False)
    for ax in axes:
        ax.set_xticks(x, labels)
        ax.grid(axis='y', alpha=.2)
    fig.suptitle(f'ECLIPSE Global-full null resource profile | seed {SEED}', fontsize=11)
    save_figure(fig, f'{SESSION_STEM}_resource_profile',
                'ECLIPSE Global-full null resource measurements, not a comparison against another model. '
                'Coach.fit includes native evaluation and I/O; preprocessing is separate. '
                'Memory excludes evaluation peaks. Cache state and runtime remain in manifests.',
                [RUN_ROOT / 'tables' / f'{SESSION_STEM}_selected_test_overview.csv'])

resource_fields=['preprocessing_seconds','fit_seconds','train_epoch_seconds_sum','subprocess_wall_seconds','peak_training_allocated_mib','peak_training_reserved_mib','memory_scope','preprocessing_cache_hit']
for key in SELECTED_DATASETS:
    if key in RESULTS:
        pd.DataFrame([{k:RESULTS[key].get(k) for k in resource_fields}]).to_csv(RUN_ROOT/'tables'/f'{stem_for(key)}_resource_summary.csv',index=False,na_rep='NOT_MEASURED')
status_rows=[]
for key in SELECTED_DATASETS:
    status_rows.append({'dataset':key,'arm':ARM,'training_seed':SEED,'actual_n':1 if key in RESULTS else 0,
       'state':'COMPLETED_PENDING_HUMAN_REVIEW' if key in RESULTS else 'NOT_MEASURED'})
pd.DataFrame(status_rows).to_csv(RUN_ROOT/'tables'/f'{SESSION_STEM}_seed_status.csv',index=False)


## 11. Complete handoff ZIP / inventory / SHA256
Includes all completed dataset artifacts, stdout logs outside native directories, resolved/original config and commands, source/runtime/input provenance, tables, English captions and PDF/PNG/source data. Default run order is Baby→Sports; do not leave archiving to quota expiry. If an interrupted job stops Run All, preserve it and manually execute reporting/packaging for completed datasets; missing datasets remain unmeasured.

Report-only copies originals into a **new** export root. The exact submitted notebook is included when NOTEBOOK_SOURCE_PATH is provided; otherwise a nonrecursive source template is included and the executed Kaggle notebook should be downloaded separately. No checkpoint tensors are loaded. Human cross-read remains pending.


In [ ]:
import zipfile
write_json(RUN_ROOT/'provenance'/'notebook_source_template.ipynb',NOTEBOOK_SOURCE_TEMPLATE)
script='\n\n'.join('# CELL '+str(i)+'\n'+''.join(c['source']) for i,c in enumerate(NOTEBOOK_SOURCE_TEMPLATE['cells']) if c['cell_type']=='code')
(RUN_ROOT/'helpers'/'orchestration_source.py').write_text(script,encoding='utf-8')
if NOTEBOOK_SOURCE_PATH:
    original=Path(NOTEBOOK_SOURCE_PATH).expanduser().resolve()
    if not original.is_file():raise FileNotFoundError(original)
    shutil.copy2(original,RUN_ROOT/'provenance'/'submitted_notebook.ipynb')
    write_json(RUN_ROOT/'provenance'/'submitted_notebook_receipt.json',{'path':str(original),'sha256':file_sha256(original)})
for key in SELECTED_DATASETS:
    if key not in RESULTS:continue
    stem=stem_for(key);native=ARTIFACT_PATHS[key]
    if RUN_MODE=='report_only':
        copied=RUN_ROOT/'artifacts'/stem
        shutil.copytree(native,copied,dirs_exist_ok=False)
        log=LOG_PATHS[key]
        if log and log.is_file():shutil.copy2(log,RUN_ROOT/'logs'/f'{stem}.log')
        m=json.loads((native/'manifest.json').read_text(encoding='utf-8'))
        write_json(RUN_ROOT/'provenance'/f'{stem}_report_only_origin.json',{'artifact':str(native),'copied_to':str(copied),
          'stdout_log':str(log) if log else None,'raw_artifacts_modified':False,'recovered_from_native_manifest':True,
          'original_yaml_available':False,'original_dirty_diff_available':False,'note':'Do not invent unavailable historical provenance.'})
        write_json(RUN_ROOT/'configs'/f'{stem}_recorded_native_config.json',{'v8_config':m.get('v8_config'),'icaart_config':m.get('icaart_config'),
           'checkpoint_config':m.get('checkpoint_contract',{}).get('config'),'argv':m.get('argv')})
        write_json(RUN_ROOT/'provenance'/f'{stem}_recorded_native_environment.json',{k:m.get(k) for k in ('python','torch','freerec','gpu','commit','precision')})
    for kind in ('learning_curves','training_memory','calibration_diagnostics'):
        for extension in ('pdf','png'):
            path=RUN_ROOT/'figures'/f'{stem}_{kind}.{extension}'
            if not path.is_file():raise FileNotFoundError(f'Missing measured figure {path}; generate plotting cells, never fabricate values.')
entries=json.loads((RUN_ROOT/'figures'/'figure_manifest.json').read_text(encoding='utf-8')) if (RUN_ROOT/'figures'/'figure_manifest.json').is_file() else []
caption_text='# ECLIPSE Global-full null figure captions\n\n'
for entry in entries:
    caption_text+='## '+entry['stem']+'\n\n'+entry['caption']+'\n\nSources:\n'+'\n'.join('- '+p for p in entry['sources'])+'\n\n'
(RUN_ROOT/'figures'/'FIGURE_CAPTIONS.md').write_text(caption_text,encoding='utf-8')
run=json.loads((RUN_ROOT/'run_manifest.json').read_text(encoding='utf-8'))
run.update(status='COMPLETED_PENDING_HUMAN_REVIEW' if len(RESULTS)==len(SELECTED_DATASETS) else 'PARTIAL_PENDING_REVIEW',
    datasets={key:{'state':'COMPLETED_PENDING_HUMAN_REVIEW' if key in RESULTS else 'NOT_MEASURED','actual_n':1 if key in RESULTS else 0,
      'selected_epoch':RESULTS.get(key,{}).get('selected_epoch'),'comparability':RESULTS.get(key,{}).get('comparability',{}).get('status')} for key in SELECTED_DATASETS},
    reviewed_by=None,matched_full_comparison='NOT_COMPUTED')
write_json(RUN_ROOT/'run_manifest.json',run)
(RUN_ROOT/'README.md').write_text(f"""# ECLIPSE Global-full null | training seed {SEED}
Selected datasets: {', '.join(SELECTED_DATASETS)}. Mode: {RUN_MODE}; attempt/export: {SESSION_ID}.
Protocol: 64D/500epochs; pooled_global_excess, delta1/tau2/halves0; eta.1/LC.01 retained.
Global excess score: [c-mu_global]+/(c+mu_global+4), mu_global=n_i*n_j/M_active; no user halves/activity strata.
Native selected TEST: artifacts/eclipse_<dataset>_global_full_null_seed{SEED}/selected_metrics.json.
Tables/ contains selected TEST/resource/diagnostic data; figures/ has PDF vector/PNG300dpi + captions.
Source/input/runtime/config/commands under provenance/ and configs/; stdout in logs/.
Algorithm source: base {RUNTIME_COMMIT} + declared Global-full overlay {GLOBAL_OVERLAY_ID}. See source digests and diff. Source template excludes recursive self-embedding.
Human review pending. Actual current seed only per dataset: no seed SD/p-values or phantom completed runs.
Coach.fit, preprocessing, subprocess wall and epoch sums have distinct scopes; missing data NOT_MEASURED.
Memory is child training-epoch allocator, not total-VRAM/whole-run/evaluation peak.
""",encoding='utf-8')
inventory=[]
for path in sorted(RUN_ROOT.rglob('*')):
    if path.is_file() and path.name!='file_inventory.json':inventory.append({'path':path.relative_to(RUN_ROOT).as_posix(),'bytes':path.stat().st_size,'sha256':file_sha256(path)})
write_json(RUN_ROOT/'file_inventory.json',inventory)
archive=RUN_ROOT.parent/f'{RUN_ROOT.name}.zip'
with zipfile.ZipFile(archive,'x',compression=zipfile.ZIP_DEFLATED,compresslevel=1) as z:
    for path in sorted(RUN_ROOT.rglob('*')):
        if path.is_file():z.write(path,arcname=str(Path(RUN_ROOT.name)/path.relative_to(RUN_ROOT)))
with zipfile.ZipFile(archive) as z:
    if z.testzip() is not None:raise RuntimeError('Archive CRC failed')
archive_receipt={'archive':str(archive),'bytes':archive.stat().st_size,'sha256':file_sha256(archive),'file_count':len(inventory)+1,
    'status':'PACKAGED_PENDING_HUMAN_REVIEW','run_root':str(RUN_ROOT)}
write_json(archive.with_suffix('.archive_receipt.json'),archive_receipt)
print(json.dumps(archive_receipt,indent=2))
try:
    from IPython.display import FileLink
    display(FileLink(str(archive)));display(FileLink(str(archive.with_suffix('.archive_receipt.json'))))
except ImportError:pass
